In [1]:
import gzip
import json
from pathlib import Path

import pandas as pd


# ============================================================
# 0. 경로
# ============================================================

ROOT = Path.cwd()
DATA_DIR = ROOT / "open"
LAW_DIR = DATA_DIR / "data" / "법령패키지" / "법령"
NOTICE_DIR = DATA_DIR / "data" / "법령패키지" / "중기부고시"

DEV_PATH = DATA_DIR / "dev.jsonl.gz"
LABEL_PATH = DATA_DIR / "dev_labels.csv"
ITEM_PATH = DATA_DIR / "data" / "항목표.json"

print("[0] ROOT =", ROOT)


# ============================================================
# 1. 공식 데이터 로드
# ============================================================

with gzip.open(DEV_PATH, "rt", encoding="utf-8") as f:
    dev = [json.loads(line) for line in f]

labels = pd.read_csv(LABEL_PATH)

with open(ITEM_PATH, "r", encoding="utf-8") as f:
    item_table = json.load(f)

items = item_table["항목"]
absence_items = set(item_table["부재탐지항목"])

dev_by_id = {row["id"]: row for row in dev}


# ============================================================
# 2. 기본 무결성 검증
# ============================================================

assert len(dev) == 200, f"dev 건수 이상: {len(dev)}"
assert len(labels) == 200, f"label 건수 이상: {len(labels)}"
assert len(items) == 24, f"항목 수 이상: {len(items)}"

expected_v = [f"v{i}" for i in range(1, 25)]
expected_e = [f"e{i}" for i in range(1, 25)]

for col in ["id"] + expected_v + expected_e:
    assert col in labels.columns, f"라벨 컬럼 없음: {col}"

assert set(dev_by_id) == set(labels["id"]), "dev / labels ID 불일치"

assert absence_items == {
    "v10", "v11", "v16", "v18", "v20"
}, f"부재탐지 항목 예상과 다름: {absence_items}"

print("[1] 공식 데이터 무결성 PASS")


# ============================================================
# 3. 법령패키지 파일 목록 검증
# ============================================================

law_files = sorted(LAW_DIR.glob("*.txt"))

if len(law_files) != 23:
    raise ValueError(
        f"공식 법령 TXT는 23개여야 하는데 현재 {len(law_files)}개"
    )

notice_files = sorted(
    p for p in NOTICE_DIR.iterdir()
    if p.is_file()
)

print("[2] 법령 TXT:", len(law_files))
print("[2] 중기부고시 파일:", len(notice_files))


# ============================================================
# 4. 공고 하나를 분석용 구조로 변환
# ============================================================

def summarize_case(case_id, violation):

    record = dev_by_id[case_id]

    label_row = labels.loc[
        labels["id"] == case_id
    ].iloc[0]

    e_col = "e" + violation[1:]
    evidence = label_row[e_col]

    if pd.isna(evidence):
        evidence = ""

    docs = record.get("docs") or []

    doc_types = [
        doc.get("type")
        for doc in docs
    ]

    doc_lengths = {
        doc.get("doc_id", f"doc_{i}"): len(doc.get("text") or "")
        for i, doc in enumerate(docs)
    }

    return {
        "id": case_id,

        "meta": record.get("meta") or {},

        "doc_types": doc_types,

        "doc_lengths": doc_lengths,

        "dropped_doc_counts":
            record.get("dropped_doc_counts") or {},

        "input_completeness":
            record.get("input_completeness") or {},

        "evidence": evidence,
    }


# ============================================================
# 5. 항목별 positive 사례 수집
# ============================================================

def get_positive_ids(v):

    return labels.loc[
        labels[v] == 1,
        "id"
    ].tolist()


# ============================================================
# 6. negative 후보 선정
#
# 아직 "법적으로 가까운 negative"라고 단정하지 않는다.
# 먼저 positive와 문서/meta 구조가 비슷한 후보를 뽑는다.
# 이후 Stage 6-2에서 실제 텍스트/법령을 보고 near-negative를 확정한다.
# ============================================================

META_KEYS_FOR_SIMILARITY = [
    "적용계약법",
    "업무구분",
    "계약방법",
    "낙찰방법",
    "정보화사업여부",
    "지역제한여부",
    "업종제한여부",
    "조달방식",
]


def case_signature(record):

    meta = record.get("meta") or {}

    return {
        key: meta.get(key)
        for key in META_KEYS_FOR_SIMILARITY
    }


def similarity_score(a, b):

    sig_a = case_signature(a)
    sig_b = case_signature(b)

    score = 0

    for key in META_KEYS_FOR_SIMILARITY:

        va = sig_a.get(key)
        vb = sig_b.get(key)

        # 둘 다 값이 존재하고 동일한 경우만 점수 부여
        if va is not None and vb is not None and va == vb:
            score += 1

    types_a = {
        d.get("type")
        for d in (a.get("docs") or [])
    }

    types_b = {
        d.get("type")
        for d in (b.get("docs") or [])
    }

    # 같은 문서 종류가 많을수록 약간 가점
    score += 0.25 * len(types_a & types_b)

    return score


def select_negative_candidates(v, n=12):

    positive_ids = get_positive_ids(v)

    positive_records = [
        dev_by_id[x]
        for x in positive_ids
    ]

    negative_ids = labels.loc[
        labels[v] == 0,
        "id"
    ].tolist()

    ranked = []

    for case_id in negative_ids:

        candidate = dev_by_id[case_id]

        # 여러 positive 중 가장 비슷한 구조를 기준으로 사용
        best_score = max(
            similarity_score(candidate, pos)
            for pos in positive_records
        )

        ranked.append(
            (best_score, case_id)
        )

    ranked.sort(
        key=lambda x: (-x[0], x[1])
    )

    return [
        {
            "id": case_id,
            "similarity_score": score,
            **summarize_case(case_id, v),
        }
        for score, case_id in ranked[:n]
    ]


# ============================================================
# 7. 항목표의 원문 필드를 그대로 보존하면서
#    24개 분석용 spec 생성
# ============================================================

specs = {}

for v in expected_v:

    official = items[v]

    positive_ids = get_positive_ids(v)

    positives = [
        summarize_case(case_id, v)
        for case_id in positive_ids
    ]

    negatives = select_negative_candidates(
        v,
        n=12
    )

    evidence_present = sum(
        bool(case["evidence"])
        for case in positives
    )

    dropped_positive = sum(
        bool(case["dropped_doc_counts"])
        for case in positives
    )

    incomplete_positive = sum(
        not (
            case["input_completeness"].get(
                "완전관측",
                False
            )
        )
        for case in positives
    )

    specs[v] = {

        # ----------------------------------------------------
        # 공식 항목표 원문
        # 임의 해석하지 않고 그대로 보존
        # ----------------------------------------------------
        "official": official,

        "absence_detection":
            v in absence_items,

        # ----------------------------------------------------
        # dev 통계
        # ----------------------------------------------------
        "dev_stats": {
            "positive_count":
                len(positive_ids),

            "positive_evidence_present":
                evidence_present,

            "positive_evidence_missing":
                len(positive_ids)
                - evidence_present,

            "positive_with_dropped_docs":
                dropped_positive,

            "positive_not_fully_observed":
                incomplete_positive,
        },

        # ----------------------------------------------------
        # 모든 positive
        # ----------------------------------------------------
        "positives": positives,

        # ----------------------------------------------------
        # 구조적으로 비슷한 negative 후보
        # 아직 법적 near-negative 확정 아님
        # ----------------------------------------------------
        "negative_candidates": negatives,

        # ----------------------------------------------------
        # 다음 단계에서 채울 판정 명세
        # 추측 금지 → 아직은 UNRESOLVED
        # ----------------------------------------------------
        "decision_spec": {

            "applicability":
                "UNRESOLVED",

            "required_meta":
                "UNRESOLVED",

            "required_documents":
                "UNRESOLVED",

            "violation_condition":
                "UNRESOLVED",

            "non_violation_condition":
                "UNRESOLVED",

            "exceptions":
                "UNRESOLVED",

            "incomplete_input_policy":
                "UNRESOLVED",

            "legal_retrieval":
                "UNRESOLVED",

            "python_role":
                "UNRESOLVED",

            "llm_role":
                "UNRESOLVED",

            "evidence_policy":
                (
                    "EMPTY"
                    if v in absence_items
                    else "UNRESOLVED"
                ),
        },
    }


# ============================================================
# 8. 검증
# ============================================================

assert set(specs) == set(expected_v)

for v in expected_v:

    s = specs[v]

    assert (
        s["dev_stats"]["positive_count"]
        == int(labels[v].sum())
    )

    if v in absence_items:

        # 공식 규칙상 부재탐지 항목의 evidence는 항상 빈칸
        positive_evidence = [
            x["evidence"]
            for x in s["positives"]
        ]

        assert all(
            e == ""
            for e in positive_evidence
        ), (
            f"{v}: 부재탐지인데 "
            "positive evidence가 존재함"
        )


print("[3] 24개 spec 기본 검증 PASS")


# ============================================================
# 9. 저장
# ============================================================

ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUT_PATH = (
    ARTIFACT_DIR
    / "item_spec_workbench.json"
)

with open(
    OUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        specs,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 10. 사람이 검토할 요약표
# ============================================================

summary_rows = []

for v in expected_v:

    s = specs[v]

    official = s["official"]

    # 항목표의 실제 필드명이 버전/구조에 따라 다를 수 있으므로
    # 요약에서는 전체 official 객체를 훼손하지 않는다.
    item_name = (
        official.get("항목명")
        if isinstance(official, dict)
        else str(official)
    )

    stats = s["dev_stats"]

    summary_rows.append({
        "item": v,
        "name": item_name,
        "absence":
            s["absence_detection"],

        "positive":
            stats["positive_count"],

        "evidence_present":
            stats[
                "positive_evidence_present"
            ],

        "dropped_docs":
            stats[
                "positive_with_dropped_docs"
            ],

        "not_fully_observed":
            stats[
                "positive_not_fully_observed"
            ],
    })


summary_df = pd.DataFrame(
    summary_rows
)

display(summary_df)


print()
print("=" * 70)
print("Stage 6-1 완료")
print("=" * 70)

print("항목 수       :", len(specs))
print("부재탐지      :", sorted(absence_items))
print("법령 TXT      :", len(law_files))
print("출력          :", OUT_PATH)

print()
print(
    "주의: negative_candidates는 "
    "구조적 유사 후보일 뿐 "
    "아직 법적 near-negative가 아닙니다."
)

print(
    "주의: decision_spec의 UNRESOLVED는 "
    "의도적으로 남겨둔 값입니다."
)

[0] ROOT = /home/alparka/nara-ai
[1] 공식 데이터 무결성 PASS
[2] 법령 TXT: 23
[2] 중기부고시 파일: 2
[3] 24개 spec 기본 검증 PASS


,item,name,absence,positive,evidence_present,dropped_docs,not_fully_observed
0,v1,참가자격 특정기관 제한,False,7,1,1,1
1,v2,고시금액 미만 실적제한,False,7,7,1,1
2,v3,실적제한 1배수 이상,False,8,8,0,0
3,v4,"고시금액 이상 특정기관, 특정실적",False,6,5,0,0
4,v5,고시금액 이상 지역제한,False,7,1,0,0
5,v6,"고시금액 미만 지역제한 시,군,구",False,6,1,0,0
6,v7,고시금액 미만 지역제한 인접 확대,False,7,7,0,0
7,v8,중복제한 (실적+지역),False,6,2,0,0
8,v9,과업지시서 특정 모델명 명시,False,6,3,0,0
9,v10,중기간 경쟁제품 입찰 직생 없음,True,7,0,1,1



Stage 6-1 완료
항목 수       : 24
부재탐지      : ['v10', 'v11', 'v16', 'v18', 'v20']
법령 TXT      : 23
출력          : /home/alparka/nara-ai/artifacts/item_spec_workbench.json

주의: negative_candidates는 구조적 유사 후보일 뿐 아직 법적 near-negative가 아닙니다.
주의: decision_spec의 UNRESOLVED는 의도적으로 남겨둔 값입니다.


In [2]:
# ============================================================
# Stage 6-2
# 공식 항목표의 법적 근거 구조 Audit
#
# 목적:
# 1. v1~v24의 공식 근거를 원문 그대로 확인
# 2. 항목표의 실제 field 구조 확인
# 3. 근거에 등장하는 법령/예규 이름 inventory 생성
# 4. 공식 23개 txt와 연결 가능한지 검사
#
# 주의:
# - 아직 조문을 자동 추출하지 않는다.
# - 아직 판정 규칙을 만들지 않는다.
# - fuzzy matching 결과를 법적 근거로 확정하지 않는다.
# ============================================================

import json
import re
from pathlib import Path
from difflib import SequenceMatcher

import pandas as pd


# ------------------------------------------------------------
# 0. 경로
# ------------------------------------------------------------

ROOT = Path.cwd()

DATA_DIR = ROOT / "open"
ITEM_PATH = DATA_DIR / "data" / "항목표.json"
LAW_DIR = DATA_DIR / "data" / "법령패키지" / "법령"
NOTICE_DIR = DATA_DIR / "data" / "법령패키지" / "중기부고시"


# ------------------------------------------------------------
# 1. 공식 항목표 다시 로드
# ------------------------------------------------------------

with open(ITEM_PATH, "r", encoding="utf-8") as f:
    item_table_audit = json.load(f)

items_audit = item_table_audit["항목"]

assert len(items_audit) == 24


# ------------------------------------------------------------
# 2. 공식 법령 파일 inventory
# ------------------------------------------------------------

law_paths = sorted(LAW_DIR.glob("*.txt"))

assert len(law_paths) == 23

law_names = [p.stem for p in law_paths]


print("=" * 100)
print("[A] 공식 법령 TXT 23개")
print("=" * 100)

for i, name in enumerate(law_names, 1):
    print(f"{i:02d}. {name}")


# ------------------------------------------------------------
# 3. 중기부 고시 파일 inventory
# ------------------------------------------------------------

notice_paths = sorted(
    p for p in NOTICE_DIR.iterdir()
    if p.is_file()
)

print()
print("=" * 100)
print("[B] 중기부고시 파일")
print("=" * 100)

for i, p in enumerate(notice_paths, 1):
    print(f"{i:02d}. {p.name}")


# ------------------------------------------------------------
# 4. 항목표 실제 field 구조 확인
# ------------------------------------------------------------

print()
print("=" * 100)
print("[C] 항목표 실제 field 구조")
print("=" * 100)

all_field_sets = {}

for v in [f"v{i}" for i in range(1, 25)]:

    obj = items_audit[v]

    if not isinstance(obj, dict):
        raise TypeError(
            f"{v}: dict가 아님 → {type(obj)}"
        )

    fields = tuple(obj.keys())

    all_field_sets.setdefault(
        fields,
        []
    ).append(v)


for fields, vs in all_field_sets.items():

    print()
    print("항목:", ", ".join(vs))
    print("fields:")

    for field in fields:
        print("   -", repr(field))


# ------------------------------------------------------------
# 5. v1~v24 공식 항목표 전체 출력
#
# 여기서는 field 이름을 추측하지 않는다.
# 공식 JSON 내용을 그대로 보여준다.
# ------------------------------------------------------------

print()
print("=" * 100)
print("[D] v1~v24 공식 정의 / 근거 / 비고")
print("=" * 100)

for i in range(1, 25):

    v = f"v{i}"
    obj = items_audit[v]

    print()
    print("#" * 100)
    print(v)
    print("#" * 100)

    for key, value in obj.items():

        print(f"\n[{key}]")

        if isinstance(value, (dict, list)):
            print(
                json.dumps(
                    value,
                    ensure_ascii=False,
                    indent=2
                )
            )
        else:
            print(value)


# ------------------------------------------------------------
# 6. 법령명 비교용 정규화
#
# 중요:
# 이것은 자동 확정용이 아니다.
# 사람이 alias 후보를 찾기 쉽게 하기 위한 용도다.
# ------------------------------------------------------------

def normalize_law_name(text):

    if text is None:
        return ""

    text = str(text)

    # 일반적인 괄호/공백 차이만 완화
    text = re.sub(r"\s+", "", text)

    text = text.replace("ㆍ", "·")
    text = text.replace("․", "·")

    return text


# ------------------------------------------------------------
# 7. 항목표 안의 모든 문자열 수집
# ------------------------------------------------------------

def collect_strings(obj):

    result = []

    if isinstance(obj, str):

        result.append(obj)

    elif isinstance(obj, dict):

        for value in obj.values():
            result.extend(
                collect_strings(value)
            )

    elif isinstance(obj, list):

        for value in obj:
            result.extend(
                collect_strings(value)
            )

    return result


official_strings = []

for v, obj in items_audit.items():

    for text in collect_strings(obj):

        official_strings.append(
            {
                "item": v,
                "text": text,
            }
        )


# ------------------------------------------------------------
# 8. 공식 문자열 ↔ 법령 파일명 후보 비교
#
# 정확히 같은 이름이 아닐 수 있으므로
# fuzzy score를 참고용으로만 계산한다.
#
# 절대 이 결과만으로 자동 grounding하지 않는다.
# ------------------------------------------------------------

def similarity(a, b):

    a = normalize_law_name(a)
    b = normalize_law_name(b)

    if not a or not b:
        return 0.0

    # 한쪽 이름이 다른 쪽 문자열 안에 그대로 있으면 강한 후보
    if a in b or b in a:
        return 1.0

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()


candidate_rows = []

for entry in official_strings:

    v = entry["item"]
    text = entry["text"]

    scored = []

    for law_name in law_names:

        score = similarity(
            text,
            law_name
        )

        scored.append(
            (score, law_name)
        )

    scored.sort(reverse=True)

    best_score, best_law = scored[0]

    # 아무 문자열이나 법령처럼 출력되는 것을 막기 위해
    # 어느 정도 유사성이 있는 것만 후보로 보존
    if best_score >= 0.45:

        candidate_rows.append(
            {
                "item": v,
                "official_text": text,
                "candidate_law_file": best_law,
                "similarity": round(
                    best_score,
                    3
                ),
            }
        )


candidate_df = pd.DataFrame(
    candidate_rows
)


print()
print("=" * 100)
print("[E] 법령 파일 연결 후보")
print("※ 후보일 뿐 확정 grounding 아님")
print("=" * 100)

if len(candidate_df):

    display(
        candidate_df.sort_values(
            ["item", "similarity"],
            ascending=[True, False]
        ).reset_index(drop=True)
    )

else:

    print("후보 없음")


# ------------------------------------------------------------
# 9. 공식 항목별 audit record 생성
#
# 아직 legal_grounding은 확정하지 않는다.
# ------------------------------------------------------------

legal_audit = {}

for i in range(1, 25):

    v = f"v{i}"

    legal_audit[v] = {

        "official_item_table":
            items_audit[v],

        "legal_grounding_status":
            "UNRESOLVED",

        "resolved_sources": [],

        "unresolved_sources": [],

        "notes": [],
    }


# ------------------------------------------------------------
# 10. 저장
# ------------------------------------------------------------

ARTIFACT_DIR = ROOT / "artifacts"

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_PATH = (
    ARTIFACT_DIR
    / "legal_grounding_audit.json"
)

with open(
    AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        legal_audit,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 11. 마지막 검증
# ------------------------------------------------------------

assert len(legal_audit) == 24

assert all(
    legal_audit[f"v{i}"][
        "legal_grounding_status"
    ] == "UNRESOLVED"
    for i in range(1, 25)
)


print()
print("=" * 100)
print("Stage 6-2A AUDIT 생성 완료")
print("=" * 100)

print("항목 수          :", len(legal_audit))
print("공식 법령 TXT    :", len(law_paths))
print("중기부고시 파일   :", len(notice_paths))
print("출력             :", AUDIT_PATH)

print()
print(
    "중요: 현재 legal_grounding_status는 "
    "24개 모두 의도적으로 UNRESOLVED입니다."
)

print(
    "법령 파일 연결 후보 역시 "
    "자동 확정값이 아닙니다."
)

# Stage 6-2A 핵심 확인: 항목표 실제 구조만 출력

print("=== top-level keys ===")
print(item_table_audit.keys())

print("\n=== v1 ===")
print(json.dumps(
    items_audit["v1"],
    ensure_ascii=False,
    indent=2
))

print("\n=== v10 (부재탐지 예시) ===")
print(json.dumps(
    items_audit["v10"],
    ensure_ascii=False,
    indent=2
))

print("\n=== v23 (적용조건 특수 예시) ===")
print(json.dumps(
    items_audit["v23"],
    ensure_ascii=False,
    indent=2
))

print("\n=== v24 (법령근거 없는 비교항목) ===")
print(json.dumps(
    items_audit["v24"],
    ensure_ascii=False,
    indent=2
))

[A] 공식 법령 TXT 23개
01. (계약예규) 공동계약운용요령
02. (계약예규) 정부 입찰·계약 집행기준
03. 국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액
04. 국가를 당사자로 하는 계약에 관한 법률 시행규칙
05. 국가를 당사자로 하는 계약에 관한 법률 시행령
06. 국가를 당사자로 하는 계약에 관한 법률
07. 소상공인기본법 시행령
08. 소상공인기본법
09. 소프트웨어 진흥법 시행령
10. 소프트웨어 진흥법
11. 중소 소프트웨어사업자의 사업 참여 지원에 관한 지침
12. 중소기업기본법 시행령
13. 중소기업기본법
14. 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역
15. 중소기업자간 경쟁제품 직접생산 확인기준
16. 중소기업제품 구매촉진 및 판로지원에 관한 법률 시행규칙
17. 중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령
18. 중소기업제품 구매촉진 및 판로지원에 관한 법률
19. 지방자치단체 입찰 및 계약 집행기준
20. 지방자치단체 입찰시 낙찰자 결정기준
21. 지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙
22. 지방자치단체를 당사자로 하는 계약에 관한 법률 시행령
23. 지방자치단체를 당사자로 하는 계약에 관한 법률

[B] 중기부고시 파일
01. 중기부고시_경쟁제품_세부품명.csv
02. 중기부고시_경쟁제품_제2025-96호.hwpx

[C] 항목표 실제 field 구조

항목: v1, v2, v3, v4, v5, v6, v7, v8, v9, v10, v11, v12, v13, v14, v15, v16, v17, v18, v19, v20, v21, v22, v23, v24
fields:
   - '항목명'
   - '부재탐지'
   - '국가계약법'
   - '지방계약법'
   - '비고'

[D] v1~v24 공식 정의 / 근거 / 비고

#####################################################################

,item,official_text,candidate_law_file,similarity
0,v10,중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사...,중소기업제품 구매촉진 및 판로지원에 관한 법률,1.000
1,v10,중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사...,중소기업제품 구매촉진 및 판로지원에 관한 법률,1.000
2,v10,중기간 경쟁제품 입찰 직생 없음,중소기업자간 경쟁제품 직접생산 확인기준,0.581
3,v11,국가계약법 시행령 제21조 중소기업제품 구매촉진 및 판로지원에 관한 법률 제7조 제...,중소기업제품 구매촉진 및 판로지원에 관한 법률,1.000
4,v11,지방계약법 시행령 제20조 중소기업제품 구매촉진 및 판로지원에 관한 법률 제7조 제...,중소기업제품 구매촉진 및 판로지원에 관한 법률,1.000
5,v11,중기간 경쟁제품 입찰 중소 없음,중소기업자간 경쟁제품 직접생산 확인기준,0.452
6,v12,국가계약법 시행령 제21조,국가를 당사자로 하는 계약에 관한 법률 시행령,0.516
7,v12,중기간경쟁제품 고시 참고,중소기업자간 경쟁제품 직접생산 확인기준,0.483
8,v12,지방계약법 시행령 제20조,지방자치단체를 당사자로 하는 계약에 관한 법률 시행령,0.457
9,v13,국가계약법 시행령 제21조 중소기업제품 구매촉진 및 판로지원에 관한 법률 제7조 제1항,중소기업제품 구매촉진 및 판로지원에 관한 법률,1.000



Stage 6-2A AUDIT 생성 완료
항목 수          : 24
공식 법령 TXT    : 23
중기부고시 파일   : 2
출력             : /home/alparka/nara-ai/artifacts/legal_grounding_audit.json

중요: 현재 legal_grounding_status는 24개 모두 의도적으로 UNRESOLVED입니다.
법령 파일 연결 후보 역시 자동 확정값이 아닙니다.
=== top-level keys ===
dict_keys(['기준일', '표기규칙', '부재탐지항목', '항목'])

=== v1 ===
{
  "항목명": "참가자격 특정기관 제한",
  "부재탐지": false,
  "국가계약법": "국가계약법 시행령 제12조 국가계약법 시행령 제21조",
  "지방계약법": "지방계약법 시행령 제13조 지방계약법 시행령 제20조",
  "비고": ""
}

=== v10 (부재탐지 예시) ===
{
  "항목명": "중기간 경쟁제품 입찰 직생 없음",
  "부재탐지": true,
  "국가계약법": "중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역",
  "지방계약법": "중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역",
  "비고": ""
}

=== v23 (적용조건 특수 예시) ===
{
  "항목명": "현장설명회 (공고기간) * 계약방법 협상 + 계약법 지방만 적용",
  "부재탐지": false,
  "국가계약법": "해당 없음(위반 성립 불가) — 항목명 주석대로 계약방법 협상 + 지방계약 건에만 적용.",
  "지방계약법": "지방계약법시행령 제35조 (행안부예규) 지방자치단체 입찰시 낙찰자 결정기준 제7장 협상에 의한 계약 낙찰자 결정기준 제3절 입찰과 계약상대자 결정절차",
  "비고": ""
}

=== v24 (법령근거 없는 비교항목

In [ ]:
# ============================================================
# 03_baseline_dev.ipynb
# Resume Point: Stage 6-3B
# Stage 6-1 ~ 6-3A 결과는 artifacts에서 복원
# ============================================================

from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
ART = ROOT / "artifacts"

required = [
    "item_spec_workbench.json",
    "official_24_basis.csv",
    "legal_grounding_source_aware.json",
    "legal_grounding_source_aware_audit.csv",
    "judgment_spec_source_pack.json",
]

print("=" * 80)
print("PROJECT RESUME CHECK")
print("=" * 80)

for name in required:
    path = ART / name
    print(
        f"{name:<45}",
        "PASS" if path.exists() else "MISSING"
    )
    assert path.exists(), f"필수 artifact 없음: {path}"

# 최신 Source Pack 로드
with open(
    ART / "judgment_spec_source_pack.json",
    "r",
    encoding="utf-8"
) as f:
    pack = json.load(f)

assert len(pack) == 24

expected_items = [
    f"v{i}"
    for i in range(1, 25)
]

assert set(pack.keys()) == set(expected_items)

absence_items = [
    v
    for v in expected_items
    if pack[v]["absence_detection"]
]

assert absence_items == [
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
]

print()
print("ITEMS         =", len(pack))
print("ABSENCE ITEMS =", absence_items)

print()
print("[PASS] Stage 6-3A 상태 복원 완료")
print("NEXT = Stage 6-3B (v1~v8 judgment specification)")
# ============================================================
# Stage 6-2B-1
# 24개 공식 법적 근거 문자열을 compact하게 전부 확인
# ============================================================

rows = []

for i in range(1, 25):
    v = f"v{i}"
    x = items_audit[v]

    rows.append({
        "item": v,
        "name": x["항목명"],
        "absence": x["부재탐지"],
        "national": x["국가계약법"],
        "local": x["지방계약법"],
        "note": x["비고"],
    })

official_basis_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 300)

display(official_basis_df)

# CSV로도 저장
basis_path = ROOT / "artifacts" / "official_24_basis.csv"

official_basis_df.to_csv(
    basis_path,
    index=False,
    encoding="utf-8"
)

print("\n[PASS] 24개 공식 근거표 생성")
print("rows :", len(official_basis_df))
print("file :", basis_path)

assert len(official_basis_df) == 24
assert official_basis_df["item"].tolist() == [
    f"v{i}" for i in range(1, 25)
]
assert official_basis_df["name"].notna().all()

PROJECT RESUME CHECK
item_spec_workbench.json                      PASS
official_24_basis.csv                         PASS
legal_grounding_source_aware.json             PASS
legal_grounding_source_aware_audit.csv        PASS
judgment_spec_source_pack.json                PASS

ITEMS         = 24
ABSENCE ITEMS = ['v10', 'v11', 'v16', 'v18', 'v20']

[PASS] Stage 6-3A 상태 복원 완료
NEXT = Stage 6-3B (v1~v8 judgment specification)


In [7]:
# ============================================================
# Stage 6-2B REVISED
# 공식 항목표의 법적 근거 → 실제 법령 파일 연결 검증
#
# 이번 단계의 목표:
#   1) 항목표 v1~v24의 국가/지방 근거를 읽는다.
#   2) 공식 근거에 등장하는 법령 약칭을 실제 23개 TXT와 연결한다.
#   3) 연결되지 않은 근거를 숨기지 않고 출력한다.
#   4) locator(조/항/호/장/절/별표)를 보존한다.
#
# 주의:
#   - 아직 실제 조문 본문을 추출하지 않는다.
#   - 아직 판정 규칙을 만들지 않는다.
#   - RESOLVED = 파일 수준 연결 성공이라는 뜻이다.
# ============================================================

import json
import re
from pathlib import Path

import pandas as pd


# ============================================================
# 0. 경로
# ============================================================

ROOT = Path.cwd()

DATA_DIR = ROOT / "open"
ITEM_PATH = DATA_DIR / "data" / "항목표.json"
LAW_DIR = DATA_DIR / "data" / "법령패키지" / "법령"
ARTIFACT_DIR = ROOT / "artifacts"

print("=" * 90)
print("Stage 6-2B REVISED 시작")
print("=" * 90)
print("ROOT     :", ROOT)
print("ITEM_PATH:", ITEM_PATH)
print("LAW_DIR  :", LAW_DIR)


# ============================================================
# 1. 필수 파일 존재 확인
# ============================================================

assert ITEM_PATH.exists(), f"항목표 없음: {ITEM_PATH}"
assert LAW_DIR.exists(), f"법령 폴더 없음: {LAW_DIR}"

print("\n[1] 경로 검증 PASS")


# ============================================================
# 2. 항목표 로드
# ============================================================

with open(ITEM_PATH, "r", encoding="utf-8") as f:
    item_table_62b = json.load(f)

assert "항목" in item_table_62b

items_62b = item_table_62b["항목"]

EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(1, 25)
]

assert set(items_62b.keys()) == set(EXPECTED_ITEMS), (
    f"항목 key 이상: {items_62b.keys()}"
)

print("[2] 항목표 24개 로드 PASS")


# ============================================================
# 3. 실제 법령 TXT 23개 로드
# ============================================================

law_paths_62b = sorted(
    LAW_DIR.glob("*.txt")
)

assert len(law_paths_62b) == 23, (
    f"법령 TXT가 23개가 아님: {len(law_paths_62b)}"
)

LAW_FILES_62B = {
    p.stem: p
    for p in law_paths_62b
}

print("[3] 법령 TXT 23개 확인 PASS")


# ============================================================
# 4. 명시적 alias
#
# fuzzy matching을 사용하지 않는다.
# ============================================================

LAW_ALIASES_62B = {

    # --------------------------------------------------------
    # 국가계약
    # --------------------------------------------------------

    "국가계약법 시행규칙":
        "국가를 당사자로 하는 계약에 관한 법률 시행규칙",

    "국가계약법 시행령":
        "국가를 당사자로 하는 계약에 관한 법률 시행령",

    "국가계약법":
        "국가를 당사자로 하는 계약에 관한 법률",

    # --------------------------------------------------------
    # 지방계약
    # --------------------------------------------------------

    "지방계약법 시행규칙":
        "지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙",

    "지방계약법 시행령":
        "지방자치단체를 당사자로 하는 계약에 관한 법률 시행령",

    "지방계약법":
        "지방자치단체를 당사자로 하는 계약에 관한 법률",

    # --------------------------------------------------------
    # 국가 예규
    # --------------------------------------------------------

    "정부 입찰·계약 집행기준":
        "(계약예규) 정부 입찰·계약 집행기준",

    "정부입찰계약집행기준":
        "(계약예규) 정부 입찰·계약 집행기준",

    "공동계약운용요령":
        "(계약예규) 공동계약운용요령",

    # --------------------------------------------------------
    # 지방 예규
    # --------------------------------------------------------

    "지방자치단체 입찰 및 계약 집행기준":
        "지방자치단체 입찰 및 계약 집행기준",

    "지방자치단체 입찰 및 계약집행기준":
        "지방자치단체 입찰 및 계약 집행기준",

    "지방자치단체 입찰시 낙찰자 결정기준":
        "지방자치단체 입찰시 낙찰자 결정기준",

    # --------------------------------------------------------
    # 판로지원법
    # --------------------------------------------------------

    "판로지원법 시행규칙":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행규칙",

    "판로지원법 시행령":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령",

    "판로지원법":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률",

    # --------------------------------------------------------
    # 소프트웨어
    # --------------------------------------------------------

    "소프트웨어 진흥법 시행령":
        "소프트웨어 진흥법 시행령",

    "소프트웨어진흥법 시행령":
        "소프트웨어 진흥법 시행령",

    "소프트웨어 진흥법":
        "소프트웨어 진흥법",

    "소프트웨어진흥법":
        "소프트웨어 진흥법",

    "중소 소프트웨어사업자의 사업 참여 지원에 관한 지침":
        "중소 소프트웨어사업자의 사업 참여 지원에 관한 지침",

    "중소 SW사업자의 사업 참여 지원 지침":
        "중소 소프트웨어사업자의 사업 참여 지원에 관한 지침",

    # --------------------------------------------------------
    # 중기간 경쟁제품
    # --------------------------------------------------------

    "중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역":
        "중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역",

    "중기간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역":
        "중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역",

    # --------------------------------------------------------
    # 고시금액
    # --------------------------------------------------------

    "고시금액":
        "국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액",
}


# ============================================================
# 5. alias가 가리키는 파일이 실제 존재하는지 검증
# ============================================================

bad_aliases = []

for alias, target in LAW_ALIASES_62B.items():

    if target not in LAW_FILES_62B:

        bad_aliases.append({
            "alias": alias,
            "target": target,
        })


if bad_aliases:

    print("\n[ERROR] 잘못된 alias target")

    for x in bad_aliases:
        print(x)

    raise RuntimeError(
        "LAW_ALIASES_62B에 실제 존재하지 않는 파일명이 있습니다."
    )


print("[4] alias → 실제 파일 검증 PASS")


# ============================================================
# 6. locator 추출
#
# 이 단계에서는 '존재 여부'를 판정하지 않는다.
# 공식 항목표에 적힌 locator를 구조화해서 보존만 한다.
# ============================================================

LOCATOR_PATTERNS = {

    "article": re.compile(
        r"제\s*\d+\s*조(?:\s*의\s*\d+)?"
    ),

    "paragraph": re.compile(
        r"제\s*\d+\s*항"
    ),

    "subparagraph": re.compile(
        r"제\s*\d+\s*호"
    ),

    "chapter": re.compile(
        r"제\s*\d+\s*장"
    ),

    "section": re.compile(
        r"제\s*\d+\s*절"
    ),

    "appendix": re.compile(
        r"별표\s*\d*"
    ),
}


def extract_locators_62b(text):

    text = str(text or "")

    result = {}

    for name, pattern in LOCATOR_PATTERNS.items():

        matches = [
            re.sub(r"\s+", "", x)
            for x in pattern.findall(text)
        ]

        # 중복 제거 + 원래 순서 유지
        result[name] = list(
            dict.fromkeys(matches)
        )

    return result


# ============================================================
# 7. 하나의 공식 근거 문자열에서 법령 occurrence 찾기
#
# 긴 alias를 먼저 찾는다.
#
# 예:
# 국가계약법 시행령
#
# 에서
# 국가계약법
#
# 을 중복으로 잡지 않도록 한다.
# ============================================================

ALIASES_SORTED_62B = sorted(
    LAW_ALIASES_62B.keys(),
    key=len,
    reverse=True
)


def find_law_occurrences_62b(text):

    text = str(text or "")

    candidates = []

    for alias in ALIASES_SORTED_62B:

        for match in re.finditer(
            re.escape(alias),
            text
        ):

            candidates.append({
                "alias": alias,
                "file": LAW_ALIASES_62B[alias],
                "start": match.start(),
                "end": match.end(),
            })

    # 긴 alias 우선
    candidates.sort(
        key=lambda x: (
            x["start"],
            -(x["end"] - x["start"])
        )
    )

    accepted = []

    for candidate in candidates:

        overlap = False

        for prev in accepted:

            if not (
                candidate["end"] <= prev["start"]
                or
                candidate["start"] >= prev["end"]
            ):
                overlap = True
                break

        if not overlap:
            accepted.append(candidate)

    accepted.sort(
        key=lambda x: x["start"]
    )

    return accepted


# ============================================================
# 8. 법령별 segment 분리
# ============================================================

def split_basis_62b(text):

    text = str(text or "").strip()

    # ----------------------------------------
    # 공식적으로 근거 없음
    # ----------------------------------------

    if not text:
        return {
            "kind": "EMPTY",
            "sources": [],
            "unmatched_text": "",
        }

    if "해당 없음" in text:

        return {
            "kind": "NOT_APPLICABLE",
            "sources": [],
            "unmatched_text": "",
        }

    # ----------------------------------------
    # 법령 occurrence
    # ----------------------------------------

    occurrences = find_law_occurrences_62b(
        text
    )

    if not occurrences:

        return {
            "kind": "UNRESOLVED",
            "sources": [],
            "unmatched_text": text,
        }

    sources = []

    for idx, occ in enumerate(occurrences):

        if idx + 1 < len(occurrences):
            segment_end = occurrences[
                idx + 1
            ]["start"]
        else:
            segment_end = len(text)

        segment = text[
            occ["start"]:segment_end
        ].strip()

        sources.append({
            "alias": occ["alias"],
            "file": occ["file"],
            "segment": segment,
            "locators": extract_locators_62b(
                segment
            ),
        })

    # ----------------------------------------
    # 첫 법령명 앞에 의미 있는 텍스트가 있는지도 확인
    # ----------------------------------------

    prefix = text[
        :occurrences[0]["start"]
    ].strip(" ,;/")

    return {
        "kind": "RESOLVED",
        "sources": sources,
        "unmatched_text": prefix,
    }


# ============================================================
# 9. 24개 전체 grounding
# ============================================================

grounding_62b = {}
audit_rows_62b = []

for i in range(1, 25):

    v = f"v{i}"
    obj = items_62b[v]

    national_text = str(
        obj.get("국가계약법") or ""
    ).strip()

    local_text = str(
        obj.get("지방계약법") or ""
    ).strip()

    national = split_basis_62b(
        national_text
    )

    local = split_basis_62b(
        local_text
    )

    problems = []

    if national["kind"] == "UNRESOLVED":
        problems.append(
            "national_no_law_match"
        )

    if local["kind"] == "UNRESOLVED":
        problems.append(
            "local_no_law_match"
        )

    # 법령 앞에 예상치 못한 문자열이 있는 경우도
    # 그냥 버리지 않는다.
    if national["unmatched_text"]:
        problems.append(
            "national_prefix_unmatched"
        )

    if local["unmatched_text"]:
        problems.append(
            "local_prefix_unmatched"
        )

    if problems:
        status = "UNRESOLVED"
    else:
        status = "RESOLVED"

    grounding_62b[v] = {

        "item":
            v,

        "item_name":
            obj["항목명"],

        "absence_detection":
            bool(obj["부재탐지"]),

        "note":
            obj.get("비고", ""),

        "official": {
            "national":
                national_text,

            "local":
                local_text,
        },

        "national":
            national,

        "local":
            local,

        "status":
            status,

        "problems":
            problems,
    }

    audit_rows_62b.append({

        "item":
            v,

        "name":
            obj["항목명"],

        "national_kind":
            national["kind"],

        "national_sources":
            len(national["sources"]),

        "local_kind":
            local["kind"],

        "local_sources":
            len(local["sources"]),

        "status":
            status,

        "problems":
            ", ".join(problems),
    })


print("[5] v1~v24 grounding 생성 PASS")


# ============================================================
# 10. 실제 연결된 파일 재검증
# ============================================================

for v, spec in grounding_62b.items():

    for side in [
        "national",
        "local"
    ]:

        for source in spec[side]["sources"]:

            target = source["file"]

            assert target in LAW_FILES_62B, (
                f"{v}: 실제 파일 없음 → {target}"
            )


print("[6] 연결된 모든 법령파일 존재 검증 PASS")


# ============================================================
# 11. Audit DataFrame
# ============================================================

audit_df_62b = pd.DataFrame(
    audit_rows_62b
)

print()
print("=" * 90)
print("[7] 24개 파일-level grounding 결과")
print("=" * 90)

display(audit_df_62b)


# ============================================================
# 12. UNRESOLVED 상세
# ============================================================

unresolved_62b = [
    v
    for v in EXPECTED_ITEMS
    if grounding_62b[v]["status"]
       != "RESOLVED"
]


print()
print("=" * 90)
print("[8] UNRESOLVED 상세")
print("=" * 90)


if not unresolved_62b:

    print("없음")

else:

    for v in unresolved_62b:

        spec = grounding_62b[v]

        print()
        print("-" * 90)
        print(
            v,
            spec["item_name"]
        )

        print(
            "문제:",
            spec["problems"]
        )

        print(
            "국가 원문:",
            spec["official"]["national"]
        )

        print(
            "국가 분석:",
            spec["national"]
        )

        print(
            "지방 원문:",
            spec["official"]["local"]
        )

        print(
            "지방 분석:",
            spec["local"]
        )


# ============================================================
# 13. 연결된 실제 파일 목록
# ============================================================

used_files_62b = set()

for spec in grounding_62b.values():

    for side in [
        "national",
        "local"
    ]:

        for source in spec[side]["sources"]:
            used_files_62b.add(
                source["file"]
            )


print()
print("=" * 90)
print("[9] 실제 사용되는 법령파일")
print("=" * 90)

for filename in sorted(
    used_files_62b
):
    print("-", filename)


# ============================================================
# 14. 저장
# ============================================================

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GROUNDING_PATH_62B = (
    ARTIFACT_DIR
    / "legal_grounding_file_level.json"
)

AUDIT_CSV_PATH_62B = (
    ARTIFACT_DIR
    / "legal_grounding_file_level_audit.csv"
)


with open(
    GROUNDING_PATH_62B,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        grounding_62b,
        f,
        ensure_ascii=False,
        indent=2
    )


audit_df_62b.to_csv(
    AUDIT_CSV_PATH_62B,
    index=False,
    encoding="utf-8"
)


print("[10] 결과 저장 PASS")


# ============================================================
# 15. 최종 결과
# ============================================================

resolved_count_62b = (
    24 - len(unresolved_62b)
)

unresolved_count_62b = (
    len(unresolved_62b)
)


print()
print("=" * 90)
print("Stage 6-2B 최종 결과")
print("=" * 90)

print(
    "항목 수        :",
    len(grounding_62b)
)

print(
    "RESOLVED       :",
    resolved_count_62b
)

print(
    "UNRESOLVED     :",
    unresolved_count_62b
)

print(
    "Grounding JSON :",
    GROUNDING_PATH_62B
)

print(
    "Audit CSV      :",
    AUDIT_CSV_PATH_62B
)


if unresolved_count_62b == 0:

    print()
    print(
        "[PASS] Stage 6-2B 파일-level grounding 완료"
    )

    print(
        "다음 단계: Stage 6-2C 실제 원문 locator 검증"
    )

else:

    print()
    print(
        "[FAIL] 아직 해결되지 않은 공식 근거가 있습니다."
    )

    print(
        "Stage 6-2C로 넘어가기 전에 위 [8] 내용을 확인해야 합니다."
    )

Stage 6-2B REVISED 시작
ROOT     : /home/alparka/nara-ai
ITEM_PATH: /home/alparka/nara-ai/open/data/항목표.json
LAW_DIR  : /home/alparka/nara-ai/open/data/법령패키지/법령

[1] 경로 검증 PASS
[2] 항목표 24개 로드 PASS
[3] 법령 TXT 23개 확인 PASS
[4] alias → 실제 파일 검증 PASS
[5] v1~v24 grounding 생성 PASS
[6] 연결된 모든 법령파일 존재 검증 PASS

[7] 24개 파일-level grounding 결과


,item,name,national_kind,national_sources,local_kind,local_sources,status,problems
0,v1,참가자격 특정기관 제한,RESOLVED,2,RESOLVED,2,RESOLVED,
1,v2,고시금액 미만 실적제한,RESOLVED,2,RESOLVED,2,RESOLVED,
2,v3,실적제한 1배수 이상,RESOLVED,3,RESOLVED,2,RESOLVED,
3,v4,"고시금액 이상 특정기관, 특정실적",RESOLVED,1,RESOLVED,1,UNRESOLVED,"national_prefix_unmatched, local_prefix_unmatched"
4,v5,고시금액 이상 지역제한,RESOLVED,2,RESOLVED,2,RESOLVED,
5,v6,"고시금액 미만 지역제한 시,군,구",RESOLVED,1,RESOLVED,2,RESOLVED,
6,v7,고시금액 미만 지역제한 인접 확대,RESOLVED,1,RESOLVED,2,RESOLVED,
7,v8,중복제한 (실적+지역),RESOLVED,1,RESOLVED,2,RESOLVED,
8,v9,과업지시서 특정 모델명 명시,RESOLVED,1,RESOLVED,1,UNRESOLVED,"national_prefix_unmatched, local_prefix_unmatched"
9,v10,중기간 경쟁제품 입찰 직생 없음,RESOLVED,1,RESOLVED,1,UNRESOLVED,"national_prefix_unmatched, local_prefix_unmatched"



[8] UNRESOLVED 상세

------------------------------------------------------------------------------------------
v4 고시금액 이상 특정기관, 특정실적
문제: ['national_prefix_unmatched', 'local_prefix_unmatched']
국가 원문: (계약예규) 정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조
국가 분석: {'kind': 'RESOLVED', 'sources': [{'alias': '정부입찰계약집행기준', 'file': '(계약예규) 정부 입찰·계약 집행기준', 'segment': '정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조', 'locators': {'article': ['제5조'], 'paragraph': [], 'subparagraph': [], 'chapter': ['제2장'], 'section': [], 'appendix': []}}], 'unmatched_text': '(계약예규)'}
지방 원문: (행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준
지방 분석: {'kind': 'RESOLVED', 'sources': [{'alias': '지방자치단체 입찰 및 계약 집행기준', 'file': '지방자치단체 입찰 및 계약 집행기준', 'segment': '지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준', 'locators': {'article': [], 'paragraph': [], 'subparagraph': [], 'chapter': ['제1장'], 'section': [], 'appendix': []}}], 'unmatched_text': '(행안부예규)'}

------------------------------------------------------------------------------------------
v9 과업지시서 특정 모델명 명시
문제: [

In [8]:
print("RESOLVED   =", resolved_count_62b)
print("UNRESOLVED =", unresolved_count_62b)
print("ITEMS      =", unresolved_62b)

RESOLVED   = 20
UNRESOLVED = 4
ITEMS      = ['v4', 'v9', 'v10', 'v21']


In [9]:
for v in ["v4", "v9", "v10", "v21"]:
    x = grounding_62b[v]

    print("\n" + "=" * 100)
    print(v, "-", x["item_name"])
    print("=" * 100)

    print("STATUS   :", x["status"])
    print("PROBLEMS :", x["problems"])

    print("\n[국가 공식 근거]")
    print(x["official"]["national"])

    print("\n[국가 분석]")
    print(x["national"])

    print("\n[지방 공식 근거]")
    print(x["official"]["local"])

    print("\n[지방 분석]")
    print(x["local"])


v4 - 고시금액 이상 특정기관, 특정실적
STATUS   : UNRESOLVED
PROBLEMS : ['national_prefix_unmatched', 'local_prefix_unmatched']

[국가 공식 근거]
(계약예규) 정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조

[국가 분석]
{'kind': 'RESOLVED', 'sources': [{'alias': '정부입찰계약집행기준', 'file': '(계약예규) 정부 입찰·계약 집행기준', 'segment': '정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조', 'locators': {'article': ['제5조'], 'paragraph': [], 'subparagraph': [], 'chapter': ['제2장'], 'section': [], 'appendix': []}}], 'unmatched_text': '(계약예규)'}

[지방 공식 근거]
(행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준

[지방 분석]
{'kind': 'RESOLVED', 'sources': [{'alias': '지방자치단체 입찰 및 계약 집행기준', 'file': '지방자치단체 입찰 및 계약 집행기준', 'segment': '지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준', 'locators': {'article': [], 'paragraph': [], 'subparagraph': [], 'chapter': ['제1장'], 'section': [], 'appendix': []}}], 'unmatched_text': '(행안부예규)'}

v9 - 과업지시서 특정 모델명 명시
STATUS   : UNRESOLVED
PROBLEMS : ['national_prefix_unmatched', 'local_prefix_unmatched']

[국가 공식 근거]
(계약예규) 정부 입찰·계약 집행기준 제2장 제한경쟁입찰의 운용 제5조

[국가 분석]
{'kind': 

In [4]:
# ============================================================
# Stage 6-2B 복구
# 저장된 결과를 직접 읽어서 커널 변수 복원
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

GROUNDING_PATH = (
    ARTIFACT_DIR
    / "legal_grounding_file_level.json"
)

assert GROUNDING_PATH.exists(), (
    f"파일을 찾을 수 없습니다: {GROUNDING_PATH}"
)

with open(
    GROUNDING_PATH,
    "r",
    encoding="utf-8"
) as f:
    grounding_62b = json.load(f)

assert len(grounding_62b) == 24

print("복구 완료")
print("항목 수 =", len(grounding_62b))

print("\n남은 4개 공식 근거")

for v in ["v4", "v9", "v10", "v21"]:

    x = grounding_62b[v]

    print("\n" + "=" * 90)
    print(v, "-", x["item_name"])
    print("=" * 90)

    print("STATUS   :", x["status"])
    print("PROBLEMS :", x["problems"])

    print("국가 :", x["official"]["national"])
    print("지방 :", x["official"]["local"])
    print("비고 :", x.get("note", ""))

복구 완료
항목 수 = 24

남은 4개 공식 근거

v4 - 고시금액 이상 특정기관, 특정실적
STATUS   : UNRESOLVED
PROBLEMS : ['national_prefix_unmatched', 'local_prefix_unmatched']
국가 : (계약예규) 정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조
지방 : (행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준
비고 : 특정기관 표현 다양

v9 - 과업지시서 특정 모델명 명시
STATUS   : UNRESOLVED
PROBLEMS : ['national_prefix_unmatched', 'local_prefix_unmatched']
국가 : (계약예규) 정부 입찰·계약 집행기준 제2장 제한경쟁입찰의 운용 제5조
지방 : (행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약일반기준 7. 계약담당자 주의사항
비고 : 

v10 - 중기간 경쟁제품 입찰 직생 없음
STATUS   : UNRESOLVED
PROBLEMS : ['national_prefix_unmatched', 'local_prefix_unmatched']
국가 : 중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역
지방 : 중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역
비고 : 

v21 - 공동 5% (10%)
STATUS   : UNRESOLVED
PROBLEMS : ['national_prefix_unmatched', 'local_prefix_unmatched']
국가 : (계약예규) 공동계약운용요령
지방 : (행안부예규) 지방자치단체 입찰 및 계약 집행기준 제6장 공동계약 운영요령
비고 : 


In [5]:
# ============================================================
# Stage 6-2C
# 공식 24개 근거 → 실제 배포 source / locator 검증
#
# 목표:
#   1) 일반 법령       : 조문 존재 확인
#   2) 예규/집행기준   : 장/절/조/명칭 anchor 확인
#   3) 중기간 지정내역 : 중기부고시 structured source 확인
#   4) v24             : 법령 retrieval 대상이 아닌 comparison으로 처리
#
# 중요:
#   - 이전 커널 변수에 의존하지 않음
#   - 외부 법령 사용하지 않음
#   - legal_specs.json 사용하지 않음
#   - 자동으로 못 찾은 것은 억지 RESOLVED 하지 않음
# ============================================================

from pathlib import Path
import pandas as pd
import json
import re
import unicodedata


# ------------------------------------------------------------
# 0. 경로
# ------------------------------------------------------------

ROOT = Path.cwd()

ARTIFACT_DIR = ROOT / "artifacts"
DATA_DIR = ROOT / "open" / "data"
LAW_PACKAGE_DIR = DATA_DIR / "법령패키지"
LAW_DIR = LAW_PACKAGE_DIR / "법령"
SME_DIR = LAW_PACKAGE_DIR / "중기부고시"

OFFICIAL_BASIS_PATH = (
    ARTIFACT_DIR / "official_24_basis.csv"
)

assert OFFICIAL_BASIS_PATH.exists(), OFFICIAL_BASIS_PATH
assert LAW_DIR.exists(), LAW_DIR
assert SME_DIR.exists(), SME_DIR


# ------------------------------------------------------------
# 1. 공식 24개 근거 로드
# ------------------------------------------------------------

basis = pd.read_csv(OFFICIAL_BASIS_PATH)

required_cols = {
    "item",
    "name",
    "absence",
    "national",
    "local",
    "note",
}

missing_cols = required_cols - set(basis.columns)

assert not missing_cols, (
    f"official_24_basis.csv 컬럼 부족: {missing_cols}"
)

assert len(basis) == 24
assert set(basis["item"]) == {
    f"v{i}" for i in range(1, 25)
}


# ------------------------------------------------------------
# 2. 실제 배포 파일 inventory
# ------------------------------------------------------------

law_files = sorted(LAW_DIR.glob("*.txt"))

sme_files = sorted(
    p for p in SME_DIR.iterdir()
    if p.is_file()
)

assert len(law_files) == 23, (
    f"예상 법령 TXT=23, 실제={len(law_files)}"
)

print("=" * 100)
print("Stage 6-2C SOURCE INVENTORY")
print("=" * 100)

print("법령 TXT :", len(law_files))
print("중기부 자료:", len(sme_files))

for p in sme_files:
    print("  SME:", p.name)


# ------------------------------------------------------------
# 3. 텍스트 정규화
# ------------------------------------------------------------

def norm_text(s):
    s = unicodedata.normalize(
        "NFC",
        str(s or "")
    )

    # whitespace 차이는 locator 판정에 불필요
    s = re.sub(r"\s+", " ", s)

    return s.strip()


def compact(s):
    """
    띄어쓰기/일부 구두점 차이를 무시한 비교용.
    원문 자체를 수정하는 용도가 아니다.
    """
    s = norm_text(s)

    return re.sub(
        r"[\s·ㆍ・,()\[\]「」『』<>]",
        "",
        s
    )


# ------------------------------------------------------------
# 4. 법령 파일 로드
# ------------------------------------------------------------

laws = {}

for p in law_files:

    text = p.read_text(
        encoding="utf-8",
        errors="replace"
    )

    laws[p.stem] = {
        "path": p,
        "text": text,
        "norm": norm_text(text),
        "compact": compact(text),
    }


# ------------------------------------------------------------
# 5. 공식 명칭 → 실제 파일
#
# fuzzy matching으로 최종 확정하지 않는다.
# 실제 배포 파일 stem과 명시적으로 연결한다.
# ------------------------------------------------------------

def find_file_by_keywords(*keyword_groups):
    """
    keyword_groups 중 하나라도 모두 포함하는
    파일 stem을 찾는다.

    자동으로 여러 개가 잡히면 확정하지 않는다.
    """

    candidates = []

    for stem in laws:

        cstem = compact(stem)

        matched = False

        for group in keyword_groups:

            if all(
                compact(k) in cstem
                for k in group
            ):
                matched = True
                break

        if matched:
            candidates.append(stem)

    return candidates


# ------------------------------------------------------------
# 6. 공식 근거에서 locator 추출
# ------------------------------------------------------------

RE_CHAPTER = re.compile(
    r"제\s*(\d+)\s*장"
)

RE_SECTION = re.compile(
    r"제\s*(\d+)\s*절"
)

RE_ARTICLE = re.compile(
    r"제\s*(\d+)\s*조(?:의\s*(\d+))?"
)

RE_APPENDIX = re.compile(
    r"(별표|별지)\s*제?\s*(\d+)"
)


def extract_locators(text):

    text = str(text or "")

    return {
        "chapters": [
            m.group(0)
            for m in RE_CHAPTER.finditer(text)
        ],

        "sections": [
            m.group(0)
            for m in RE_SECTION.finditer(text)
        ],

        "articles": [
            m.group(0)
            for m in RE_ARTICLE.finditer(text)
        ],

        "appendices": [
            m.group(0)
            for m in RE_APPENDIX.finditer(text)
        ],
    }


# ------------------------------------------------------------
# 7. locator가 실제 파일 안에 존재하는지 검사
# ------------------------------------------------------------

def locator_exists(locator, source_text):

    return (
        compact(locator)
        in compact(source_text)
    )


# ------------------------------------------------------------
# 8. 공식 근거에 등장하는 source 분류
# ------------------------------------------------------------

def classify_basis_source(text):

    t = str(text or "").strip()
    ct = compact(t)

    if not t:
        return "EMPTY"

    if "해당없음" in ct:
        return "COMPARISON_OR_NA"

    if (
        "중소기업자간경쟁제품및공사용자재"
        in ct
        or
        "직접구매대상품목지정내역"
        in ct
    ):
        return "SME_MIXED"

    if (
        "집행기준" in t
        or "운용요령" in t
        or "운영요령" in t
        or "계약예규" in t
        or "행안부예규" in t
    ):
        return "GUIDELINE"

    return "STATUTE"


# ------------------------------------------------------------
# 9. source 파일 후보 찾기
# ------------------------------------------------------------

SOURCE_PATTERNS = [
    (
        "국가계약법 시행령",
        [
            ("국가", "계약", "시행령"),
        ]
    ),

    (
        "국가계약법 시행규칙",
        [
            ("국가", "계약", "시행규칙"),
        ]
    ),

    (
        "지방계약법 시행령",
        [
            ("지방", "계약", "시행령"),
        ]
    ),

    (
        "지방계약법 시행규칙",
        [
            ("지방", "계약", "시행규칙"),
        ]
    ),

    (
        "정부 입찰·계약 집행기준",
        [
            ("정부", "입찰", "계약", "집행기준"),
        ]
    ),

    (
        "정부입찰계약집행기준",
        [
            ("정부", "입찰", "계약", "집행기준"),
        ]
    ),

    (
        "지방자치단체 입찰 및 계약 집행기준",
        [
            ("지방자치단체", "입찰", "계약", "집행기준"),
        ]
    ),

    (
        "공동계약운용요령",
        [
            ("공동", "계약", "운용요령"),
            ("공동", "계약", "운영요령"),
        ]
    ),

    (
        "중소기업제품 구매촉진 및 판로지원에 관한 법률",
        [
            ("중소기업제품", "구매촉진", "판로지원", "법률"),
        ]
    ),

    (
        "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령",
        [
            (
                "중소기업제품",
                "구매촉진",
                "판로지원",
                "시행령",
            ),
        ]
    ),

    (
        "소프트웨어 진흥법",
        [
            ("소프트웨어", "진흥법"),
        ]
    ),
]


def detect_sources(text):

    tcompact = compact(text)

    found = []

    for official_name, groups in SOURCE_PATTERNS:

        if compact(official_name) not in tcompact:
            continue

        candidates = find_file_by_keywords(
            *groups
        )

        found.append({
            "official_name":
                official_name,

            "file_candidates":
                candidates,
        })

    return found


# ------------------------------------------------------------
# 10. hierarchical anchor 후보
#
# 조문 번호 외에도 공식 근거 문자열에 적힌
# 장/항목 명칭을 확인하기 위한 핵심 anchor
# ------------------------------------------------------------

KNOWN_ANCHORS = [
    "제한경쟁입찰의 운용",
    "입찰 및 계약 일반기준",
    "입찰 및 계약일반기준",
    "계약담당자 주의사항",
    "공동계약 운영요령",
    "공동계약운용요령",
    "수의계약 운영요령",
]


def detect_named_anchors(text):

    return [
        a
        for a in KNOWN_ANCHORS
        if compact(a) in compact(text)
    ]


# ------------------------------------------------------------
# 11. SME structured source 확인
# ------------------------------------------------------------

sme_csvs = [
    p for p in sme_files
    if p.suffix.lower() == ".csv"
]

sme_originals = [
    p for p in sme_files
    if p.suffix.lower() != ".csv"
]

SME_STRUCTURED_AVAILABLE = (
    len(sme_csvs) >= 1
)


# ------------------------------------------------------------
# 12. 한 side 검증
# ------------------------------------------------------------

def verify_side(item, side, basis_text):

    text = str(basis_text or "").strip()

    result = {
        "item": item,
        "side": side,
        "official_basis": text,
        "source_type":
            classify_basis_source(text),

        "sources": [],
        "locators":
            extract_locators(text),

        "named_anchors":
            detect_named_anchors(text),

        "sme_structured_source": None,

        "status": "UNRESOLVED",
        "problems": [],
    }

    # ------------------------
    # v24 / 해당 없음
    # ------------------------

    if item == "v24":

        result["source_type"] = "COMPARISON"

        result["status"] = "RESOLVED"

        return result

    if not text:

        result["problems"].append(
            "empty_basis"
        )

        return result

    # ------------------------
    # 일반 법령 / guideline
    # ------------------------

    detected = detect_sources(text)

    for source in detected:

        candidates = source[
            "file_candidates"
        ]

        entry = {
            "official_name":
                source["official_name"],

            "file_candidates":
                candidates,

            "resolved_file":
                None,

            "locator_checks":
                {},

            "anchor_checks":
                {},
        }

        if len(candidates) == 1:

            resolved_file = candidates[0]

            entry["resolved_file"] = (
                resolved_file
            )

            source_text = laws[
                resolved_file
            ]["text"]

            for loc_type, locs in (
                result["locators"].items()
            ):

                for loc in locs:

                    entry[
                        "locator_checks"
                    ][loc] = locator_exists(
                        loc,
                        source_text
                    )

            for anchor in (
                result["named_anchors"]
            ):

                entry[
                    "anchor_checks"
                ][anchor] = (
                    compact(anchor)
                    in compact(source_text)
                )

        elif len(candidates) == 0:

            result["problems"].append(
                "source_file_not_found:"
                + source["official_name"]
            )

        else:

            result["problems"].append(
                "source_file_ambiguous:"
                + source["official_name"]
            )

        result["sources"].append(entry)

    # ------------------------
    # SME 지정품목 별도 처리
    # ------------------------

    if result["source_type"] == "SME_MIXED":

        result["sme_structured_source"] = {
            "csv_available":
                SME_STRUCTURED_AVAILABLE,

            "csv_files":
                [p.name for p in sme_csvs],

            "original_files":
                [p.name for p in sme_originals],
        }

        if not SME_STRUCTURED_AVAILABLE:

            result["problems"].append(
                "sme_designated_product_csv_missing"
            )

    # ------------------------
    # source 자체가 하나도 안 잡힘
    # ------------------------

    if (
        not result["sources"]
        and result["source_type"]
        not in {
            "COMPARISON",
        }
    ):

        # SME mixed라도 법률 + 지정내역 중
        # 법률 source는 있어야 한다.
        result["problems"].append(
            "no_legal_source_detected"
        )

    # ------------------------
    # locator 실패 검사
    # ------------------------

    for source in result["sources"]:

        if not source["resolved_file"]:
            continue

        for locator, ok in (
            source["locator_checks"].items()
        ):

            if not ok:

                result["problems"].append(
                    f"locator_not_found:"
                    f"{source['resolved_file']}:"
                    f"{locator}"
                )

        for anchor, ok in (
            source["anchor_checks"].items()
        ):

            if not ok:

                result["problems"].append(
                    f"anchor_not_found:"
                    f"{source['resolved_file']}:"
                    f"{anchor}"
                )

    # ------------------------
    # 최종 상태
    # ------------------------

    if not result["problems"]:

        result["status"] = "RESOLVED"

    else:

        # 적어도 source가 하나 연결됐으면
        # 완전 실패와 구분한다.
        has_resolved_source = any(
            s["resolved_file"]
            for s in result["sources"]
        )

        if has_resolved_source:
            result["status"] = "PARTIAL"

        else:
            result["status"] = "FAILED"

    return result


# ------------------------------------------------------------
# 13. 24개 전체 검증
# ------------------------------------------------------------

locator_audit = {}

summary_rows = []

for _, row in basis.iterrows():

    item = row["item"]

    national = verify_side(
        item,
        "national",
        row["national"]
    )

    local = verify_side(
        item,
        "local",
        row["local"]
    )

    # v24는 양쪽 모두 comparison
    if item == "v24":

        overall = "RESOLVED"

    else:

        side_statuses = {
            national["status"],
            local["status"],
        }

        if side_statuses == {"RESOLVED"}:

            overall = "RESOLVED"

        elif "FAILED" in side_statuses:

            overall = "FAILED"

        else:

            overall = "PARTIAL"

    locator_audit[item] = {
        "item":
            item,

        "name":
            row["name"],

        "absence":
            bool(row["absence"]),

        "note":
            (
                ""
                if pd.isna(row["note"])
                else str(row["note"])
            ),

        "national":
            national,

        "local":
            local,

        "overall_status":
            overall,
    }

    summary_rows.append({
        "item":
            item,

        "name":
            row["name"],

        "national":
            national["status"],

        "local":
            local["status"],

        "overall":
            overall,

        "national_problems":
            " | ".join(
                national["problems"]
            ),

        "local_problems":
            " | ".join(
                local["problems"]
            ),
    })


# ------------------------------------------------------------
# 14. 저장
# ------------------------------------------------------------

OUT_JSON = (
    ARTIFACT_DIR
    / "legal_grounding_locator_audit.json"
)

OUT_CSV = (
    ARTIFACT_DIR
    / "legal_grounding_locator_audit.csv"
)


with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        locator_audit,
        f,
        ensure_ascii=False,
        indent=2
    )


summary_df = pd.DataFrame(
    summary_rows
)

summary_df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 15. 결과 출력
# ------------------------------------------------------------

display(
    summary_df[
        [
            "item",
            "name",
            "national",
            "local",
            "overall",
        ]
    ]
)


counts = (
    summary_df["overall"]
    .value_counts()
    .to_dict()
)

resolved_items = (
    summary_df.loc[
        summary_df["overall"]
        == "RESOLVED",
        "item"
    ].tolist()
)

partial_items = (
    summary_df.loc[
        summary_df["overall"]
        == "PARTIAL",
        "item"
    ].tolist()
)

failed_items = (
    summary_df.loc[
        summary_df["overall"]
        == "FAILED",
        "item"
    ].tolist()
)


print()
print("=" * 100)
print("Stage 6-2C 최종 결과")
print("=" * 100)

print(
    "RESOLVED =",
    len(resolved_items)
)

print(
    "PARTIAL  =",
    len(partial_items)
)

print(
    "FAILED   =",
    len(failed_items)
)

print(
    "PARTIAL ITEMS =",
    partial_items
)

print(
    "FAILED ITEMS  =",
    failed_items
)

print()
print("JSON =", OUT_JSON)
print("CSV  =", OUT_CSV)


# ------------------------------------------------------------
# 16. 문제 항목만 상세 요약
# ------------------------------------------------------------

problem_items = (
    partial_items
    + failed_items
)

if problem_items:

    print()
    print("=" * 100)
    print("검토 필요한 항목")
    print("=" * 100)

    for item in problem_items:

        x = locator_audit[item]

        print()
        print(
            item,
            "-",
            x["name"]
        )

        print(
            "  국가:",
            x["national"]["status"]
        )

        for p in x[
            "national"
        ]["problems"]:
            print(
                "    -",
                p
            )

        print(
            "  지방:",
            x["local"]["status"]
        )

        for p in x[
            "local"
        ]["problems"]:
            print(
                "    -",
                p
            )

Stage 6-2C SOURCE INVENTORY
법령 TXT : 23
중기부 자료: 2
  SME: 중기부고시_경쟁제품_세부품명.csv
  SME: 중기부고시_경쟁제품_제2025-96호.hwpx


,item,name,national,local,overall
0,v1,참가자격 특정기관 제한,RESOLVED,RESOLVED,RESOLVED
1,v2,고시금액 미만 실적제한,RESOLVED,PARTIAL,PARTIAL
2,v3,실적제한 1배수 이상,PARTIAL,RESOLVED,PARTIAL
3,v4,"고시금액 이상 특정기관, 특정실적",RESOLVED,RESOLVED,RESOLVED
4,v5,고시금액 이상 지역제한,RESOLVED,RESOLVED,RESOLVED
5,v6,"고시금액 미만 지역제한 시,군,구",RESOLVED,PARTIAL,PARTIAL
6,v7,고시금액 미만 지역제한 인접 확대,RESOLVED,PARTIAL,PARTIAL
7,v8,중복제한 (실적+지역),RESOLVED,PARTIAL,PARTIAL
8,v9,과업지시서 특정 모델명 명시,RESOLVED,RESOLVED,RESOLVED
9,v10,중기간 경쟁제품 입찰 직생 없음,FAILED,FAILED,FAILED



Stage 6-2C 최종 결과
RESOLVED = 8
PARTIAL  = 13
FAILED   = 3
PARTIAL ITEMS = ['v2', 'v3', 'v6', 'v7', 'v8', 'v11', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19']
FAILED ITEMS  = ['v10', 'v20', 'v23']

JSON = /home/alparka/nara-ai/artifacts/legal_grounding_locator_audit.json
CSV  = /home/alparka/nara-ai/artifacts/legal_grounding_locator_audit.csv

검토 필요한 항목

v2 - 고시금액 미만 실적제한
  국가: RESOLVED
  지방: PARTIAL
    - locator_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행령:제1절
    - anchor_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행령:입찰 및 계약 일반기준
    - anchor_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행령:입찰 및 계약일반기준
    - anchor_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행령:계약담당자 주의사항
    - locator_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙:제1절
    - anchor_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙:입찰 및 계약 일반기준
    - anchor_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙:입찰 및 계약일반기준
    - anchor_not_found:지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙:계약담당자 주의사항

v3 - 실적제한 1배수 이상
  국가: PARTIAL
    - anchor_not_found:국가를 당사자로 하는 계약에 관한 법률 시행령:제한경

In [1]:
# ============================================================
# Stage 6-2D
# PARTIAL / FAILED 항목의 공식 근거 원문 + 실제 파일 inventory
# 한 번에 출력하여 source-segment 매핑 확정
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
ARTIFACT_DIR = ROOT / "artifacts"
LAW_DIR = ROOT / "open" / "data" / "법령패키지" / "법령"
SME_DIR = ROOT / "open" / "data" / "법령패키지" / "중기부고시"

basis = pd.read_csv(
    ARTIFACT_DIR / "official_24_basis.csv"
)

problem_items = [
    "v2", "v3", "v6", "v7", "v8",
    "v10", "v11", "v13", "v14", "v15",
    "v16", "v17", "v18", "v19",
    "v20", "v23"
]

# ------------------------------------------------------------
# A. 실제 법령 파일명
# ------------------------------------------------------------

print("=" * 110)
print("[A] 실제 법령 TXT 파일")
print("=" * 110)

law_files = sorted(LAW_DIR.glob("*.txt"))

for i, p in enumerate(law_files, 1):
    print(f"{i:02d}. {p.stem}")

print("\n총", len(law_files), "개")


# ------------------------------------------------------------
# B. 실제 중기부고시 파일명
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("[B] 실제 중기부고시 파일")
print("=" * 110)

sme_files = sorted(
    p for p in SME_DIR.iterdir()
    if p.is_file()
)

for i, p in enumerate(sme_files, 1):
    print(f"{i:02d}. {p.name}")


# ------------------------------------------------------------
# C. 문제 16개 공식 근거 원문
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("[C] PARTIAL / FAILED 항목 공식 근거 원문")
print("=" * 110)

for item in problem_items:

    row = basis.loc[
        basis["item"] == item
    ].iloc[0]

    print("\n" + "-" * 110)
    print(
        f"{item} | {row['name']} "
        f"| absence={row['absence']}"
    )
    print("-" * 110)

    print("[국가]")
    print(row["national"])

    print("\n[지방]")
    print(row["local"])

    print("\n[비고]")
    print(
        "" if pd.isna(row["note"])
        else row["note"]
    )

[A] 실제 법령 TXT 파일
01. (계약예규) 공동계약운용요령
02. (계약예규) 정부 입찰·계약 집행기준
03. 국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액
04. 국가를 당사자로 하는 계약에 관한 법률 시행규칙
05. 국가를 당사자로 하는 계약에 관한 법률 시행령
06. 국가를 당사자로 하는 계약에 관한 법률
07. 소상공인기본법 시행령
08. 소상공인기본법
09. 소프트웨어 진흥법 시행령
10. 소프트웨어 진흥법
11. 중소 소프트웨어사업자의 사업 참여 지원에 관한 지침
12. 중소기업기본법 시행령
13. 중소기업기본법
14. 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역
15. 중소기업자간 경쟁제품 직접생산 확인기준
16. 중소기업제품 구매촉진 및 판로지원에 관한 법률 시행규칙
17. 중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령
18. 중소기업제품 구매촉진 및 판로지원에 관한 법률
19. 지방자치단체 입찰 및 계약 집행기준
20. 지방자치단체 입찰시 낙찰자 결정기준
21. 지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙
22. 지방자치단체를 당사자로 하는 계약에 관한 법률 시행령
23. 지방자치단체를 당사자로 하는 계약에 관한 법률

총 23 개

[B] 실제 중기부고시 파일
01. 중기부고시_경쟁제품_세부품명.csv
02. 중기부고시_경쟁제품_제2025-96호.hwpx

[C] PARTIAL / FAILED 항목 공식 근거 원문

--------------------------------------------------------------------------------------------------------------
v2 | 고시금액 미만 실적제한 | absence=False
--------------------------------------------------------------------------------------------------------------


In [2]:
# ============================================================
# Stage 6-2D
# Source-aware legal grounding rebuild
#
# 핵심:
#   - locator를 전체 basis 문자열에 적용하지 않는다.
#   - 각 법령/source 뒤에 붙은 locator는 그 source에만 귀속한다.
#   - 예규 prefix는 source qualifier로 처리한다.
#   - SME 지정품목은 별도 structured source로 처리한다.
#   - v23 국가 = NOT_APPLICABLE
#   - v24 = COMPARISON
#
# 이전 커널 변수에 의존하지 않는다.
# ============================================================

from pathlib import Path
import pandas as pd
import json
import re
import unicodedata


# ------------------------------------------------------------
# 0. PATH
# ------------------------------------------------------------

ROOT = Path.cwd()

ARTIFACT_DIR = ROOT / "artifacts"
LAW_DIR = ROOT / "open" / "data" / "법령패키지" / "법령"
SME_DIR = ROOT / "open" / "data" / "법령패키지" / "중기부고시"

BASIS_PATH = ARTIFACT_DIR / "official_24_basis.csv"

assert BASIS_PATH.exists()
assert LAW_DIR.exists()
assert SME_DIR.exists()

basis = pd.read_csv(BASIS_PATH)

assert len(basis) == 24


# ------------------------------------------------------------
# 1. NORMALIZATION
# ------------------------------------------------------------

def norm(s):
    return unicodedata.normalize(
        "NFC",
        str(s or "")
    ).strip()


def compact(s):
    s = norm(s)

    return re.sub(
        r"[\s·ㆍ・,()\[\]「」『』<>—\-]",
        "",
        s
    )


# ------------------------------------------------------------
# 2. 실제 법령 파일
# ------------------------------------------------------------

law_paths = sorted(LAW_DIR.glob("*.txt"))

assert len(law_paths) == 23

law_texts = {}

for p in law_paths:

    law_texts[p.stem] = p.read_text(
        encoding="utf-8",
        errors="replace"
    )


# ------------------------------------------------------------
# 3. 실제 파일을 정확한 canonical source로 등록
# ------------------------------------------------------------

CANONICAL = {
    "국가계약법":
        "국가를 당사자로 하는 계약에 관한 법률",

    "국가계약법 시행령":
        "국가를 당사자로 하는 계약에 관한 법률 시행령",

    "국가계약법 시행규칙":
        "국가를 당사자로 하는 계약에 관한 법률 시행규칙",

    "지방계약법":
        "지방자치단체를 당사자로 하는 계약에 관한 법률",

    "지방계약법 시행령":
        "지방자치단체를 당사자로 하는 계약에 관한 법률 시행령",

    "지방계약법 시행규칙":
        "지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙",

    "정부 입찰·계약 집행기준":
        "(계약예규) 정부 입찰·계약 집행기준",

    "정부입찰계약집행기준":
        "(계약예규) 정부 입찰·계약 집행기준",

    "공동계약운용요령":
        "(계약예규) 공동계약운용요령",

    "지방자치단체 입찰 및 계약 집행기준":
        "지방자치단체 입찰 및 계약 집행기준",

    "지방자치단체 입찰 및 계약집행기준":
        "지방자치단체 입찰 및 계약 집행기준",

    "지방자치단체 입찰시 낙찰자 결정기준":
        "지방자치단체 입찰시 낙찰자 결정기준",

    "중소기업제품 구매촉진 및 판로지원에 관한 법률":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률",

    "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령",

    "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행규칙":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행규칙",

    "소프트웨어진흥법":
        "소프트웨어 진흥법",

    "소프트웨어 진흥법":
        "소프트웨어 진흥법",

    "소프트웨어 진흥법 시행령":
        "소프트웨어 진흥법 시행령",

    "중소 소프트웨어사업자의 사업 참여 지원에 관한 지침":
        "중소 소프트웨어사업자의 사업 참여 지원에 관한 지침",

    "중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역":
        "중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역",
}


# 실제 파일 존재 검증
for alias, stem in CANONICAL.items():

    assert stem in law_texts, (
        f"canonical source 파일 없음: "
        f"{alias} -> {stem}"
    )


# ------------------------------------------------------------
# 4. source alias
#
# 긴 이름부터 매칭해야
# '... 법률 시행령'을 '... 법률'로 잘못 자르지 않는다.
# ------------------------------------------------------------

ALIASES = sorted(
    CANONICAL.keys(),
    key=len,
    reverse=True
)


# ------------------------------------------------------------
# 5. qualifier 제거
# ------------------------------------------------------------

QUALIFIERS = [
    "(계약예규)",
    "(행안부예규)",
]


def remove_qualifiers(text):

    t = norm(text)

    for q in QUALIFIERS:
        t = t.replace(q, " ")

    return re.sub(
        r"\s+",
        " ",
        t
    ).strip()


# ------------------------------------------------------------
# 6. 특수 structured source
# ------------------------------------------------------------

SME_DESIGNATED_NAME = (
    "중소기업자간 경쟁제품 및 공사용자재 "
    "직접구매 대상 품목 지정 내역"
)

SME_CSVS = sorted(
    SME_DIR.glob("*.csv")
)

SME_HWPX = sorted(
    SME_DIR.glob("*.hwpx")
)

assert SME_CSVS, "중기부 세부품명 CSV 없음"


# ------------------------------------------------------------
# 7. basis 문자열 → source segment 분리
# ------------------------------------------------------------

def find_source_occurrences(text):

    clean = remove_qualifiers(text)

    candidates = []

    for alias in ALIASES:

        start = 0

        while True:

            pos = clean.find(alias, start)

            if pos < 0:
                break

            end = pos + len(alias)

            candidates.append({
                "start": pos,
                "end": end,
                "alias": alias,
                "file": CANONICAL[alias],
            })

            start = end

    # 같은 위치에서 긴 alias 우선
    candidates.sort(
        key=lambda x: (
            x["start"],
            -(x["end"] - x["start"])
        )
    )

    selected = []

    for c in candidates:

        overlap = any(
            not (
                c["end"] <= s["start"]
                or c["start"] >= s["end"]
            )
            for s in selected
        )

        if not overlap:
            selected.append(c)

    selected.sort(
        key=lambda x: x["start"]
    )

    return clean, selected


def split_sources(text):

    original = norm(text)

    # --------------------------------
    # 명시적 해당 없음
    # --------------------------------

    if compact(original).startswith("해당없음"):

        return [{
            "source_type": "NOT_APPLICABLE",
            "source": None,
            "segment": original,
            "locator_text": "",
        }]

    clean, occs = find_source_occurrences(
        original
    )

    if not occs:

        return [{
            "source_type": "UNRESOLVED",
            "source": None,
            "segment": clean,
            "locator_text": clean,
        }]

    result = []

    for i, occ in enumerate(occs):

        seg_end = (
            occs[i + 1]["start"]
            if i + 1 < len(occs)
            else len(clean)
        )

        segment = clean[
            occ["start"]:seg_end
        ].strip()

        locator_text = clean[
            occ["end"]:seg_end
        ].strip()

        result.append({
            "source_type": "LEGAL_FILE",
            "source": occ["file"],
            "alias": occ["alias"],
            "segment": segment,
            "locator_text": locator_text,
        })

    return result


# ------------------------------------------------------------
# 8. locator 추출
# ------------------------------------------------------------

LOCATOR_PATTERNS = {
    "chapter":
        re.compile(r"제\s*\d+\s*장"),

    "section":
        re.compile(r"제\s*\d+\s*절"),

    "article":
        re.compile(
            r"제\s*\d+\s*조(?:의\s*\d+)?"
        ),

    "paragraph":
        re.compile(r"제\s*\d+\s*항"),

    "appendix":
        re.compile(
            r"별표\s*제?\s*\d+|별표\s*\d+"
        ),
}


def extract_locators(text):

    result = {}

    for kind, pattern in LOCATOR_PATTERNS.items():

        result[kind] = [
            m.group(0)
            for m in pattern.finditer(
                str(text or "")
            )
        ]

    return result


# ------------------------------------------------------------
# 9. 명칭 anchor
# ------------------------------------------------------------

ANCHORS = [
    "제한경쟁입찰의 운용",
    "입찰 및 계약 일반기준",
    "입찰 및 계약일반기준",
    "총칙",
    "계약담당자 주의사항",
    "수의계약 운영요령",
    "수의계약 대상과 운영요령",
    "금액기준에 따른 2인 이상 견적서 제출 수의계약",
    "수의계약 요령",
    "공동계약 운영요령",
    "공동계약운용요령",
    "협상에 의한 계약 낙찰자 결정기준",
    "입찰과 계약상대자 결정절차",
]


def extract_anchors(text):

    ct = compact(text)

    return [
        a
        for a in ANCHORS
        if compact(a) in ct
    ]


# ------------------------------------------------------------
# 10. source 안에서 locator 검증
#
# 중요:
# 제2조의3은 이제 그 조문이 붙어 있는
# 판로지원법 시행령에서만 검사한다.
# ------------------------------------------------------------

def verify_segment(segment):

    if segment["source_type"] == "NOT_APPLICABLE":

        return {
            **segment,
            "status": "RESOLVED",
            "locators": {},
            "anchors": [],
            "problems": [],
        }

    if segment["source_type"] != "LEGAL_FILE":

        return {
            **segment,
            "status": "FAILED",
            "locators": {},
            "anchors": [],
            "problems": [
                "source_unresolved"
            ],
        }

    source = segment["source"]
    source_text = law_texts[source]

    locators = extract_locators(
        segment["locator_text"]
    )

    anchors = extract_anchors(
        segment["locator_text"]
    )

    checks = {}
    problems = []

    for kind, values in locators.items():

        for value in values:

            ok = (
                compact(value)
                in compact(source_text)
            )

            checks[value] = ok

            if not ok:
                problems.append(
                    f"{kind}_not_found:{value}"
                )

    anchor_checks = {}

    for anchor in anchors:

        ok = (
            compact(anchor)
            in compact(source_text)
        )

        anchor_checks[anchor] = ok

        if not ok:
            problems.append(
                f"anchor_not_found:{anchor}"
            )

    status = (
        "RESOLVED"
        if not problems
        else "PARTIAL"
    )

    return {
        **segment,
        "status": status,
        "locators": locators,
        "locator_checks": checks,
        "anchors": anchors,
        "anchor_checks": anchor_checks,
        "problems": problems,
    }


# ------------------------------------------------------------
# 11. 한 side 전체 검증
# ------------------------------------------------------------

def verify_basis_side(item, text):

    text = norm(text)

    # v24는 법령 retrieval 항목이 아니다.
    if item == "v24":

        return {
            "status": "RESOLVED",
            "source_type": "COMPARISON",
            "segments": [],
            "problems": [],
        }

    segments = split_sources(text)

    verified = [
        verify_segment(s)
        for s in segments
    ]

    problems = []

    for s in verified:
        problems.extend(
            s.get("problems", [])
        )

    # ------------------------------------
    # SME 지정품목 structured source
    # ------------------------------------

    has_sme_designation = (
        compact(SME_DESIGNATED_NAME)
        in compact(text)
    )

    structured_source = None

    if has_sme_designation:

        structured_source = {
            "type": "SME_DESIGNATED_PRODUCTS",
            "law_txt":
                "중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역",
            "detail_csv":
                [p.name for p in SME_CSVS],
            "official_hwpx":
                [p.name for p in SME_HWPX],
            "status": "RESOLVED",
        }

    # ------------------------------------
    # 상태
    # ------------------------------------

    if any(
        s["status"] == "FAILED"
        for s in verified
    ):
        status = "FAILED"

    elif problems:
        status = "PARTIAL"

    else:
        status = "RESOLVED"

    return {
        "status": status,
        "segments": verified,
        "structured_source":
            structured_source,
        "problems": problems,
    }


# ------------------------------------------------------------
# 12. 24개 전체 재검증
# ------------------------------------------------------------

audit = {}
rows = []

for _, row in basis.iterrows():

    item = row["item"]

    national = verify_basis_side(
        item,
        row["national"]
    )

    local = verify_basis_side(
        item,
        row["local"]
    )

    statuses = {
        national["status"],
        local["status"],
    }

    if statuses == {"RESOLVED"}:

        overall = "RESOLVED"

    elif "FAILED" in statuses:

        overall = "FAILED"

    else:

        overall = "PARTIAL"

    audit[item] = {
        "item": item,
        "name": row["name"],
        "absence": bool(row["absence"]),
        "note":
            ""
            if pd.isna(row["note"])
            else str(row["note"]),
        "national": national,
        "local": local,
        "overall_status": overall,
    }

    rows.append({
        "item": item,
        "name": row["name"],
        "national": national["status"],
        "local": local["status"],
        "overall": overall,
        "national_problems":
            " | ".join(
                national["problems"]
            ),
        "local_problems":
            " | ".join(
                local["problems"]
            ),
    })


summary = pd.DataFrame(rows)


# ------------------------------------------------------------
# 13. 저장
# ------------------------------------------------------------

JSON_PATH = (
    ARTIFACT_DIR
    / "legal_grounding_source_aware.json"
)

CSV_PATH = (
    ARTIFACT_DIR
    / "legal_grounding_source_aware_audit.csv"
)

with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        ensure_ascii=False,
        indent=2
    )

summary.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 14. 결과
# ------------------------------------------------------------

display(
    summary[
        [
            "item",
            "name",
            "national",
            "local",
            "overall",
        ]
    ]
)

resolved = summary.loc[
    summary["overall"] == "RESOLVED",
    "item"
].tolist()

partial = summary.loc[
    summary["overall"] == "PARTIAL",
    "item"
].tolist()

failed = summary.loc[
    summary["overall"] == "FAILED",
    "item"
].tolist()


print("\n" + "=" * 100)
print("Stage 6-2D SOURCE-AWARE RESULT")
print("=" * 100)

print("RESOLVED =", len(resolved))
print("PARTIAL  =", len(partial))
print("FAILED   =", len(failed))

print("PARTIAL ITEMS =", partial)
print("FAILED ITEMS  =", failed)

print("\nJSON =", JSON_PATH)
print("CSV  =", CSV_PATH)


# ------------------------------------------------------------
# 15. 문제만 간결하게 출력
# ------------------------------------------------------------

if partial or failed:

    print("\n" + "=" * 100)
    print("REMAINING PROBLEMS")
    print("=" * 100)

    for item in partial + failed:

        x = audit[item]

        print(
            f"\n{item} - {x['name']}"
        )

        for side in [
            "national",
            "local"
        ]:

            sx = x[side]

            if sx["status"] == "RESOLVED":
                continue

            print(
                f"  [{side}] "
                f"{sx['status']}"
            )

            for seg in sx["segments"]:

                if seg["status"] != "RESOLVED":

                    print(
                        "    source =",
                        seg.get("source")
                    )

                    print(
                        "    segment =",
                        seg.get("segment")
                    )

                    print(
                        "    problems =",
                        seg.get("problems")
                    )

,item,name,national,local,overall
0,v1,참가자격 특정기관 제한,RESOLVED,RESOLVED,RESOLVED
1,v2,고시금액 미만 실적제한,RESOLVED,RESOLVED,RESOLVED
2,v3,실적제한 1배수 이상,RESOLVED,RESOLVED,RESOLVED
3,v4,"고시금액 이상 특정기관, 특정실적",RESOLVED,RESOLVED,RESOLVED
4,v5,고시금액 이상 지역제한,RESOLVED,RESOLVED,RESOLVED
5,v6,"고시금액 미만 지역제한 시,군,구",RESOLVED,RESOLVED,RESOLVED
6,v7,고시금액 미만 지역제한 인접 확대,RESOLVED,RESOLVED,RESOLVED
7,v8,중복제한 (실적+지역),RESOLVED,RESOLVED,RESOLVED
8,v9,과업지시서 특정 모델명 명시,RESOLVED,RESOLVED,RESOLVED
9,v10,중기간 경쟁제품 입찰 직생 없음,RESOLVED,RESOLVED,RESOLVED



Stage 6-2D SOURCE-AWARE RESULT
RESOLVED = 24
PARTIAL  = 0
FAILED   = 0
PARTIAL ITEMS = []
FAILED ITEMS  = []

JSON = /home/alparka/nara-ai/artifacts/legal_grounding_source_aware.json
CSV  = /home/alparka/nara-ai/artifacts/legal_grounding_source_aware_audit.csv


In [3]:
# ============================================================
# Stage 6-3A
# Final Judgment Specification용 Source Pack 구축
#
# 입력
#   - official_24_basis.csv
#   - legal_grounding_source_aware.json
#   - item_spec_workbench.json
#   - 실제 배포 법령 TXT
#
# 출력
#   - artifacts/judgment_spec_source_pack.json
#
# 목적
#   24개 항목별로
#   공식 정의 + grounding + dev 사례를 한 구조에 통합
#
# 주의
#   아직 violation rule을 자동 확정하지 않는다.
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# 0. PATH
# ------------------------------------------------------------

ROOT = Path.cwd()
ART = ROOT / "artifacts"
LAW_DIR = ROOT / "open" / "data" / "법령패키지" / "법령"

BASIS_PATH = ART / "official_24_basis.csv"
GROUNDING_PATH = ART / "legal_grounding_source_aware.json"
WORKBENCH_PATH = ART / "item_spec_workbench.json"

for p in [
    BASIS_PATH,
    GROUNDING_PATH,
    WORKBENCH_PATH,
]:
    assert p.exists(), f"파일 없음: {p}"


# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

basis = pd.read_csv(BASIS_PATH)

with open(
    GROUNDING_PATH,
    "r",
    encoding="utf-8"
) as f:
    grounding = json.load(f)

with open(
    WORKBENCH_PATH,
    "r",
    encoding="utf-8"
) as f:
    workbench = json.load(f)


assert len(basis) == 24
assert len(grounding) == 24
assert len(workbench) == 24


# ------------------------------------------------------------
# 2. LAW TEXT LOAD
# ------------------------------------------------------------

law_texts = {}

for p in LAW_DIR.glob("*.txt"):

    law_texts[p.stem] = p.read_text(
        encoding="utf-8",
        errors="replace"
    )


# ------------------------------------------------------------
# 3. locator 주변 원문 추출
#
# 아직 "이 문장이 법적 판정조건이다"라고 확정하지 않는다.
# 사람이 검토할 source context를 만드는 단계.
# ------------------------------------------------------------

def compact(s):
    import re
    import unicodedata

    s = unicodedata.normalize(
        "NFC",
        str(s or "")
    )

    return re.sub(
        r"\s+",
        "",
        s
    )


def find_context(text, target, window=1200):

    if not target:
        return None

    ct = compact(text)
    cq = compact(target)

    pos = ct.find(cq)

    if pos < 0:
        return None

    # compact index와 원문 index가 다르므로
    # 원문에서 다시 느슨하게 찾을 후보를 만든다.
    import re

    pattern = r"\s*".join(
        map(re.escape, list(str(target)))
    )

    m = re.search(
        pattern,
        text
    )

    if not m:
        return None

    start = max(
        0,
        m.start() - window
    )

    end = min(
        len(text),
        m.end() + window
    )

    return text[start:end]


# ------------------------------------------------------------
# 4. 한 grounding segment의 source context
# ------------------------------------------------------------

def build_segment_context(seg):

    if seg.get("source_type") != "LEGAL_FILE":

        return {
            **seg,
            "source_context": None,
        }

    source = seg.get("source")

    if source not in law_texts:

        return {
            **seg,
            "source_context": None,
            "context_problem":
                "source_file_missing",
        }

    text = law_texts[source]

    locator_candidates = []

    locators = seg.get(
        "locators",
        {}
    )

    # 가장 구체적인 위치부터
    for kind in [
        "article",
        "section",
        "chapter",
        "appendix",
    ]:

        for x in locators.get(
            kind,
            []
        ):
            locator_candidates.append(x)

    # named anchor도 사용
    for x in seg.get(
        "anchors",
        []
    ):
        locator_candidates.append(x)

    contexts = []

    seen = set()

    for target in locator_candidates:

        if target in seen:
            continue

        seen.add(target)

        ctx = find_context(
            text,
            target
        )

        contexts.append({
            "target": target,
            "found": ctx is not None,
            "context": ctx,
        })

    return {
        **seg,
        "source_contexts": contexts,
    }


# ------------------------------------------------------------
# 5. 24개 Source Pack
# ------------------------------------------------------------

source_pack = {}

for _, row in basis.iterrows():

    v = row["item"]

    g = grounding[v]
    w = workbench[v]

    national_segments = [
        build_segment_context(seg)
        for seg in g["national"]["segments"]
    ]

    local_segments = [
        build_segment_context(seg)
        for seg in g["local"]["segments"]
    ]

    source_pack[v] = {

        # -------------------------
        # 공식 항목
        # -------------------------
        "item":
            v,

        "name":
            row["name"],

        "absence_detection":
            bool(row["absence"]),

        "official_note":
            ""
            if pd.isna(row["note"])
            else str(row["note"]),

        # -------------------------
        # 공식 근거
        # -------------------------
        "official_basis": {
            "national":
                row["national"],

            "local":
                row["local"],
        },

        # -------------------------
        # 검증된 source
        # -------------------------
        "grounding": {
            "national":
                national_segments,

            "local":
                local_segments,

            "national_structured":
                g["national"].get(
                    "structured_source"
                ),

            "local_structured":
                g["local"].get(
                    "structured_source"
                ),
        },

        # -------------------------
        # DEV 정보
        # -------------------------
        "dev": {
            "positive_count":
                w.get(
                    "dev_stats",
                    {}
                ).get(
                    "positive_count"
                ),

            "positives":
                w.get(
                    "positives",
                    []
                ),

            "negative_candidates":
                w.get(
                    "negative_candidates",
                    []
                ),
        },

        # -------------------------
        # 최종 spec 자리
        # 아직 확정하지 않음
        # -------------------------
        "decision_spec": {

            "applicable_contract_law":
                "UNRESOLVED",

            "applicable_work_type":
                "UNRESOLVED",

            "amount_condition":
                "UNRESOLVED",

            "other_applicability":
                "UNRESOLVED",

            "required_meta":
                [],

            "required_docs":
                [],

            "violation_condition":
                "UNRESOLVED",

            "nonviolation_condition":
                "UNRESOLVED",

            "exceptions":
                [],

            "absence_detection":
                bool(row["absence"]),

            "incomplete_docs_policy":
                "UNRESOLVED",

            "python_role":
                [],

            "gemma_role":
                [],

            "evidence_policy":
                (
                    "EMPTY"
                    if bool(row["absence"])
                    else "UNRESOLVED"
                ),

            "status":
                "UNRESOLVED",
        },
    }


# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

assert set(source_pack) == {
    f"v{i}" for i in range(1, 25)
}

absence_items = {
    v
    for v, x in source_pack.items()
    if x["absence_detection"]
}

assert absence_items == {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}


# ------------------------------------------------------------
# 7. SOURCE CONTEXT 통계
# ------------------------------------------------------------

rows = []

for v, x in source_pack.items():

    legal_segments = []
    found_contexts = 0
    missing_contexts = 0

    for side in [
        "national",
        "local"
    ]:

        for seg in x[
            "grounding"
        ][side]:

            if (
                seg.get("source_type")
                != "LEGAL_FILE"
            ):
                continue

            legal_segments.append(seg)

            for ctx in seg.get(
                "source_contexts",
                []
            ):

                if ctx["found"]:
                    found_contexts += 1
                else:
                    missing_contexts += 1

    rows.append({
        "item": v,
        "name": x["name"],
        "legal_segments":
            len(legal_segments),
        "contexts_found":
            found_contexts,
        "contexts_missing":
            missing_contexts,
        "positive_count":
            x["dev"][
                "positive_count"
            ],
        "absence":
            x["absence_detection"],
    })


audit_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# 8. SAVE
# ------------------------------------------------------------

OUT_PATH = (
    ART
    / "judgment_spec_source_pack.json"
)

with open(
    OUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        source_pack,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 9. OUTPUT
# ------------------------------------------------------------

display(audit_df)

print()
print("=" * 100)
print("Stage 6-3A SOURCE PACK RESULT")
print("=" * 100)

print(
    "ITEMS            =",
    len(source_pack)
)

print(
    "ABSENCE ITEMS    =",
    sorted(absence_items)
)

print(
    "MISSING CONTEXTS =",
    int(
        audit_df[
            "contexts_missing"
        ].sum()
    )
)

print(
    "OUTPUT           =",
    OUT_PATH
)

problem = audit_df[
    audit_df["contexts_missing"] > 0
]

if len(problem) == 0:

    print(
        "\n[PASS] "
        "24개 specification source pack 생성 완료"
    )

else:

    print(
        "\n[CHECK] "
        "locator context 미발견 항목:"
    )

    print(
        problem[
            [
                "item",
                "name",
                "contexts_missing",
            ]
        ].to_string(
            index=False
        )
    )

,item,name,legal_segments,contexts_found,contexts_missing,positive_count,absence
0,v1,참가자격 특정기관 제한,4,4,0,7,False
1,v2,고시금액 미만 실적제한,4,10,0,7,False
2,v3,실적제한 1배수 이상,5,7,0,8,False
3,v4,"고시금액 이상 특정기관, 특정실적",2,6,0,6,False
4,v5,고시금액 이상 지역제한,3,3,0,7,False
5,v6,"고시금액 미만 지역제한 시,군,구",3,8,0,6,False
6,v7,고시금액 미만 지역제한 인접 확대,3,8,0,7,False
7,v8,중복제한 (실적+지역),3,8,0,6,False
8,v9,과업지시서 특정 모델명 명시,2,7,0,6,False
9,v10,중기간 경쟁제품 입찰 직생 없음,4,2,0,7,True



Stage 6-3A SOURCE PACK RESULT
ITEMS            = 24
ABSENCE ITEMS    = ['v10', 'v11', 'v16', 'v18', 'v20']
MISSING CONTEXTS = 0
OUTPUT           = /home/alparka/nara-ai/artifacts/judgment_spec_source_pack.json

[PASS] 24개 specification source pack 생성 완료


In [5]:
# ============================================================
# Stage 6-3B INPUT
# v1~v8 최종 판정명세 작성에 필요한 자료를 한 번에 출력
# ============================================================

from pathlib import Path
import json

ROOT = Path.cwd()
PACK_PATH = ROOT / "artifacts" / "judgment_spec_source_pack.json"

with open(PACK_PATH, "r", encoding="utf-8") as f:
    pack = json.load(f)

for v in [f"v{i}" for i in range(1, 9)]:
    x = pack[v]

    print("\n" + "=" * 120)
    print(f"{v} | {x['name']}")
    print("=" * 120)

    print("부재탐지:", x["absence_detection"])
    print("비고:", x["official_note"])

    print("\n[공식 근거]")
    print("국가:", x["official_basis"]["national"])
    print("지방:", x["official_basis"]["local"])

    # --------------------------------------------------------
    # 실제 법령 context
    # 너무 긴 중복 context는 앞뒤 포함 최대 3000자
    # --------------------------------------------------------

    print("\n[검증된 법령 원문 CONTEXT]")

    for side in ["national", "local"]:
        print(f"\n--- {side.upper()} ---")

        for seg in x["grounding"][side]:
            if seg.get("source_type") != "LEGAL_FILE":
                continue

            print("\nSOURCE:", seg.get("source"))
            print("SEGMENT:", seg.get("segment"))

            contexts = seg.get("source_contexts", [])

            if not contexts:
                print("CONTEXT: 없음")
                continue

            seen = set()

            for ctx in contexts:
                text = ctx.get("context")

                if not text:
                    continue

                # 같은 source에서 동일 context 중복 출력 방지
                key = text[:500]

                if key in seen:
                    continue

                seen.add(key)

                print(
                    f"\nTARGET: {ctx.get('target')}"
                )

                print(
                    text[:3000]
                )

    # --------------------------------------------------------
    # DEV positive
    # --------------------------------------------------------

    print("\n[DEV POSITIVES]")

    positives = x["dev"]["positives"]

    print("COUNT:", len(positives))

    for j, case in enumerate(positives, 1):

        print(f"\nPOSITIVE #{j}")

        # 너무 거대한 원문 전체를 무작정 출력하지 않고
        # workbench에 저장된 구조 그대로 확인
        print(
            json.dumps(
                case,
                ensure_ascii=False,
                indent=2
            )[:5000]
        )

    # --------------------------------------------------------
    # DEV negative candidates
    # --------------------------------------------------------

    print("\n[DEV NEGATIVE CANDIDATES]")

    negatives = x["dev"]["negative_candidates"]

    print("COUNT:", len(negatives))

    for j, case in enumerate(negatives[:8], 1):

        print(f"\nNEGATIVE CANDIDATE #{j}")

        print(
            json.dumps(
                case,
                ensure_ascii=False,
                indent=2
            )[:3500]
        )


v1 | 참가자격 특정기관 제한
부재탐지: False
비고: 

[공식 근거]
국가: 국가계약법 시행령 제12조 국가계약법 시행령 제21조
지방: 지방계약법 시행령 제13조 지방계약법 시행령 제20조

[검증된 법령 원문 CONTEXT]

--- NATIONAL ---

SOURCE: 국가를 당사자로 하는 계약에 관한 법률 시행령
SEGMENT: 국가계약법 시행령 제12조

TARGET: 제12조
 2008.12.31>

제9조(예정가격의 결정기준)
  ①각 중앙관서의 장 또는 계약담당공무원은 다음 각 호의 가격을 기준으로 하여 예정가격을 결정하여야 한다. <개정 2014.11.4>
    1. 적정한 거래가 형성된 경우에는 그 거래실례가격(법령의 규정에 의하여 가격이 결정된 경우에는 그 결정가격의 범위안에서의 거래실례가격)
    2. 신규개발품이거나 특수규격품등의 특수한 물품ㆍ공사ㆍ용역등 계약의 특수성으로 인하여 적정한 거래실례가격이 없는 경우에는 원가계산에 의한 가격. 이 경우 원가계산에 의한 가격은 계약의 목적이 되는 물품ㆍ공사ㆍ용역등을 구성하는 재료비ㆍ노무비ㆍ경비와 일반관리비 및 이윤으로 이를 계산한다.
    3. 공사의 경우 이미 수행한 공사의 종류별 시장거래가격 등을 토대로 산정한 표준시장단가로서 중앙관서의 장이 인정한 가격
    4. 제1호 내지 제3호의 규정에 의한 가격에 의할 수 없는 경우에는 감정가격, 유사한 물품ㆍ공사ㆍ용역등의 거래실례가격 또는 견적가격
  ②제1항의 규정에 불구하고 해외로부터 수입하고 있는 군용물자부품을 국산화한 업체와 계약을 체결하려는 경우에는 그 수입가격 등을 고려하여 방위사업청장이 인정한 가격을 기준으로 하여 예정가격을 결정할 수 있다. <신설 2003.12.11, 2006.2.8, 2007.10.10>
  ③각 중앙관서의 장 또는 계약담당공무원은 제1항의 규정에 의하여 예정가격을 결정함에 있어서는 계약수량, 이행기간, 수급상황, 계약조건 기타 제반여건을 참작하여야 한다.
  ④제1항 내지 제3항외에 예정가격의

In [8]:
# ============================================================
# Stage 6-3B INPUT
# v1~v8 최종 판정명세 작성용 자료 출력
#
# 입력:
#   artifacts/judgment_spec_source_pack.json
#
# 목적:
#   v1~v8 각각에 대해
#   1) 공식 정의/근거
#   2) 실제 배포 법령 원문
#   3) DEV positive 사례
#   4) 구조적으로 유사한 negative 후보
#   를 한 번에 확인
#
# 주의:
#   아직 violation_condition을 자동 생성하지 않는다.
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# 0. PATH
# ------------------------------------------------------------

ROOT = Path.cwd()

PACK_PATH = (
    ROOT
    / "artifacts"
    / "judgment_spec_source_pack.json"
)

assert PACK_PATH.exists(), (
    f"파일 없음: {PACK_PATH}"
)


# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

with open(
    PACK_PATH,
    "r",
    encoding="utf-8"
) as f:

    pack = json.load(f)


EXPECTED = [
    f"v{i}"
    for i in range(1, 9)
]

for v in EXPECTED:
    assert v in pack, f"{v} 없음"


print(
    "[1] Source Pack LOAD PASS"
)

print(
    "대상:",
    EXPECTED
)


# ------------------------------------------------------------
# 2. 출력 길이 제한 helper
# ------------------------------------------------------------

def print_json(obj, limit=5000):

    text = json.dumps(
        obj,
        ensure_ascii=False,
        indent=2
    )

    if len(text) > limit:
        text = (
            text[:limit]
            + "\n... [TRUNCATED]"
        )

    print(text)


def print_context(text, limit=3500):

    if not text:
        print("없음")
        return

    if len(text) > limit:
        print(
            text[:limit]
            + "\n... [TRUNCATED]"
        )
    else:
        print(text)


# ------------------------------------------------------------
# 3. v1~v8
# ------------------------------------------------------------

for v in EXPECTED:

    x = pack[v]

    print(
        "\n\n"
        + "=" * 120
    )

    print(
        f"{v} | {x['name']}"
    )

    print(
        "=" * 120
    )

    # --------------------------------------------------------
    # 기본정보
    # --------------------------------------------------------

    print(
        "\n[기본 정보]"
    )

    print(
        "부재탐지:",
        x["absence_detection"]
    )

    print(
        "비고:",
        x.get(
            "official_note",
            ""
        )
    )


    # --------------------------------------------------------
    # 공식 근거
    # --------------------------------------------------------

    print(
        "\n[공식 법적 근거]"
    )

    print(
        "\n국가:"
    )

    print(
        x["official_basis"][
            "national"
        ]
    )

    print(
        "\n지방:"
    )

    print(
        x["official_basis"][
            "local"
        ]
    )


    # --------------------------------------------------------
    # 실제 법령 원문
    # --------------------------------------------------------

    print(
        "\n[검증된 법령 원문]"
    )

    for side in [
        "national",
        "local"
    ]:

        print(
            "\n"
            + "-" * 100
        )

        print(
            side.upper()
        )

        print(
            "-" * 100
        )

        segments = (
            x["grounding"][side]
        )

        if not segments:

            print(
                "segment 없음"
            )

            continue

        for seg_no, seg in enumerate(
            segments,
            start=1
        ):

            print(
                f"\nSEGMENT #{seg_no}"
            )

            print(
                "SOURCE TYPE:",
                seg.get(
                    "source_type"
                )
            )

            print(
                "SOURCE:",
                seg.get(
                    "source"
                )
            )

            print(
                "SEGMENT:",
                seg.get(
                    "segment"
                )
            )

            # LEGAL_FILE만 실제 context 출력
            if (
                seg.get("source_type")
                != "LEGAL_FILE"
            ):

                print(
                    "※ 일반 법령 TXT가 아닌 "
                    "structured/special source"
                )

                continue

            contexts = seg.get(
                "source_contexts",
                []
            )

            if not contexts:

                print(
                    "CONTEXT 없음"
                )

                continue

            seen = set()

            for ctx_no, ctx in enumerate(
                contexts,
                start=1
            ):

                target = ctx.get(
                    "target"
                )

                text = ctx.get(
                    "context"
                )

                if not text:
                    continue

                # 같은 context 중복 방지
                signature = text[:500]

                if signature in seen:
                    continue

                seen.add(
                    signature
                )

                print(
                    f"\n  TARGET #{ctx_no}:",
                    target
                )

                print(
                    "  FOUND:",
                    ctx.get(
                        "found"
                    )
                )

                print()

                print_context(
                    text,
                    limit=3500
                )


    # --------------------------------------------------------
    # DEV POSITIVE
    # --------------------------------------------------------

    positives = (
        x["dev"]
        .get(
            "positives",
            []
        )
    )

    print(
        "\n"
        + "-" * 100
    )

    print(
        "[DEV POSITIVES]"
    )

    print(
        "COUNT:",
        len(positives)
    )

    for j, case in enumerate(
        positives,
        start=1
    ):

        print(
            f"\nPOSITIVE #{j}"
        )

        print_json(
            case,
            limit=5000
        )


    # --------------------------------------------------------
    # DEV NEGATIVE CANDIDATES
    # --------------------------------------------------------

    negatives = (
        x["dev"]
        .get(
            "negative_candidates",
            []
        )
    )

    print(
        "\n"
        + "-" * 100
    )

    print(
        "[DEV NEGATIVE CANDIDATES]"
    )

    print(
        "COUNT:",
        len(negatives)
    )

    # 최대 8개만 출력
    for j, case in enumerate(
        negatives[:8],
        start=1
    ):

        print(
            f"\nNEGATIVE CANDIDATE #{j}"
        )

        print_json(
            case,
            limit=3500
        )


# ------------------------------------------------------------
# 4. 마지막 검증
# ------------------------------------------------------------

print(
    "\n\n"
    + "=" * 120
)

print(
    "Stage 6-3B INPUT EXTRACTION RESULT"
)

print(
    "=" * 120
)

print(
    "ITEMS =",
    len(EXPECTED)
)

print(
    "TARGET =",
    EXPECTED
)

missing_positive = [
    v
    for v in EXPECTED
    if not pack[v]["dev"].get(
        "positives"
    )
]

print(
    "NO POSITIVE ITEMS =",
    missing_positive
)

if not missing_positive:

    print(
        "\n[PASS] "
        "v1~v8 판정명세 검토자료 출력 완료"
    )

else:

    print(
        "\n[CHECK] "
        "positive 사례가 없는 항목 존재"
    )

[1] Source Pack LOAD PASS
대상: ['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8']


v1 | 참가자격 특정기관 제한

[기본 정보]
부재탐지: False
비고: 

[공식 법적 근거]

국가:
국가계약법 시행령 제12조 국가계약법 시행령 제21조

지방:
지방계약법 시행령 제13조 지방계약법 시행령 제20조

[검증된 법령 원문]

----------------------------------------------------------------------------------------------------
NATIONAL
----------------------------------------------------------------------------------------------------

SEGMENT #1
SOURCE TYPE: LEGAL_FILE
SOURCE: 국가를 당사자로 하는 계약에 관한 법률 시행령
SEGMENT: 국가계약법 시행령 제12조

  TARGET #1: 제12조
  FOUND: True

 2008.12.31>

제9조(예정가격의 결정기준)
  ①각 중앙관서의 장 또는 계약담당공무원은 다음 각 호의 가격을 기준으로 하여 예정가격을 결정하여야 한다. <개정 2014.11.4>
    1. 적정한 거래가 형성된 경우에는 그 거래실례가격(법령의 규정에 의하여 가격이 결정된 경우에는 그 결정가격의 범위안에서의 거래실례가격)
    2. 신규개발품이거나 특수규격품등의 특수한 물품ㆍ공사ㆍ용역등 계약의 특수성으로 인하여 적정한 거래실례가격이 없는 경우에는 원가계산에 의한 가격. 이 경우 원가계산에 의한 가격은 계약의 목적이 되는 물품ㆍ공사ㆍ용역등을 구성하는 재료비ㆍ노무비ㆍ경비와 일반관리비 및 이윤으로 이를 계산한다.
    3. 공사의 경우 이미 수행한 공사의 종류별 시장거래가격 등을 토대로 산정한 표준시장단가로서 중앙관서의 장이 인정한 가격
    4. 제1호 내지 제3호

In [1]:
# ============================================================
# 03_baseline_dev.ipynb
# Resume Point: Stage 6-3B
# Stage 6-1 ~ 6-3A 결과는 artifacts에서 복원
# ============================================================

from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
ART = ROOT / "artifacts"

required = [
    "item_spec_workbench.json",
    "official_24_basis.csv",
    "legal_grounding_source_aware.json",
    "legal_grounding_source_aware_audit.csv",
    "judgment_spec_source_pack.json",
]

print("=" * 80)
print("PROJECT RESUME CHECK")
print("=" * 80)

for name in required:
    path = ART / name
    print(
        f"{name:<45}",
        "PASS" if path.exists() else "MISSING"
    )
    assert path.exists(), f"필수 artifact 없음: {path}"

# 최신 Source Pack 로드
with open(
    ART / "judgment_spec_source_pack.json",
    "r",
    encoding="utf-8"
) as f:
    pack = json.load(f)

assert len(pack) == 24

expected_items = [
    f"v{i}"
    for i in range(1, 25)
]

assert set(pack.keys()) == set(expected_items)

absence_items = [
    v
    for v in expected_items
    if pack[v]["absence_detection"]
]

assert absence_items == [
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
]

print()
print("ITEMS         =", len(pack))
print("ABSENCE ITEMS =", absence_items)

print()
print("[PASS] Stage 6-3A 상태 복원 완료")
print("NEXT = Stage 6-3B (v1~v8 judgment specification)")

PROJECT RESUME CHECK
item_spec_workbench.json                      PASS
official_24_basis.csv                         PASS
legal_grounding_source_aware.json             PASS
legal_grounding_source_aware_audit.csv        PASS
judgment_spec_source_pack.json                PASS

ITEMS         = 24
ABSENCE ITEMS = ['v10', 'v11', 'v16', 'v18', 'v20']

[PASS] Stage 6-3A 상태 복원 완료
NEXT = Stage 6-3B (v1~v8 judgment specification)


In [2]:
# ============================================================
# Stage 6-3B-1
# v1~v8 Judgment Specification Input Validation
# ============================================================

TARGET_ITEMS = [f"v{i}" for i in range(1, 9)]

rows = []

for v in TARGET_ITEMS:
    x = pack[v]

    national = x["grounding"]["national"]
    local = x["grounding"]["local"]

    # 실제 법령 context 개수
    def count_contexts(segments):
        n = 0

        for seg in segments:
            if seg.get("source_type") != "LEGAL_FILE":
                continue

            for ctx in seg.get("source_contexts", []):
                if ctx.get("found") and ctx.get("context"):
                    n += 1

        return n

    national_contexts = count_contexts(national)
    local_contexts = count_contexts(local)

    positives = x["dev"].get("positives", [])
    negatives = x["dev"].get("negative_candidates", [])

    rows.append({
        "item": v,
        "name": x["name"],
        "absence": x["absence_detection"],
        "national_segments": len(national),
        "local_segments": len(local),
        "national_contexts": national_contexts,
        "local_contexts": local_contexts,
        "dev_positive": len(positives),
        "dev_negative_candidates": len(negatives),
    })


check_63b = pd.DataFrame(rows)

display(check_63b)


# ------------------------------------------------------------
# 검증
# ------------------------------------------------------------

assert len(check_63b) == 8

assert check_63b["item"].tolist() == TARGET_ITEMS

# v1~v8에는 부재탐지 항목이 없어야 함
assert not check_63b["absence"].any()

# 모든 항목에 dev positive 존재
assert (check_63b["dev_positive"] > 0).all()

# 공식 grounding 자체가 없는 항목 검사
no_grounding = []

for v in TARGET_ITEMS:

    x = pack[v]

    if (
        not x["grounding"]["national"]
        and not x["grounding"]["local"]
    ):
        no_grounding.append(v)


print()
print("=" * 90)
print("Stage 6-3B-1 VALIDATION RESULT")
print("=" * 90)

print("ITEMS             =", len(check_63b))
print("TARGET            =", TARGET_ITEMS)
print("ABSENCE ITEMS     =", check_63b.loc[
    check_63b["absence"], "item"
].tolist())

print("NO GROUNDING      =", no_grounding)

print(
    "TOTAL POSITIVES   =",
    int(check_63b["dev_positive"].sum())
)

print(
    "TOTAL NEG CAND.   =",
    int(check_63b["dev_negative_candidates"].sum())
)

if not no_grounding:
    print()
    print(
        "[PASS] v1~v8 judgment specification "
        "입력자료 검증 완료"
    )
else:
    print()
    print(
        "[FAIL] grounding 없는 항목 존재:",
        no_grounding
    )

,item,name,absence,national_segments,local_segments,national_contexts,local_contexts,dev_positive,dev_negative_candidates
0,v1,참가자격 특정기관 제한,False,2,2,2,2,7,12
1,v2,고시금액 미만 실적제한,False,2,2,2,8,7,12
2,v3,실적제한 1배수 이상,False,3,2,5,2,8,12
3,v4,"고시금액 이상 특정기관, 특정실적",False,1,1,3,3,6,12
4,v5,고시금액 이상 지역제한,False,1,2,1,2,7,12
5,v6,"고시금액 미만 지역제한 시,군,구",False,1,2,1,7,6,12
6,v7,고시금액 미만 지역제한 인접 확대,False,1,2,1,7,7,12
7,v8,중복제한 (실적+지역),False,1,2,1,7,6,12



Stage 6-3B-1 VALIDATION RESULT
ITEMS             = 8
TARGET            = ['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8']
ABSENCE ITEMS     = []
NO GROUNDING      = []
TOTAL POSITIVES   = 54
TOTAL NEG CAND.   = 96

[PASS] v1~v8 judgment specification 입력자료 검증 완료


In [3]:
# ============================================================
# Stage 6-3B-2
# v1~v8 Final Spec Review Pack 생성
#
# 목적:
# - 최종 decision_spec을 작성하기 전에
# - 검증된 법령 원문 + DEV positive + negative candidate를
#   하나의 파일로 보존
#
# 이 파일을 기반으로 v1~v8 최종 명세를 작성한다.
# ============================================================

from pathlib import Path
import json

TARGET_ITEMS = [f"v{i}" for i in range(1, 9)]

OUT_PATH = (
    ROOT
    / "artifacts"
    / "v1_v8_final_spec_review.json"
)

review_pack = {}


for v in TARGET_ITEMS:

    x = pack[v]

    # --------------------------------------------------------
    # 1. 법령 context 정리
    # --------------------------------------------------------

    legal_contexts = {
        "national": [],
        "local": [],
    }

    for side in ["national", "local"]:

        for seg in x["grounding"][side]:

            source_info = {
                "source_type": seg.get("source_type"),
                "source": seg.get("source"),
                "segment": seg.get("segment"),
                "contexts": [],
            }

            for ctx in seg.get("source_contexts", []):

                if not ctx.get("found"):
                    continue

                text = ctx.get("context")

                if not text:
                    continue

                source_info["contexts"].append({
                    "target": ctx.get("target"),
                    "context": text,
                })

            legal_contexts[side].append(
                source_info
            )


    # --------------------------------------------------------
    # 2. DEV 사례
    # --------------------------------------------------------

    positives = x["dev"].get(
        "positives",
        []
    )

    negative_candidates = x["dev"].get(
        "negative_candidates",
        []
    )


    # --------------------------------------------------------
    # 3. review pack
    # --------------------------------------------------------

    review_pack[v] = {

        "item": v,

        "name":
            x["name"],

        "absence_detection":
            x["absence_detection"],

        "official_note":
            x.get(
                "official_note",
                ""
            ),

        "official_basis":
            x["official_basis"],

        "legal_contexts":
            legal_contexts,

        "dev_positive_count":
            len(positives),

        "dev_positives":
            positives,

        "dev_negative_candidate_count":
            len(negative_candidates),

        "dev_negative_candidates":
            negative_candidates,

        # 아직 자동으로 판정규칙을 만들지 않는다.
        "decision_spec": {
            "status": "PENDING_FINAL_REVIEW"
        },
    }


# ------------------------------------------------------------
# 4. 저장
# ------------------------------------------------------------

with open(
    OUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        review_pack,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 5. 저장 후 재검증
# ------------------------------------------------------------

with open(
    OUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    check = json.load(f)


assert list(check.keys()) == TARGET_ITEMS
assert len(check) == 8


for v in TARGET_ITEMS:

    assert (
        check[v]["item"]
        == v
    )

    assert (
        check[v]["absence_detection"]
        is False
    )

    assert (
        check[v]["dev_positive_count"]
        > 0
    )

    assert (
        check[v]["official_basis"]
    )

    assert (
        check[v]["legal_contexts"]
    )


# ------------------------------------------------------------
# 6. Audit
# ------------------------------------------------------------

print("=" * 90)
print("Stage 6-3B-2 FINAL SPEC REVIEW PACK")
print("=" * 90)

for v in TARGET_ITEMS:

    x = check[v]

    national_ctx = sum(
        len(s["contexts"])
        for s in x["legal_contexts"]["national"]
    )

    local_ctx = sum(
        len(s["contexts"])
        for s in x["legal_contexts"]["local"]
    )

    print(
        f"{v:>3} | "
        f"{x['name']:<25} | "
        f"POS={x['dev_positive_count']:>2} | "
        f"NEG={x['dev_negative_candidate_count']:>2} | "
        f"N_CTX={national_ctx:>2} | "
        f"L_CTX={local_ctx:>2}"
    )


print()
print("OUTPUT =", OUT_PATH)

print()
print(
    "[PASS] v1~v8 final-spec review pack 생성 완료"
)

Stage 6-3B-2 FINAL SPEC REVIEW PACK
 v1 | 참가자격 특정기관 제한              | POS= 7 | NEG=12 | N_CTX= 2 | L_CTX= 2
 v2 | 고시금액 미만 실적제한              | POS= 7 | NEG=12 | N_CTX= 2 | L_CTX= 8
 v3 | 실적제한 1배수 이상               | POS= 8 | NEG=12 | N_CTX= 5 | L_CTX= 2
 v4 | 고시금액 이상 특정기관, 특정실적        | POS= 6 | NEG=12 | N_CTX= 3 | L_CTX= 3
 v5 | 고시금액 이상 지역제한              | POS= 7 | NEG=12 | N_CTX= 1 | L_CTX= 2
 v6 | 고시금액 미만 지역제한 시,군,구        | POS= 6 | NEG=12 | N_CTX= 1 | L_CTX= 7
 v7 | 고시금액 미만 지역제한 인접 확대        | POS= 7 | NEG=12 | N_CTX= 1 | L_CTX= 7
 v8 | 중복제한 (실적+지역)              | POS= 6 | NEG=12 | N_CTX= 1 | L_CTX= 7

OUTPUT = /home/alparka/nara-ai/artifacts/v1_v8_final_spec_review.json

[PASS] v1~v8 final-spec review pack 생성 완료


In [ ]:
# ============================================================
# Stage 6-3B FINAL
# v1~v8 exact legal grounding repair + final decision specs
#
# 핵심:
# 1) 기존 substring context를 판정 근거로 신뢰하지 않는다.
# 2) 실제 23개 법령 파일에서 "줄 시작의 정확한 조문 제목"을 다시 찾는다.
# 3) v2 / v5의 합쳐진 source를 명시적으로 분리한다.
# 4) v1~v8 최종 decision_spec을 작성한다.
# 5) 애매한 것은 추측하지 않고 unresolved에 명시한다.
# ============================================================

from pathlib import Path
import json
import re
import pandas as pd


# ============================================================
# 0. 경로
# ============================================================

ROOT = Path.cwd()

REVIEW_PATH = (
    ROOT
    / "artifacts"
    / "v1_v8_final_spec_review.json"
)

LAW_DIR = (
    ROOT
    / "open"
    / "data"
    / "법령패키지"
    / "법령"
)

OUT_PATH = (
    ROOT
    / "artifacts"
    / "judgment_specs_v1_v8.json"
)

assert REVIEW_PATH.exists(), REVIEW_PATH
assert LAW_DIR.exists(), LAW_DIR


with open(REVIEW_PATH, "r", encoding="utf-8") as f:
    review = json.load(f)


TARGET_ITEMS = [f"v{i}" for i in range(1, 9)]

assert list(review.keys()) == TARGET_ITEMS


# ============================================================
# 1. 실제 법령 23개
# ============================================================

law_paths = sorted(LAW_DIR.glob("*.txt"))

assert len(law_paths) == 23, (
    f"법령 TXT 개수 이상: {len(law_paths)}"
)


def normalize_name(s):
    return re.sub(
        r"[^0-9A-Za-z가-힣]",
        "",
        str(s)
    )


def find_law_file(alias):

    a = normalize_name(alias)

    # --------------------------------------------------------
    # 1순위: 정규화 후 완전 일치
    # --------------------------------------------------------
    exact = [
        p
        for p in law_paths
        if normalize_name(p.stem) == a
    ]

    if len(exact) == 1:
        return exact[0]

    if len(exact) > 1:
        raise AssertionError(
            f"\n법령 파일 exact match 중복\n"
            f"alias={alias}\n"
            f"candidates={[x.name for x in exact]}"
        )


    # --------------------------------------------------------
    # 2순위: 부분 일치
    #
    # 약칭/문장부호 차이가 있는 경우를 위한 fallback.
    # 여러 개면 더 구체적인(이름이 긴) 후보 우선.
    # --------------------------------------------------------
    candidates = [
        p
        for p in law_paths
        if (
            a in normalize_name(p.stem)
            or
            normalize_name(p.stem) in a
        )
    ]

    if not candidates:
        raise AssertionError(
            f"\n법령 파일 식별 실패\n"
            f"alias={alias}\n"
            f"candidates=[]"
        )

    # 가장 구체적인 파일명 우선
    candidates = sorted(
        candidates,
        key=lambda p: len(normalize_name(p.stem)),
        reverse=True
    )

    best = candidates[0]

    # 가장 긴 후보와 같은 길이 후보가 여러 개면
    # 임의 선택하지 않고 실패시킨다.
    best_len = len(
        normalize_name(best.stem)
    )

    tied = [
        p
        for p in candidates
        if len(normalize_name(p.stem)) == best_len
    ]

    assert len(tied) == 1, (
        f"\n법령 파일 fallback match 모호함\n"
        f"alias={alias}\n"
        f"candidates={[x.name for x in candidates]}"
    )

    return best


def load_law(alias):

    p = find_law_file(alias)

    return {
        "source": p.stem,
        "file": p.name,
        "text": p.read_text(
            encoding="utf-8",
            errors="replace"
        )
    }


# ------------------------------------------------------------
# 필요한 source
# ------------------------------------------------------------

NAT_DECREE = load_law(
    "국가를 당사자로 하는 계약에 관한 법률 시행령"
)

NAT_RULE = load_law(
    "국가를 당사자로 하는 계약에 관한 법률 시행규칙"
)

LOC_DECREE = load_law(
    "지방자치단체를 당사자로 하는 계약에 관한 법률 시행령"
)

LOC_RULE = load_law(
    "지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙"
)

GOV_GUIDE = load_law(
    "(계약예규) 정부 입찰·계약 집행기준"
)

LOC_GUIDE = load_law(
    "지방자치단체 입찰 및 계약 집행기준"
)

THRESHOLD_NOTICE = load_law(
    "국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액"
)


# ============================================================
# 2. 정확한 조문 추출
#
# 중요:
# "제5조"라는 문자열 출현을 찾는 것이 아니라
#
#     ^제5조(
#
# 형태의 실제 조문 제목만 찾는다.
# ============================================================

ARTICLE_HEADER = re.compile(
    r"(?m)^[ \t]*제\d+조(?:의\d+)?\("
)


def article_header_text(article):
    """
    입력 예:
      "7"    -> "제7조"
      "21"   -> "제21조"
      "2의2" -> "제2조의2"
      "5의3" -> "제5조의3"
    """

    article = str(article).strip()

    if "의" in article:

        base, sub = article.split(
            "의",
            1
        )

        assert (
            base.isdigit()
            and
            sub.isdigit()
        ), (
            f"잘못된 가지조문 형식: {article}"
        )

        return (
            f"제{base}조의{sub}"
        )

    assert article.isdigit(), (
        f"잘못된 조문 형식: {article}"
    )

    return f"제{article}조"


def extract_article(
    text,
    article
):

    header = article_header_text(
        article
    )

    # 실제 조문 제목만 탐색
    # 예:
    #   제7조(
    #   제2조의2(
    #   제5조의3(
    pattern = re.compile(
        rf"(?m)^[ \t]*"
        rf"{re.escape(header)}\("
    )

    matches = list(
        pattern.finditer(text)
    )

    assert matches, (
        f"{header} 실제 제목 미발견"
    )

    # 보통 법령 파일의 실제 본문 첫 출현 사용
    m = matches[0]

    start = m.start()

    nxt = ARTICLE_HEADER.search(
        text,
        m.end()
    )

    end = (
        nxt.start()
        if nxt
        else len(text)
    )

    result = (
        text[start:end]
        .strip()
    )

    assert result.startswith(
        f"{header}("
    ), (
        f"\n조문 추출 시작점 이상\n"
        f"expected={header}(\n"
        f"actual={result[:100]}"
    )

    return result


# ============================================================
# ARTICLE LABEL SANITY CHECK
# ============================================================

assert article_header_text("7") == "제7조"
assert article_header_text("21") == "제21조"
assert article_header_text("2의2") == "제2조의2"
assert article_header_text("2의3") == "제2조의3"
assert article_header_text("5의3") == "제5조의3"

print(
    "[PASS] 일반조문/가지조문 표기 변환 검증 완료"
)


def extract_region(
    text,
    start_pattern,
    end_pattern=None,
    flags=re.M
):

    start_re = re.compile(
        start_pattern,
        flags
    )

    starts = list(
        start_re.finditer(text)
    )

    assert starts, (
        f"START 미발견: {start_pattern}"
    )

    # 목차가 아니라 실제 본문에 가까운 마지막 출현을 우선.
    start_match = starts[-1]
    start = start_match.start()

    if end_pattern:

        end_re = re.compile(
            end_pattern,
            flags
        )

        end_match = end_re.search(
            text,
            start_match.end()
        )

        end = (
            end_match.start()
            if end_match
            else len(text)
        )

    else:

        end = len(text)

    return text[start:end].strip()


def extract_chapter_article(
    text,
    chapter_title_regex,
    next_chapter_regex,
    article_no
):

    chapter = extract_region(
        text,
        chapter_title_regex,
        next_chapter_regex
    )

    return extract_article(
        chapter,
        article_no
    )


# ============================================================
# 3. 예규의 정확한 특수 구간
# ============================================================

# 정부 입찰·계약 집행기준
# 제2장 제한경쟁입찰의 운용 → 제5조
GOV_CH2_ART5 = extract_chapter_article(
    GOV_GUIDE["text"],
    r"(?m)^[ \t]*제2장[ \t]+제한경쟁입찰의[ \t]+운용[ \t]*$",
    r"(?m)^[ \t]*제3장\b",
    5
)

assert GOV_CH2_ART5.startswith(
    "제5조("
)


# 지방 집행기준
# 제1장 내 "7. 계약담당자 주의사항"
LOC_CAUTION = extract_region(
    LOC_GUIDE["text"],
    r"(?m)^[ \t]*7\.[ \t]*계약담당자[ \t]*주의사항[ \t]*$",
    r"(?m)^[ \t]*8\."
)

assert "계약담당자 주의사항" in LOC_CAUTION
assert "실적" in LOC_CAUTION


# 지방 소액수의
# "1. 금액기준에 따른 2인 이상 견적서 제출 수의계약"
small_start = re.search(
    r"(?m)^[ \t]*1\.[ \t]*금액기준에[ \t]*따른"
    r"[ \t]*2인[ \t]*이상[ \t]*견적서[ \t]*제출"
    r"[ \t]*수의계약[ \t]*$",
    LOC_GUIDE["text"]
)

assert small_start, (
    "지방 수의계약 금액기준 본문 미발견"
)

small_tail = LOC_GUIDE["text"][
    small_start.start():
]

small_end = re.search(
    r"(?m)^[ \t]*2\.",
    small_tail[
        small_start.end() - small_start.start():
    ]
)

if small_end:

    relative_start = (
        small_start.end()
        - small_start.start()
    )

    end_pos = (
        relative_start
        + small_end.start()
    )

    LOC_SMALL_VALUE = (
        small_tail[:end_pos]
        .strip()
    )

else:

    LOC_SMALL_VALUE = (
        small_tail[:12000]
        .strip()
    )


assert "2인 이상" in LOC_SMALL_VALUE
assert "견적" in LOC_SMALL_VALUE
assert "시·군" in LOC_SMALL_VALUE or "시․군" in LOC_SMALL_VALUE
assert "인접" in LOC_SMALL_VALUE


# ============================================================
# 4. 정확한 법령 블록 생성
# ============================================================

LEGAL = {

    "nat_decree_12": {
        "source": NAT_DECREE["source"],
        "locator": "제12조",
        "text": extract_article(
            NAT_DECREE["text"], 12
        )
    },

    "nat_decree_21": {
        "source": NAT_DECREE["source"],
        "locator": "제21조",
        "text": extract_article(
            NAT_DECREE["text"], 21
        )
    },

    "nat_rule_25": {
        "source": NAT_RULE["source"],
        "locator": "제25조",
        "text": extract_article(
            NAT_RULE["text"], 25
        )
    },

    "loc_decree_13": {
        "source": LOC_DECREE["source"],
        "locator": "제13조",
        "text": extract_article(
            LOC_DECREE["text"], 13
        )
    },

    "loc_decree_20": {
        "source": LOC_DECREE["source"],
        "locator": "제20조",
        "text": extract_article(
            LOC_DECREE["text"], 20
        )
    },

    "loc_rule_24": {
        "source": LOC_RULE["source"],
        "locator": "제24조",
        "text": extract_article(
            LOC_RULE["text"], 24
        )
    },

    "loc_rule_25": {
        "source": LOC_RULE["source"],
        "locator": "제25조",
        "text": extract_article(
            LOC_RULE["text"], 25
        )
    },

    "gov_guide_ch2_art5": {
        "source": GOV_GUIDE["source"],
        "locator":
            "제2장 제한경쟁입찰의 운용 제5조",
        "text":
            GOV_CH2_ART5
    },

    "loc_caution": {
        "source": LOC_GUIDE["source"],
        "locator":
            "제1장 7. 계약담당자 주의사항",
        "text":
            LOC_CAUTION
    },

    "loc_small_value": {
        "source": LOC_GUIDE["source"],
        "locator":
            "제5장 수의계약 운영요령 "
            "제3절 금액기준에 따른 "
            "2인 이상 견적서 제출 수의계약",
        "text":
            LOC_SMALL_VALUE
    },

    "national_threshold_notice": {
        "source":
            THRESHOLD_NOTICE["source"],

        "locator":
            "고시금액",

        # 별도 고시는 그 자체가 판단표이므로
        # 파일 전체를 보존한다.
        "text":
            THRESHOLD_NOTICE["text"].strip()
    },
}


# ============================================================
# 5. exact grounding 검증
# ============================================================

assert LEGAL[
    "nat_decree_12"
]["text"].startswith(
    "제12조("
)

assert LEGAL[
    "nat_decree_21"
]["text"].startswith(
    "제21조("
)

assert LEGAL[
    "nat_rule_25"
]["text"].startswith(
    "제25조("
)

assert LEGAL[
    "loc_rule_24"
]["text"].startswith(
    "제24조("
)

assert LEGAL[
    "loc_rule_25"
]["text"].startswith(
    "제25조("
)

assert LEGAL[
    "gov_guide_ch2_art5"
]["text"].startswith(
    "제5조("
)

assert (
    "지역제한입찰"
    in LEGAL["loc_rule_24"]["text"]
)

assert (
    "계약담당자 주의사항"
    in LEGAL["loc_caution"]["text"]
)

assert (
    "인접"
    in LEGAL["loc_small_value"]["text"]
)


# ============================================================
# 6. v1~v8에서 사용할 exact law key
# ============================================================

LEGAL_KEYS = {

    "v1": [
        "nat_decree_12",
        "nat_decree_21",
        "loc_decree_13",
        "loc_decree_20",
    ],

    "v2": [
        "nat_decree_21",
        "nat_rule_25",
        "loc_decree_20",
        "loc_rule_25",
        "loc_caution",
    ],

    "v3": [
        "nat_decree_21",
        "nat_rule_25",
        "gov_guide_ch2_art5",
        "loc_decree_20",
        "loc_rule_25",
    ],

    "v4": [
        "gov_guide_ch2_art5",
        "loc_caution",
    ],

    "v5": [
        "nat_decree_21",
        "national_threshold_notice",
        "loc_decree_20",
        "loc_rule_24",
    ],

    "v6": [
        "nat_rule_25",
        "loc_rule_25",
        "loc_small_value",
    ],

    "v7": [
        "nat_rule_25",
        "loc_rule_25",
        "loc_small_value",
    ],

    "v8": [
        "nat_rule_25",
        "loc_rule_25",
        "loc_small_value",
    ],
}


# ============================================================
# 7. 공통 정책
# ============================================================

COMMON = {

    "absence_detection": False,

    "document_scope": (
        "공고문만 보지 말고 제공된 모든 docs를 사용한다. "
        "공고문·규격서·과업지시서·제안요청서·예외공표서를 "
        "모두 판정 후보로 본다."
    ),

    "presence_rule": (
        "v1~v8은 부재탐지형이 아니다. "
        "제공된 문서에서 위반 제한조건을 실제로 확인했을 때만 "
        "1 후보로 올린다. 문서가 누락되었다는 이유만으로 "
        "위반을 추론하지 않는다."
    ),

    "incomplete_rule": (
        "dropped_doc_counts가 있거나 완전관측=False라도 "
        "현재 보이는 문서에서 위반조건이 명시적으로 확인되면 "
        "위반 판정은 가능하다. 반대로 위반문구를 찾지 못한 경우 "
        "누락 문서에 있을 것이라고 추측하여 1로 만들지 않는다."
    ),

    "evidence_rule": (
        "위반일 경우 제공된 docs의 원문에 실제로 존재하는 "
        "부분문자열만 evidence로 사용한다. "
        "가능하면 위반 제한조건을 직접 보여주는 가장 짧은 "
        "연속 구간을 선택하고 500자를 넘기지 않는다."
    ),
}


# ============================================================
# 8. FINAL DECISION SPECS
# ============================================================

FINAL = {}


# ------------------------------------------------------------
# v1 참가자격 특정기관 제한
# ------------------------------------------------------------

FINAL["v1"] = {

    "status":
        "REVIEWED",

    "applicability": (
        "국가계약법 또는 지방계약법 적용 공고에서 "
        "입찰참가자격 자체를 특정 기관·기관유형에 속한 자로 "
        "제한하는 경우를 검사한다."
    ),

    "required_meta": [
        "적용계약법",
        "업무구분",
        "계약방법",
        "면허업종제한목록",
        "업종제한여부",
    ],

    "facts_to_extract": [
        "참가자격을 규정하는 문장",
        "참가 가능한 기관 또는 기관유형",
        "해당 제한이 법정 허가·인가·면허·등록 요건인지 여부",
    ],

    "amount_condition":
        "없음",

    "violation_condition": (
        "입찰에 참가하는 업체 자체를 대학, 산학협력단, "
        "특정 공공기관·기관유형 등으로 제한하고, "
        "그 제한이 계약 수행에 필요한 법정 자격요건으로 "
        "직접 정당화되지 않는 경우."
    ),

    "nonviolation_condition": (
        "기관명이 단순 발주기관·수요기관·실적 발주기관으로 "
        "언급된 것뿐이거나, 다른 법령에 따른 허가·인가·면허·"
        "등록 등 필수 자격을 요구하는 경우."
    ),

    "exceptions": [
        "다른 법령이 해당 기관유형 또는 법정 자격을 직접 요구하는 경우",
    ],

    "python_role": [
        "참가자격 섹션 후보 추출",
        "기관유형 표현 후보 추출",
        "meta 자격정보 정리",
        "evidence exact-substring 검증",
    ],

    "gemma_role": [
        "문장이 참가업체 자체의 자격을 제한하는지 판단",
        "단순 기관 언급과 실제 참가자격 제한을 구분",
        "법정 필수자격과 임의 기관제한을 구분",
    ],

    "unresolved": [],
}


# ------------------------------------------------------------
# v2 고시금액 미만 실적제한
# ------------------------------------------------------------

FINAL["v2"] = {

    "status":
        "REVIEWED_WITH_RUNTIME_THRESHOLD",

    "applicability": (
        "과거 수행·납품·용역 실적을 입찰참가자격으로 요구하고, "
        "해당 공고의 추정가격이 적용되는 법령상 고시금액 "
        "미만인지 검사한다."
    ),

    "required_meta": [
        "적용계약법",
        "업무구분",
        "계약방법",
        "입찰추정가격",
        "배정예산금액",
    ],

    "facts_to_extract": [
        "실적 요구 존재 여부",
        "실적 요구가 참가자격인지 단순 평가요소인지",
        "계약 종류",
        "추정가격",
        "적용 고시금액",
    ],

    "amount_condition": (
        "입찰추정가격 < 해당 계약에 적용되는 고시금액. "
        "단일 숫자를 하드코딩하지 않고 배포 법령패키지의 "
        "적용 기준을 사용한다."
    ),

    "violation_condition": (
        "고시금액 미만인 공고에서 과거 동종·유사 수행실적, "
        "납품실적 또는 일정 금액 이상의 계약실적을 "
        "입찰참가자격으로 요구하는 경우."
    ),

    "nonviolation_condition": (
        "실적 요구가 없거나, 실적이 참가자격 제한이 아닌 "
        "평가자료에 불과하거나, 해당 계약이 실적제한을 "
        "허용하는 법적 범위에 해당하는 경우."
    ),

    "exceptions": [
        "지방계약의 소액수의 특례는 별도 법령분기 후 판단",
        "특수한 기술·공법 등 법령상 별도 제한근거가 있는 경우는 Gemma가 원문과 함께 판단",
    ],

    "python_role": [
        "입찰추정가격 추출",
        "계약유형별 고시금액 lookup",
        "금액 비교",
        "실적 금액·기간 표현 후보 추출",
        "evidence 검증",
    ],

    "gemma_role": [
        "실적 문장이 실제 참가자격 제한인지 판단",
        "실적과 평가기준을 구분",
        "특례·법정 제한근거의 적용 여부 판단",
    ],

    "unresolved": [
        "계약유형별 고시금액은 runtime에 배포 법령패키지에서 결정하며 전역 상수로 두지 않음",
    ],
}


# ------------------------------------------------------------
# v3 실적제한 1배수 이상
# ------------------------------------------------------------

FINAL["v3"] = {

    "status":
        "REVIEWED",

    "applicability": (
        "입찰참가자격으로 과거 실적의 금액·규모를 요구하는 "
        "공고를 대상으로 한다."
    ),

    "required_meta": [
        "배정예산금액",
        "입찰추정가격",
        "적용계약법",
        "업무구분",
    ],

    "facts_to_extract": [
        "최소 요구 실적금액 또는 비교 가능한 실적규모",
        "사업예산",
        "VAT 포함·별도 표현",
        "실적조건이 참가자격인지 여부",
    ],

    "amount_condition": (
        "공식 항목 비고에 따라 사업예산을 기준으로 한다. "
        "비교 가능한 실적 요구액이 사업예산의 1배 이상인지 계산한다."
    ),

    "violation_condition": (
        "참가자격으로 요구하는 최소 실적금액 또는 명시된 "
        "비교가능 실적규모가 사업예산의 100% 이상인 경우."
    ),

    "nonviolation_condition": (
        "실적 제한이 없거나, 비교 가능한 최소 실적기준이 "
        "사업예산의 100% 미만인 경우."
    ),

    "exceptions": [
        "서로 단위가 다른 실적규모를 임의 환산하지 않는다",
    ],

    "python_role": [
        "배정예산금액 사용",
        "문서의 실적금액 숫자 후보 추출",
        "VAT 표현 보존",
        "실적요구액 / 사업예산 비율 계산",
        "evidence 검증",
    ],

    "gemma_role": [
        "여러 숫자 중 실제 최소 실적요구액 식별",
        "숫자가 참가자격 조건인지 판단",
        "비교 단위가 동일한지 판단",
    ],

    "unresolved": [
        "VAT 포함/별도 때문에 직접비교가 불가능한 경우 Python이 임의 보정하지 않고 Gemma 판단 대상으로 유지",
    ],
}


# ------------------------------------------------------------
# v4 고시금액 이상 특정기관·특정실적
# ------------------------------------------------------------

FINAL["v4"] = {

    "status":
        "REVIEWED_WITH_OPEN_AMOUNT_EDGE",

    "applicability": (
        "과거 실적을 참가자격으로 요구하면서 실적의 "
        "발주기관 범위 또는 인정되는 실적 자체를 부당하게 "
        "특정하는지 검사한다."
    ),

    "required_meta": [
        "적용계약법",
        "업무구분",
        "입찰추정가격",
        "계약방법",
    ],

    "facts_to_extract": [
        "실적 발주기관 제한",
        "실적 대상기관·수요처 제한",
        "특정 명칭·행사·사업형태만 인정하는 제한",
        "동등·유사실적 인정 여부",
    ],

    "amount_condition": (
        "항목명에는 고시금액 이상이 포함되어 있으나 "
        "DEV에서는 v2와 v4가 동시에 양성인 사례가 존재하므로, "
        "입찰추정가격 >= 고시금액을 단독 hard gate로 사용하지 않는다. "
        "금액은 feature로 제공하고 의미판단은 법령근거와 함께 수행한다."
    ),

    "violation_condition": (
        "실적을 국가기관·공공기관·대학병원 등 특정 발주기관의 "
        "실적으로만 인정하거나, 실제 동등한 실적을 배제하도록 "
        "특정 명칭·대상·수행환경의 실적으로 과도하게 좁혀 "
        "참가자격을 제한하는 경우."
    ),

    "nonviolation_condition": (
        "발주기관 종류와 관계없이 동등·유사 실적을 폭넓게 "
        "인정하거나, 특정 표현이 계약 수행에 객관적으로 "
        "필수적인 요건이면서 법적 근거가 확인되는 경우."
    ),

    "exceptions": [
        "계약 목적상 특정 성격의 실적이 객관적으로 불가피한 경우는 법령 문맥과 함께 판단",
    ],

    "python_role": [
        "기관명·기관유형 후보 추출",
        "실적 관련 문장 후보 추출",
        "입찰추정가격 및 threshold feature 제공",
        "evidence 검증",
    ],

    "gemma_role": [
        "특정기관 발주실적 제한인지 판단",
        "특정명칭·특정수요처에 의한 부당한 실적 제한인지 판단",
        "동등·유사 실적 인정 가능성을 문맥으로 판단",
    ],

    "unresolved": [
        "항목명의 '고시금액 이상'을 독립 hard gate로 적용할지는 DEV FP/FN audit에서 재검증",
    ],
}


# ------------------------------------------------------------
# v5 고시금액 이상 지역제한
# ------------------------------------------------------------

FINAL["v5"] = {

    "status":
        "REVIEWED_WITH_RUNTIME_THRESHOLD",

    "applicability": (
        "입찰참가자의 본점·주된 영업소 소재지를 특정 지역으로 "
        "제한하는 공고에서, 추정가격이 해당 지역제한 허용금액 "
        "이상인지 검사한다."
    ),

    "required_meta": [
        "적용계약법",
        "업무구분",
        "입찰추정가격",
        "지역제한여부",
        "제한지역코드목록",
        "소관구분",
    ],

    "facts_to_extract": [
        "실제 문서의 소재지 제한",
        "제한 지역의 행정단위",
        "추정가격",
        "계약유형별 지역제한 금액기준",
    ],

    "amount_condition": (
        "입찰추정가격 >= 해당 계약에 적용되는 지역제한 허용금액. "
        "국가계약은 배포 고시금액 파일을, 지방계약은 "
        "지방계약법 시행규칙 제24조의 계약종류별 금액을 사용한다."
    ),

    "violation_condition": (
        "법령상 지역제한 허용금액 이상인 계약에서 "
        "본점·주된 영업소 소재지를 특정 지역으로 제한하는 경우."
    ),

    "nonviolation_condition": (
        "지역제한 자체가 없거나, 법령상 허용금액 미만에서 "
        "허용된 범위의 지역제한을 하는 경우."
    ),

    "exceptions": [
        "법령상 별도 지역제한 허용사유가 명시적으로 존재하는 경우",
    ],

    "python_role": [
        "추정가격 추출",
        "계약유형 판별",
        "배포 법령패키지에서 적용 금액 lookup",
        "지역 meta와 문서 region 후보 정리",
        "evidence 검증",
    ],

    "gemma_role": [
        "문서의 지역표현이 실제 참가자격 제한인지 판단",
        "단순 사업장소·납품장소와 업체 소재지 제한을 구분",
    ],

    "unresolved": [
        "국가계약 고시금액은 별도 고시 source에서 runtime lookup",
        "지방계약은 계약유형 및 기관조건에 따라 시행규칙 제24조 분기를 적용",
    ],
}


# ------------------------------------------------------------
# v6 고시금액 미만 지역제한 시·군·구
# ------------------------------------------------------------

FINAL["v6"] = {

    "status":
        "REVIEWED",

    "applicability": (
        "고시금액 미만 계약에서 참가업체 소재지를 "
        "기초지방자치단체 수준(시·군·구)으로 제한하는지 검사한다."
    ),

    "required_meta": [
        "적용계약법",
        "계약방법",
        "입찰추정가격",
        "지역제한여부",
        "제한지역코드목록",
    ],

    "facts_to_extract": [
        "지역제한 존재 여부",
        "광역/기초 행정단위",
        "계약방법",
        "소액수의 여부",
    ],

    "amount_condition": (
        "고시금액 미만 영역. 단, 지방 소액수의는 별도 예외분기를 적용한다."
    ),

    "violation_condition": (
        "일반적인 경쟁계약 등에서 업체 소재지를 특정 시·군·구 "
        "등 기초단위로 과도하게 좁혀 참가자격을 제한하는 경우."
    ),

    "nonviolation_condition": (
        "지방계약의 적법한 2인 이상 견적 수의계약으로서 "
        "배포된 수의계약 운영요령이 허용한 관할 시·군 제한에 "
        "정확히 해당하거나, 기초단위 제한이 존재하지 않는 경우."
    ),

    "exceptions": [
        "지방 소액수의 2인 이상 견적에서 집행기준이 명시적으로 허용한 지역범위",
    ],

    "python_role": [
        "지역 token의 단위=기초 여부 추출",
        "계약방법 및 추정가격 확인",
        "수의계약 예외 branch 후보 생성",
        "evidence 검증",
    ],

    "gemma_role": [
        "지역 문장이 업체 소재지 참가제한인지 판단",
        "수의계약 특례의 실제 적용 여부 판단",
    ],

    "unresolved": [],
}


# ------------------------------------------------------------
# v7 고시금액 미만 지역제한 인접 확대
# ------------------------------------------------------------

FINAL["v7"] = {

    "status":
        "REVIEWED",

    "applicability": (
        "고시금액 미만 계약에서 참가업체 소재지 제한을 "
        "복수 지역·인접 지역으로 확대한 형태를 검사한다."
    ),

    "required_meta": [
        "적용계약법",
        "계약방법",
        "입찰추정가격",
        "지역제한여부",
        "제한지역코드목록",
    ],

    "facts_to_extract": [
        "제한지역 개수",
        "광역/기초 단위",
        "또는/및 등 결합방식",
        "수의계약 여부",
    ],

    "amount_condition": (
        "고시금액 미만 영역. 지방 소액수의의 인접지역 허용은 "
        "별도 예외분기로 처리한다."
    ),

    "violation_condition": (
        "통상 허용되는 단일 지역범위를 넘어 복수 광역지역, "
        "인접 지역 등의 형태로 참가가능 지역을 임의 확대하여 "
        "지역제한을 설정한 경우."
    ),

    "nonviolation_condition": (
        "지방 소액수의 2인 이상 견적에서 배포 집행기준이 "
        "허용하는 관할 시·군 + 인접 시·군, 관할 시·도 + "
        "인접 시·도 범위에 정확히 해당하는 경우."
    ),

    "exceptions": [
        "지방 수의계약 운영요령이 명시적으로 허용하는 인접지역 확대",
    ],

    "python_role": [
        "복수 지역 후보 추출",
        "광역/기초 token 구조화",
        "계약방법 확인",
        "외부 지리 데이터 없이 제공된 지역표현만 사용",
        "evidence 검증",
    ],

    "gemma_role": [
        "복수 지역 표현이 실제 참가자격 제한인지 판단",
        "허용된 소액수의 인접지역 예외인지 판단",
    ],

    "unresolved": [
        "외부 지리정보로 인접성을 새로 보강하지 않음; 대회 제공 정보만 사용",
    ],
}


# ------------------------------------------------------------
# v8 중복제한 (실적 + 지역)
# ------------------------------------------------------------

FINAL["v8"] = {

    "status":
        "REVIEWED_WITH_EXCEPTION_CHECK",

    "applicability": (
        "같은 공고의 참가자격에 실적제한과 지역제한이 "
        "동시에 적용되는지 검사한다."
    ),

    "required_meta": [
        "적용계약법",
        "계약방법",
        "입찰추정가격",
        "지역제한여부",
        "제한지역코드목록",
    ],

    "facts_to_extract": [
        "실적 참가자격 제한 존재",
        "업체 소재지 지역제한 존재",
        "두 제한이 같은 입찰의 참가요건으로 누적되는지",
        "지방 소액수의 특례 여부",
    ],

    "amount_condition": (
        "실적+지역 제한의 결합 여부가 핵심이며, "
        "지방 소액수의 가능성은 별도 예외분기로 확인한다."
    ),

    "violation_condition": (
        "동일 공고에서 입찰참가자에게 과거 실적조건과 "
        "업체 소재지 지역조건을 동시에 충족하도록 요구하는 경우."
    ),

    "nonviolation_condition": (
        "둘 중 하나만 존재하거나, 실적 문구가 단순 평가자료이고 "
        "참가자격 제한이 아니거나, 배포 법령이 허용하는 "
        "예외 조합으로 확인되는 경우."
    ),

    "exceptions": [
        "지방 소액수의 특례가 적용될 가능성이 있으면 자동확정하지 않고 exact law block과 함께 판정",
    ],

    "python_role": [
        "실적 후보와 지역 후보를 각각 구조화",
        "동일 공고 내 동시 존재 flag 생성",
        "계약방법·추정가격 제공",
        "evidence exact-substring 검증",
    ],

    "gemma_role": [
        "두 조건이 모두 실제 참가자격 제한인지 확인",
        "단순 평가실적과 참가자격 실적을 구분",
        "지방 소액수의 예외 적용 여부 판단",
    ],

    "unresolved": [
        "지방 소액수의에서 실적+지역 조합이 허용되는 정확한 경계는 DEV FP/FN audit에서 별도 검증",
    ],
}


# ============================================================
# 9. 공통정보 + 법령 + DEV 근거 결합
# ============================================================

FINAL_PACK = {}


for v in TARGET_ITEMS:

    src = review[v]

    spec = {
        "item":
            v,

        "name":
            src["name"],

        "official_note":
            src["official_note"],

        "official_basis":
            src["official_basis"],

        **COMMON,

        "decision_spec":
            FINAL[v],

        "exact_legal_keys":
            LEGAL_KEYS[v],

        "exact_legal_text": {
            key: LEGAL[key]
            for key in LEGAL_KEYS[v]
        },

        "dev": {
            "positive_count":
                src["dev_positive_count"],

            "positive_ids": [
                x["id"]
                for x in src["dev_positives"]
            ],

            "positive_evidence_examples": [
                x["evidence"]
                for x in src["dev_positives"]
                if x.get("evidence")
            ],

            # 중요:
            # 이것들은 구조 유사 후보일 뿐
            # 법적으로 검증된 near-negative가 아니다.
            "negative_candidate_count":
                src["dev_negative_candidate_count"],

            "negative_candidate_ids": [
                x["id"]
                for x in src[
                    "dev_negative_candidates"
                ]
            ],

            "negative_candidate_semantics":
                "STRUCTURAL_CANDIDATES_ONLY",
        },
    }

    FINAL_PACK[v] = spec


# ============================================================
# 10. 최종 validation
# ============================================================

assert list(
    FINAL_PACK.keys()
) == TARGET_ITEMS


for v in TARGET_ITEMS:

    x = FINAL_PACK[v]

    assert (
        x["absence_detection"]
        is False
    )

    assert (
        x["decision_spec"]["status"]
        != "PENDING_FINAL_REVIEW"
    )

    assert (
        x["exact_legal_keys"]
    )

    assert (
        x["dev"]["positive_count"]
        > 0
    )

    for key in x["exact_legal_keys"]:

        assert key in LEGAL

        assert (
            LEGAL[key]["text"].strip()
        )


# ------------------------------------------------------------
# source 분리 핵심검증
# ------------------------------------------------------------

assert (
    LEGAL["loc_rule_25"]["source"]
    !=
    LEGAL["loc_decree_20"]["source"]
)

assert (
    LEGAL["national_threshold_notice"]["source"]
    !=
    LEGAL["nat_decree_21"]["source"]
)


# ============================================================
# 11. 저장
# ============================================================

with open(
    OUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_PACK,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 12. Audit
# ============================================================

rows = []

for v in TARGET_ITEMS:

    x = FINAL_PACK[v]

    rows.append({

        "item":
            v,

        "name":
            x["name"],

        "status":
            x["decision_spec"]["status"],

        "legal_blocks":
            len(
                x["exact_legal_keys"]
            ),

        "positive_count":
            x["dev"]["positive_count"],

        "evidence_examples":
            len(
                x["dev"][
                    "positive_evidence_examples"
                ]
            ),

        "unresolved":
            len(
                x["decision_spec"][
                    "unresolved"
                ]
            ),
    })


audit_63b = pd.DataFrame(rows)

display(audit_63b)


print()
print("=" * 100)
print("Stage 6-3B FINAL RESULT")
print("=" * 100)

print(
    "ITEMS              =",
    len(FINAL_PACK)
)

print(
    "EXACT LEGAL BLOCKS =",
    len(LEGAL)
)

print(
    "TOTAL POSITIVES    =",
    sum(
        x["dev"]["positive_count"]
        for x in FINAL_PACK.values()
    )
)

print(
    "PENDING SPECS      =",
    [
        v
        for v, x in FINAL_PACK.items()
        if x[
            "decision_spec"
        ][
            "status"
        ] == "PENDING_FINAL_REVIEW"
    ]
)

print(
    "OUTPUT             =",
    OUT_PATH
)

print()
print(
    "[PASS] Stage 6-3B v1~v8 exact grounding repair + "
    "final judgment specification 완료"
)

,item,name,status,legal_blocks,positive_count,evidence_examples,unresolved
0,v1,참가자격 특정기관 제한,REVIEWED,4,7,1,0
1,v2,고시금액 미만 실적제한,REVIEWED_WITH_RUNTIME_THRESHOLD,5,7,7,1
2,v3,실적제한 1배수 이상,REVIEWED,5,8,8,1
3,v4,"고시금액 이상 특정기관, 특정실적",REVIEWED_WITH_OPEN_AMOUNT_EDGE,2,6,5,1
4,v5,고시금액 이상 지역제한,REVIEWED_WITH_RUNTIME_THRESHOLD,4,7,1,2
5,v6,"고시금액 미만 지역제한 시,군,구",REVIEWED,3,6,1,0
6,v7,고시금액 미만 지역제한 인접 확대,REVIEWED,3,7,7,1
7,v8,중복제한 (실적+지역),REVIEWED_WITH_EXCEPTION_CHECK,3,6,2,1



Stage 6-3B FINAL RESULT
ITEMS              = 8
EXACT LEGAL BLOCKS = 11
TOTAL POSITIVES    = 54
PENDING SPECS      = []
OUTPUT             = /home/alparka/nara-ai/artifacts/judgment_specs_v1_v8.json

[PASS] Stage 6-3B v1~v8 exact grounding repair + final judgment specification 완료
LAW FILE MATCH CHECK
국가를 당사자로 하는 계약에 관한 법률 시행령
  -> 국가를 당사자로 하는 계약에 관한 법률 시행령.txt
국가를 당사자로 하는 계약에 관한 법률 시행규칙
  -> 국가를 당사자로 하는 계약에 관한 법률 시행규칙.txt
지방자치단체를 당사자로 하는 계약에 관한 법률 시행령
  -> 지방자치단체를 당사자로 하는 계약에 관한 법률 시행령.txt
지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙
  -> 지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙.txt
(계약예규) 정부 입찰·계약 집행기준
  -> (계약예규) 정부 입찰·계약 집행기준.txt
지방자치단체 입찰 및 계약 집행기준
  -> 지방자치단체 입찰 및 계약 집행기준.txt
국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액
  -> 국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액.txt

[PASS] 법령 파일명 매칭 검증 완료


AssertionError: 제5의3조 실제 제목 미발견

In [6]:
# ============================================================
# Stage 6-3C FINAL - ROBUST VERSION
# v9~v19 exact grounding + final judgment specifications
#
# 중요 수정:
# 1) 법령 파일 fuzzy matching 제거 -> 정확 파일명 사용
# 2) 제N조 / 제N조의M 을 line-based parser로 정확히 처리
# 3) 공백, NBSP, XML CDATA가 있어도 조문 제목 인식
# 4) 예규는 장(Chapter) 범위를 먼저 찾은 뒤 조문 검색
# 5) v19 공식 locator "제5조의3"이 실제 package에 없으면
#    억지로 만들지 않고 source 내 실제 관련 문구를 anchor grounding
# 6) 중기간 경쟁제품은 배포 CSV를 structured source로 사용
# 7) v10/v11/v16/v18은 부재탐지로 유지
# ============================================================

from pathlib import Path
import json
import re
import math
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

DATA_DIR = ROOT / "open"

LAW_DIR = (
    DATA_DIR
    / "data"
    / "법령패키지"
    / "법령"
)

SME_DIR = (
    DATA_DIR
    / "data"
    / "법령패키지"
    / "중기부고시"
)

SOURCE_PACK_PATH = (
    ROOT
    / "artifacts"
    / "judgment_spec_source_pack.json"
)

BASIS_PATH = (
    ROOT
    / "artifacts"
    / "official_24_basis.csv"
)

OUT_PATH = (
    ROOT
    / "artifacts"
    / "judgment_specs_v9_v19.json"
)


assert LAW_DIR.exists(), LAW_DIR
assert SME_DIR.exists(), SME_DIR
assert SOURCE_PACK_PATH.exists(), SOURCE_PACK_PATH
assert BASIS_PATH.exists(), BASIS_PATH


TARGET_ITEMS = [
    f"v{i}"
    for i in range(9, 20)
]


# ============================================================
# 1. LOAD SOURCE PACK
# ============================================================

with open(
    SOURCE_PACK_PATH,
    "r",
    encoding="utf-8"
) as f:

    source_pack = json.load(f)


assert all(
    v in source_pack
    for v in TARGET_ITEMS
)


basis_df = pd.read_csv(
    BASIS_PATH,
    dtype=str
)

basis_df = basis_df.set_index(
    "item",
    drop=False
)


assert all(
    v in basis_df.index
    for v in TARGET_ITEMS
)


def safe_str(x):

    if x is None:
        return ""

    if isinstance(x, float):
        if math.isnan(x):
            return ""

    if pd.isna(x):
        return ""

    return str(x)


print("[1] SOURCE PACK / BASIS LOAD PASS")


# ============================================================
# 2. EXACT 법령 파일
#
# fuzzy matching을 하지 않는다.
# 이미 실제 package 파일명을 확인했기 때문에
# 명시적으로 연결한다.
# ============================================================

LAW_FILE_NAMES = {

    "nat_decree":
        "국가를 당사자로 하는 계약에 관한 법률 시행령.txt",

    "nat_rule":
        "국가를 당사자로 하는 계약에 관한 법률 시행규칙.txt",

    "loc_decree":
        "지방자치단체를 당사자로 하는 계약에 관한 법률 시행령.txt",

    "loc_rule":
        "지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙.txt",

    "gov_guide":
        "(계약예규) 정부 입찰·계약 집행기준.txt",

    "loc_guide":
        "지방자치단체 입찰 및 계약 집행기준.txt",

    "sme_act":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률.txt",

    "sme_decree":
        "중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령.txt",

    "threshold_notice":
        "국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액.txt",
}


LAW = {}


for key, filename in LAW_FILE_NAMES.items():

    path = LAW_DIR / filename

    assert path.exists(), (
        f"법령 파일 없음:\n{path}"
    )

    LAW[key] = {

        "path":
            path,

        "source":
            path.stem,

        "text":
            path.read_text(
                encoding="utf-8",
                errors="replace"
            )
    }


assert len(
    list(LAW_DIR.glob("*.txt"))
) == 23


print("[2] 정확 법령 파일 연결 PASS")


# ============================================================
# 3. TEXT NORMALIZATION
# ============================================================

def clean_line(line):

    s = str(line)

    # NBSP 계열
    s = s.replace("\u00a0", " ")
    s = s.replace("\u2007", " ")
    s = s.replace("\u202f", " ")

    # XML CDATA wrapper 제거
    s = s.replace("<![CDATA[", "")
    s = s.replace("]]>", "")

    return s.strip()


def compact_text(s):

    s = clean_line(s)

    return re.sub(
        r"\s+",
        "",
        s
    )


def make_lines(text):

    return text.splitlines()


for x in LAW.values():

    x["lines"] = make_lines(
        x["text"]
    )


# ============================================================
# 4. ROBUST ARTICLE PARSER
#
# 허용:
# 제5조(
# 제 5 조 (
# 제5조의3(
# 제5조의 3 (
# 제 5 조 의 3 (
#
# XML CDATA wrapper도 clean_line에서 제거됨.
# ============================================================

ARTICLE_LINE_RE = re.compile(
    r"^제\s*"
    r"(?P<base>\d+)"
    r"\s*조"
    r"(?:\s*의\s*(?P<sub>\d+))?"
    r"\s*(?:\(|（)"
)


def article_key_from_line(line):

    s = clean_line(line)

    m = ARTICLE_LINE_RE.match(s)

    if not m:
        return None

    base = m.group("base")
    sub = m.group("sub")

    if sub is None:
        return base

    return f"{base}의{sub}"


def article_display(key):

    key = str(key).strip()

    if "의" in key:

        base, sub = key.split(
            "의",
            1
        )

        return f"제{base}조의{sub}"

    return f"제{key}조"


def article_index(
    lines,
    start=0,
    end=None
):

    if end is None:
        end = len(lines)

    result = []

    for i in range(
        start,
        end
    ):

        key = article_key_from_line(
            lines[i]
        )

        if key is not None:

            result.append({
                "line":
                    i,

                "key":
                    key,

                "header":
                    clean_line(
                        lines[i]
                    ),
            })

    return result


def extract_article(
    lines,
    key,
    start=0,
    end=None,
    required=True
):

    if end is None:
        end = len(lines)

    indexes = article_index(
        lines,
        start=start,
        end=end
    )

    matches = [
        x
        for x in indexes
        if x["key"] == str(key)
    ]

    if not matches:

        if required:

            available = [
                x["key"]
                for x in indexes
                if (
                    x["key"].startswith(
                        str(key).split("의")[0]
                    )
                )
            ]

            raise AssertionError(
                "\n"
                f"{article_display(key)} 실제 조문 제목 미발견\n"
                f"근처 family={available[:30]}"
            )

        return None

    # 일반 법령에서는 첫 번째 실제 본문 조문 사용
    target = matches[0]

    pos = indexes.index(target)

    begin = target["line"]

    if pos + 1 < len(indexes):
        finish = indexes[pos + 1]["line"]
    else:
        finish = end

    text = "\n".join(
        lines[begin:finish]
    ).strip()

    assert (
        article_key_from_line(
            lines[begin]
        )
        ==
        str(key)
    )

    return {

        "key":
            str(key),

        "header":
            clean_line(
                lines[begin]
            ),

        "start_line":
            begin,

        "end_line":
            finish,

        "text":
            text,
    }


# ============================================================
# 5. CHAPTER PARSER
#
# 목차와 본문에 같은 장 제목이 반복될 수 있으므로,
# 가장 많은 실제 조문을 포함하는 candidate를 선택.
# ============================================================

def find_heading_candidates(
    lines,
    target_compact
):

    out = []

    for i, line in enumerate(lines):

        c = compact_text(line)

        if c.startswith(
            target_compact
        ):

            out.append(i)

    return out


def find_next_chapter(
    lines,
    start,
    next_chapter_prefix
):

    for i in range(
        start + 1,
        len(lines)
    ):

        c = compact_text(
            lines[i]
        )

        if c.startswith(
            next_chapter_prefix
        ):

            return i

    return len(lines)


def extract_best_chapter(
    lines,
    chapter_prefix,
    next_chapter_prefix
):

    candidates = find_heading_candidates(
        lines,
        chapter_prefix
    )

    assert candidates, (
        f"chapter heading 미발견: "
        f"{chapter_prefix}"
    )

    scored = []

    for start in candidates:

        end = find_next_chapter(
            lines,
            start,
            next_chapter_prefix
        )

        articles = article_index(
            lines,
            start,
            end
        )

        scored.append(
            (
                len(articles),
                end - start,
                start,
                end
            )
        )

    # 실제 본문은 목차보다 실제 조문 수가 훨씬 많음
    scored.sort(
        reverse=True
    )

    _, _, start, end = scored[0]

    return {
        "start":
            start,

        "end":
            end,

        "text":
            "\n".join(
                lines[start:end]
            ).strip(),

        "article_keys":
            [
                x["key"]
                for x in article_index(
                    lines,
                    start,
                    end
                )
            ],
    }


# ============================================================
# 6. 정부 집행기준 제2장
# ============================================================

GOV_LINES = LAW[
    "gov_guide"
]["lines"]


GOV_CH2 = extract_best_chapter(

    GOV_LINES,

    "제2장제한경쟁입찰의운용",

    "제3장"
)


assert "5" in GOV_CH2[
    "article_keys"
]


GOV_ART5 = extract_article(
    GOV_LINES,
    "5",
    start=GOV_CH2["start"],
    end=GOV_CH2["end"],
    required=True
)


# ------------------------------------------------------------
# v19 공식 항목표에는 제5조의3이라고 적혀 있음.
#
# 하지만 현재 package 실제 TXT에 없을 가능성을 허용한다.
# 절대 가짜 조문을 만들지 않는다.
# ------------------------------------------------------------

GOV_ART5_3 = extract_article(
    GOV_LINES,
    "5의3",
    start=GOV_CH2["start"],
    end=GOV_CH2["end"],
    required=False
)


# ============================================================
# 7. ANCHOR CONTEXT EXTRACTOR
# ============================================================

def extract_anchor_contexts(
    lines,
    must_any=None,
    must_all=None,
    radius=8,
    max_contexts=10
):

    must_any = (
        must_any or []
    )

    must_all = (
        must_all or []
    )

    hits = []

    for i, line in enumerate(lines):

        s = clean_line(line)

        any_ok = (
            True
            if not must_any
            else any(
                k in s
                for k in must_any
            )
        )

        all_ok = all(
            k in s
            for k in must_all
        )

        if (
            any_ok
            and
            all_ok
        ):

            hits.append(i)

    contexts = []
    seen = set()

    for i in hits:

        start = max(
            0,
            i - radius
        )

        end = min(
            len(lines),
            i + radius + 1
        )

        text = "\n".join(
            lines[start:end]
        ).strip()

        # 같은 영역 중복 제거
        signature = re.sub(
            r"\s+",
            "",
            text
        )[:500]

        if signature in seen:
            continue

        seen.add(signature)

        contexts.append({
            "hit_line":
                i,

            "start_line":
                start,

            "end_line":
                end,

            "text":
                text,
        })

        if len(contexts) >= max_contexts:
            break

    return contexts


# ------------------------------------------------------------
# v19 관련 실제 source anchor
# ------------------------------------------------------------

V19_GOV_ANCHORS = extract_anchor_contexts(

    GOV_LINES,

    must_any=[
        "물품공급",
        "물품공급ㆍ",
        "물품공급․",
        "기술지원",
        "확약서",
    ],

    radius=8,

    max_contexts=12
)


assert V19_GOV_ANCHORS, (
    "정부 집행기준에서 "
    "물품공급/기술지원 관련 문구를 전혀 찾지 못함"
)


# ------------------------------------------------------------
# 제2조의6도 현재 package에서 직접 확인
# ------------------------------------------------------------

GOV_ART2_6 = extract_article(
    GOV_LINES,
    "2의6",
    required=False
)


# ============================================================
# 8. 지방 집행기준
# ============================================================

LOC_GUIDE_LINES = LAW[
    "loc_guide"
]["lines"]


LOC_CH1 = extract_best_chapter(

    LOC_GUIDE_LINES,

    "제1장입찰및계약일반기준",

    "제2장"
)


def extract_numbered_block(
    lines,
    start,
    end,
    number,
    title_keyword
):

    start_idx = None

    pattern = re.compile(
        rf"^{number}\s*\."
    )

    for i in range(
        start,
        end
    ):

        s = clean_line(
            lines[i]
        )

        c = compact_text(s)

        if (
            pattern.match(s)
            and
            compact_text(
                title_keyword
            ) in c
        ):

            start_idx = i
            break

    assert start_idx is not None, (
        f"{number}. {title_keyword} "
        f"본문 미발견"
    )

    next_pattern = re.compile(
        rf"^{number + 1}\s*\."
    )

    finish = end

    for i in range(
        start_idx + 1,
        end
    ):

        s = clean_line(
            lines[i]
        )

        if next_pattern.match(s):

            finish = i
            break

    return {
        "start_line":
            start_idx,

        "end_line":
            finish,

        "text":
            "\n".join(
                lines[
                    start_idx:finish
                ]
            ).strip(),
    }


LOC_CAUTION = extract_numbered_block(

    LOC_GUIDE_LINES,

    LOC_CH1["start"],

    LOC_CH1["end"],

    7,

    "계약담당자 주의사항"
)


assert (
    "계약담당자"
    in LOC_CAUTION["text"]
)


# ============================================================
# 9. 일반 법령 정확 조문
# ============================================================

def get_article(
    source_key,
    article_key
):

    data = LAW[source_key]

    obj = extract_article(
        data["lines"],
        article_key,
        required=True
    )

    return {
        "source":
            data["source"],

        "locator":
            article_display(
                article_key
            ),

        "header":
            obj["header"],

        "text":
            obj["text"],
    }


LEGAL = {}


# ------------------------------------------------------------
# 국가 / 지방 계약
# ------------------------------------------------------------

LEGAL["nat_decree_12"] = get_article(
    "nat_decree",
    "12"
)

LEGAL["nat_decree_21"] = get_article(
    "nat_decree",
    "21"
)

LEGAL["nat_rule_17"] = get_article(
    "nat_rule",
    "17"
)

LEGAL["loc_decree_13"] = get_article(
    "loc_decree",
    "13"
)

LEGAL["loc_decree_20"] = get_article(
    "loc_decree",
    "20"
)

LEGAL["loc_rule_17"] = get_article(
    "loc_rule",
    "17"
)


# ------------------------------------------------------------
# 판로지원법
# ------------------------------------------------------------

LEGAL["sme_act_7"] = get_article(
    "sme_act",
    "7"
)

LEGAL["sme_act_9"] = get_article(
    "sme_act",
    "9"
)

LEGAL["sme_decree_2_2"] = get_article(
    "sme_decree",
    "2의2"
)

LEGAL["sme_decree_2_3"] = get_article(
    "sme_decree",
    "2의3"
)


# ------------------------------------------------------------
# 정부 집행기준 v9
# ------------------------------------------------------------

LEGAL["gov_ch2_art5"] = {

    "source":
        LAW["gov_guide"]["source"],

    "locator":
        "제2장 제한경쟁입찰의 운용 제5조",

    "header":
        GOV_ART5["header"],

    "text":
        GOV_ART5["text"],
}


# ------------------------------------------------------------
# 지방 주의사항
# ------------------------------------------------------------

LEGAL["local_caution"] = {

    "source":
        LAW["loc_guide"]["source"],

    "locator":
        "제1장 입찰 및 계약 일반기준 "
        "7. 계약담당자 주의사항",

    "header":
        "7. 계약담당자 주의사항",

    "text":
        LOC_CAUTION["text"],
}


# ------------------------------------------------------------
# 고시금액
# ------------------------------------------------------------

LEGAL["threshold_notice"] = {

    "source":
        LAW[
            "threshold_notice"
        ]["source"],

    "locator":
        "고시금액 고시 전체",

    "header":
        "",

    "text":
        LAW[
            "threshold_notice"
        ]["text"].strip(),
}


# ============================================================
# 10. v19 정부 근거를 안전하게 구성
# ============================================================

v19_material_parts = []


# 실제 제5조의3이 있으면 우선 보존
if GOV_ART5_3 is not None:

    v19_material_parts.append(
        {
            "kind":
                "EXACT_OFFICIAL_LOCATOR",

            "locator":
                "제2장 제한경쟁입찰의 운용 제5조의3",

            "text":
                GOV_ART5_3["text"],
        }
    )


# 현재 package에 제2조의6이 있고 관련 내용이 있으면 보존
if (
    GOV_ART2_6 is not None
    and
    (
        "물품공급"
        in GOV_ART2_6["text"]
        or
        "기술지원"
        in GOV_ART2_6["text"]
    )
):

    v19_material_parts.append(
        {
            "kind":
                "CURRENT_PACKAGE_RELEVANT_ARTICLE",

            "locator":
                GOV_ART2_6["header"],

            "text":
                GOV_ART2_6["text"],
        }
    )


# anchor 근거도 함께 보존
for ctx in V19_GOV_ANCHORS:

    v19_material_parts.append(
        {
            "kind":
                "SOURCE_ANCHOR_CONTEXT",

            "locator":
                (
                    f"line "
                    f"{ctx['start_line']}"
                    f"-"
                    f"{ctx['end_line']}"
                ),

            "text":
                ctx["text"],
        }
    )


assert v19_material_parts


LEGAL["gov_v19_material"] = {

    "source":
        LAW[
            "gov_guide"
        ]["source"],

    "official_locator":
        (
            "제2장 제한경쟁입찰의 운용 "
            "제5조의3"
        ),

    "official_locator_found":
        (
            GOV_ART5_3
            is not None
        ),

    "materials":
        v19_material_parts,
}


# ============================================================
# 11. 중기부 경쟁제품 CSV
# ============================================================

SME_CSV_PATH = (
    SME_DIR
    / "중기부고시_경쟁제품_세부품명.csv"
)


assert SME_CSV_PATH.exists(), (
    SME_CSV_PATH
)


def read_csv_flexible(path):

    last_error = None

    for enc in [
        "utf-8-sig",
        "utf-8",
        "cp949",
    ]:

        try:

            return pd.read_csv(
                path,
                dtype=str,
                encoding=enc
            )

        except Exception as e:

            last_error = e

    raise last_error


sme_products = read_csv_flexible(
    SME_CSV_PATH
)


assert len(
    sme_products
) > 0


# ------------------------------------------------------------
# 어떤 열이 세부품명번호인지
# 하드코딩하지 않고 이름을 확인
# ------------------------------------------------------------

code_columns = [

    col
    for col in sme_products.columns

    if (
        "세부품명번호"
        in str(col)
        or
        "품명번호"
        in str(col)
    )
]


assert code_columns, (
    "경쟁제품 CSV에서 "
    "세부품명번호 계열 열을 찾지 못함\n"
    f"columns={list(sme_products.columns)}"
)


SME_STRUCTURED_SOURCE = {

    "source_type":
        "STRUCTURED_CSV",

    "file":
        SME_CSV_PATH.name,

    "row_count":
        len(sme_products),

    "columns":
        list(sme_products.columns),

    "product_code_columns":
        code_columns,

    "purpose":
        (
            "meta의 세부품명번호목록과 대조하여 "
            "중소기업자간 경쟁제품 여부 판정"
        ),
}


print(
    "[3] 중기부 경쟁제품 CSV PASS:",
    SME_CSV_PATH.name
)

print(
    "    rows   =",
    len(sme_products)
)

print(
    "    code columns =",
    code_columns
)


# ============================================================
# 12. EXACT PARSER SANITY CHECK
# ============================================================

assert (
    LEGAL[
        "sme_act_7"
    ]["header"]
    .replace(" ", "")
    .startswith("제7조(")
)

assert (
    LEGAL[
        "sme_act_9"
    ]["header"]
    .replace(" ", "")
    .startswith("제9조(")
)

assert (
    article_key_from_line(
        LEGAL[
            "sme_decree_2_2"
        ]["header"]
    )
    ==
    "2의2"
)

assert (
    article_key_from_line(
        LEGAL[
            "sme_decree_2_3"
        ]["header"]
    )
    ==
    "2의3"
)

assert (
    article_key_from_line(
        LEGAL[
            "gov_ch2_art5"
        ]["header"]
    )
    ==
    "5"
)


print("[4] exact article parser PASS")


# ============================================================
# 13. OFFICIAL LOCATOR / SOURCE AUDIT
# ============================================================

print()
print("=" * 100)
print("v19 GOVERNMENT GUIDE LOCATOR CHECK")
print("=" * 100)

print(
    "공식 locator = "
    "제2장 제한경쟁입찰의 운용 제5조의3"
)

print(
    "실제 literal heading 발견 =",
    (
        GOV_ART5_3
        is not None
    )
)

print(
    "정부집행기준 제2장 article keys =",
    GOV_CH2[
        "article_keys"
    ]
)

print(
    "v19 관련 anchor contexts =",
    len(V19_GOV_ANCHORS)
)


if GOV_ART5_3 is None:

    print()
    print(
        "[WARN] 공식 항목표에는 제5조의3이 적혀 있으나 "
        "현재 배포 TXT의 제2장에서는 literal heading이 "
        "확인되지 않았습니다."
    )

    print(
        "가짜 조문을 생성하지 않고, "
        "동일 공식 source의 실제 물품공급·기술지원 "
        "관련 문구를 grounding으로 보존합니다."
    )


# ============================================================
# 14. LEGAL KEY MAPPING
# ============================================================

LEGAL_KEYS = {

    "v9": [
        "gov_ch2_art5",
        "local_caution",
    ],

    "v10": [
        "sme_act_9",
    ],

    "v11": [
        "nat_decree_21",
        "loc_decree_20",
        "sme_act_7",
    ],

    "v12": [
        "nat_decree_21",
        "loc_decree_20",
    ],

    "v13": [
        "nat_decree_21",
        "loc_decree_20",
        "sme_act_7",
    ],

    "v14": [
        "nat_decree_21",
        "loc_decree_20",
        "sme_decree_2_2",
        "sme_decree_2_3",
        "threshold_notice",
    ],

    "v15": [
        "nat_decree_21",
        "loc_decree_20",
        "sme_decree_2_2",
        "sme_decree_2_3",
        "threshold_notice",
    ],

    "v16": [
        "nat_decree_21",
        "loc_decree_20",
        "sme_decree_2_2",
        "sme_decree_2_3",
        "threshold_notice",
    ],

    "v17": [
        "nat_decree_21",
        "loc_decree_20",
        "sme_decree_2_2",
        "sme_decree_2_3",
    ],

    "v18": [
        "nat_decree_21",
        "loc_decree_20",
        "sme_decree_2_2",
        "sme_decree_2_3",
    ],

    "v19": [
        "nat_decree_12",
        "nat_rule_17",
        "gov_v19_material",
        "loc_decree_13",
        "loc_rule_17",
        "local_caution",
    ],
}


STRUCTURED_ITEMS = {
    "v10",
    "v11",
    "v12",
    "v13",
    "v14",
    "v15",
    "v16",
    "v17",
    "v18",
}


# ============================================================
# 15. COMMON SPEC
# ============================================================

COMMON = {

    "document_scope": (
        "공고문만 사용하지 않는다. 제공된 공고문, 규격서, "
        "과업지시서, 제안요청서, 예외공표서 전체를 판정대상으로 한다."
    ),

    "positive_presence_policy": (
        "비부재탐지 항목은 실제 제공 문서에서 위반조건이 "
        "확인될 때만 1 후보로 판단한다."
    ),

    "incomplete_input_handling": (
        "dropped_doc_counts가 있거나 완전관측=False인 경우 "
        "누락 문서의 내용을 추측하지 않는다. "
        "현재 보이는 문서에서 위반이 명시되면 위반 판정은 가능하지만, "
        "부재탐지 항목에서 단순히 문구가 안 보인다는 이유만으로 "
        "즉시 위반 확정하지 않는다."
    ),

    "evidence_rule": (
        "비부재탐지 항목의 evidence는 제공 docs에 실제 존재하는 "
        "연속 부분문자열이어야 한다. 요약·변형 금지, 최대 500자. "
        "부재탐지 v10/v11/v16/v18은 evidence를 빈 문자열로 둔다."
    ),
}


# ============================================================
# 16. DECISION SPECS
# ============================================================

FINAL = {}


# ------------------------------------------------------------
# v9
# ------------------------------------------------------------

FINAL["v9"] = {

    "status":
        "REVIEWED",

    "applicability":
        (
            "공고문·규격서·과업지시서·제안요청서 등에 "
            "제품의 제조사, 브랜드, 모델명, 고유 규격 등이 "
            "기재된 물품 또는 관련 계약."
        ),

    "required_meta": [
        "업무구분",
        "계약방법",
    ],

    "facts_to_extract": [
        "제조사명",
        "브랜드명",
        "모델명",
        "제품번호",
        "동등 이상 허용 여부",
        "특별한 필요사유 여부",
    ],

    "violation_condition":
        (
            "특별한 정당화 없이 특정 제조사·상표·모델·"
            "고유 제품을 사실상 지정하고, 동등 이상의 제품을 "
            "허용하지 않아 경쟁을 제한하는 경우."
        ),

    "nonviolation_condition":
        (
            "참고모델 또는 예시이거나 동등 이상 제품을 "
            "실질적으로 허용하거나, 특별한 사유가 공식 자료상 "
            "확인되는 경우."
        ),

    "python_role": [
        "모델번호 형태 후보 추출",
        "브랜드/제조사 후보 추출",
        "동등 이상 허용 표현 후보 추출",
        "evidence exact-substring 검증",
    ],

    "gemma_role": [
        "특정 제품의 강제 지정인지 판단",
        "참고·예시인지 실제 참가/납품 제한인지 판단",
        "특별사유 존재 여부 판단",
    ],

    "unresolved": [],
}


# ------------------------------------------------------------
# v10
# ------------------------------------------------------------

FINAL["v10"] = {

    "status":
        "REVIEWED_WITH_STRUCTURED_LOOKUP",

    "applicability":
        (
            "meta의 세부품명번호가 배포된 중기부 경쟁제품 "
            "CSV상 중소기업자간 경쟁제품에 해당하는 물품 입찰."
        ),

    "required_meta": [
        "업무구분",
        "세부품명번호목록",
    ],

    "facts_to_extract": [
        "중기간 경쟁제품 여부",
        "직접생산확인증명서 또는 동등 직접생산 자격 요구 여부",
    ],

    "violation_condition":
        (
            "중소기업자간 경쟁제품인데 필요한 직접생산 "
            "확인 자격요건이 공고 전체에서 확인되지 않는 경우."
        ),

    "nonviolation_condition":
        (
            "경쟁제품이 아니거나 필요한 직접생산 확인 "
            "요건이 공고에 실제로 존재하는 경우."
        ),

    "python_role": [
        "세부품명번호 → 경쟁제품 CSV exact lookup",
        "직접생산 관련 표현 후보 탐색",
        "input completeness 제공",
    ],

    "gemma_role": [
        "직생 관련 문구가 실제 참가자격 요건인지 확인",
        "표현 변형의 의미 동일성 판단",
    ],

    "evidence":
        "EMPTY",

    "unresolved": [
        "부재탐지이므로 문서가 탈락된 경우 단순 absence로 확정하지 않음",
    ],
}


# ------------------------------------------------------------
# v11
# ------------------------------------------------------------

FINAL["v11"] = {

    "status":
        "REVIEWED_WITH_STRUCTURED_LOOKUP",

    "applicability":
        (
            "배포 경쟁제품 CSV상 중소기업자간 경쟁제품에 "
            "해당하는 물품 입찰."
        ),

    "required_meta": [
        "세부품명번호목록",
        "조항호내용",
    ],

    "facts_to_extract": [
        "경쟁제품 여부",
        "중소기업자 참가제한 여부",
    ],

    "violation_condition":
        (
            "중소기업자간 경쟁제품인데 중소기업자 "
            "참가제한이 공고에서 확인되지 않는 경우."
        ),

    "nonviolation_condition":
        (
            "경쟁제품이 아니거나 중소기업자 참가제한이 "
            "공고에 실제 존재하는 경우."
        ),

    "python_role": [
        "경쟁제품 CSV lookup",
        "중소기업/중기업/소기업/소상공인 표현 후보 추출",
        "완전관측 여부 제공",
    ],

    "gemma_role": [
        "기업규모 문구가 실제 입찰참가자격인지 판단",
        "법적 예외 문맥 판단",
    ],

    "evidence":
        "EMPTY",

    "unresolved": [
        "예외 사유가 문서에 있으면 법령 context와 함께 판단",
    ],
}


# ------------------------------------------------------------
# v12
# ------------------------------------------------------------

FINAL["v12"] = {

    "status":
        "REVIEWED_WITH_STRUCTURED_LOOKUP",

    "applicability":
        (
            "배포 경쟁제품 CSV에 해당하지 않는 일반제품."
        ),

    "required_meta": [
        "세부품명번호목록",
    ],

    "facts_to_extract": [
        "경쟁제품 여부",
        "직접생산확인증명서 요구 여부",
        "해당 요구가 참가자격인지 여부",
    ],

    "violation_condition":
        (
            "일반제품임에도 직접생산확인증명서 등을 "
            "입찰참가자격으로 요구하여 참가자를 제한하는 경우."
        ),

    "nonviolation_condition":
        (
            "경쟁제품이거나 직접생산 관련 문구가 "
            "참가자격 제한이 아닌 경우."
        ),

    "python_role": [
        "경쟁제품 CSV lookup",
        "직접생산 표현 후보 추출",
        "evidence 검증",
    ],

    "gemma_role": [
        "직접생산 문구의 실제 기능이 참가자격 제한인지 판단",
    ],

    "unresolved": [],
}


# ------------------------------------------------------------
# v13
# ------------------------------------------------------------

FINAL["v13"] = {

    "status":
        "REVIEWED_WITH_STRUCTURED_LOOKUP",

    "applicability":
        (
            "중소기업자간 경쟁제품으로 지정된 물품 입찰."
        ),

    "required_meta": [
        "세부품명번호목록",
        "조항호내용",
    ],

    "facts_to_extract": [
        "경쟁제품 여부",
        "기업규모 제한",
        "소기업·소상공인 한정 여부",
    ],

    "violation_condition":
        (
            "중소기업자간 경쟁제품의 참가범위를 "
            "중소기업자 전체가 아니라 소기업·소상공인으로 "
            "부당하게 축소한 경우."
        ),

    "nonviolation_condition":
        (
            "중소기업자 범위로 제한하거나 공식 예외가 "
            "적용되는 경우."
        ),

    "python_role": [
        "경쟁제품 CSV lookup",
        "기업규모 표현 구조화",
        "evidence 검증",
    ],

    "gemma_role": [
        "중소기업 제한과 소기업·소상공인 한정을 구분",
        "예외 적용 여부 판단",
    ],

    "unresolved": [
        "예외가 존재하는 경우 제7조 및 관련 package context와 함께 판단",
    ],
}


# ------------------------------------------------------------
# v14
# ------------------------------------------------------------

FINAL["v14"] = {

    "status":
        "REVIEWED_WITH_RUNTIME_THRESHOLD",

    "applicability":
        (
            "중기간 경쟁제품이 아닌 일반물품이며 "
            "추정가격이 적용 고시금액 이상인 경우."
        ),

    "required_meta": [
        "입찰추정가격",
        "세부품명번호목록",
        "조항호내용",
        "적용계약법",
    ],

    "facts_to_extract": [
        "일반제품 여부",
        "추정가격",
        "적용 고시금액",
        "중소기업 제한 여부",
    ],

    "violation_condition":
        (
            "일반물품이고 고시금액 이상인데 "
            "입찰참가자를 중소기업자로 제한한 경우."
        ),

    "nonviolation_condition":
        (
            "경쟁제품이거나 중소기업 제한이 없거나 "
            "법령상 예외가 적용되는 경우."
        ),

    "python_role": [
        "경쟁제품 CSV lookup",
        "고시금액 source를 이용한 금액구간 판정",
        "기업규모 표현 추출",
        "evidence 검증",
    ],

    "gemma_role": [
        "기업규모 문장이 실제 참가제한인지 판단",
        "제2조의3 예외 적용 여부 판단",
    ],

    "unresolved": [
        "고시금액은 전역 숫자로 고정하지 않고 package 기준으로 판정",
    ],
}


# ------------------------------------------------------------
# v15
# ------------------------------------------------------------

FINAL["v15"] = {

    "status":
        "REVIEWED_WITH_EXCEPTION_CHECK",

    "applicability":
        (
            "일반물품이고 추정가격이 1억원 이상 "
            "고시금액 미만인 경우."
        ),

    "required_meta": [
        "입찰추정가격",
        "세부품명번호목록",
        "조항호내용",
    ],

    "facts_to_extract": [
        "일반제품 여부",
        "금액구간",
        "소기업·소상공인 한정 여부",
        "예외사유 여부",
    ],

    "violation_condition":
        (
            "1억원 이상 고시금액 미만 일반물품에서 "
            "중소기업자 전체가 아니라 소기업·소상공인만으로 "
            "참가범위를 제한한 경우."
        ),

    "nonviolation_condition":
        (
            "중소기업자 범위로 적정하게 제한하거나 "
            "제2조의3 등 공식 예외가 적용되는 경우."
        ),

    "python_role": [
        "금액구간 판정",
        "경쟁제품 제외",
        "기업규모 표현 추출",
        "evidence 검증",
    ],

    "gemma_role": [
        "중소기업 vs 소기업·소상공인 제한 구분",
        "예외 사유 판단",
    ],

    "unresolved": [
        "제2조의3 예외는 반드시 별도 feature로 보존",
    ],
}


# ------------------------------------------------------------
# v16
# ------------------------------------------------------------

FINAL["v16"] = {

    "status":
        "REVIEWED_WITH_EXCEPTION_CHECK",

    "applicability":
        (
            "일반물품이고 추정가격이 1억원 이상 "
            "고시금액 미만인 경우."
        ),

    "required_meta": [
        "입찰추정가격",
        "세부품명번호목록",
        "조항호내용",
    ],

    "facts_to_extract": [
        "일반제품 여부",
        "금액구간",
        "중소기업자 제한 존재 여부",
        "예외사유 여부",
    ],

    "violation_condition":
        (
            "해당 금액구간의 일반물품인데 필요한 "
            "중소기업자 참가제한이 공고에서 확인되지 않고, "
            "공식 예외도 확인되지 않는 경우."
        ),

    "nonviolation_condition":
        (
            "중소기업자 제한이 존재하거나 법령상 예외가 "
            "확인되는 경우."
        ),

    "python_role": [
        "금액구간 판정",
        "경쟁제품 제외",
        "중소기업 제한 표현 후보 탐색",
        "완전관측 정보 제공",
    ],

    "gemma_role": [
        "실질적인 중소기업자 제한 존재 여부 판단",
        "예외 사유 판단",
    ],

    "evidence":
        "EMPTY",

    "unresolved": [
        "부재탐지이므로 incomplete docs에서 단순 문구부재=위반 금지",
    ],
}


# ------------------------------------------------------------
# v17
# ------------------------------------------------------------

FINAL["v17"] = {

    "status":
        "REVIEWED_WITH_EXCEPTION_CHECK",

    "applicability":
        (
            "일반물품이고 추정가격이 1억원 미만인 경우."
        ),

    "required_meta": [
        "입찰추정가격",
        "세부품명번호목록",
        "조항호내용",
    ],

    "facts_to_extract": [
        "일반제품 여부",
        "중소기업 전체 제한 여부",
        "소기업·소상공인 제한 여부",
        "예외사유 여부",
    ],

    "violation_condition":
        (
            "1억원 미만 일반물품에서 참가범위를 "
            "소기업·소상공인보다 넓은 중소기업자 전체로 "
            "제한한 경우."
        ),

    "nonviolation_condition":
        (
            "소기업·소상공인으로 적정하게 제한했거나 "
            "공식 예외가 적용되는 경우."
        ),

    "python_role": [
        "1억원 미만 판정",
        "경쟁제품 제외",
        "기업규모 표현 구조화",
        "evidence 검증",
    ],

    "gemma_role": [
        "중소기업 전체와 소기업·소상공인 제한 구분",
        "예외 적용 여부 판단",
    ],

    "unresolved": [
        "제2조의3 예외 적용 여부를 별도 판단",
    ],
}


# ------------------------------------------------------------
# v18
# ------------------------------------------------------------

FINAL["v18"] = {

    "status":
        "REVIEWED_WITH_EXCEPTION_CHECK",

    "applicability":
        (
            "일반물품이고 추정가격이 1억원 미만인 경우."
        ),

    "required_meta": [
        "입찰추정가격",
        "세부품명번호목록",
        "조항호내용",
    ],

    "facts_to_extract": [
        "일반제품 여부",
        "소기업·소상공인 제한 존재 여부",
        "예외사유 여부",
    ],

    "violation_condition":
        (
            "해당 금액구간에서 필요한 소기업·소상공인 "
            "참가제한이 공고에서 확인되지 않고 "
            "공식 예외도 확인되지 않는 경우."
        ),

    "nonviolation_condition":
        (
            "소기업·소상공인 제한이 존재하거나 "
            "공식 예외가 적용되는 경우."
        ),

    "python_role": [
        "1억원 미만 판정",
        "경쟁제품 제외",
        "소기업/소상공인 표현 후보 탐색",
        "완전관측 정보 제공",
    ],

    "gemma_role": [
        "실질적인 소기업·소상공인 참가제한 여부 판단",
        "예외 적용 여부 판단",
    ],

    "evidence":
        "EMPTY",

    "unresolved": [
        "부재탐지 + 예외조항이므로 키워드 부재만으로 위반 확정 금지",
    ],
}


# ------------------------------------------------------------
# v19
# ------------------------------------------------------------

v19_unresolved = []


if GOV_ART5_3 is None:

    v19_unresolved.append(
        (
            "공식 항목표의 정부 집행기준 "
            "제2장 제5조의3 literal heading이 "
            "현재 배포 TXT에서 확인되지 않음. "
            "동일 source의 실제 물품공급·기술지원 "
            "관련 조문/anchor를 근거로 보존."
        )
    )


FINAL["v19"] = {

    "status":
        (
            "REVIEWED_WITH_SOURCE_LOCATOR_DRIFT"
            if GOV_ART5_3 is None
            else
            "REVIEWED"
        ),

    "applicability":
        (
            "물품공급·기술지원 확약서, 협약서, "
            "공급확인서 등의 제출을 요구하는 공고."
        ),

    "required_meta": [
        "업무구분",
        "계약방법",
    ],

    "facts_to_extract": [
        "요구 문서명",
        "발급 주체",
        "제출 시점",
        "입찰참가자격 단계인지 여부",
        "계약체결/낙찰 후 단계인지 여부",
    ],

    "violation_condition":
        (
            "물품공급·기술지원 관련 확약서나 협약서를 "
            "입찰 단계의 참가자격 서류로 제출하도록 요구하는 경우."
        ),

    "nonviolation_condition":
        (
            "낙찰 후 또는 계약체결 단계에서 제출하게 하거나, "
            "발주기관이 필요한 사전 협약을 체결하는 구조이거나, "
            "해당 서류를 입찰참가자격으로 요구하지 않는 경우."
        ),

    "python_role": [
        "확약서/협약서/공급확인서 표현 후보 추출",
        "입찰시/투찰시/서류제출시/계약시/낙찰후 등의 시점표현 추출",
        "evidence exact-substring 검증",
    ],

    "gemma_role": [
        "서류가 실제 참가자격 제출서류인지 판단",
        "입찰단계와 낙찰·계약단계를 구분",
        "물품공급 확약서와 일반 제품증빙을 구분",
    ],

    "unresolved":
        v19_unresolved,
}


# ============================================================
# 17. DEV CASE ACCESSOR
#
# Stage 6-1 / 6-3A 과정에서 만들어진 artifact의
# 실제 구조 차이를 안전하게 처리한다.
#
# 절대:
#   없는 key -> [] 로 조용히 처리하지 않는다.
#
# 반드시:
#   실제 양성 사례가 있는 구조를 찾아 검증한다.
# ============================================================

def extract_dev_cases(src, item):

    """
    지원하는 구조:

    A)
    {
        "dev_positives": [...],
        "dev_negative_candidates": [...]
    }

    B)
    {
        "dev": {
            "positives": [...],
            "negative_candidates": [...]
        }
    }

    C)
    {
        "dev": {
            "positive_cases": [...],
            "negative_candidates": [...]
        }
    }

    D)
    {
        "positives": [...],
        "negative_candidates": [...]
    }

    존재하지 않는 구조를 빈 리스트로 숨기지 않는다.
    """

    positives = None
    negatives = None
    positive_source = None
    negative_source = None


    # --------------------------------------------------------
    # A. top-level dev_positives
    # --------------------------------------------------------

    if (
        "dev_positives" in src
        and
        isinstance(
            src["dev_positives"],
            list
        )
    ):

        positives = src[
            "dev_positives"
        ]

        positive_source = (
            "top-level.dev_positives"
        )


    if (
        "dev_negative_candidates"
        in src
        and
        isinstance(
            src[
                "dev_negative_candidates"
            ],
            list
        )
    ):

        negatives = src[
            "dev_negative_candidates"
        ]

        negative_source = (
            "top-level."
            "dev_negative_candidates"
        )


    # --------------------------------------------------------
    # B/C. nested dev
    # --------------------------------------------------------

    dev = src.get(
        "dev"
    )

    if isinstance(
        dev,
        dict
    ):

        if positives is None:

            for key in [
                "positives",
                "positive_cases",
                "dev_positives",
            ]:

                if (
                    key in dev
                    and
                    isinstance(
                        dev[key],
                        list
                    )
                ):

                    positives = dev[key]

                    positive_source = (
                        f"dev.{key}"
                    )

                    break


        if negatives is None:

            for key in [
                "negative_candidates",
                "negatives",
                "dev_negative_candidates",
            ]:

                if (
                    key in dev
                    and
                    isinstance(
                        dev[key],
                        list
                    )
                ):

                    negatives = dev[key]

                    negative_source = (
                        f"dev.{key}"
                    )

                    break


    # --------------------------------------------------------
    # D. direct top-level
    # --------------------------------------------------------

    if positives is None:

        if (
            "positives" in src
            and
            isinstance(
                src["positives"],
                list
            )
        ):

            positives = src[
                "positives"
            ]

            positive_source = (
                "top-level.positives"
            )


    if negatives is None:

        if (
            "negative_candidates"
            in src
            and
            isinstance(
                src[
                    "negative_candidates"
                ],
                list
            )
        ):

            negatives = src[
                "negative_candidates"
            ]

            negative_source = (
                "top-level."
                "negative_candidates"
            )


    # --------------------------------------------------------
    # 없으면 조용히 [] 처리하지 말고 즉시 실패
    # --------------------------------------------------------

    if positives is None:

        raise AssertionError(
            "\n"
            f"{item}: positive case 구조를 찾지 못했습니다.\n"
            f"top-level keys = "
            f"{list(src.keys())}"
        )


    if negatives is None:

        raise AssertionError(
            "\n"
            f"{item}: negative candidate 구조를 찾지 못했습니다.\n"
            f"top-level keys = "
            f"{list(src.keys())}"
        )


    # --------------------------------------------------------
    # 실제 case 구조 검증
    # --------------------------------------------------------

    assert isinstance(
        positives,
        list
    )

    assert isinstance(
        negatives,
        list
    )


    for case in positives:

        assert isinstance(
            case,
            dict
        )

        assert "id" in case, (
            f"{item}: positive에 id 없음"
        )


    for case in negatives:

        assert isinstance(
            case,
            dict
        )

        assert "id" in case, (
            f"{item}: negative candidate에 id 없음"
        )


    return {
        "positives":
            positives,

        "negatives":
            negatives,

        "positive_source":
            positive_source,

        "negative_source":
            negative_source,
    }


# ============================================================
# 18. DEV SOURCE 구조 실제 검증
#
# 여기서 v9~v19 양성 개수가
# 우리가 기존에 확인한 값과 정확히 같은지 확인한다.
# ============================================================

EXPECTED_POSITIVE_COUNTS = {

    "v9": 6,
    "v10": 7,
    "v11": 6,
    "v12": 6,
    "v13": 6,
    "v14": 8,
    "v15": 6,
    "v16": 6,
    "v17": 6,
    "v18": 7,
    "v19": 6,
}


DEV_CASES = {}


dev_audit_rows = []


for v in TARGET_ITEMS:

    extracted = extract_dev_cases(
        source_pack[v],
        v
    )

    DEV_CASES[v] = extracted

    pos_count = len(
        extracted[
            "positives"
        ]
    )

    neg_count = len(
        extracted[
            "negatives"
        ]
    )


    assert (
        pos_count
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    ), (
        f"\n{v} positive count mismatch\n"
        f"expected="
        f"{EXPECTED_POSITIVE_COUNTS[v]}\n"
        f"actual={pos_count}\n"
        f"source="
        f"{extracted['positive_source']}"
    )


    dev_audit_rows.append({

        "item":
            v,

        "positive":
            pos_count,

        "negative_candidates":
            neg_count,

        "positive_source":
            extracted[
                "positive_source"
            ],

        "negative_source":
            extracted[
                "negative_source"
            ],
    })


dev_structure_audit = pd.DataFrame(
    dev_audit_rows
)


display(
    dev_structure_audit
)


print()
print("=" * 100)
print("DEV CASE STRUCTURE CHECK")
print("=" * 100)

print(
    "TOTAL POSITIVES =",
    sum(
        len(
            DEV_CASES[v][
                "positives"
            ]
        )
        for v
        in TARGET_ITEMS
    )
)

print(
    "EXPECTED TOTAL =",
    sum(
        EXPECTED_POSITIVE_COUNTS.values()
    )
)

print(
    "[PASS] v9~v19 DEV 사례 구조 확인 완료"
)


# ============================================================
# 19. FINAL PACK BUILD
# ============================================================

EXPECTED_ABSENCE = {
    "v10",
    "v11",
    "v16",
    "v18",
}


FINAL_PACK = {}


for v in TARGET_ITEMS:

    src = source_pack[v]

    absence = bool(
        src[
            "absence_detection"
        ]
    )


    # --------------------------------------------------------
    # 부재탐지 정확성
    # --------------------------------------------------------

    assert (
        absence
        ==
        (v in EXPECTED_ABSENCE)
    ), (
        f"{v}: absence mismatch"
    )


    # --------------------------------------------------------
    # exact 법령 블록
    # --------------------------------------------------------

    exact_legal = {

        key:
            LEGAL[key]

        for key
        in LEGAL_KEYS[v]
    }


    # --------------------------------------------------------
    # DEV 사례
    # --------------------------------------------------------

    positive_cases = (
        DEV_CASES[v][
            "positives"
        ]
    )

    negative_cases = (
        DEV_CASES[v][
            "negatives"
        ]
    )


    # --------------------------------------------------------
    # official name
    # --------------------------------------------------------

    item_name = src.get(
        "name"
    )

    if not item_name:

        item_name = safe_str(
            basis_df.loc[
                v,
                "name"
            ]
        )


    assert item_name, (
        f"{v}: item name 없음"
    )


    # --------------------------------------------------------
    # build
    # --------------------------------------------------------

    FINAL_PACK[v] = {

        "item":
            v,

        "name":
            item_name,

        "absence_detection":
            absence,

        "official_basis": {

            "national":
                safe_str(
                    basis_df.loc[
                        v,
                        "national"
                    ]
                ),

            "local":
                safe_str(
                    basis_df.loc[
                        v,
                        "local"
                    ]
                ),

            "note":
                safe_str(
                    basis_df.loc[
                        v,
                        "note"
                    ]
                ),
        },

        **COMMON,

        "decision_spec":
            FINAL[v],

        "exact_legal_keys":
            LEGAL_KEYS[v],

        "exact_legal_text":
            exact_legal,

        "structured_sources":
            (
                [
                    SME_STRUCTURED_SOURCE
                ]
                if v
                in STRUCTURED_ITEMS
                else []
            ),

        "dev": {

            "positive_count":
                len(
                    positive_cases
                ),

            "positive_ids": [
                x["id"]
                for x
                in positive_cases
            ],

            "positive_evidence_examples": [
                x.get(
                    "evidence"
                )
                for x
                in positive_cases
                if x.get(
                    "evidence"
                )
            ],

            "negative_candidate_count":
                len(
                    negative_cases
                ),

            "negative_candidate_ids": [
                x["id"]
                for x
                in negative_cases
            ],

            "negative_candidate_semantics":
                "STRUCTURAL_CANDIDATES_ONLY",

            "source_schema": {
                "positives":
                    DEV_CASES[v][
                        "positive_source"
                    ],

                "negatives":
                    DEV_CASES[v][
                        "negative_source"
                    ],
            },
        },
    }


# ============================================================
# 20. FINAL VALIDATION
# ============================================================

assert (
    list(
        FINAL_PACK.keys()
    )
    ==
    TARGET_ITEMS
)


# ------------------------------------------------------------
# expected positives
# ------------------------------------------------------------

for v in TARGET_ITEMS:

    x = FINAL_PACK[v]

    expected_positive = (
        EXPECTED_POSITIVE_COUNTS[v]
    )


    assert (
        x[
            "dev"
        ][
            "positive_count"
        ]
        ==
        expected_positive
    ), (
        f"{v}: final positive mismatch"
    )


    assert (
        x[
            "decision_spec"
        ][
            "status"
        ]
        !=
        "PENDING_FINAL_REVIEW"
    )


    assert (
        x[
            "absence_detection"
        ]
        ==
        (
            v
            in EXPECTED_ABSENCE
        )
    )


    assert (
        len(
            x[
                "exact_legal_keys"
            ]
        )
        > 0
    )


    for key in x[
        "exact_legal_keys"
    ]:

        assert key in LEGAL, (
            f"{v}: 없는 LEGAL key {key}"
        )


# ------------------------------------------------------------
# 전체 양성 수 검증
# ------------------------------------------------------------

EXPECTED_TOTAL_POSITIVES = sum(
    EXPECTED_POSITIVE_COUNTS.values()
)

ACTUAL_TOTAL_POSITIVES = sum(

    x[
        "dev"
    ][
        "positive_count"
    ]

    for x
    in FINAL_PACK.values()
)


assert (
    ACTUAL_TOTAL_POSITIVES
    ==
    EXPECTED_TOTAL_POSITIVES
), (
    f"total positive mismatch: "
    f"{ACTUAL_TOTAL_POSITIVES} "
    f"!= "
    f"{EXPECTED_TOTAL_POSITIVES}"
)


# ------------------------------------------------------------
# 중기간 structured source
# ------------------------------------------------------------

for v in STRUCTURED_ITEMS:

    assert (
        len(
            FINAL_PACK[v][
                "structured_sources"
            ]
        )
        == 1
    ), (
        f"{v}: SME structured source 없음"
    )


# ------------------------------------------------------------
# 부재탐지 evidence 정책
# ------------------------------------------------------------

for v in EXPECTED_ABSENCE:

    assert (
        FINAL_PACK[v][
            "decision_spec"
        ].get(
            "evidence"
        )
        ==
        "EMPTY"
    ), (
        f"{v}: absence evidence policy 오류"
    )


# ------------------------------------------------------------
# v19 근거 실제 존재 검증
# ------------------------------------------------------------

v19_blob = json.dumps(
    FINAL_PACK[
        "v19"
    ][
        "exact_legal_text"
    ],
    ensure_ascii=False
)


assert (
    "물품공급"
    in v19_blob
    or
    "기술지원"
    in v19_blob
    or
    "확약서"
    in v19_blob
), (
    "v19 관련 실제 source context 없음"
)


print()
print(
    "[PASS] FINAL_PACK 구조 검증 완료"
)


# ============================================================
# 21. SAVE
# ============================================================

with open(
    OUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_PACK,
        f,
        ensure_ascii=False,
        indent=2
    )


assert OUT_PATH.exists()


# 저장 후 다시 읽어서 검증
with open(
    OUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    saved = json.load(f)


assert (
    list(
        saved.keys()
    )
    ==
    TARGET_ITEMS
)


for v in TARGET_ITEMS:

    assert (
        saved[v][
            "dev"
        ][
            "positive_count"
        ]
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    )


print(
    "[PASS] JSON SAVE / RELOAD 검증 완료"
)


# ============================================================
# 22. AUDIT TABLE
# ============================================================

rows = []


for v in TARGET_ITEMS:

    x = FINAL_PACK[v]

    rows.append({

        "item":
            v,

        "name":
            x["name"],

        "absence":
            x[
                "absence_detection"
            ],

        "status":
            x[
                "decision_spec"
            ][
                "status"
            ],

        "legal_blocks":
            len(
                x[
                    "exact_legal_keys"
                ]
            ),

        "structured_sources":
            len(
                x[
                    "structured_sources"
                ]
            ),

        "positives":
            x[
                "dev"
            ][
                "positive_count"
            ],

        "negative_candidates":
            x[
                "dev"
            ][
                "negative_candidate_count"
            ],

        "evidence_examples":
            len(
                x[
                    "dev"
                ][
                    "positive_evidence_examples"
                ]
            ),

        "unresolved":
            len(
                x[
                    "decision_spec"
                ][
                    "unresolved"
                ]
            ),
    })


audit_63c = pd.DataFrame(
    rows
)


display(
    audit_63c
)


# ============================================================
# 23. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 6-3C FINAL RESULT")
print("=" * 100)


print(
    "ITEMS                =",
    len(
        FINAL_PACK
    )
)


print(
    "TARGET               =",
    TARGET_ITEMS
)


print(
    "ABSENCE ITEMS        =",
    sorted(
        v
        for v, x
        in FINAL_PACK.items()
        if x[
            "absence_detection"
        ]
    )
)


print(
    "TOTAL POSITIVES      =",
    ACTUAL_TOTAL_POSITIVES
)


print(
    "EXPECTED POSITIVES   =",
    EXPECTED_TOTAL_POSITIVES
)


print(
    "SME CSV              =",
    SME_CSV_PATH.name
)


print(
    "SME CSV ROWS         =",
    len(
        sme_products
    )
)


print(
    "SME CODE COLUMNS     =",
    code_columns
)


print(
    "V19 제5조의3 FOUND   =",
    (
        GOV_ART5_3
        is not None
    )
)


print(
    "V19 ANCHOR CONTEXTS  =",
    len(
        V19_GOV_ANCHORS
    )
)


print(
    "PENDING SPECS        =",
    [
        v

        for v, x
        in FINAL_PACK.items()

        if x[
            "decision_spec"
        ][
            "status"
        ]
        ==
        "PENDING_FINAL_REVIEW"
    ]
)


print(
    "OUTPUT               =",
    OUT_PATH
)


print()


if (
    GOV_ART5_3
    is None
):

    print(
        "[PASS/WARN] Stage 6-3C v9~v19 완료"
    )

    print(
        "v19 공식 locator 제5조의3은 "
        "현재 package TXT에서 literal heading 미검출."
    )

    print(
        "해당 차이는 v19 unresolved에 명시적으로 보존됨."
    )

else:

    print(
        "[PASS] Stage 6-3C v9~v19 "
        "final judgment specification 완료"
    )

[1] SOURCE PACK / BASIS LOAD PASS
[2] 정확 법령 파일 연결 PASS
[3] 중기부 경쟁제품 CSV PASS: 중기부고시_경쟁제품_세부품명.csv
    rows   = 616
    code columns = ['제품명번호', '세부품명번호']
[4] exact article parser PASS

v19 GOVERNMENT GUIDE LOCATOR CHECK
공식 locator = 제2장 제한경쟁입찰의 운용 제5조의3
실제 literal heading 발견 = True
정부집행기준 제2장 article keys = ['3', '4', '5', '5의2', '5의3']
v19 관련 anchor contexts = 11


,item,positive,negative_candidates,positive_source,negative_source
0,v9,6,12,dev.positives,dev.negative_candidates
1,v10,7,12,dev.positives,dev.negative_candidates
2,v11,6,12,dev.positives,dev.negative_candidates
3,v12,6,12,dev.positives,dev.negative_candidates
4,v13,6,12,dev.positives,dev.negative_candidates
5,v14,8,12,dev.positives,dev.negative_candidates
6,v15,6,12,dev.positives,dev.negative_candidates
7,v16,6,12,dev.positives,dev.negative_candidates
8,v17,6,12,dev.positives,dev.negative_candidates
9,v18,7,12,dev.positives,dev.negative_candidates



DEV CASE STRUCTURE CHECK
TOTAL POSITIVES = 70
EXPECTED TOTAL = 70
[PASS] v9~v19 DEV 사례 구조 확인 완료

[PASS] FINAL_PACK 구조 검증 완료
[PASS] JSON SAVE / RELOAD 검증 완료


,item,name,absence,status,legal_blocks,structured_sources,positives,negative_candidates,evidence_examples,unresolved
0,v9,과업지시서 특정 모델명 명시,False,REVIEWED,2,0,6,12,3,0
1,v10,중기간 경쟁제품 입찰 직생 없음,True,REVIEWED_WITH_STRUCTURED_LOOKUP,1,1,7,12,0,1
2,v11,중기간 경쟁제품 입찰 중소 없음,True,REVIEWED_WITH_STRUCTURED_LOOKUP,3,1,6,12,0,1
3,v12,일반제품 직생 제한,False,REVIEWED_WITH_STRUCTURED_LOOKUP,2,1,6,12,1,0
4,v13,"중기간 경쟁제품 소기업, 소상공인 제한",False,REVIEWED_WITH_STRUCTURED_LOOKUP,3,1,6,12,1,1
5,v14,고시금액 이상 일반물품 중소기업 제한,False,REVIEWED_WITH_RUNTIME_THRESHOLD,5,1,8,12,1,1
6,v15,1억 이상- 고시금액미만 소기업 제한,False,REVIEWED_WITH_EXCEPTION_CHECK,5,1,6,12,2,1
7,v16,1억 이상- 고시금액미만 중소기업 제한 없음,True,REVIEWED_WITH_EXCEPTION_CHECK,5,1,6,12,0,1
8,v17,1억원 미만 일반물품 중소기업 제한,False,REVIEWED_WITH_EXCEPTION_CHECK,4,1,6,12,1,1
9,v18,1억원 미만 일반물품 소기업 제한 없음,True,REVIEWED_WITH_EXCEPTION_CHECK,4,1,7,12,0,1



Stage 6-3C FINAL RESULT
ITEMS                = 11
TARGET               = ['v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19']
ABSENCE ITEMS        = ['v10', 'v11', 'v16', 'v18']
TOTAL POSITIVES      = 70
EXPECTED POSITIVES   = 70
SME CSV              = 중기부고시_경쟁제품_세부품명.csv
SME CSV ROWS         = 616
SME CODE COLUMNS     = ['제품명번호', '세부품명번호']
V19 제5조의3 FOUND   = True
V19 ANCHOR CONTEXTS  = 11
PENDING SPECS        = []
OUTPUT               = /home/alparka/nara-ai/artifacts/judgment_specs_v9_v19.json

[PASS] Stage 6-3C v9~v19 final judgment specification 완료


In [1]:
# ============================================================
# Stage 6-3D FINAL
# v20~v24 exact grounding + final judgment specifications
#
# v20 : 입찰참가자격 (SW) [absence]
# v21 : 공동 5% (10%)
# v22 : 현장설명회 참석업체 자격 제한
# v23 : 현장설명회 공고기간
# v24 : 공고서 ↔ 나라장터 meta 불일치
#
# 원칙
# - 대회 배포 법령패키지만 사용
# - 외부 법령/웹 자료 사용 X
# - 실제 source text에서 조문/anchor를 확인
# - 없는 조문을 임의 생성하지 않음
# - DEV positive 수를 기존 검증값과 재대조
# - v24는 법령 RAG가 아니라 docs ↔ meta 비교
# ============================================================

from pathlib import Path
import json
import re
import math
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

DATA_DIR = ROOT / "open"

LAW_DIR = (
    DATA_DIR
    / "data"
    / "법령패키지"
    / "법령"
)

SOURCE_PACK_PATH = (
    ROOT
    / "artifacts"
    / "judgment_spec_source_pack.json"
)

BASIS_PATH = (
    ROOT
    / "artifacts"
    / "official_24_basis.csv"
)

OUT_PATH = (
    ROOT
    / "artifacts"
    / "judgment_specs_v20_v24.json"
)


assert LAW_DIR.exists(), LAW_DIR
assert SOURCE_PACK_PATH.exists(), SOURCE_PACK_PATH
assert BASIS_PATH.exists(), BASIS_PATH


TARGET_ITEMS = [
    "v20",
    "v21",
    "v22",
    "v23",
    "v24",
]


# ============================================================
# 1. SOURCE PACK / BASIS LOAD
# ============================================================

with open(
    SOURCE_PACK_PATH,
    "r",
    encoding="utf-8"
) as f:

    source_pack = json.load(f)


assert all(
    v in source_pack
    for v in TARGET_ITEMS
)


basis_df = pd.read_csv(
    BASIS_PATH,
    dtype=str
)

basis_df = basis_df.set_index(
    "item",
    drop=False
)


assert all(
    v in basis_df.index
    for v in TARGET_ITEMS
)


def safe_str(x):

    if x is None:
        return ""

    if isinstance(x, float):
        if math.isnan(x):
            return ""

    if pd.isna(x):
        return ""

    return str(x)


print("[1] SOURCE PACK / BASIS LOAD PASS")


# ============================================================
# 2. EXACT LAW FILES
# ============================================================

LAW_FILE_NAMES = {

    "sw_act":
        "소프트웨어 진흥법.txt",

    "sw_support":
        "중소 소프트웨어사업자의 사업 참여 지원에 관한 지침.txt",

    "joint_national":
        "(계약예규) 공동계약운용요령.txt",

    "local_guide":
        "지방자치단체 입찰 및 계약 집행기준.txt",

    "nat_decree":
        "국가를 당사자로 하는 계약에 관한 법률 시행령.txt",

    "loc_decree":
        "지방자치단체를 당사자로 하는 계약에 관한 법률 시행령.txt",

    "local_award":
        "지방자치단체 입찰시 낙찰자 결정기준.txt",
}


LAW = {}


for key, filename in LAW_FILE_NAMES.items():

    path = LAW_DIR / filename

    assert path.exists(), (
        f"\n법령 파일 없음\n{path}"
    )

    text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    LAW[key] = {
        "path": path,
        "source": path.stem,
        "text": text,
        "lines": text.splitlines(),
    }


assert len(
    list(
        LAW_DIR.glob("*.txt")
    )
) == 23


print("[2] 정확 법령파일 연결 PASS")


# ============================================================
# 3. TEXT / ARTICLE PARSER
# ============================================================

def clean_line(line):

    s = str(line)

    s = s.replace(
        "\u00a0",
        " "
    )

    s = s.replace(
        "\u2007",
        " "
    )

    s = s.replace(
        "\u202f",
        " "
    )

    s = s.replace(
        "<![CDATA[",
        ""
    )

    s = s.replace(
        "]]>",
        ""
    )

    return s.strip()


def compact(s):

    return re.sub(
        r"\s+",
        "",
        clean_line(s)
    )


ARTICLE_RE = re.compile(
    r"^제\s*"
    r"(?P<base>\d+)"
    r"\s*조"
    r"(?:\s*의\s*(?P<sub>\d+))?"
    r"\s*(?:\(|（)"
)


def article_key_from_line(line):

    m = ARTICLE_RE.match(
        clean_line(line)
    )

    if not m:
        return None

    base = m.group(
        "base"
    )

    sub = m.group(
        "sub"
    )

    if sub:
        return (
            f"{base}의{sub}"
        )

    return base


def article_display(key):

    key = str(
        key
    ).strip()

    if "의" in key:

        base, sub = key.split(
            "의",
            1
        )

        return (
            f"제{base}조의{sub}"
        )

    return (
        f"제{key}조"
    )


def article_index(
    lines,
    start=0,
    end=None
):

    if end is None:
        end = len(lines)

    out = []

    for i in range(
        start,
        end
    ):

        key = article_key_from_line(
            lines[i]
        )

        if key is not None:

            out.append({
                "line": i,
                "key": key,
                "header":
                    clean_line(
                        lines[i]
                    ),
            })

    return out


def extract_article(
    lines,
    key,
    start=0,
    end=None,
    required=True
):

    if end is None:
        end = len(lines)

    index = article_index(
        lines,
        start,
        end
    )

    matches = [
        x
        for x in index
        if x["key"] == str(key)
    ]


    if not matches:

        if required:

            raise AssertionError(
                f"\n{article_display(key)} "
                f"실제 제목 미발견"
            )

        return None


    target = matches[0]

    pos = index.index(
        target
    )

    begin = target[
        "line"
    ]


    if pos + 1 < len(index):

        finish = index[
            pos + 1
        ][
            "line"
        ]

    else:

        finish = end


    text = "\n".join(
        lines[
            begin:finish
        ]
    ).strip()


    return {

        "key":
            str(key),

        "header":
            target["header"],

        "start_line":
            begin,

        "end_line":
            finish,

        "text":
            text,
    }


# ============================================================
# 4. ANCHOR CONTEXT
# ============================================================

def extract_anchor_contexts(
    lines,
    keywords,
    radius=8,
    max_contexts=12
):

    hits = []


    for i, line in enumerate(
        lines
    ):

        s = clean_line(
            line
        )

        if any(
            keyword in s
            for keyword
            in keywords
        ):

            hits.append(i)


    contexts = []

    seen = set()


    for i in hits:

        start = max(
            0,
            i - radius
        )

        end = min(
            len(lines),
            i + radius + 1
        )

        text = "\n".join(
            lines[
                start:end
            ]
        ).strip()


        signature = re.sub(
            r"\s+",
            "",
            text
        )[:800]


        if signature in seen:
            continue


        seen.add(
            signature
        )


        contexts.append({

            "hit_line":
                i,

            "start_line":
                start,

            "end_line":
                end,

            "text":
                text,
        })


        if (
            len(contexts)
            >= max_contexts
        ):

            break


    return contexts


# ============================================================
# 5. CHAPTER EXTRACTION
#
# 목차와 본문이 중복되는 문서는
# 가장 내용이 긴 candidate를 실제 본문으로 선택
# ============================================================

def extract_chapter(
    lines,
    chapter_prefix,
    next_chapter_prefix
):

    starts = []


    for i, line in enumerate(
        lines
    ):

        if compact(
            line
        ).startswith(
            chapter_prefix
        ):

            starts.append(i)


    assert starts, (
        f"chapter 없음: "
        f"{chapter_prefix}"
    )


    candidates = []


    for start in starts:

        end = len(
            lines
        )


        for j in range(
            start + 1,
            len(lines)
        ):

            if compact(
                lines[j]
            ).startswith(
                next_chapter_prefix
            ):

                end = j
                break


        length = (
            end - start
        )

        candidates.append(
            (
                length,
                start,
                end
            )
        )


    candidates.sort(
        reverse=True
    )


    _, start, end = (
        candidates[0]
    )


    return {

        "start":
            start,

        "end":
            end,

        "text":
            "\n".join(
                lines[
                    start:end
                ]
            ).strip(),
    }


# ============================================================
# 6. v20 SW MATERIAL
# ============================================================

SW_ACT_LINES = (
    LAW[
        "sw_act"
    ][
        "lines"
    ]
)

SW_SUPPORT_LINES = (
    LAW[
        "sw_support"
    ][
        "lines"
    ]
)


# ------------------------------------------------------------
# 공식 locator: 지침 제2조
# ------------------------------------------------------------

SW_SUPPORT_ART2 = (
    extract_article(
        SW_SUPPORT_LINES,
        "2",
        required=True
    )
)


# ------------------------------------------------------------
# 공식 locator: 별표1
#
# 표 구조가 article 형식이 아닐 수 있으므로
# 별표/금액 anchor로 직접 확보
# ------------------------------------------------------------

SW_APPENDIX_CONTEXTS = (
    extract_anchor_contexts(

        SW_SUPPORT_LINES,

        keywords=[
            "별표 1",
            "별표1",
            "20억원",
            "40억원",
            "80억원",
            "사업금액",
            "참여할 수 있는 사업금액",
        ],

        radius=15,

        max_contexts=20
    )
)


assert (
    SW_APPENDIX_CONTEXTS
), (
    "SW 지침 별표/금액 context 미발견"
)


# ------------------------------------------------------------
# SW법 자체 관련 조문/문맥
# ------------------------------------------------------------

SW_ACT_CONTEXTS = (
    extract_anchor_contexts(

        SW_ACT_LINES,

        keywords=[
            "중소 소프트웨어사업자",
            "사업 참여",
            "소프트웨어사업자",
            "대기업",
        ],

        radius=10,

        max_contexts=12
    )
)


assert (
    SW_ACT_CONTEXTS
), (
    "소프트웨어진흥법 관련 context 미발견"
)


# ============================================================
# 7. v21 JOINT CONTRACT MATERIAL
# ============================================================

NATIONAL_JOINT_LINES = (
    LAW[
        "joint_national"
    ][
        "lines"
    ]
)


NATIONAL_JOINT_CONTEXTS = (
    extract_anchor_contexts(

        NATIONAL_JOINT_LINES,

        keywords=[
            "최소지분율",
            "출자비율",
            "분담비율",
            "100분의 5",
            "100분의 10",
            "5%",
            "10%",
        ],

        radius=12,

        max_contexts=20
    )
)


assert (
    NATIONAL_JOINT_CONTEXTS
), (
    "국가 공동계약 비율 context 미발견"
)


# ------------------------------------------------------------
# 지방 집행기준 제6장
# ------------------------------------------------------------

LOCAL_GUIDE_LINES = (
    LAW[
        "local_guide"
    ][
        "lines"
    ]
)


LOCAL_CH6 = extract_chapter(

    LOCAL_GUIDE_LINES,

    "제6장공동계약운영요령",

    "제7장"
)


LOCAL_JOINT_CONTEXTS = (
    extract_anchor_contexts(

        LOCAL_CH6[
            "text"
        ].splitlines(),

        keywords=[
            "최소지분율",
            "출자비율",
            "분담비율",
            "100분의 5",
            "100분의 10",
            "5%",
            "10%",
        ],

        radius=12,

        max_contexts=20
    )
)


assert (
    LOCAL_JOINT_CONTEXTS
), (
    "지방 공동계약 비율 context 미발견"
)


# ============================================================
# 8. v22 SITE-BRIEFING PARTICIPATION RESTRICTION
# ============================================================

NAT_DECREE_LINES = (
    LAW[
        "nat_decree"
    ][
        "lines"
    ]
)

LOC_DECREE_LINES = (
    LAW[
        "loc_decree"
    ][
        "lines"
    ]
)


NAT_ART43 = (
    extract_article(
        NAT_DECREE_LINES,
        "43",
        required=True
    )
)


LOC_ART43 = (
    extract_article(
        LOC_DECREE_LINES,
        "43",
        required=True
    )
)


# 삭제된 항까지 포함되어 있는지
# 현장설명 anchor도 별도 확보
NAT_SITE_CONTEXTS = (
    extract_anchor_contexts(

        NAT_DECREE_LINES,

        keywords=[
            "현장설명",
            "설명회 참석",
        ],

        radius=10,

        max_contexts=12
    )
)


LOC_SITE_CONTEXTS = (
    extract_anchor_contexts(

        LOC_DECREE_LINES,

        keywords=[
            "현장설명",
            "설명회 참석",
        ],

        radius=10,

        max_contexts=12
    )
)


# ============================================================
# 9. v23 SITE-BRIEFING NOTICE PERIOD
# ============================================================

LOC_ART35 = (
    extract_article(
        LOC_DECREE_LINES,
        "35",
        required=True
    )
)


LOCAL_AWARD_LINES = (
    LAW[
        "local_award"
    ][
        "lines"
    ]
)


LOCAL_AWARD_SITE_CONTEXTS = (
    extract_anchor_contexts(

        LOCAL_AWARD_LINES,

        keywords=[
            "현장설명",
            "설명회",
            "제안요청",
            "공고기간",
            "제안서 제출",
        ],

        radius=14,

        max_contexts=20
    )
)


assert (
    LOCAL_AWARD_SITE_CONTEXTS
), (
    "지방 낙찰자 결정기준 "
    "현장설명/공고기간 context 미발견"
)


# ============================================================
# 10. LEGAL MATERIALS
# ============================================================

LEGAL = {

    "sw_act_material": {

        "source":
            LAW[
                "sw_act"
            ][
                "source"
            ],

        "locator":
            "소프트웨어 진흥법",

        "contexts":
            SW_ACT_CONTEXTS,
    },


    "sw_support_art2": {

        "source":
            LAW[
                "sw_support"
            ][
                "source"
            ],

        "locator":
            "제2조",

        "header":
            SW_SUPPORT_ART2[
                "header"
            ],

        "text":
            SW_SUPPORT_ART2[
                "text"
            ],
    },


    "sw_support_appendix1": {

        "source":
            LAW[
                "sw_support"
            ][
                "source"
            ],

        "locator":
            "별표1",

        "contexts":
            SW_APPENDIX_CONTEXTS,
    },


    "joint_national": {

        "source":
            LAW[
                "joint_national"
            ][
                "source"
            ],

        "locator":
            "공동수급체 구성 / 참여비율",

        "contexts":
            NATIONAL_JOINT_CONTEXTS,
    },


    "joint_local": {

        "source":
            LAW[
                "local_guide"
            ][
                "source"
            ],

        "locator":
            "제6장 공동계약 운영요령",

        "contexts":
            LOCAL_JOINT_CONTEXTS,
    },


    "nat_decree_43": {

        "source":
            LAW[
                "nat_decree"
            ][
                "source"
            ],

        "locator":
            "제43조",

        "header":
            NAT_ART43[
                "header"
            ],

        "text":
            NAT_ART43[
                "text"
            ],

        "site_contexts":
            NAT_SITE_CONTEXTS,
    },


    "loc_decree_43": {

        "source":
            LAW[
                "loc_decree"
            ][
                "source"
            ],

        "locator":
            "제43조",

        "header":
            LOC_ART43[
                "header"
            ],

        "text":
            LOC_ART43[
                "text"
            ],

        "site_contexts":
            LOC_SITE_CONTEXTS,
    },


    "loc_decree_35": {

        "source":
            LAW[
                "loc_decree"
            ][
                "source"
            ],

        "locator":
            "제35조",

        "header":
            LOC_ART35[
                "header"
            ],

        "text":
            LOC_ART35[
                "text"
            ],
    },


    "local_award_site": {

        "source":
            LAW[
                "local_award"
            ][
                "source"
            ],

        "locator":
            (
                "제7장 협상계약 관련 "
                "현장설명/공고기간"
            ),

        "contexts":
            LOCAL_AWARD_SITE_CONTEXTS,
    },
}


# ============================================================
# 11. DEV CASE ACCESSOR
#
# 6-3C에서 검증한 방식 그대로 사용.
# 없는 key를 []로 조용히 바꾸지 않는다.
# ============================================================

def extract_dev_cases(
    src,
    item
):

    positives = None
    negatives = None

    positive_source = None
    negative_source = None


    # A. top-level
    if (
        "dev_positives"
        in src
        and
        isinstance(
            src[
                "dev_positives"
            ],
            list
        )
    ):

        positives = (
            src[
                "dev_positives"
            ]
        )

        positive_source = (
            "top-level.dev_positives"
        )


    if (
        "dev_negative_candidates"
        in src
        and
        isinstance(
            src[
                "dev_negative_candidates"
            ],
            list
        )
    ):

        negatives = (
            src[
                "dev_negative_candidates"
            ]
        )

        negative_source = (
            "top-level."
            "dev_negative_candidates"
        )


    # B. nested dev
    dev = src.get(
        "dev"
    )


    if isinstance(
        dev,
        dict
    ):

        if positives is None:

            for key in [
                "positives",
                "positive_cases",
                "dev_positives",
            ]:

                if (
                    key in dev
                    and
                    isinstance(
                        dev[key],
                        list
                    )
                ):

                    positives = dev[
                        key
                    ]

                    positive_source = (
                        f"dev.{key}"
                    )

                    break


        if negatives is None:

            for key in [
                "negative_candidates",
                "negatives",
                "dev_negative_candidates",
            ]:

                if (
                    key in dev
                    and
                    isinstance(
                        dev[key],
                        list
                    )
                ):

                    negatives = dev[
                        key
                    ]

                    negative_source = (
                        f"dev.{key}"
                    )

                    break


    if positives is None:

        raise AssertionError(
            f"\n{item}: "
            "positive 사례 구조 미발견\n"
            f"keys={list(src.keys())}"
        )


    if negatives is None:

        raise AssertionError(
            f"\n{item}: "
            "negative candidate 구조 미발견\n"
            f"keys={list(src.keys())}"
        )


    for case in positives:

        assert (
            isinstance(
                case,
                dict
            )
            and
            "id"
            in case
        )


    for case in negatives:

        assert (
            isinstance(
                case,
                dict
            )
            and
            "id"
            in case
        )


    return {

        "positives":
            positives,

        "negatives":
            negatives,

        "positive_source":
            positive_source,

        "negative_source":
            negative_source,
    }


# ============================================================
# 12. DEV COUNTS
# ============================================================

EXPECTED_POSITIVE_COUNTS = {

    "v20": 5,
    "v21": 6,
    "v22": 5,
    "v23": 5,
    "v24": 8,
}


DEV_CASES = {}


for v in TARGET_ITEMS:

    data = extract_dev_cases(
        source_pack[v],
        v
    )

    DEV_CASES[v] = data

    actual = len(
        data[
            "positives"
        ]
    )


    assert (
        actual
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    ), (
        f"{v}: positive mismatch "
        f"{actual} != "
        f"{EXPECTED_POSITIVE_COUNTS[v]}"
    )


print(
    "[3] DEV positive counts PASS"
)


# ============================================================
# 13. COMMON POLICIES
# ============================================================

COMMON = {

    "document_scope":
        (
            "공고문뿐 아니라 규격서, 과업지시서, "
            "제안요청서, 예외공표서 전체를 사용한다."
        ),

    "incomplete_input_handling":
        (
            "보이는 문서에서 위반사항이 명시적으로 확인되면 "
            "판정 가능하다. 그러나 dropped_doc_counts가 있거나 "
            "완전관측=False일 때 누락 문서 내용을 임의 추측하지 않는다. "
            "특히 부재탐지 항목은 문구가 보이지 않는다는 사실만으로 "
            "즉시 위반 확정하지 않는다."
        ),

    "evidence_rule":
        (
            "비부재탐지 항목에서 위반이면 제공 docs에 실제 존재하는 "
            "연속 부분문자열만 evidence로 사용한다. "
            "요약·생성 금지. v20은 부재탐지이므로 evidence는 빈 문자열."
        ),
}


# ============================================================
# 14. DECISION SPEC
# ============================================================

FINAL = {}


# ------------------------------------------------------------
# v20 입찰참가자격(SW)
# ------------------------------------------------------------

FINAL["v20"] = {

    "status":
        "REVIEWED_WITH_AMOUNT_TABLE",

    "absence_detection":
        True,

    "applicability":
        (
            "정보화사업 또는 소프트웨어사업으로 판단되는 공고 중 "
            "중소 소프트웨어사업자 참여지원 지침의 적용을 받는 사업."
        ),

    "required_meta": [
        "정보화사업여부",
        "업무구분",
        "배정예산금액",
        "입찰추정가격",
        "조항호내용",
    ],

    "facts_to_extract": [
        "SW/정보화사업 여부",
        "사업금액",
        "참여 가능한 사업자 규모",
        "대기업 참여제한 또는 중소 SW사업자 참여자격 기재 여부",
        "법령상 예외 존재 여부",
    ],

    "amount_condition":
        (
            "20억/40억/80억 등 금액 구간을 문자열로 임의 하드코딩하여 "
            "판정하지 않고, 배포된 지침 제2조 및 별표1의 실제 표를 "
            "판정 context로 사용한다."
        ),

    "violation_condition":
        (
            "SW사업에 대해 지침상 사업금액 및 사업자 규모에 따라 "
            "기재되어야 하는 입찰참가자격 또는 참여제한이 필요한데, "
            "공고 전체에서 해당 자격 제한이 확인되지 않고 "
            "공식 예외도 확인되지 않는 경우."
        ),

    "nonviolation_condition":
        (
            "필요한 SW사업자 참여자격/참여제한이 기재되어 있거나, "
            "해당 지침 적용대상이 아니거나 공식 예외가 적용되는 경우."
        ),

    "python_role": [
        "정보화사업여부와 사업금액 구조화",
        "지침 별표1 금액구간 source 제공",
        "SW사업자/대기업/중소기업 관련 표현 후보 탐색",
        "input completeness 제공",
    ],

    "gemma_role": [
        "실제 SW사업인지 판단",
        "별표1 기준과 사업금액을 결합하여 필요한 참가자격 판단",
        "공고에 실질적인 참가제한이 존재하는지 판단",
        "예외 적용 여부 판단",
    ],

    "evidence":
        "EMPTY",

    "unresolved": [
        "별표1 금액 구간은 이후 structured extractor에서 source table을 직접 구조화",
    ],
}


# ------------------------------------------------------------
# v21 공동 5%(10%)
# ------------------------------------------------------------

FINAL["v21"] = {

    "status":
        "REVIEWED_WITH_SOURCE_RATIO",

    "absence_detection":
        False,

    "applicability":
        (
            "공동수급체 또는 공동도급을 허용·요구하는 공고."
        ),

    "required_meta": [
        "공동도급구성방식",
        "적용계약법",
        "업무구분",
        "계약방법",
    ],

    "facts_to_extract": [
        "공동계약 허용 여부",
        "공동수급체 구성방식",
        "구성원별 최소 출자비율",
        "구성원별 최소 분담비율",
        "공고에 기재된 5%/10% 등 참여비율",
    ],

    "violation_condition":
        (
            "공고가 공동수급체 구성원의 최소 참여·출자·분담비율을 "
            "적용되는 국가/지방 공동계약 기준의 5% 또는 10% 기준과 "
            "불일치하게 제한하는 경우."
        ),

    "nonviolation_condition":
        (
            "공동계약이 아니거나, 공고의 최소 참여비율이 "
            "해당 국가/지방 및 공동계약 형태에 적용되는 "
            "공식 source 기준과 일치하는 경우."
        ),

    "python_role": [
        "공동도급구성방식 확인",
        "5%, 10%, 출자비율, 분담비율 숫자 후보 추출",
        "적용계약법별 source 선택",
        "evidence exact substring 검증",
    ],

    "gemma_role": [
        "숫자가 공동수급체 구성원의 최소 참여비율인지 판단",
        "국가/지방 및 공동이행/분담이행 문맥에 맞는 기준 선택",
        "공고 조건과 공식 source 기준의 일치 여부 판단",
    ],

    "unresolved": [
        "5%/10%의 적용 분기는 Stage 7 structured extraction에서 exact source 문맥으로 구조화",
    ],
}


# ------------------------------------------------------------
# v22 현장설명회 참석업체 자격 제한
# 협상계약만 적용
# ------------------------------------------------------------

FINAL["v22"] = {

    "status":
        "REVIEWED",

    "absence_detection":
        False,

    "applicability":
        (
            "협상에 의한 계약인 공고에서 현장설명회·제안요청설명회 "
            "등의 참석 여부를 입찰참가자격으로 사용하는지 검사."
        ),

    "required_meta": [
        "낙찰방법",
        "계약방법",
        "적용계약법",
    ],

    "facts_to_extract": [
        "협상계약 여부",
        "현장설명회/제안요청설명회 존재 여부",
        "설명회 참석 의무 여부",
        "불참자를 입찰참가 불가로 처리하는지 여부",
    ],

    "violation_condition":
        (
            "협상에 의한 계약에서 현장설명회 또는 유사 설명회 "
            "참석자를 입찰참가자격 요건으로 한정하거나, "
            "불참 업체의 입찰참가를 배제하는 경우."
        ),

    "nonviolation_condition":
        (
            "협상계약이 아니거나, 설명회를 실시하더라도 참석 여부가 "
            "입찰참가자격과 무관하고 불참 업체의 참가를 배제하지 않는 경우."
        ),

    "python_role": [
        "낙찰방법/계약방법에서 협상계약 후보 판정",
        "현장설명회/제안요청설명회 표현 탐색",
        "'참석자에 한함', '불참 시 입찰불가' 등의 강제표현 후보 추출",
        "evidence 검증",
    ],

    "gemma_role": [
        "설명회 참석이 단순 안내인지 참가자격 제한인지 판단",
        "불참 업체의 실제 입찰 배제 여부 판단",
    ],

    "unresolved": [],
}


# ------------------------------------------------------------
# v23 현장설명회 공고기간
# 협상 + 지방계약만 적용
# ------------------------------------------------------------

FINAL["v23"] = {

    "status":
        "REVIEWED_WITH_PERIOD_RULE",

    "absence_detection":
        False,

    "applicability":
        (
            "지방계약법 적용 + 협상에 의한 계약이며 "
            "현장설명회 또는 제안요청 설명회를 실시하는 경우."
        ),

    "required_meta": [
        "적용계약법",
        "낙찰방법",
        "계약방법",
        "공고게시일자",
        "개찰예정일자",
    ],

    "facts_to_extract": [
        "지방계약 여부",
        "협상계약 여부",
        "설명회 일자",
        "제안서 제출마감일",
        "입찰공고일",
        "법령상 요구되는 최소 공고기간",
    ],

    "violation_condition":
        (
            "지방계약의 협상계약에서 설명회를 실시하면서 "
            "입찰공고일부터 설명회 또는 제안서 제출 관련 기한까지의 "
            "기간이 지방계약법 시행령 제35조 및 배포 낙찰자결정기준이 "
            "요구하는 최소기간보다 짧은 경우."
        ),

    "nonviolation_condition":
        (
            "국가계약이거나 협상계약이 아니거나, 설명회가 없거나, "
            "지방계약 공식 기준의 최소 공고기간을 충족하는 경우."
        ),

    "python_role": [
        "적용계약법 == 지방계약법 확인",
        "협상계약 여부 확인",
        "공고일/설명회일/제안서마감일 날짜 추출",
        "날짜 차이 계산",
        "evidence exact substring 검증",
    ],

    "gemma_role": [
        "어떤 날짜가 법령상 비교대상인지 판단",
        "해당 공고에 적용되는 최소기간 규칙 선택",
        "긴급공고 등 예외 문맥 판단",
    ],

    "unresolved": [
        "최소 일수는 Stage 7에서 제35조/낙찰자결정기준 source를 구조화하여 적용",
    ],
}


# ------------------------------------------------------------
# v24 공고서와 나라장터 입력값 상이
# ------------------------------------------------------------

FINAL["v24"] = {

    "status":
        "REVIEWED_COMPARISON",

    "absence_detection":
        False,

    "applicability":
        "모든 공고.",

    "required_meta": [
        "배정예산금액",
        "입찰추정가격",
        "계약방법",
        "낙찰방법",
        "지역제한여부",
        "제한지역코드목록",
        "업종제한여부",
        "면허업종제한목록",
    ],

    "facts_to_extract": [
        "공고문/첨부의 사업예산 또는 추정가격",
        "공고문/첨부의 계약방법",
        "공고문/첨부의 지역제한",
        "공고문/첨부의 업종·면허 제한",
        "각 값과 meta의 대응값",
    ],

    "violation_condition":
        (
            "공고문 또는 첨부문서에 명시된 예산·계약방법·지역제한·"
            "업종제한 등의 값과 나라장터 meta의 대응값이 "
            "동일 공고 내에서 실질적으로 서로 다른 경우."
        ),

    "nonviolation_condition":
        (
            "문서와 meta가 의미상 일치하거나, 표기방식 차이·"
            "VAT 포함/별도·지역코드 표현 차이 등으로 동일 내용을 "
            "다르게 표현한 것에 불과한 경우."
        ),

    "python_role": [
        "meta 비교대상 필드 구조화",
        "문서의 금액·계약방법·지역·업종 후보 추출",
        "금액 정규화",
        "지역 token/표현 정규화",
        "업종 코드/명칭 후보 정규화",
        "명백한 exact mismatch flag 생성",
        "evidence exact substring 검증",
    ],

    "gemma_role": [
        "문서 표현과 meta 값의 의미상 대응관계 판단",
        "실질적 불일치와 단순 표기차이를 구분",
        "여러 문서 중 최종 공고조건을 식별",
    ],

    "unresolved": [
        "v24는 법령 RAG 대상이 아니라 docs↔meta comparison 전용으로 구현",
    ],
}


# ============================================================
# 15. LEGAL KEY MAP
# ============================================================

LEGAL_KEYS = {

    "v20": [
        "sw_act_material",
        "sw_support_art2",
        "sw_support_appendix1",
    ],

    "v21": [
        "joint_national",
        "joint_local",
    ],

    "v22": [
        "nat_decree_43",
        "loc_decree_43",
    ],

    "v23": [
        "loc_decree_35",
        "local_award_site",
    ],

    # v24는 법령 근거가 없는 비교형
    "v24": [],
}


# ============================================================
# 16. FINAL PACK BUILD
# ============================================================

FINAL_PACK = {}


for v in TARGET_ITEMS:

    src = source_pack[v]

    absence = bool(
        src[
            "absence_detection"
        ]
    )


    expected_absence = (
        v == "v20"
    )


    assert (
        absence
        ==
        expected_absence
    ), (
        f"{v}: absence mismatch"
    )


    dev_cases = (
        DEV_CASES[v]
    )


    exact_legal = {

        key:
            LEGAL[key]

        for key
        in LEGAL_KEYS[v]
    }


    FINAL_PACK[v] = {

        "item":
            v,

        "name":
            src[
                "name"
            ],

        "absence_detection":
            absence,

        "official_basis": {

            "national":
                safe_str(
                    basis_df.loc[
                        v,
                        "national"
                    ]
                ),

            "local":
                safe_str(
                    basis_df.loc[
                        v,
                        "local"
                    ]
                ),

            "note":
                safe_str(
                    basis_df.loc[
                        v,
                        "note"
                    ]
                ),
        },

        **COMMON,

        "decision_spec":
            FINAL[v],

        "exact_legal_keys":
            LEGAL_KEYS[v],

        "exact_legal_text":
            exact_legal,

        "dev": {

            "positive_count":
                len(
                    dev_cases[
                        "positives"
                    ]
                ),

            "positive_ids": [
                x["id"]

                for x
                in dev_cases[
                    "positives"
                ]
            ],

            "positive_evidence_examples": [
                x.get(
                    "evidence"
                )

                for x
                in dev_cases[
                    "positives"
                ]

                if x.get(
                    "evidence"
                )
            ],

            "negative_candidate_count":
                len(
                    dev_cases[
                        "negatives"
                    ]
                ),

            "negative_candidate_ids": [
                x["id"]

                for x
                in dev_cases[
                    "negatives"
                ]
            ],

            "negative_candidate_semantics":
                "STRUCTURAL_CANDIDATES_ONLY",

            "source_schema": {

                "positives":
                    dev_cases[
                        "positive_source"
                    ],

                "negatives":
                    dev_cases[
                        "negative_source"
                    ],
            },
        },
    }


# ============================================================
# 17. VALIDATION
# ============================================================

assert list(
    FINAL_PACK.keys()
) == TARGET_ITEMS


for v in TARGET_ITEMS:

    x = FINAL_PACK[v]

    assert (
        x[
            "dev"
        ][
            "positive_count"
        ]
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    )


    assert (
        x[
            "decision_spec"
        ][
            "status"
        ]
        !=
        "PENDING_FINAL_REVIEW"
    )


# v20만 absence
assert [
    v
    for v, x
    in FINAL_PACK.items()
    if x[
        "absence_detection"
    ]
] == [
    "v20"
]


# v20 evidence blank
assert (
    FINAL_PACK[
        "v20"
    ][
        "decision_spec"
    ][
        "evidence"
    ]
    ==
    "EMPTY"
)


# v24 법령 block 없어야 함
assert (
    FINAL_PACK[
        "v24"
    ][
        "exact_legal_keys"
    ]
    ==
    []
)


assert (
    FINAL_PACK[
        "v24"
    ][
        "exact_legal_text"
    ]
    ==
    {}
)


# v20 실제 SW 자료 존재
sw_blob = json.dumps(
    FINAL_PACK[
        "v20"
    ][
        "exact_legal_text"
    ],
    ensure_ascii=False
)

assert (
    "소프트웨어"
    in sw_blob
)


# v21 비율 source 존재
joint_blob = json.dumps(
    FINAL_PACK[
        "v21"
    ][
        "exact_legal_text"
    ],
    ensure_ascii=False
)

assert (
    "출자"
    in joint_blob
    or
    "분담"
    in joint_blob
    or
    "지분"
    in joint_blob
)


# v23 지방 시행령 제35조
assert (
    article_key_from_line(
        LEGAL[
            "loc_decree_35"
        ][
            "header"
        ]
    )
    ==
    "35"
)


TOTAL_POSITIVES = sum(
    x[
        "dev"
    ][
        "positive_count"
    ]

    for x
    in FINAL_PACK.values()
)


assert (
    TOTAL_POSITIVES
    ==
    29
)


print(
    "[4] FINAL validation PASS"
)


# ============================================================
# 18. SAVE
# ============================================================

with open(
    OUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_PACK,
        f,
        ensure_ascii=False,
        indent=2
    )


assert OUT_PATH.exists()


# reload validation
with open(
    OUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    saved = json.load(f)


assert list(
    saved.keys()
) == TARGET_ITEMS


for v in TARGET_ITEMS:

    assert (
        saved[v][
            "dev"
        ][
            "positive_count"
        ]
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    )


print(
    "[5] JSON SAVE / RELOAD PASS"
)


# ============================================================
# 19. AUDIT TABLE
# ============================================================

rows = []


for v in TARGET_ITEMS:

    x = FINAL_PACK[v]

    rows.append({

        "item":
            v,

        "name":
            x["name"],

        "absence":
            x[
                "absence_detection"
            ],

        "status":
            x[
                "decision_spec"
            ][
                "status"
            ],

        "legal_blocks":
            len(
                x[
                    "exact_legal_keys"
                ]
            ),

        "positives":
            x[
                "dev"
            ][
                "positive_count"
            ],

        "negative_candidates":
            x[
                "dev"
            ][
                "negative_candidate_count"
            ],

        "evidence_examples":
            len(
                x[
                    "dev"
                ][
                    "positive_evidence_examples"
                ]
            ),

        "unresolved":
            len(
                x[
                    "decision_spec"
                ][
                    "unresolved"
                ]
            ),
    })


audit_63d = pd.DataFrame(
    rows
)


display(
    audit_63d
)


# ============================================================
# 20. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 6-3D FINAL RESULT")
print("=" * 100)


print(
    "ITEMS               =",
    len(
        FINAL_PACK
    )
)


print(
    "TARGET              =",
    TARGET_ITEMS
)


print(
    "ABSENCE ITEMS       =",
    [
        v
        for v, x
        in FINAL_PACK.items()
        if x[
            "absence_detection"
        ]
    ]
)


print(
    "TOTAL POSITIVES     =",
    TOTAL_POSITIVES
)


print(
    "EXPECTED POSITIVES  =",
    sum(
        EXPECTED_POSITIVE_COUNTS.values()
    )
)


print(
    "SW APPENDIX CONTEXT =",
    len(
        SW_APPENDIX_CONTEXTS
    )
)


print(
    "NATIONAL JOINT CTX  =",
    len(
        NATIONAL_JOINT_CONTEXTS
    )
)


print(
    "LOCAL JOINT CTX     =",
    len(
        LOCAL_JOINT_CONTEXTS
    )
)


print(
    "V22 NAT SITE CTX    =",
    len(
        NAT_SITE_CONTEXTS
    )
)


print(
    "V22 LOCAL SITE CTX  =",
    len(
        LOC_SITE_CONTEXTS
    )
)


print(
    "V23 AWARD CTX       =",
    len(
        LOCAL_AWARD_SITE_CONTEXTS
    )
)


print(
    "PENDING SPECS       =",
    [
        v

        for v, x
        in FINAL_PACK.items()

        if x[
            "decision_spec"
        ][
            "status"
        ]
        ==
        "PENDING_FINAL_REVIEW"
    ]
)


print(
    "OUTPUT              =",
    OUT_PATH
)


print()

print(
    "[PASS] Stage 6-3D v20~v24 "
    "final judgment specification 완료"
)

[1] SOURCE PACK / BASIS LOAD PASS
[2] 정확 법령파일 연결 PASS
[3] DEV positive counts PASS
[4] FINAL validation PASS
[5] JSON SAVE / RELOAD PASS


,item,name,absence,status,legal_blocks,positives,negative_candidates,evidence_examples,unresolved
0,v20,입찰참가자격 (SW),True,REVIEWED_WITH_AMOUNT_TABLE,3,5,12,0,1
1,v21,공동 5% (10%),False,REVIEWED_WITH_SOURCE_RATIO,2,6,12,1,1
2,v22,현장설명회 참석업체 (자격으로 제한 여부) * 계약방법 협상만 적용,False,REVIEWED,2,5,12,5,0
3,v23,현장설명회 (공고기간) * 계약방법 협상 + 계약법 지방만 적용,False,REVIEWED_WITH_PERIOD_RULE,2,5,12,5,1
4,v24,공고서와 나라장터 입력값 상이,False,REVIEWED_COMPARISON,0,8,12,1,1



Stage 6-3D FINAL RESULT
ITEMS               = 5
TARGET              = ['v20', 'v21', 'v22', 'v23', 'v24']
ABSENCE ITEMS       = ['v20']
TOTAL POSITIVES     = 29
EXPECTED POSITIVES  = 29
SW APPENDIX CONTEXT = 16
NATIONAL JOINT CTX  = 20
LOCAL JOINT CTX     = 20
V22 NAT SITE CTX    = 6
V22 LOCAL SITE CTX  = 6
V23 AWARD CTX       = 20
PENDING SPECS       = []
OUTPUT              = /home/alparka/nara-ai/artifacts/judgment_specs_v20_v24.json

[PASS] Stage 6-3D v20~v24 final judgment specification 완료


In [2]:
# ============================================================
# Stage 6-3E FINAL
# v1~v24 Judgment Specification Integration
#
# 목적:
# - v1~v8 / v9~v19 / v20~v24 산출물 통합
# - 24개 누락/중복 검사
# - 공식 항목명/부재탐지 여부 검증
# - DEV positive count 재검증
# - 법령 grounding 구조 검증
# - PENDING spec 존재 여부 검증
#
# 이 단계에서는 새로운 법적 판단규칙을 만들지 않는다.
# 기존에 확정한 spec의 통합 무결성만 검사한다.
# ============================================================

from pathlib import Path
import json
import math
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

ARTIFACT_DIR = (
    ROOT
    / "artifacts"
)

V1_V8_PATH = (
    ARTIFACT_DIR
    / "judgment_specs_v1_v8.json"
)

V9_V19_PATH = (
    ARTIFACT_DIR
    / "judgment_specs_v9_v19.json"
)

V20_V24_PATH = (
    ARTIFACT_DIR
    / "judgment_specs_v20_v24.json"
)

BASIS_PATH = (
    ARTIFACT_DIR
    / "official_24_basis.csv"
)

SOURCE_PACK_PATH = (
    ARTIFACT_DIR
    / "judgment_spec_source_pack.json"
)

OUT_JSON = (
    ARTIFACT_DIR
    / "judgment_specs_v1_v24.json"
)

OUT_AUDIT = (
    ARTIFACT_DIR
    / "judgment_specs_v1_v24_audit.csv"
)


for path in [
    V1_V8_PATH,
    V9_V19_PATH,
    V20_V24_PATH,
    BASIS_PATH,
    SOURCE_PACK_PATH,
]:

    assert path.exists(), (
        f"필수 artifact 없음: {path}"
    )


print("[1] 필수 artifact 존재 PASS")


# ============================================================
# 1. LOAD
# ============================================================

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


spec_1_8 = load_json(
    V1_V8_PATH
)

spec_9_19 = load_json(
    V9_V19_PATH
)

spec_20_24 = load_json(
    V20_V24_PATH
)

source_pack = load_json(
    SOURCE_PACK_PATH
)


basis = pd.read_csv(
    BASIS_PATH,
    dtype=str
)

basis = basis.set_index(
    "item",
    drop=False
)


print("[2] artifact load PASS")


# ============================================================
# 2. EXPECTED STRUCTURE
# ============================================================

EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(
        1,
        25
    )
]


EXPECTED_GROUPS = {

    "v1_v8": [
        f"v{i}"
        for i in range(
            1,
            9
        )
    ],

    "v9_v19": [
        f"v{i}"
        for i in range(
            9,
            20
        )
    ],

    "v20_v24": [
        f"v{i}"
        for i in range(
            20,
            25
        )
    ],
}


EXPECTED_ABSENCE = {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}


EXPECTED_POSITIVE_COUNTS = {

    "v1": 7,
    "v2": 7,
    "v3": 8,
    "v4": 6,
    "v5": 7,
    "v6": 6,
    "v7": 7,
    "v8": 6,

    "v9": 6,
    "v10": 7,
    "v11": 6,
    "v12": 6,
    "v13": 6,
    "v14": 8,
    "v15": 6,
    "v16": 6,
    "v17": 6,
    "v18": 7,
    "v19": 6,

    "v20": 5,
    "v21": 6,
    "v22": 5,
    "v23": 5,
    "v24": 8,
}


assert (
    sum(
        EXPECTED_POSITIVE_COUNTS.values()
    )
    ==
    153
)


print("[3] expected structure 정의 PASS")


# ============================================================
# 3. GROUP 자체 검증
# ============================================================

assert (
    list(
        spec_1_8.keys()
    )
    ==
    EXPECTED_GROUPS[
        "v1_v8"
    ]
), (
    "v1~v8 파일 item 구성 이상"
)


assert (
    list(
        spec_9_19.keys()
    )
    ==
    EXPECTED_GROUPS[
        "v9_v19"
    ]
), (
    "v9~v19 파일 item 구성 이상"
)


assert (
    list(
        spec_20_24.keys()
    )
    ==
    EXPECTED_GROUPS[
        "v20_v24"
    ]
), (
    "v20~v24 파일 item 구성 이상"
)


print("[4] 세 그룹 item 범위 PASS")


# ============================================================
# 4. MERGE
# ============================================================

ALL_SPECS = {}


for group in [
    spec_1_8,
    spec_9_19,
    spec_20_24,
]:

    for v, spec in group.items():

        assert (
            v not in ALL_SPECS
        ), (
            f"중복 item 발견: {v}"
        )

        ALL_SPECS[v] = spec


assert (
    list(
        ALL_SPECS.keys()
    )
    ==
    EXPECTED_ITEMS
)


assert (
    len(
        ALL_SPECS
    )
    ==
    24
)


print("[5] v1~v24 merge PASS")


# ============================================================
# 5. 공식 BASIS 구조 검증
# ============================================================

assert (
    list(
        basis.index
    )
    ==
    EXPECTED_ITEMS
), (
    "official_24_basis.csv item 순서/구성 이상"
)


assert (
    set(
        source_pack.keys()
    )
    ==
    set(
        EXPECTED_ITEMS
    )
)


def safe_str(x):

    if x is None:
        return ""

    if isinstance(
        x,
        float
    ):

        if math.isnan(x):
            return ""

    if pd.isna(x):
        return ""

    return str(x).strip()


print("[6] official basis/source pack PASS")


# ============================================================
# 6. REQUIRED KEYS
# ============================================================

REQUIRED_TOP_KEYS = {

    "item",
    "name",
    "absence_detection",
    "official_basis",
    "decision_spec",
    "exact_legal_keys",
    "exact_legal_text",
    "dev",
}


REQUIRED_DEV_KEYS = {

    "positive_count",
    "positive_ids",
    "negative_candidate_count",
    "negative_candidate_ids",
}


# ============================================================
# 7. ITEM-BY-ITEM INTEGRITY VALIDATION
# ============================================================

audit_rows = []


for v in EXPECTED_ITEMS:

    spec = ALL_SPECS[v]


    # --------------------------------------------------------
    # 7-1. top-level
    # --------------------------------------------------------

    missing_top = (
        REQUIRED_TOP_KEYS
        -
        set(
            spec.keys()
        )
    )

    assert not missing_top, (
        f"{v}: top-level key 누락 "
        f"{sorted(missing_top)}"
    )


    assert (
        spec["item"]
        ==
        v
    ), (
        f"{v}: item field mismatch"
    )


    # --------------------------------------------------------
    # 7-2. 공식 항목명
    # --------------------------------------------------------

    official_name = safe_str(
        basis.loc[
            v,
            "name"
        ]
    )

    actual_name = safe_str(
        spec[
            "name"
        ]
    )


    assert (
        actual_name
        ==
        official_name
    ), (
        f"\n{v}: 항목명 불일치\n"
        f"official={official_name}\n"
        f"spec={actual_name}"
    )


    # --------------------------------------------------------
    # 7-3. absence
    # --------------------------------------------------------

    expected_absence = (
        v
        in EXPECTED_ABSENCE
    )

    actual_absence = bool(
        spec[
            "absence_detection"
        ]
    )


    assert (
        actual_absence
        ==
        expected_absence
    ), (
        f"{v}: absence mismatch"
    )


    # source pack과도 대조
    assert (
        actual_absence
        ==
        bool(
            source_pack[v][
                "absence_detection"
            ]
        )
    ), (
        f"{v}: source_pack absence mismatch"
    )


    # --------------------------------------------------------
    # 7-4. decision_spec
    # --------------------------------------------------------

    decision = spec[
        "decision_spec"
    ]


    assert isinstance(
        decision,
        dict
    )


    assert (
        "status"
        in decision
    ), (
        f"{v}: decision_spec.status 없음"
    )


    status = safe_str(
        decision[
            "status"
        ]
    )


    assert status, (
        f"{v}: status empty"
    )


    assert (
        status
        !=
        "PENDING_FINAL_REVIEW"
    ), (
        f"{v}: 아직 PENDING"
    )


    # 최소한 판정조건은 반드시 있어야 함
    assert (
        safe_str(
            decision.get(
                "violation_condition"
            )
        )
    ), (
        f"{v}: violation_condition 없음"
    )


    assert (
        safe_str(
            decision.get(
                "nonviolation_condition"
            )
        )
    ), (
        f"{v}: nonviolation_condition 없음"
    )


    # --------------------------------------------------------
    # 7-5. DEV 구조
    # --------------------------------------------------------

    dev = spec[
        "dev"
    ]


    missing_dev = (
        REQUIRED_DEV_KEYS
        -
        set(
            dev.keys()
        )
    )


    assert not missing_dev, (
        f"{v}: dev key 누락 "
        f"{sorted(missing_dev)}"
    )


    positive_count = int(
        dev[
            "positive_count"
        ]
    )


    assert (
        positive_count
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    ), (
        f"{v}: positive count "
        f"{positive_count} != "
        f"{EXPECTED_POSITIVE_COUNTS[v]}"
    )


    positive_ids = dev[
        "positive_ids"
    ]


    assert (
        len(
            positive_ids
        )
        ==
        positive_count
    ), (
        f"{v}: positive ID 개수 mismatch"
    )


    assert (
        len(
            set(
                positive_ids
            )
        )
        ==
        len(
            positive_ids
        )
    ), (
        f"{v}: positive ID 중복"
    )


    negative_ids = dev[
        "negative_candidate_ids"
    ]


    assert (
        len(
            negative_ids
        )
        ==
        int(
            dev[
                "negative_candidate_count"
            ]
        )
    ), (
        f"{v}: negative candidate count mismatch"
    )


    # positive / negative 후보가 겹치면 안 됨
    overlap = (
        set(
            positive_ids
        )
        &
        set(
            negative_ids
        )
    )


    assert not overlap, (
        f"{v}: positive와 negative candidate "
        f"ID 중복 {sorted(overlap)}"
    )


    # --------------------------------------------------------
    # 7-6. evidence 정책
    # --------------------------------------------------------

    if expected_absence:

        # decision_spec에 evidence key가 있는 파일은
        # EMPTY여야 함
        if (
            "evidence"
            in decision
        ):

            assert (
                decision[
                    "evidence"
                ]
                ==
                "EMPTY"
            ), (
                f"{v}: absence evidence policy 오류"
            )


    # --------------------------------------------------------
    # 7-7. legal grounding
    # --------------------------------------------------------

    legal_keys = spec[
        "exact_legal_keys"
    ]

    legal_text = spec[
        "exact_legal_text"
    ]


    assert isinstance(
        legal_keys,
        list
    )

    assert isinstance(
        legal_text,
        dict
    )


    if v == "v24":

        # 비교형 항목
        assert (
            legal_keys
            ==
            []
        )

        assert (
            legal_text
            ==
            {}
        )

    else:

        assert (
            len(
                legal_keys
            )
            > 0
        ), (
            f"{v}: legal grounding 없음"
        )


        assert (
            set(
                legal_keys
            )
            ==
            set(
                legal_text.keys()
            )
        ), (
            f"{v}: legal key/text mismatch"
        )


        for legal_key in legal_keys:

            block = legal_text[
                legal_key
            ]

            assert isinstance(
                block,
                dict
            ), (
                f"{v}/{legal_key}: "
                "legal block이 dict 아님"
            )


            # legal block 전체가 텅 비어 있으면 실패
            blob = json.dumps(
                block,
                ensure_ascii=False
            ).strip()


            assert (
                len(
                    blob
                )
                > 10
            ), (
                f"{v}/{legal_key}: "
                "legal context 비어 있음"
            )


    # --------------------------------------------------------
    # 7-8. official basis
    # --------------------------------------------------------

    official_basis = spec[
        "official_basis"
    ]


    assert isinstance(
        official_basis,
        dict
    )


    for side in [
        "national",
        "local",
    ]:

        assert (
            side
            in official_basis
        ), (
            f"{v}: official_basis.{side} 없음"
        )


    # --------------------------------------------------------
    # 7-9. audit
    # --------------------------------------------------------

    audit_rows.append({

        "item":
            v,

        "name":
            actual_name,

        "absence":
            actual_absence,

        "status":
            status,

        "positive":
            positive_count,

        "negative_candidates":
            len(
                negative_ids
            ),

        "legal_blocks":
            len(
                legal_keys
            ),

        "unresolved":
            len(
                decision.get(
                    "unresolved",
                    []
                )
            ),
    })


print("[7] 24개 item integrity PASS")


# ============================================================
# 8. GLOBAL VALIDATION
# ============================================================

# ------------------------------------------------------------
# absence set
# ------------------------------------------------------------

actual_absence_items = {

    v
    for v, spec
    in ALL_SPECS.items()

    if spec[
        "absence_detection"
    ]
}


assert (
    actual_absence_items
    ==
    EXPECTED_ABSENCE
), (
    f"absence set mismatch\n"
    f"expected={sorted(EXPECTED_ABSENCE)}\n"
    f"actual={sorted(actual_absence_items)}"
)


# ------------------------------------------------------------
# positive total
# ------------------------------------------------------------

TOTAL_POSITIVES = sum(

    int(
        spec[
            "dev"
        ][
            "positive_count"
        ]
    )

    for spec
    in ALL_SPECS.values()
)


assert (
    TOTAL_POSITIVES
    ==
    153
), (
    f"positive total mismatch: "
    f"{TOTAL_POSITIVES}"
)


# ------------------------------------------------------------
# pending
# ------------------------------------------------------------

PENDING = [

    v
    for v, spec
    in ALL_SPECS.items()

    if spec[
        "decision_spec"
    ][
        "status"
    ]
    ==
    "PENDING_FINAL_REVIEW"
]


assert (
    PENDING
    ==
    []
)


# ------------------------------------------------------------
# exact item coverage
# ------------------------------------------------------------

assert (
    set(
        ALL_SPECS.keys()
    )
    ==
    set(
        EXPECTED_ITEMS
    )
)


assert (
    len(
        ALL_SPECS
    )
    ==
    24
)


print("[8] global consistency PASS")


# ============================================================
# 9. GROUP-SPECIFIC IMPORTANT CHECKS
# ============================================================

# ------------------------------------------------------------
# v1~v8 모두 presence-type
# ------------------------------------------------------------

assert all(
    not ALL_SPECS[v][
        "absence_detection"
    ]

    for v
    in EXPECTED_GROUPS[
        "v1_v8"
    ]
)


# ------------------------------------------------------------
# v9~v19 absence 정확히 4개
# ------------------------------------------------------------

assert {
    v
    for v
    in EXPECTED_GROUPS[
        "v9_v19"
    ]

    if ALL_SPECS[v][
        "absence_detection"
    ]
} == {
    "v10",
    "v11",
    "v16",
    "v18",
}


# ------------------------------------------------------------
# v20~v24 absence = v20
# ------------------------------------------------------------

assert {
    v
    for v
    in EXPECTED_GROUPS[
        "v20_v24"
    ]

    if ALL_SPECS[v][
        "absence_detection"
    ]
} == {
    "v20"
}


# ------------------------------------------------------------
# v24 comparison-only
# ------------------------------------------------------------

assert (
    ALL_SPECS[
        "v24"
    ][
        "decision_spec"
    ][
        "status"
    ]
    ==
    "REVIEWED_COMPARISON"
)


assert (
    ALL_SPECS[
        "v24"
    ][
        "exact_legal_keys"
    ]
    ==
    []
)


# ------------------------------------------------------------
# SME structured source가 있어야 할 항목
# ------------------------------------------------------------

for v in [
    "v10",
    "v11",
    "v12",
    "v13",
    "v14",
    "v15",
    "v16",
    "v17",
    "v18",
]:

    assert (
        "structured_sources"
        in ALL_SPECS[v]
    )

    assert (
        len(
            ALL_SPECS[v][
                "structured_sources"
            ]
        )
        >= 1
    ), (
        f"{v}: SME structured source 없음"
    )


print("[9] item-family specific checks PASS")


# ============================================================
# 10. SAVE
# ============================================================

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ALL_SPECS,
        f,
        ensure_ascii=False,
        indent=2
    )


audit_df = pd.DataFrame(
    audit_rows
)


audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 11. SAVE / RELOAD VALIDATION
# ============================================================

with open(
    OUT_JSON,
    "r",
    encoding="utf-8"
) as f:

    saved = json.load(f)


assert (
    list(
        saved.keys()
    )
    ==
    EXPECTED_ITEMS
)


assert (
    len(
        saved
    )
    ==
    24
)


for v in EXPECTED_ITEMS:

    assert (
        saved[v][
            "item"
        ]
        ==
        v
    )

    assert (
        saved[v][
            "dev"
        ][
            "positive_count"
        ]
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    )


saved_audit = pd.read_csv(
    OUT_AUDIT
)


assert (
    len(
        saved_audit
    )
    ==
    24
)


print("[10] SAVE / RELOAD PASS")


# ============================================================
# 12. AUDIT DISPLAY
# ============================================================

display(
    audit_df
)


# ============================================================
# 13. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 6-3E FINAL RESULT")
print("=" * 100)


print(
    "ITEMS               =",
    len(
        ALL_SPECS
    )
)


print(
    "FIRST / LAST        =",
    list(
        ALL_SPECS.keys()
    )[0],
    "/",
    list(
        ALL_SPECS.keys()
    )[-1]
)


print(
    "ABSENCE ITEMS       =",
    sorted(
        actual_absence_items,
        key=lambda x:
            int(
                x[1:]
            )
    )
)


print(
    "TOTAL POSITIVES     =",
    TOTAL_POSITIVES
)


print(
    "EXPECTED POSITIVES  =",
    153
)


print(
    "PENDING SPECS       =",
    PENDING
)


print(
    "V24 LEGAL BLOCKS    =",
    len(
        ALL_SPECS[
            "v24"
        ][
            "exact_legal_keys"
        ]
    )
)


print(
    "OUTPUT JSON         =",
    OUT_JSON
)


print(
    "AUDIT CSV           =",
    OUT_AUDIT
)


print()

print(
    "[PASS] Stage 6-3E v1~v24 "
    "integrated judgment specification 완료"
)

[1] 필수 artifact 존재 PASS
[2] artifact load PASS
[3] expected structure 정의 PASS
[4] 세 그룹 item 범위 PASS
[5] v1~v24 merge PASS
[6] official basis/source pack PASS
[7] 24개 item integrity PASS
[8] global consistency PASS
[9] item-family specific checks PASS
[10] SAVE / RELOAD PASS


,item,name,absence,status,positive,negative_candidates,legal_blocks,unresolved
0,v1,참가자격 특정기관 제한,False,REVIEWED,7,12,4,0
1,v2,고시금액 미만 실적제한,False,REVIEWED_WITH_RUNTIME_THRESHOLD,7,12,5,1
2,v3,실적제한 1배수 이상,False,REVIEWED,8,12,5,1
3,v4,"고시금액 이상 특정기관, 특정실적",False,REVIEWED_WITH_OPEN_AMOUNT_EDGE,6,12,2,1
4,v5,고시금액 이상 지역제한,False,REVIEWED_WITH_RUNTIME_THRESHOLD,7,12,4,2
5,v6,"고시금액 미만 지역제한 시,군,구",False,REVIEWED,6,12,3,0
6,v7,고시금액 미만 지역제한 인접 확대,False,REVIEWED,7,12,3,1
7,v8,중복제한 (실적+지역),False,REVIEWED_WITH_EXCEPTION_CHECK,6,12,3,1
8,v9,과업지시서 특정 모델명 명시,False,REVIEWED,6,12,2,0
9,v10,중기간 경쟁제품 입찰 직생 없음,True,REVIEWED_WITH_STRUCTURED_LOOKUP,7,12,1,1



Stage 6-3E FINAL RESULT
ITEMS               = 24
FIRST / LAST        = v1 / v24
ABSENCE ITEMS       = ['v10', 'v11', 'v16', 'v18', 'v20']
TOTAL POSITIVES     = 153
EXPECTED POSITIVES  = 153
PENDING SPECS       = []
V24 LEGAL BLOCKS    = 0
OUTPUT JSON         = /home/alparka/nara-ai/artifacts/judgment_specs_v1_v24.json
AUDIT CSV           = /home/alparka/nara-ai/artifacts/judgment_specs_v1_v24_audit.csv

[PASS] Stage 6-3E v1~v24 integrated judgment specification 완료


In [1]:
# ============================================================
# Stage 6-3E FINAL
# v1~v24 Judgment Specification Integration
#
# 목적:
# - v1~v8 / v9~v19 / v20~v24 산출물 통합
# - 24개 누락/중복 검사
# - 공식 항목명/부재탐지 여부 검증
# - DEV positive count 재검증
# - 법령 grounding 구조 검증
# - PENDING spec 존재 여부 검증
#
# 이 단계에서는 새로운 법적 판단규칙을 만들지 않는다.
# 기존에 확정한 spec의 통합 무결성만 검사한다.
# ============================================================

from pathlib import Path
import json
import math
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

ARTIFACT_DIR = (
    ROOT
    / "artifacts"
)

V1_V8_PATH = (
    ARTIFACT_DIR
    / "judgment_specs_v1_v8.json"
)

V9_V19_PATH = (
    ARTIFACT_DIR
    / "judgment_specs_v9_v19.json"
)

V20_V24_PATH = (
    ARTIFACT_DIR
    / "judgment_specs_v20_v24.json"
)

BASIS_PATH = (
    ARTIFACT_DIR
    / "official_24_basis.csv"
)

SOURCE_PACK_PATH = (
    ARTIFACT_DIR
    / "judgment_spec_source_pack.json"
)

OUT_JSON = (
    ARTIFACT_DIR
    / "judgment_specs_v1_v24.json"
)

OUT_AUDIT = (
    ARTIFACT_DIR
    / "judgment_specs_v1_v24_audit.csv"
)


for path in [
    V1_V8_PATH,
    V9_V19_PATH,
    V20_V24_PATH,
    BASIS_PATH,
    SOURCE_PACK_PATH,
]:

    assert path.exists(), (
        f"필수 artifact 없음: {path}"
    )


print("[1] 필수 artifact 존재 PASS")


# ============================================================
# 1. LOAD
# ============================================================

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


spec_1_8 = load_json(
    V1_V8_PATH
)

spec_9_19 = load_json(
    V9_V19_PATH
)

spec_20_24 = load_json(
    V20_V24_PATH
)

source_pack = load_json(
    SOURCE_PACK_PATH
)


basis = pd.read_csv(
    BASIS_PATH,
    dtype=str
)

basis = basis.set_index(
    "item",
    drop=False
)


print("[2] artifact load PASS")


# ============================================================
# 2. EXPECTED STRUCTURE
# ============================================================

EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(
        1,
        25
    )
]


EXPECTED_GROUPS = {

    "v1_v8": [
        f"v{i}"
        for i in range(
            1,
            9
        )
    ],

    "v9_v19": [
        f"v{i}"
        for i in range(
            9,
            20
        )
    ],

    "v20_v24": [
        f"v{i}"
        for i in range(
            20,
            25
        )
    ],
}


EXPECTED_ABSENCE = {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}


EXPECTED_POSITIVE_COUNTS = {

    "v1": 7,
    "v2": 7,
    "v3": 8,
    "v4": 6,
    "v5": 7,
    "v6": 6,
    "v7": 7,
    "v8": 6,

    "v9": 6,
    "v10": 7,
    "v11": 6,
    "v12": 6,
    "v13": 6,
    "v14": 8,
    "v15": 6,
    "v16": 6,
    "v17": 6,
    "v18": 7,
    "v19": 6,

    "v20": 5,
    "v21": 6,
    "v22": 5,
    "v23": 5,
    "v24": 8,
}


assert (
    sum(
        EXPECTED_POSITIVE_COUNTS.values()
    )
    ==
    153
)


print("[3] expected structure 정의 PASS")


# ============================================================
# 3. GROUP 자체 검증
# ============================================================

assert (
    list(
        spec_1_8.keys()
    )
    ==
    EXPECTED_GROUPS[
        "v1_v8"
    ]
), (
    "v1~v8 파일 item 구성 이상"
)


assert (
    list(
        spec_9_19.keys()
    )
    ==
    EXPECTED_GROUPS[
        "v9_v19"
    ]
), (
    "v9~v19 파일 item 구성 이상"
)


assert (
    list(
        spec_20_24.keys()
    )
    ==
    EXPECTED_GROUPS[
        "v20_v24"
    ]
), (
    "v20~v24 파일 item 구성 이상"
)


print("[4] 세 그룹 item 범위 PASS")


# ============================================================
# 4. MERGE
# ============================================================

ALL_SPECS = {}


for group in [
    spec_1_8,
    spec_9_19,
    spec_20_24,
]:

    for v, spec in group.items():

        assert (
            v not in ALL_SPECS
        ), (
            f"중복 item 발견: {v}"
        )

        ALL_SPECS[v] = spec


assert (
    list(
        ALL_SPECS.keys()
    )
    ==
    EXPECTED_ITEMS
)


assert (
    len(
        ALL_SPECS
    )
    ==
    24
)


print("[5] v1~v24 merge PASS")


# ============================================================
# 5. 공식 BASIS 구조 검증
# ============================================================

assert (
    list(
        basis.index
    )
    ==
    EXPECTED_ITEMS
), (
    "official_24_basis.csv item 순서/구성 이상"
)


assert (
    set(
        source_pack.keys()
    )
    ==
    set(
        EXPECTED_ITEMS
    )
)


def safe_str(x):

    if x is None:
        return ""

    if isinstance(
        x,
        float
    ):

        if math.isnan(x):
            return ""

    if pd.isna(x):
        return ""

    return str(x).strip()


print("[6] official basis/source pack PASS")


# ============================================================
# 6. REQUIRED KEYS
# ============================================================

REQUIRED_TOP_KEYS = {

    "item",
    "name",
    "absence_detection",
    "official_basis",
    "decision_spec",
    "exact_legal_keys",
    "exact_legal_text",
    "dev",
}


REQUIRED_DEV_KEYS = {

    "positive_count",
    "positive_ids",
    "negative_candidate_count",
    "negative_candidate_ids",
}


# ============================================================
# 7. ITEM-BY-ITEM INTEGRITY VALIDATION
# ============================================================

audit_rows = []


for v in EXPECTED_ITEMS:

    spec = ALL_SPECS[v]


    # --------------------------------------------------------
    # 7-1. top-level
    # --------------------------------------------------------

    missing_top = (
        REQUIRED_TOP_KEYS
        -
        set(
            spec.keys()
        )
    )

    assert not missing_top, (
        f"{v}: top-level key 누락 "
        f"{sorted(missing_top)}"
    )


    assert (
        spec["item"]
        ==
        v
    ), (
        f"{v}: item field mismatch"
    )


    # --------------------------------------------------------
    # 7-2. 공식 항목명
    # --------------------------------------------------------

    official_name = safe_str(
        basis.loc[
            v,
            "name"
        ]
    )

    actual_name = safe_str(
        spec[
            "name"
        ]
    )


    assert (
        actual_name
        ==
        official_name
    ), (
        f"\n{v}: 항목명 불일치\n"
        f"official={official_name}\n"
        f"spec={actual_name}"
    )


    # --------------------------------------------------------
    # 7-3. absence
    # --------------------------------------------------------

    expected_absence = (
        v
        in EXPECTED_ABSENCE
    )

    actual_absence = bool(
        spec[
            "absence_detection"
        ]
    )


    assert (
        actual_absence
        ==
        expected_absence
    ), (
        f"{v}: absence mismatch"
    )


    # source pack과도 대조
    assert (
        actual_absence
        ==
        bool(
            source_pack[v][
                "absence_detection"
            ]
        )
    ), (
        f"{v}: source_pack absence mismatch"
    )


    # --------------------------------------------------------
    # 7-4. decision_spec
    # --------------------------------------------------------

    decision = spec[
        "decision_spec"
    ]


    assert isinstance(
        decision,
        dict
    )


    assert (
        "status"
        in decision
    ), (
        f"{v}: decision_spec.status 없음"
    )


    status = safe_str(
        decision[
            "status"
        ]
    )


    assert status, (
        f"{v}: status empty"
    )


    assert (
        status
        !=
        "PENDING_FINAL_REVIEW"
    ), (
        f"{v}: 아직 PENDING"
    )


    # 최소한 판정조건은 반드시 있어야 함
    assert (
        safe_str(
            decision.get(
                "violation_condition"
            )
        )
    ), (
        f"{v}: violation_condition 없음"
    )


    assert (
        safe_str(
            decision.get(
                "nonviolation_condition"
            )
        )
    ), (
        f"{v}: nonviolation_condition 없음"
    )


    # --------------------------------------------------------
    # 7-5. DEV 구조
    # --------------------------------------------------------

    dev = spec[
        "dev"
    ]


    missing_dev = (
        REQUIRED_DEV_KEYS
        -
        set(
            dev.keys()
        )
    )


    assert not missing_dev, (
        f"{v}: dev key 누락 "
        f"{sorted(missing_dev)}"
    )


    positive_count = int(
        dev[
            "positive_count"
        ]
    )


    assert (
        positive_count
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    ), (
        f"{v}: positive count "
        f"{positive_count} != "
        f"{EXPECTED_POSITIVE_COUNTS[v]}"
    )


    positive_ids = dev[
        "positive_ids"
    ]


    assert (
        len(
            positive_ids
        )
        ==
        positive_count
    ), (
        f"{v}: positive ID 개수 mismatch"
    )


    assert (
        len(
            set(
                positive_ids
            )
        )
        ==
        len(
            positive_ids
        )
    ), (
        f"{v}: positive ID 중복"
    )


    negative_ids = dev[
        "negative_candidate_ids"
    ]


    assert (
        len(
            negative_ids
        )
        ==
        int(
            dev[
                "negative_candidate_count"
            ]
        )
    ), (
        f"{v}: negative candidate count mismatch"
    )


    # positive / negative 후보가 겹치면 안 됨
    overlap = (
        set(
            positive_ids
        )
        &
        set(
            negative_ids
        )
    )


    assert not overlap, (
        f"{v}: positive와 negative candidate "
        f"ID 중복 {sorted(overlap)}"
    )


    # --------------------------------------------------------
    # 7-6. evidence 정책
    # --------------------------------------------------------

    if expected_absence:

        # decision_spec에 evidence key가 있는 파일은
        # EMPTY여야 함
        if (
            "evidence"
            in decision
        ):

            assert (
                decision[
                    "evidence"
                ]
                ==
                "EMPTY"
            ), (
                f"{v}: absence evidence policy 오류"
            )


    # --------------------------------------------------------
    # 7-7. legal grounding
    # --------------------------------------------------------

    legal_keys = spec[
        "exact_legal_keys"
    ]

    legal_text = spec[
        "exact_legal_text"
    ]


    assert isinstance(
        legal_keys,
        list
    )

    assert isinstance(
        legal_text,
        dict
    )


    if v == "v24":

        # 비교형 항목
        assert (
            legal_keys
            ==
            []
        )

        assert (
            legal_text
            ==
            {}
        )

    else:

        assert (
            len(
                legal_keys
            )
            > 0
        ), (
            f"{v}: legal grounding 없음"
        )


        assert (
            set(
                legal_keys
            )
            ==
            set(
                legal_text.keys()
            )
        ), (
            f"{v}: legal key/text mismatch"
        )


        for legal_key in legal_keys:

            block = legal_text[
                legal_key
            ]

            assert isinstance(
                block,
                dict
            ), (
                f"{v}/{legal_key}: "
                "legal block이 dict 아님"
            )


            # legal block 전체가 텅 비어 있으면 실패
            blob = json.dumps(
                block,
                ensure_ascii=False
            ).strip()


            assert (
                len(
                    blob
                )
                > 10
            ), (
                f"{v}/{legal_key}: "
                "legal context 비어 있음"
            )


    # --------------------------------------------------------
    # 7-8. official basis
    # --------------------------------------------------------

    official_basis = spec[
        "official_basis"
    ]


    assert isinstance(
        official_basis,
        dict
    )


    for side in [
        "national",
        "local",
    ]:

        assert (
            side
            in official_basis
        ), (
            f"{v}: official_basis.{side} 없음"
        )


    # --------------------------------------------------------
    # 7-9. audit
    # --------------------------------------------------------

    audit_rows.append({

        "item":
            v,

        "name":
            actual_name,

        "absence":
            actual_absence,

        "status":
            status,

        "positive":
            positive_count,

        "negative_candidates":
            len(
                negative_ids
            ),

        "legal_blocks":
            len(
                legal_keys
            ),

        "unresolved":
            len(
                decision.get(
                    "unresolved",
                    []
                )
            ),
    })


print("[7] 24개 item integrity PASS")


# ============================================================
# 8. GLOBAL VALIDATION
# ============================================================

# ------------------------------------------------------------
# absence set
# ------------------------------------------------------------

actual_absence_items = {

    v
    for v, spec
    in ALL_SPECS.items()

    if spec[
        "absence_detection"
    ]
}


assert (
    actual_absence_items
    ==
    EXPECTED_ABSENCE
), (
    f"absence set mismatch\n"
    f"expected={sorted(EXPECTED_ABSENCE)}\n"
    f"actual={sorted(actual_absence_items)}"
)


# ------------------------------------------------------------
# positive total
# ------------------------------------------------------------

TOTAL_POSITIVES = sum(

    int(
        spec[
            "dev"
        ][
            "positive_count"
        ]
    )

    for spec
    in ALL_SPECS.values()
)


assert (
    TOTAL_POSITIVES
    ==
    153
), (
    f"positive total mismatch: "
    f"{TOTAL_POSITIVES}"
)


# ------------------------------------------------------------
# pending
# ------------------------------------------------------------

PENDING = [

    v
    for v, spec
    in ALL_SPECS.items()

    if spec[
        "decision_spec"
    ][
        "status"
    ]
    ==
    "PENDING_FINAL_REVIEW"
]


assert (
    PENDING
    ==
    []
)


# ------------------------------------------------------------
# exact item coverage
# ------------------------------------------------------------

assert (
    set(
        ALL_SPECS.keys()
    )
    ==
    set(
        EXPECTED_ITEMS
    )
)


assert (
    len(
        ALL_SPECS
    )
    ==
    24
)


print("[8] global consistency PASS")


# ============================================================
# 9. GROUP-SPECIFIC IMPORTANT CHECKS
# ============================================================

# ------------------------------------------------------------
# v1~v8 모두 presence-type
# ------------------------------------------------------------

assert all(
    not ALL_SPECS[v][
        "absence_detection"
    ]

    for v
    in EXPECTED_GROUPS[
        "v1_v8"
    ]
)


# ------------------------------------------------------------
# v9~v19 absence 정확히 4개
# ------------------------------------------------------------

assert {
    v
    for v
    in EXPECTED_GROUPS[
        "v9_v19"
    ]

    if ALL_SPECS[v][
        "absence_detection"
    ]
} == {
    "v10",
    "v11",
    "v16",
    "v18",
}


# ------------------------------------------------------------
# v20~v24 absence = v20
# ------------------------------------------------------------

assert {
    v
    for v
    in EXPECTED_GROUPS[
        "v20_v24"
    ]

    if ALL_SPECS[v][
        "absence_detection"
    ]
} == {
    "v20"
}


# ------------------------------------------------------------
# v24 comparison-only
# ------------------------------------------------------------

assert (
    ALL_SPECS[
        "v24"
    ][
        "decision_spec"
    ][
        "status"
    ]
    ==
    "REVIEWED_COMPARISON"
)


assert (
    ALL_SPECS[
        "v24"
    ][
        "exact_legal_keys"
    ]
    ==
    []
)


# ------------------------------------------------------------
# SME structured source가 있어야 할 항목
# ------------------------------------------------------------

for v in [
    "v10",
    "v11",
    "v12",
    "v13",
    "v14",
    "v15",
    "v16",
    "v17",
    "v18",
]:

    assert (
        "structured_sources"
        in ALL_SPECS[v]
    )

    assert (
        len(
            ALL_SPECS[v][
                "structured_sources"
            ]
        )
        >= 1
    ), (
        f"{v}: SME structured source 없음"
    )


print("[9] item-family specific checks PASS")


# ============================================================
# 10. SAVE
# ============================================================

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ALL_SPECS,
        f,
        ensure_ascii=False,
        indent=2
    )


audit_df = pd.DataFrame(
    audit_rows
)


audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 11. SAVE / RELOAD VALIDATION
# ============================================================

with open(
    OUT_JSON,
    "r",
    encoding="utf-8"
) as f:

    saved = json.load(f)


assert (
    list(
        saved.keys()
    )
    ==
    EXPECTED_ITEMS
)


assert (
    len(
        saved
    )
    ==
    24
)


for v in EXPECTED_ITEMS:

    assert (
        saved[v][
            "item"
        ]
        ==
        v
    )

    assert (
        saved[v][
            "dev"
        ][
            "positive_count"
        ]
        ==
        EXPECTED_POSITIVE_COUNTS[v]
    )


saved_audit = pd.read_csv(
    OUT_AUDIT
)


assert (
    len(
        saved_audit
    )
    ==
    24
)


print("[10] SAVE / RELOAD PASS")


# ============================================================
# 12. AUDIT DISPLAY
# ============================================================

display(
    audit_df
)


# ============================================================
# 13. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 6-3E FINAL RESULT")
print("=" * 100)


print(
    "ITEMS               =",
    len(
        ALL_SPECS
    )
)


print(
    "FIRST / LAST        =",
    list(
        ALL_SPECS.keys()
    )[0],
    "/",
    list(
        ALL_SPECS.keys()
    )[-1]
)


print(
    "ABSENCE ITEMS       =",
    sorted(
        actual_absence_items,
        key=lambda x:
            int(
                x[1:]
            )
    )
)


print(
    "TOTAL POSITIVES     =",
    TOTAL_POSITIVES
)


print(
    "EXPECTED POSITIVES  =",
    153
)


print(
    "PENDING SPECS       =",
    PENDING
)


print(
    "V24 LEGAL BLOCKS    =",
    len(
        ALL_SPECS[
            "v24"
        ][
            "exact_legal_keys"
        ]
    )
)


print(
    "OUTPUT JSON         =",
    OUT_JSON
)


print(
    "AUDIT CSV           =",
    OUT_AUDIT
)


print()

print(
    "[PASS] Stage 6-3E v1~v24 "
    "integrated judgment specification 완료"
)

[1] 필수 artifact 존재 PASS
[2] artifact load PASS
[3] expected structure 정의 PASS
[4] 세 그룹 item 범위 PASS
[5] v1~v24 merge PASS
[6] official basis/source pack PASS
[7] 24개 item integrity PASS
[8] global consistency PASS
[9] item-family specific checks PASS
[10] SAVE / RELOAD PASS


,item,name,absence,status,positive,negative_candidates,legal_blocks,unresolved
0,v1,참가자격 특정기관 제한,False,REVIEWED,7,12,4,0
1,v2,고시금액 미만 실적제한,False,REVIEWED_WITH_RUNTIME_THRESHOLD,7,12,5,1
2,v3,실적제한 1배수 이상,False,REVIEWED,8,12,5,1
3,v4,"고시금액 이상 특정기관, 특정실적",False,REVIEWED_WITH_OPEN_AMOUNT_EDGE,6,12,2,1
4,v5,고시금액 이상 지역제한,False,REVIEWED_WITH_RUNTIME_THRESHOLD,7,12,4,2
5,v6,"고시금액 미만 지역제한 시,군,구",False,REVIEWED,6,12,3,0
6,v7,고시금액 미만 지역제한 인접 확대,False,REVIEWED,7,12,3,1
7,v8,중복제한 (실적+지역),False,REVIEWED_WITH_EXCEPTION_CHECK,6,12,3,1
8,v9,과업지시서 특정 모델명 명시,False,REVIEWED,6,12,2,0
9,v10,중기간 경쟁제품 입찰 직생 없음,True,REVIEWED_WITH_STRUCTURED_LOOKUP,7,12,1,1



Stage 6-3E FINAL RESULT
ITEMS               = 24
FIRST / LAST        = v1 / v24
ABSENCE ITEMS       = ['v10', 'v11', 'v16', 'v18', 'v20']
TOTAL POSITIVES     = 153
EXPECTED POSITIVES  = 153
PENDING SPECS       = []
V24 LEGAL BLOCKS    = 0
OUTPUT JSON         = /home/alparka/nara-ai/artifacts/judgment_specs_v1_v24.json
AUDIT CSV           = /home/alparka/nara-ai/artifacts/judgment_specs_v1_v24_audit.csv

[PASS] Stage 6-3E v1~v24 integrated judgment specification 완료


In [2]:
# ============================================================
# Stage 6-4 FINAL
# 24개 Judgment Specification Consistency Validation
#
# 목표
# 1) 통합 spec 24개 구조 검증
# 2) 공식 dev_labels와 positive 수 재검증
# 3) 부재탐지/evidence 정책 검증
# 4) 금액/상품/SW/현장설명회/v24 family 정합성 검증
# 5) 관련 항목의 DEV 동시양성(co-occurrence) 확인
# 6) 이후 Stage 7/10에서 사용할 consistency manifest 저장
#
# 주의
# - DEV에서 동시양성이 없다고 무조건 mutex로 만들지 않는다.
# - 한 공고에 여러 품목이 있을 수 있으므로 notice-level 강제 mutex 금지.
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

SPEC_PATH = (
    ROOT
    / "artifacts"
    / "judgment_specs_v1_v24.json"
)

DEV_LABEL_PATH = (
    ROOT
    / "open"
    / "dev_labels.csv"
)

OUT_RULES = (
    ROOT
    / "artifacts"
    / "judgment_spec_consistency_rules.json"
)

OUT_AUDIT = (
    ROOT
    / "artifacts"
    / "judgment_spec_consistency_audit.csv"
)

OUT_STAGE6 = (
    ROOT
    / "artifacts"
    / "stage6_final_manifest.json"
)


assert SPEC_PATH.exists(), SPEC_PATH
assert DEV_LABEL_PATH.exists(), DEV_LABEL_PATH


with open(
    SPEC_PATH,
    "r",
    encoding="utf-8"
) as f:
    specs = json.load(f)


labels = pd.read_csv(
    DEV_LABEL_PATH
)


EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(1, 25)
]

ABSENCE_ITEMS = {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}


assert list(specs.keys()) == EXPECTED_ITEMS
assert len(labels) == 200

print("[1] INPUT LOAD PASS")


# ============================================================
# 1. DEV LABEL VALIDATION
# ============================================================

assert all(
    v in labels.columns
    for v in EXPECTED_ITEMS
)

assert all(
    f"e{i}" in labels.columns
    for i in range(1, 25)
)


for v in EXPECTED_ITEMS:

    vals = set(
        labels[v]
        .dropna()
        .astype(int)
        .unique()
    )

    assert vals <= {0, 1}, (
        f"{v}: label 값 이상 {vals}"
    )


DEV_POSITIVE_COUNTS = {
    v: int(labels[v].sum())
    for v in EXPECTED_ITEMS
}


SPEC_POSITIVE_COUNTS = {
    v: int(
        specs[v]["dev"]["positive_count"]
    )
    for v in EXPECTED_ITEMS
}


assert (
    DEV_POSITIVE_COUNTS
    ==
    SPEC_POSITIVE_COUNTS
), (
    "통합 spec positive count와 "
    "dev_labels.csv 불일치"
)


assert (
    sum(
        DEV_POSITIVE_COUNTS.values()
    )
    ==
    153
)


print("[2] DEV LABEL / POSITIVE COUNT PASS")


# ============================================================
# 2. ABSENCE EVIDENCE VALIDATION
# ============================================================

for v in ABSENCE_ITEMS:

    idx = int(v[1:])

    e_col = f"e{idx}"

    positive_rows = (
        labels[v]
        .astype(int)
        == 1
    )

    evidence = (
        labels.loc[
            positive_rows,
            e_col
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    assert (
        evidence
        ==
        ""
    ).all(), (
        f"{v}: 부재탐지인데 "
        f"positive evidence 존재"
    )


print("[3] ABSENCE EVIDENCE POLICY PASS")


# ============================================================
# 3. ITEM-LEVEL SEMANTIC REQUIREMENTS
# ============================================================

# 반드시 필요한 meta
REQUIRED_META_EXPECTATIONS = {

    # 가격/실적
    "v2": {"입찰추정가격"},
    "v3": {"배정예산금액"},
    "v4": {"입찰추정가격"},
    "v5": {"입찰추정가격"},

    # 지역
    "v6": {
        "입찰추정가격",
        "지역제한여부",
    },
    "v7": {
        "입찰추정가격",
        "지역제한여부",
    },
    "v8": {
        "입찰추정가격",
        "지역제한여부",
    },

    # 품목
    "v10": {"세부품명번호목록"},
    "v11": {"세부품명번호목록"},
    "v12": {"세부품명번호목록"},
    "v13": {"세부품명번호목록"},

    # 일반물품 금액대
    "v14": {
        "입찰추정가격",
        "세부품명번호목록",
    },
    "v15": {
        "입찰추정가격",
        "세부품명번호목록",
    },
    "v16": {
        "입찰추정가격",
        "세부품명번호목록",
    },
    "v17": {
        "입찰추정가격",
        "세부품명번호목록",
    },
    "v18": {
        "입찰추정가격",
        "세부품명번호목록",
    },

    # SW
    "v20": {
        "정보화사업여부",
        "입찰추정가격",
    },

    # 공동계약
    "v21": {
        "공동도급구성방식",
        "적용계약법",
    },

    # 설명회
    "v22": {
        "낙찰방법",
    },

    "v23": {
        "적용계약법",
        "낙찰방법",
        "공고게시일자",
    },

    # 비교
    "v24": {
        "계약방법",
        "지역제한여부",
        "업종제한여부",
    },
}


audit_rows = []
failures = []


for v in EXPECTED_ITEMS:

    spec = specs[v]

    decision = spec[
        "decision_spec"
    ]

    issues = []
    warnings = []


    # --------------------------------------------------------
    # absence
    # --------------------------------------------------------

    expected_absence = (
        v in ABSENCE_ITEMS
    )

    if (
        bool(
            spec[
                "absence_detection"
            ]
        )
        != expected_absence
    ):
        issues.append(
            "absence_detection mismatch"
        )


    # --------------------------------------------------------
    # 핵심 판정조건
    # --------------------------------------------------------

    for key in [
        "violation_condition",
        "nonviolation_condition",
    ]:

        if not str(
            decision.get(
                key,
                ""
            )
        ).strip():

            issues.append(
                f"{key} missing"
            )


    # --------------------------------------------------------
    # required_meta
    # --------------------------------------------------------

    actual_meta = set(
        decision.get(
            "required_meta",
            []
        )
    )

    expected_meta = (
        REQUIRED_META_EXPECTATIONS.get(
            v,
            set()
        )
    )

    missing_meta = (
        expected_meta
        -
        actual_meta
    )

    if missing_meta:

        issues.append(
            "missing required_meta: "
            + ",".join(
                sorted(
                    missing_meta
                )
            )
        )


    # --------------------------------------------------------
    # 법령 grounding
    # --------------------------------------------------------

    legal_keys = spec[
        "exact_legal_keys"
    ]

    legal_text = spec[
        "exact_legal_text"
    ]

    if v == "v24":

        if legal_keys != []:
            issues.append(
                "v24 legal_keys must be empty"
            )

        if legal_text != {}:
            issues.append(
                "v24 legal_text must be empty"
            )

    else:

        if len(legal_keys) == 0:
            issues.append(
                "legal grounding empty"
            )

        if (
            set(legal_keys)
            !=
            set(
                legal_text.keys()
            )
        ):
            issues.append(
                "legal key/text mismatch"
            )


    # --------------------------------------------------------
    # structured source
    # --------------------------------------------------------

    if v in {
        "v10",
        "v11",
        "v12",
        "v13",
        "v14",
        "v15",
        "v16",
        "v17",
        "v18",
    }:

        if (
            len(
                spec.get(
                    "structured_sources",
                    []
                )
            )
            == 0
        ):

            issues.append(
                "SME structured source missing"
            )


    # --------------------------------------------------------
    # known unresolved는 FAIL이 아니라 이후 검증대상
    # --------------------------------------------------------

    unresolved = decision.get(
        "unresolved",
        []
    )

    if unresolved:
        warnings.append(
            f"unresolved={len(unresolved)}"
        )


    status = (
        "FAIL"
        if issues
        else
        "PASS"
    )


    if issues:
        failures.append(
            (
                v,
                issues
            )
        )


    audit_rows.append({

        "item":
            v,

        "name":
            spec["name"],

        "status":
            status,

        "absence":
            expected_absence,

        "positive":
            DEV_POSITIVE_COUNTS[v],

        "legal_blocks":
            len(
                legal_keys
            ),

        "required_meta":
            len(
                actual_meta
            ),

        "unresolved":
            len(
                unresolved
            ),

        "issues":
            " | ".join(
                issues
            ),

        "warnings":
            " | ".join(
                warnings
            ),
    })


audit_df = pd.DataFrame(
    audit_rows
)


assert not failures, (
    "Stage 6-4 item-level FAIL:\n"
    +
    "\n".join(
        f"{v}: {issues}"
        for v, issues
        in failures
    )
)


print("[4] ITEM SEMANTIC REQUIREMENTS PASS")


# ============================================================
# 4. DEV CO-OCCURRENCE
#
# 중요:
# 동시양성 여부를 확인만 한다.
# 0이라고 hard mutex를 만들지는 않는다.
# ============================================================

RELATED_PAIRS = [

    # 자격/실적/지역
    ("v2", "v4"),
    ("v5", "v6"),
    ("v5", "v7"),
    ("v6", "v7"),
    ("v8", "v2"),
    ("v8", "v3"),
    ("v8", "v4"),
    ("v8", "v5"),
    ("v8", "v6"),
    ("v8", "v7"),

    # 제품
    ("v10", "v12"),
    ("v11", "v13"),

    # 일반제품 가격구간
    ("v14", "v15"),
    ("v14", "v16"),
    ("v14", "v17"),
    ("v14", "v18"),
    ("v15", "v16"),
    ("v17", "v18"),

    # 설명회
    ("v22", "v23"),
]


co_rows = []


for a, b in RELATED_PAIRS:

    both = int(
        (
            (
                labels[a]
                .astype(int)
                == 1
            )
            &
            (
                labels[b]
                .astype(int)
                == 1
            )
        ).sum()
    )

    co_rows.append({

        "item_a":
            a,

        "item_b":
            b,

        "positive_a":
            DEV_POSITIVE_COUNTS[a],

        "positive_b":
            DEV_POSITIVE_COUNTS[b],

        "both_positive":
            both,
    })


co_df = pd.DataFrame(
    co_rows
)


print("[5] DEV CO-OCCURRENCE 계산 PASS")


# ============================================================
# 5. FAMILY CONSISTENCY MANIFEST
#
# 실제 inference에서 사용할 의미 관계.
#
# notice-level hard mutex가 아님.
# 한 공고에 여러 품목이 있을 수 있기 때문.
# ============================================================

CONSISTENCY_RULES = {

    "version":
        "stage6-4",

    "principles": {

        "no_notice_level_mutex": (
            "한 공고에 여러 품목/조건이 있을 수 있으므로 "
            "항목 간 무조건적인 notice-level mutex를 적용하지 않는다."
        ),

        "apply_relations_after_fact_extraction": (
            "상품구분, 금액구간, 계약법, 계약방법 등 "
            "Stage 7 structured facts가 확정된 뒤에만 "
            "항목 간 정합성 규칙을 적용한다."
        ),

        "dev_zero_overlap_is_not_proof": (
            "DEV 200건에서 동시양성이 0이어도 "
            "논리적으로 불가능하다고 자동 간주하지 않는다."
        ),
    },


    # --------------------------------------------------------
    # qualification / region
    # --------------------------------------------------------

    "qualification_region": {

        "v2_v4": (
            "DEV에서 중첩 가능성이 있으므로 "
            "고시금액만으로 상호배제시키지 않는다."
        ),

        "v5_v6_v7": (
            "가격구간과 지역제한 유형을 각각 구조화한 뒤 판단한다."
        ),

        "v8": (
            "실적제한 + 지역제한의 동시 존재 항목이므로 "
            "v2~v7과 동시양성이 가능하다."
        ),
    },


    # --------------------------------------------------------
    # product classification
    # --------------------------------------------------------

    "product_classification": {

        "competitive_product_items": [
            "v10",
            "v11",
            "v13",
        ],

        "general_product_items": [
            "v12",
            "v14",
            "v15",
            "v16",
            "v17",
            "v18",
        ],

        "rule": (
            "세부품명번호 단위로 경쟁제품 여부를 먼저 판정한다. "
            "공고 전체를 하나의 상품종류로 단정하지 않는다."
        ),
    },


    # --------------------------------------------------------
    # price bands
    # --------------------------------------------------------

    "general_product_price_bands": {

        "high": {
            "condition":
                "price >= applicable_notice_threshold",

            "items": [
                "v14"
            ],
        },

        "middle": {
            "condition":
                "100000000 <= price < applicable_notice_threshold",

            "items": [
                "v15",
                "v16",
            ],
        },

        "low": {
            "condition":
                "price < 100000000",

            "items": [
                "v17",
                "v18",
            ],
        },

        "rule": (
            "동일 판정대상 물품/금액에 대해서는 하나의 금액구간만 "
            "적용한다. 단, 공고에 여러 품목이 존재하면 공고 수준에서 "
            "여러 구간이 동시에 나타날 수 있으므로 강제 mutex 금지."
        ),
    },


    # --------------------------------------------------------
    # absence handling
    # --------------------------------------------------------

    "absence_detection": {

        "items":
            sorted(
                ABSENCE_ITEMS,
                key=lambda x:
                    int(
                        x[1:]
                    )
            ),

        "evidence":
            "always empty",

        "incomplete_docs_rule": (
            "dropped_doc_counts 또는 완전관측=False이면 "
            "단순 문구부재만으로 violation=1을 확정하지 않는다."
        ),
    },


    # --------------------------------------------------------
    # SW
    # --------------------------------------------------------

    "software": {

        "item":
            "v20",

        "required_sequence": [
            "SW/정보화사업 적용 여부",
            "사업금액",
            "사업자 규모 제한 기준",
            "실제 참가자격 문구 존재 여부",
            "예외 확인",
        ],
    },


    # --------------------------------------------------------
    # site briefing
    # --------------------------------------------------------

    "site_briefing": {

        "v22": (
            "협상계약에서 설명회 참석 자체를 "
            "입찰참가자격으로 제한하는지 판단"
        ),

        "v23": (
            "지방계약 + 협상계약에서 "
            "공고/설명회 관련 최소기간 위반 판단"
        ),

        "coexistence":
            (
                "v22와 v23은 서로 다른 위반이므로 "
                "동시양성 가능"
            ),
    },


    # --------------------------------------------------------
    # comparison
    # --------------------------------------------------------

    "v24": {

        "type":
            "docs_meta_comparison",

        "legal_rag":
            False,

        "compare_fields": [
            "예산/추정가격",
            "계약방법",
            "지역제한",
            "업종/면허제한",
        ],
    },
}


# ============================================================
# 6. GLOBAL SPEC CONSISTENCY
# ============================================================

# absence 정확히 5개
actual_absence = {
    v
    for v in EXPECTED_ITEMS
    if specs[v][
        "absence_detection"
    ]
}


assert (
    actual_absence
    ==
    ABSENCE_ITEMS
)


# v24 comparison
assert (
    specs[
        "v24"
    ][
        "decision_spec"
    ][
        "status"
    ]
    ==
    "REVIEWED_COMPARISON"
)


assert (
    specs[
        "v24"
    ][
        "exact_legal_keys"
    ]
    ==
    []
)


# 모든 spec pending 없음
pending = [

    v
    for v in EXPECTED_ITEMS

    if specs[v][
        "decision_spec"
    ][
        "status"
    ]
    ==
    "PENDING_FINAL_REVIEW"
]


assert pending == []


# 전체 positive
assert (
    sum(
        specs[v][
            "dev"
        ][
            "positive_count"
        ]
        for v
        in EXPECTED_ITEMS
    )
    ==
    153
)


print("[6] GLOBAL CONSISTENCY PASS")


# ============================================================
# 7. SAVE ARTIFACTS
# ============================================================

with open(
    OUT_RULES,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "consistency_rules":
                CONSISTENCY_RULES,

            "dev_cooccurrence":
                co_rows,
        },
        f,
        ensure_ascii=False,
        indent=2
    )


audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


STAGE6_MANIFEST = {

    "stage":
        "6",

    "status":
        "PASS",

    "items":
        24,

    "absence_items":
        sorted(
            ABSENCE_ITEMS,
            key=lambda x:
                int(
                    x[1:]
                )
        ),

    "dev_positive_total":
        153,

    "pending_specs":
        [],

    "artifacts": {

        "integrated_specs":
            str(
                SPEC_PATH
            ),

        "consistency_rules":
            str(
                OUT_RULES
            ),

        "consistency_audit":
            str(
                OUT_AUDIT
            ),
    },

    "next_stage":
        "Stage 7 - structured information extraction",
}


with open(
    OUT_STAGE6,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        STAGE6_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 8. RELOAD VALIDATION
# ============================================================

with open(
    OUT_RULES,
    "r",
    encoding="utf-8"
) as f:
    saved_rules = json.load(f)


with open(
    OUT_STAGE6,
    "r",
    encoding="utf-8"
) as f:
    saved_manifest = json.load(f)


assert (
    saved_manifest[
        "status"
    ]
    ==
    "PASS"
)


assert (
    saved_manifest[
        "items"
    ]
    ==
    24
)


assert (
    saved_manifest[
        "dev_positive_total"
    ]
    ==
    153
)


assert (
    len(
        saved_rules[
            "dev_cooccurrence"
        ]
    )
    ==
    len(
        RELATED_PAIRS
    )
)


print("[7] SAVE / RELOAD PASS")


# ============================================================
# 9. DISPLAY
# ============================================================

display(
    audit_df
)

print()
print("RELATED DEV CO-OCCURRENCE")
display(
    co_df
)


# ============================================================
# 10. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 6-4 FINAL RESULT")
print("=" * 100)

print(
    "ITEMS               =",
    len(
        specs
    )
)

print(
    "ITEM FAILURES       =",
    len(
        failures
    )
)

print(
    "ABSENCE ITEMS       =",
    sorted(
        actual_absence,
        key=lambda x:
            int(
                x[1:]
            )
    )
)

print(
    "TOTAL POSITIVES     =",
    sum(
        DEV_POSITIVE_COUNTS.values()
    )
)

print(
    "PENDING SPECS       =",
    pending
)

print(
    "RELATED PAIRS AUDIT =",
    len(
        RELATED_PAIRS
    )
)

print(
    "RULES JSON          =",
    OUT_RULES
)

print(
    "AUDIT CSV           =",
    OUT_AUDIT
)

print(
    "STAGE6 MANIFEST     =",
    OUT_STAGE6
)

print()
print(
    "[PASS] Stage 6 전체 종료"
)

print(
    "NEXT = Stage 7 structured information extraction"
)

[1] INPUT LOAD PASS
[2] DEV LABEL / POSITIVE COUNT PASS
[3] ABSENCE EVIDENCE POLICY PASS
[4] ITEM SEMANTIC REQUIREMENTS PASS
[5] DEV CO-OCCURRENCE 계산 PASS
[6] GLOBAL CONSISTENCY PASS
[7] SAVE / RELOAD PASS


,item,name,status,absence,positive,legal_blocks,required_meta,unresolved,issues,warnings
0,v1,참가자격 특정기관 제한,PASS,False,7,4,5,0,,
1,v2,고시금액 미만 실적제한,PASS,False,7,5,5,1,,unresolved=1
2,v3,실적제한 1배수 이상,PASS,False,8,5,4,1,,unresolved=1
3,v4,"고시금액 이상 특정기관, 특정실적",PASS,False,6,2,4,1,,unresolved=1
4,v5,고시금액 이상 지역제한,PASS,False,7,4,6,2,,unresolved=2
5,v6,"고시금액 미만 지역제한 시,군,구",PASS,False,6,3,5,0,,
6,v7,고시금액 미만 지역제한 인접 확대,PASS,False,7,3,5,1,,unresolved=1
7,v8,중복제한 (실적+지역),PASS,False,6,3,5,1,,unresolved=1
8,v9,과업지시서 특정 모델명 명시,PASS,False,6,2,2,0,,
9,v10,중기간 경쟁제품 입찰 직생 없음,PASS,True,7,1,2,1,,unresolved=1



RELATED DEV CO-OCCURRENCE


,item_a,item_b,positive_a,positive_b,both_positive
0,v2,v4,7,6,3
1,v5,v6,7,6,0
2,v5,v7,7,7,0
3,v6,v7,6,7,1
4,v8,v2,6,7,1
5,v8,v3,6,8,3
6,v8,v4,6,6,1
7,v8,v5,6,7,1
8,v8,v6,6,6,1
9,v8,v7,6,7,1



Stage 6-4 FINAL RESULT
ITEMS               = 24
ITEM FAILURES       = 0
ABSENCE ITEMS       = ['v10', 'v11', 'v16', 'v18', 'v20']
TOTAL POSITIVES     = 153
PENDING SPECS       = []
RELATED PAIRS AUDIT = 19
RULES JSON          = /home/alparka/nara-ai/artifacts/judgment_spec_consistency_rules.json
AUDIT CSV           = /home/alparka/nara-ai/artifacts/judgment_spec_consistency_audit.csv
STAGE6 MANIFEST     = /home/alparka/nara-ai/artifacts/stage6_final_manifest.json

[PASS] Stage 6 전체 종료
NEXT = Stage 7 structured information extraction


In [3]:
# ============================================================
# Stage 7 FINAL
# Structured Information Extraction
#
# 목적:
#   입찰공고 1건 → v1~v24 판정에 필요한 사실 후보 구조화
#
# 중요:
# - 여기서는 violation을 판정하지 않는다.
# - regex miss = 조건 없음 으로 간주하지 않는다.
# - 모든 snippet은 반드시 원문 exact substring.
# - 모든 docs/attachments를 사용한다.
# - meta null / "미입력"을 임의 의미로 변환하지 않는다.
# - 중기간 경쟁제품은 공식 배포 CSV exact lookup.
#
# Output:
#   artifacts/structured_facts_dev.jsonl
#   artifacts/structured_extraction_audit.csv
#   artifacts/structured_extraction_positive_hint_audit.csv
#   artifacts/stage7_manifest.json
# ============================================================

from pathlib import Path
from datetime import datetime
import gzip
import json
import re
import math
import hashlib
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

DEV_PATH = (
    ROOT
    / "open"
    / "dev.jsonl.gz"
)

LABEL_PATH = (
    ROOT
    / "open"
    / "dev_labels.csv"
)

SPEC_PATH = (
    ROOT
    / "artifacts"
    / "judgment_specs_v1_v24.json"
)

SME_CSV_PATH = (
    ROOT
    / "open"
    / "data"
    / "법령패키지"
    / "중기부고시"
    / "중기부고시_경쟁제품_세부품명.csv"
)

OUT_JSONL = (
    ROOT
    / "artifacts"
    / "structured_facts_dev.jsonl"
)

OUT_AUDIT = (
    ROOT
    / "artifacts"
    / "structured_extraction_audit.csv"
)

OUT_HINT_AUDIT = (
    ROOT
    / "artifacts"
    / "structured_extraction_positive_hint_audit.csv"
)

OUT_MANIFEST = (
    ROOT
    / "artifacts"
    / "stage7_manifest.json"
)


for path in [
    DEV_PATH,
    LABEL_PATH,
    SPEC_PATH,
    SME_CSV_PATH,
]:

    assert path.exists(), (
        f"필수 파일 없음: {path}"
    )


print("[1] PATH PASS")


# ============================================================
# 1. LOAD DEV / LABEL / SPECS
# ============================================================

def load_jsonl_gz(path):

    rows = []

    with gzip.open(
        path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line_no, line in enumerate(
            f,
            start=1
        ):

            line = line.strip()

            if not line:
                continue

            obj = json.loads(
                line
            )

            rows.append(
                obj
            )

    return rows


dev = load_jsonl_gz(
    DEV_PATH
)

labels = pd.read_csv(
    LABEL_PATH
)


with open(
    SPEC_PATH,
    "r",
    encoding="utf-8"
) as f:

    specs = json.load(f)


EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(
        1,
        25
    )
]


assert len(dev) == 200
assert len(labels) == 200
assert list(specs.keys()) == EXPECTED_ITEMS


dev_ids = [
    x["id"]
    for x in dev
]

label_ids = (
    labels["id"]
    .astype(str)
    .tolist()
)


assert dev_ids == label_ids
assert len(set(dev_ids)) == 200


print("[2] DEV / LABEL / SPEC LOAD PASS")


# ============================================================
# 2. META SCHEMA VALIDATION
# ============================================================

EXPECTED_META_FIELDS = {

    "적용계약법",
    "업무구분",
    "계약방법",
    "낙찰방법",
    "낙찰하한율",
    "배정예산금액",
    "입찰추정가격",
    "소관구분",
    "공동도급구성방식",
    "정보화사업여부",
    "세부품명번호목록",
    "제한지역코드목록",
    "지역제한여부",
    "면허업종제한목록",
    "업종제한여부",
    "조항호내용",
    "공고게시일자",
    "개찰예정일자",
    "긴급공고여부",
    "입찰방법",
    "조달방식",
}


# spec이 요구하는 모든 meta field
REQUIRED_META_UNION = set()


for v in EXPECTED_ITEMS:

    REQUIRED_META_UNION.update(

        specs[v][
            "decision_spec"
        ].get(
            "required_meta",
            []
        )
    )


assert (
    REQUIRED_META_UNION
    <=
    EXPECTED_META_FIELDS
), (
    "spec에서 정의되지 않은 meta를 요구함:\n"
    f"{sorted(REQUIRED_META_UNION - EXPECTED_META_FIELDS)}"
)


for row in dev:

    meta = row[
        "meta"
    ]

    missing = (
        EXPECTED_META_FIELDS
        -
        set(
            meta.keys()
        )
    )

    assert not missing, (
        f"{row['id']}: meta field 누락 "
        f"{sorted(missing)}"
    )


print(
    "[3] META SCHEMA PASS"
)

print(
    "    required meta =",
    sorted(
        REQUIRED_META_UNION
    )
)


# ============================================================
# 3. SME COMPETITIVE PRODUCT CSV
# ============================================================

def read_csv_flexible(path):

    last_error = None

    for enc in [
        "utf-8-sig",
        "utf-8",
        "cp949",
    ]:

        try:

            return pd.read_csv(
                path,
                dtype=str,
                encoding=enc
            )

        except Exception as e:

            last_error = e

    raise last_error


sme_df = read_csv_flexible(
    SME_CSV_PATH
)


assert len(sme_df) > 0


CODE_COLUMNS = [

    col
    for col
    in sme_df.columns

    if (
        "품명번호"
        in str(col)
    )
]


assert CODE_COLUMNS, (
    f"품명번호 column 없음: "
    f"{list(sme_df.columns)}"
)


def normalize_product_code(value):

    if value is None:
        return None

    s = re.sub(
        r"\D",
        "",
        str(value)
    )

    if len(s) < 8:
        return None

    return s


# ------------------------------------------------------------
# code → CSV row indices
# ------------------------------------------------------------

PRODUCT_CODE_INDEX = {}


for idx, row in sme_df.iterrows():

    for col in CODE_COLUMNS:

        code = normalize_product_code(
            row.get(
                col
            )
        )

        if not code:
            continue

        PRODUCT_CODE_INDEX.setdefault(
            code,
            []
        ).append(
            int(idx)
        )


assert PRODUCT_CODE_INDEX


print(
    "[4] SME CSV PASS"
)

print(
    "    rows        =",
    len(sme_df)
)

print(
    "    code columns=",
    CODE_COLUMNS
)

print(
    "    unique codes=",
    len(
        PRODUCT_CODE_INDEX
    )
)


# ============================================================
# 4. RAW DOCUMENT BLOCK EXTRACTION
#
# 중요한 부분:
# snippet은 원문에서 그대로 slice.
# later evidence extraction에 재사용 가능.
# ============================================================

def line_spans(text):

    """
    반환:
    [
        {
            line_no,
            start,
            end,
            text
        }
    ]

    start/end는 원문 offset.
    """

    spans = []

    offset = 0

    lines = text.splitlines(
        keepends=True
    )


    # splitlines 결과가 빈 경우
    if not lines and text:

        lines = [
            text
        ]


    for i, line in enumerate(
        lines
    ):

        start = offset

        end = (
            start
            +
            len(line)
        )

        spans.append({

            "line_no":
                i + 1,

            "start":
                start,

            "end":
                end,

            "text":
                line,
        })

        offset = end


    return spans


# ============================================================
# 5. CANDIDATE PATTERNS
#
# 이것들은 "판정규칙"이 아님.
# 관련 문맥을 놓치지 않기 위한 recall-oriented 후보검색.
# ============================================================

CATEGORY_PATTERNS = {

    # v1
    "qualification": [
        r"입찰\s*참가\s*자격",
        r"입찰참가자격",
        r"참가\s*자격",
        r"자격\s*요건",
        r"자격요건",
    ],


    # v2/v3/v4/v8
    "performance": [
        r"실적",
        r"이행\s*실적",
        r"수행\s*실적",
        r"용역\s*실적",
        r"납품\s*실적",
        r"계약\s*실적",
    ],


    # v5/v6/v7/v8/v24
    "region": [
        r"지역\s*제한",
        r"지역제한",
        r"본점\s*소재지",
        r"주된\s*영업소",
        r"사업장\s*소재지",
        r"소재지",
        r"관할\s*(?:시|군|구|도)",
    ],


    # v9
    "product_model": [
        r"모델\s*명",
        r"모델명",
        r"모델\s*번호",
        r"제품\s*번호",
        r"품\s*번",
        r"품번",
        r"제조\s*사",
        r"제조사",
        r"브랜드",
        r"상표",
    ],


    # v10/v12
    "direct_production": [
        r"직접\s*생산",
        r"직접생산",
        r"직접\s*생산\s*확인",
        r"직접생산확인",
    ],


    # v11/v13~18
    "enterprise": [
        r"중소기업자?",
        r"중기업",
        r"소기업",
        r"소상공인",
        r"중소\s*기업",
        r"소기업\s*[·ㆍ,\s및]*\s*소상공인",
    ],


    # v19
    "supply_commitment": [
        r"물품\s*공급",
        r"물품공급",
        r"기술\s*지원",
        r"기술지원",
        r"확약서",
        r"협약서",
        r"공급\s*확인서",
        r"공급확인서",
    ],


    # v20
    "software": [
        r"소프트웨어",
        r"\bSW\b",
        r"정보화\s*사업",
        r"정보화사업",
        r"대기업\s*참여",
        r"중소\s*소프트웨어",
    ],


    # v21
    "joint_contract": [
        r"공동\s*수급",
        r"공동수급",
        r"공동\s*도급",
        r"공동도급",
        r"공동\s*이행",
        r"공동이행",
        r"분담\s*이행",
        r"분담이행",
        r"공동\s*계약",
        r"공동계약",
        r"출자\s*비율",
        r"출자비율",
        r"분담\s*비율",
        r"분담비율",
        r"지분율",
    ],


    # v22/v23
    "briefing": [
        r"현장\s*설명",
        r"현장설명",
        r"제안요청\s*설명",
        r"제안\s*설명회",
        r"설명회",
    ],


    # 일정
    "deadline": [
        r"공고\s*기간",
        r"제안서\s*제출",
        r"입찰서\s*제출",
        r"제출\s*마감",
        r"마감\s*일",
        r"개찰",
    ],


    # v24 계약방법 비교
    "contract_method": [
        r"일반\s*경쟁",
        r"제한\s*경쟁",
        r"지명\s*경쟁",
        r"수의\s*계약",
        r"협상에\s*의한\s*계약",
        r"협상\s*계약",
        r"적격\s*심사",
        r"최저가",
    ],


    # v24 업종/면허
    "industry": [
        r"업종",
        r"면허",
        r"등록업체",
        r"등록\s*업체",
        r"허가",
        r"인가",
    ],


    # 금액
    "amount": [
        r"추정\s*가격",
        r"추정가격",
        r"사업\s*예산",
        r"사업예산",
        r"배정\s*예산",
        r"예산\s*금액",
        r"예산액",
        r"사업\s*금액",
        r"사업금액",
        r"사업비",
        r"기초\s*금액",
        r"기초금액",
        r"예정\s*가격",
        r"예정가격",
    ],
}


COMPILED_PATTERNS = {

    category: [
        re.compile(
            pattern,
            re.I
        )
        for pattern
        in patterns
    ]

    for category, patterns
    in CATEGORY_PATTERNS.items()
}


# ============================================================
# 6. BLOCK EXTRACTOR
# ============================================================

def extract_blocks_from_doc(
    doc,
    compiled_patterns,
    radius_lines=1,
    max_blocks=30
):

    text = (
        doc.get(
            "text"
        )
        or
        ""
    )

    if not text:
        return []


    spans = line_spans(
        text
    )


    hit_line_indices = []


    for i, line in enumerate(
        spans
    ):

        line_text = line[
            "text"
        ]

        if any(
            p.search(
                line_text
            )
            for p
            in compiled_patterns
        ):

            hit_line_indices.append(
                i
            )


    blocks = []

    seen = set()


    for i in hit_line_indices:

        left = max(
            0,
            i - radius_lines
        )

        right = min(
            len(spans) - 1,
            i + radius_lines
        )


        start = spans[
            left
        ][
            "start"
        ]

        end = spans[
            right
        ][
            "end"
        ]


        raw = text[
            start:end
        ]


        # 원문 exact substring 검증
        assert (
            text[
                start:end
            ]
            ==
            raw
        )


        key = (
            start,
            end
        )


        if key in seen:
            continue


        seen.add(
            key
        )


        matched_terms = []


        for pattern in compiled_patterns:

            for m in pattern.finditer(
                raw
            ):

                term = m.group(0)

                if term not in matched_terms:

                    matched_terms.append(
                        term
                    )


        blocks.append({

            "doc_id":
                doc.get(
                    "doc_id"
                ),

            "doc_type":
                doc.get(
                    "type"
                ),

            "start":
                start,

            "end":
                end,

            "start_line":
                spans[
                    left
                ][
                    "line_no"
                ],

            "end_line":
                spans[
                    right
                ][
                    "line_no"
                ],

            "matched_terms":
                matched_terms,

            # exact substring
            "text":
                raw,
        })


        if len(
            blocks
        ) >= max_blocks:

            break


    return blocks


def collect_category(
    docs,
    category
):

    pattern_list = (
        COMPILED_PATTERNS[
            category
        ]
    )

    result = []


    for doc in docs:

        blocks = (
            extract_blocks_from_doc(
                doc,
                pattern_list,
                radius_lines=1,
                max_blocks=30
            )
        )

        result.extend(
            blocks
        )


    return result


# ============================================================
# 7. PRODUCT CODES
# ============================================================

def extract_meta_product_codes(
    value
):

    if value is None:
        return []


    if isinstance(
        value,
        list
    ):

        text = " ".join(
            str(x)
            for x in value
            if x is not None
        )

    else:

        text = str(
            value
        )


    raw_codes = re.findall(
        r"(?<!\d)"
        r"\d{8,12}"
        r"(?!\d)",
        text
    )


    result = []


    for x in raw_codes:

        code = normalize_product_code(
            x
        )

        if (
            code
            and
            code not in result
        ):

            result.append(
                code
            )


    return result


def competitive_product_lookup(
    codes
):

    if not codes:

        return {
            "status":
                "UNKNOWN_NO_CODE",

            "is_competitive_product":
                None,

            "matched_codes":
                [],

            "unmatched_codes":
                [],

            "matched_rows":
                [],
        }


    matched = []
    unmatched = []
    matched_rows = []


    seen_rows = set()


    for code in codes:

        row_ids = (
            PRODUCT_CODE_INDEX.get(
                code,
                []
            )
        )


        if row_ids:

            matched.append(
                code
            )

            for idx in row_ids:

                if idx in seen_rows:
                    continue

                seen_rows.add(
                    idx
                )

                row = (
                    sme_df.loc[
                        idx
                    ]
                    .where(
                        pd.notna(
                            sme_df.loc[
                                idx
                            ]
                        ),
                        None
                    )
                    .to_dict()
                )

                matched_rows.append(
                    row
                )

        else:

            unmatched.append(
                code
            )


    return {

        "status":
            (
                "MATCH"
                if matched
                else
                "NO_MATCH"
            ),

        "is_competitive_product":
            bool(
                matched
            ),

        "matched_codes":
            matched,

        "unmatched_codes":
            unmatched,

        "matched_rows":
            matched_rows,
    }


# ============================================================
# 8. MONEY EXTRACTION
#
# 여기서 복합 금액을 임의 합산하지 않는다.
#
# 예:
# 3억 5천만원
# →
# 3억 후보
# 5천만원 후보
#
# Stage 7은 후보 추출.
# Stage 7/8 이후 semantic layer에서 문맥결합.
# ============================================================

MONEY_RE = re.compile(

    r"(?<!\d)"

    r"(?P<num>"
    r"\d[\d,]*"
    r"(?:\.\d+)?"
    r")"

    r"\s*"

    r"(?P<unit>"
    r"억원|억|천만원|백만원|만원|천원|원"
    r")"
)


MONEY_UNIT = {

    "억원":
        100_000_000,

    "억":
        100_000_000,

    "천만원":
        10_000_000,

    "백만원":
        1_000_000,

    "만원":
        10_000,

    "천원":
        1_000,

    "원":
        1,
}


AMOUNT_LABELS = [

    "추정가격",
    "사업예산",
    "배정예산",
    "예산금액",
    "예산액",
    "사업금액",
    "사업비",
    "기초금액",
    "예정가격",
]


def amount_label(
    text
):

    compacted = re.sub(
        r"\s+",
        "",
        text
    )


    matched = [

        label
        for label
        in AMOUNT_LABELS

        if label
        in compacted
    ]


    return matched


def extract_money_candidates(
    blocks
):

    result = []


    for block in blocks:

        raw = block[
            "text"
        ]


        for m in MONEY_RE.finditer(
            raw
        ):

            raw_number = (
                m.group(
                    "num"
                )
                .replace(
                    ",",
                    ""
                )
            )

            unit = m.group(
                "unit"
            )


            try:

                number = float(
                    raw_number
                )

                value = int(
                    round(
                        number
                        *
                        MONEY_UNIT[
                            unit
                        ]
                    )
                )

            except Exception:

                value = None


            result.append({

                "doc_id":
                    block[
                        "doc_id"
                    ],

                "doc_type":
                    block[
                        "doc_type"
                    ],

                "raw":
                    m.group(0),

                "normalized_won":
                    value,

                "labels":
                    amount_label(
                        raw
                    ),

                "context":
                    raw,
            })


    return result


# ============================================================
# 9. DATE EXTRACTION
# ============================================================

DATE_PATTERNS = [

    re.compile(
        r"(?P<y>20\d{2})"
        r"[.\-/]"
        r"(?P<m>\d{1,2})"
        r"[.\-/]"
        r"(?P<d>\d{1,2})"
    ),

    re.compile(
        r"(?P<y>20\d{2})\s*년\s*"
        r"(?P<m>\d{1,2})\s*월\s*"
        r"(?P<d>\d{1,2})\s*일"
    ),
]


def normalize_date_match(
    match
):

    try:

        y = int(
            match.group(
                "y"
            )
        )

        m = int(
            match.group(
                "m"
            )
        )

        d = int(
            match.group(
                "d"
            )
        )


        dt = datetime(
            y,
            m,
            d
        )


        return dt.strftime(
            "%Y-%m-%d"
        )

    except Exception:

        return None


def extract_date_candidates(
    blocks
):

    result = []

    seen = set()


    for block in blocks:

        raw = block[
            "text"
        ]


        for pattern in DATE_PATTERNS:

            for m in pattern.finditer(
                raw
            ):

                iso = normalize_date_match(
                    m
                )


                key = (
                    block[
                        "doc_id"
                    ],
                    m.group(0),
                    iso,
                )


                if key in seen:
                    continue


                seen.add(
                    key
                )


                result.append({

                    "doc_id":
                        block[
                            "doc_id"
                        ],

                    "doc_type":
                        block[
                            "doc_type"
                        ],

                    "raw":
                        m.group(0),

                    "iso_date":
                        iso,

                    "context":
                        raw,
                })


    return result


# ============================================================
# 10. PERCENTAGE EXTRACTION
# ============================================================

PERCENT_RE = re.compile(
    r"(?<!\d)"
    r"(\d+(?:\.\d+)?)"
    r"\s*%"
)


HUNDREDTH_RE = re.compile(
    r"100\s*분의\s*"
    r"(\d+(?:\.\d+)?)"
)


def extract_percentage_candidates(
    blocks
):

    result = []


    for block in blocks:

        raw = block[
            "text"
        ]


        for m in PERCENT_RE.finditer(
            raw
        ):

            result.append({

                "doc_id":
                    block[
                        "doc_id"
                    ],

                "doc_type":
                    block[
                        "doc_type"
                    ],

                "raw":
                    m.group(0),

                "percent":
                    float(
                        m.group(1)
                    ),

                "context":
                    raw,
            })


        for m in HUNDREDTH_RE.finditer(
            raw
        ):

            result.append({

                "doc_id":
                    block[
                        "doc_id"
                    ],

                "doc_type":
                    block[
                        "doc_type"
                    ],

                "raw":
                    m.group(0),

                "percent":
                    float(
                        m.group(1)
                    ),

                "context":
                    raw,
            })


    return result


# ============================================================
# 11. EXACT BLOCK VALIDATION
# ============================================================

def validate_exact_blocks(
    docs,
    categories
):

    doc_map = {

        doc.get(
            "doc_id"
        ):
            doc.get(
                "text"
            )
            or
            ""

        for doc
        in docs
    }


    checked = 0


    for blocks in categories.values():

        for block in blocks:

            doc_id = block[
                "doc_id"
            ]

            assert doc_id in doc_map


            original = doc_map[
                doc_id
            ]


            start = block[
                "start"
            ]

            end = block[
                "end"
            ]


            assert (
                original[
                    start:end
                ]
                ==
                block[
                    "text"
                ]
            ), (
                f"exact substring mismatch: "
                f"{doc_id}"
            )


            checked += 1


    return checked


# ============================================================
# 12. BUILD STRUCTURED FACTS
# ============================================================

def build_structured_facts(
    row
):

    row_id = row[
        "id"
    ]

    docs = row.get(
        "docs",
        []
    )

    meta = row.get(
        "meta",
        {}
    )


    # --------------------------------------------------------
    # candidate blocks
    # --------------------------------------------------------

    categories = {

        category:
            collect_category(
                docs,
                category
            )

        for category
        in CATEGORY_PATTERNS
    }


    exact_blocks_checked = (
        validate_exact_blocks(
            docs,
            categories
        )
    )


    # --------------------------------------------------------
    # product classification
    # --------------------------------------------------------

    product_codes = (
        extract_meta_product_codes(
            meta.get(
                "세부품명번호목록"
            )
        )
    )


    product_lookup = (
        competitive_product_lookup(
            product_codes
        )
    )


    # --------------------------------------------------------
    # numeric/date candidates
    # --------------------------------------------------------

    money_candidates = (
        extract_money_candidates(
            categories[
                "amount"
            ]
        )
    )


    date_source_blocks = (

        categories[
            "briefing"
        ]
        +
        categories[
            "deadline"
        ]
    )


    date_candidates = (
        extract_date_candidates(
            date_source_blocks
        )
    )


    joint_percentages = (
        extract_percentage_candidates(
            categories[
                "joint_contract"
            ]
        )
    )


    # --------------------------------------------------------
    # comparison candidates for v24
    # --------------------------------------------------------

    comparison_candidates = {

        "amount":
            money_candidates,

        "contract_method":
            categories[
                "contract_method"
            ],

        "region":
            categories[
                "region"
            ],

        "industry":
            categories[
                "industry"
            ],
    }


    # --------------------------------------------------------
    # simple non-judgmental hints
    # --------------------------------------------------------

    candidate_hints = {

        category:
            bool(
                categories[
                    category
                ]
            )

        for category
        in categories
    }


    result = {

        "id":
            row_id,


        # ----------------------------------------------------
        # 원본 META
        # ----------------------------------------------------

        "meta": {

            "raw":
                meta,

            "core": {

                "contract_law":
                    meta.get(
                        "적용계약법"
                    ),

                "work_type":
                    meta.get(
                        "업무구분"
                    ),

                "contract_method":
                    meta.get(
                        "계약방법"
                    ),

                "award_method":
                    meta.get(
                        "낙찰방법"
                    ),

                "award_lower_rate":
                    meta.get(
                        "낙찰하한율"
                    ),

                "budget_amount":
                    meta.get(
                        "배정예산금액"
                    ),

                "estimated_price":
                    meta.get(
                        "입찰추정가격"
                    ),

                "agency_category":
                    meta.get(
                        "소관구분"
                    ),

                "joint_contract_method":
                    meta.get(
                        "공동도급구성방식"
                    ),

                "is_information_project":
                    meta.get(
                        "정보화사업여부"
                    ),

                "region_restricted":
                    meta.get(
                        "지역제한여부"
                    ),

                "region_codes":
                    meta.get(
                        "제한지역코드목록"
                    ),

                "industry_restricted":
                    meta.get(
                        "업종제한여부"
                    ),

                "industry_codes":
                    meta.get(
                        "면허업종제한목록"
                    ),

                "clause_summary":
                    meta.get(
                        "조항호내용"
                    ),

                "notice_date":
                    meta.get(
                        "공고게시일자"
                    ),

                "opening_date":
                    meta.get(
                        "개찰예정일자"
                    ),

                "urgent":
                    meta.get(
                        "긴급공고여부"
                    ),

                "bid_method":
                    meta.get(
                        "입찰방법"
                    ),

                "procurement_method":
                    meta.get(
                        "조달방식"
                    ),
            },
        },


        # ----------------------------------------------------
        # DOCUMENT INFO
        # ----------------------------------------------------

        "documents": {

            "count":
                len(
                    docs
                ),

            "types": [
                doc.get(
                    "type"
                )
                for doc
                in docs
            ],

            "ids": [
                doc.get(
                    "doc_id"
                )
                for doc
                in docs
            ],

            "lengths": {

                doc.get(
                    "doc_id"
                ):
                    len(
                        doc.get(
                            "text"
                        )
                        or
                        ""
                    )

                for doc
                in docs
            },
        },


        # ----------------------------------------------------
        # COMPLETENESS
        # ----------------------------------------------------

        "completeness": {

            "dropped_doc_counts":
                row.get(
                    "dropped_doc_counts",
                    {}
                ),

            "input_completeness":
                row.get(
                    "input_completeness",
                    {}
                ),

            "fully_observed":
                bool(
                    row.get(
                        "input_completeness",
                        {}
                    ).get(
                        "완전관측",
                        False
                    )
                ),

            "has_dropped_docs":
                bool(
                    row.get(
                        "dropped_doc_counts",
                        {}
                    )
                ),
        },


        # ----------------------------------------------------
        # PRODUCT FACTS
        # ----------------------------------------------------

        "product": {

            "meta_raw":
                meta.get(
                    "세부품명번호목록"
                ),

            "codes":
                product_codes,

            "competitive_lookup":
                product_lookup,
        },


        # ----------------------------------------------------
        # EXACT TEXT CANDIDATES
        # ----------------------------------------------------

        "candidates":
            categories,


        # ----------------------------------------------------
        # NORMALIZED NUMERIC FACTS
        # ----------------------------------------------------

        "numeric_candidates": {

            "document_money":
                money_candidates,

            "joint_percentages":
                joint_percentages,
        },


        # ----------------------------------------------------
        # DATE FACTS
        # ----------------------------------------------------

        "date_candidates":
            date_candidates,


        # ----------------------------------------------------
        # v24 comparison candidates
        # ----------------------------------------------------

        "comparison_candidates":
            comparison_candidates,


        # ----------------------------------------------------
        # 이것은 decision이 아님
        # ----------------------------------------------------

        "candidate_hints":
            candidate_hints,


        # ----------------------------------------------------
        # validation metadata
        # ----------------------------------------------------

        "validation": {

            "exact_blocks_checked":
                exact_blocks_checked,

            "exact_substring_pass":
                True,
        },
    }


    return result


# ============================================================
# 13. RUN ALL DEV
# ============================================================

structured_dev = []


for i, row in enumerate(
    dev,
    start=1
):

    facts = (
        build_structured_facts(
            row
        )
    )

    structured_dev.append(
        facts
    )


    if (
        i % 25
        == 0
    ):

        print(
            f"[extract] "
            f"{i}/{len(dev)}"
        )


assert (
    len(
        structured_dev
    )
    ==
    200
)


assert (
    [
        x["id"]
        for x
        in structured_dev
    ]
    ==
    dev_ids
)


print("[5] DEV 200 STRUCTURED EXTRACTION PASS")


# ============================================================
# 14. STRUCTURAL VALIDATION
# ============================================================

total_exact_blocks = 0


for x in structured_dev:

    assert (
        x[
            "validation"
        ][
            "exact_substring_pass"
        ]
        is True
    )


    total_exact_blocks += (
        x[
            "validation"
        ][
            "exact_blocks_checked"
        ]
    )


    # 모든 required meta는 raw에 존재해야 함
    assert (
        REQUIRED_META_UNION
        <=
        set(
            x[
                "meta"
            ][
                "raw"
            ].keys()
        )
    )


    # product state invariant
    p = (
        x[
            "product"
        ][
            "competitive_lookup"
        ]
    )


    assert p[
        "status"
    ] in {
        "MATCH",
        "NO_MATCH",
        "UNKNOWN_NO_CODE",
    }


    if (
        p[
            "status"
        ]
        ==
        "UNKNOWN_NO_CODE"
    ):

        assert (
            p[
                "is_competitive_product"
            ]
            is None
        )


    if (
        p[
            "status"
        ]
        ==
        "MATCH"
    ):

        assert (
            p[
                "is_competitive_product"
            ]
            is True
        )

        assert p[
            "matched_codes"
        ]


    if (
        p[
            "status"
        ]
        ==
        "NO_MATCH"
    ):

        assert (
            p[
                "is_competitive_product"
            ]
            is False
        )


assert total_exact_blocks > 0


print("[6] STRUCTURAL VALIDATION PASS")


# ============================================================
# 15. SAVE JSONL
# ============================================================

with open(
    OUT_JSONL,
    "w",
    encoding="utf-8"
) as f:

    for obj in structured_dev:

        f.write(
            json.dumps(
                obj,
                ensure_ascii=False
            )
            +
            "\n"
        )


assert OUT_JSONL.exists()


# reload
reloaded = []


with open(
    OUT_JSONL,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():

            reloaded.append(
                json.loads(
                    line
                )
            )


assert len(reloaded) == 200


assert [
    x["id"]
    for x
    in reloaded
] == dev_ids


print("[7] JSONL SAVE / RELOAD PASS")


# ============================================================
# 16. GENERAL AUDIT
# ============================================================

audit_rows = []


for x in structured_dev:

    c = x[
        "candidates"
    ]

    product_state = (
        x[
            "product"
        ][
            "competitive_lookup"
        ]
    )


    audit_rows.append({

        "id":
            x["id"],

        "docs":
            x[
                "documents"
            ][
                "count"
            ],

        "fully_observed":
            x[
                "completeness"
            ][
                "fully_observed"
            ],

        "dropped_docs":
            x[
                "completeness"
            ][
                "has_dropped_docs"
            ],

        "product_codes":
            len(
                x[
                    "product"
                ][
                    "codes"
                ]
            ),

        "competitive_status":
            product_state[
                "status"
            ],

        "qualification":
            len(
                c[
                    "qualification"
                ]
            ),

        "performance":
            len(
                c[
                    "performance"
                ]
            ),

        "region":
            len(
                c[
                    "region"
                ]
            ),

        "product_model":
            len(
                c[
                    "product_model"
                ]
            ),

        "direct_production":
            len(
                c[
                    "direct_production"
                ]
            ),

        "enterprise":
            len(
                c[
                    "enterprise"
                ]
            ),

        "supply_commitment":
            len(
                c[
                    "supply_commitment"
                ]
            ),

        "software":
            len(
                c[
                    "software"
                ]
            ),

        "joint_contract":
            len(
                c[
                    "joint_contract"
                ]
            ),

        "briefing":
            len(
                c[
                    "briefing"
                ]
            ),

        "amount":
            len(
                x[
                    "numeric_candidates"
                ][
                    "document_money"
                ]
            ),

        "date":
            len(
                x[
                    "date_candidates"
                ]
            ),

        "exact_blocks":
            x[
                "validation"
            ][
                "exact_blocks_checked"
            ],
    })


audit_df = pd.DataFrame(
    audit_rows
)


audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


print("[8] AUDIT TABLE 생성 PASS")


# ============================================================
# 17. DEV POSITIVE → CANDIDATE HINT COVERAGE
#
# 중요:
# 이 표는 품질진단용일 뿐이다.
#
# hit=False라고:
#     prediction=0
# 으로 만들면 안 된다.
#
# Gemma는 이후 전체 문서를 다시 본다.
# ============================================================

facts_by_id = {

    x["id"]:
        x

    for x
    in structured_dev
}


def candidate_hit(
    facts,
    item
):

    c = facts[
        "candidate_hints"
    ]


    mapping = {

        "v1":
            c["qualification"],

        "v2":
            c["performance"],

        "v3":
            c["performance"],

        "v4":
            c["performance"],

        "v5":
            c["region"],

        "v6":
            c["region"],

        "v7":
            c["region"],

        "v8":
            (
                c["performance"]
                and
                c["region"]
            ),

        "v9":
            c["product_model"],

        # absence items:
        # presence candidate coverage로 평가하지 않음
        "v10":
            None,

        "v11":
            None,

        "v12":
            c["direct_production"],

        "v13":
            c["enterprise"],

        "v14":
            c["enterprise"],

        "v15":
            c["enterprise"],

        "v16":
            None,

        "v17":
            c["enterprise"],

        "v18":
            None,

        "v19":
            c["supply_commitment"],

        "v20":
            None,

        "v21":
            c["joint_contract"],

        "v22":
            c["briefing"],

        "v23":
            c["briefing"],

        "v24":
            (
                c["amount"]
                or
                c["contract_method"]
                or
                c["region"]
                or
                c["industry"]
            ),
    }


    return mapping[
        item
    ]


hint_rows = []


for v in EXPECTED_ITEMS:

    positives = (
        labels.loc[
            labels[v]
            .astype(int)
            == 1,
            "id"
        ]
        .astype(str)
        .tolist()
    )


    applicable_hits = []


    for row_id in positives:

        hit = candidate_hit(
            facts_by_id[
                row_id
            ],
            v
        )

        if hit is not None:

            applicable_hits.append(
                bool(
                    hit
                )
            )


    if applicable_hits:

        hit_count = sum(
            applicable_hits
        )

        denominator = len(
            applicable_hits
        )

        rate = (
            hit_count
            /
            denominator
        )

    else:

        hit_count = None
        denominator = None
        rate = None


    hint_rows.append({

        "item":
            v,

        "positive_count":
            len(
                positives
            ),

        "candidate_audit_applicable":
            denominator,

        "candidate_hit":
            hit_count,

        "candidate_hit_rate":
            rate,

        "hard_gate":
            False,
    })


hint_df = pd.DataFrame(
    hint_rows
)


hint_df.to_csv(
    OUT_HINT_AUDIT,
    index=False,
    encoding="utf-8"
)


print("[9] POSITIVE HINT AUDIT 생성 PASS")


# ============================================================
# 18. GLOBAL STATISTICS
# ============================================================

product_status_counts = (

    audit_df[
        "competitive_status"
    ]
    .value_counts(
        dropna=False
    )
    .to_dict()
)


fully_observed_count = int(

    audit_df[
        "fully_observed"
    ]
    .sum()
)


dropped_count = int(

    audit_df[
        "dropped_docs"
    ]
    .sum()
)


# ============================================================
# 19. STAGE 7 MANIFEST
# ============================================================

manifest = {

    "stage":
        "7",

    "status":
        "PASS",

    "records":
        200,

    "required_meta_fields":
        sorted(
            REQUIRED_META_UNION
        ),

    "candidate_categories":
        list(
            CATEGORY_PATTERNS.keys()
        ),

    "exact_blocks_checked":
        total_exact_blocks,

    "competitive_product_csv": {

        "file":
            SME_CSV_PATH.name,

        "rows":
            len(
                sme_df
            ),

        "code_columns":
            CODE_COLUMNS,

        "unique_codes":
            len(
                PRODUCT_CODE_INDEX
            ),
    },

    "completeness": {

        "fully_observed":
            fully_observed_count,

        "has_dropped_docs":
            dropped_count,
    },

    "product_lookup_status":
        product_status_counts,

    "important_policy": {

        "regex_is_not_gate":
            True,

        "missing_candidate_does_not_mean_absence":
            True,

        "exact_substrings_preserved":
            True,

        "all_docs_used":
            True,

        "absence_items_require_completeness_handling":
            [
                "v10",
                "v11",
                "v16",
                "v18",
                "v20",
            ],
    },

    "artifacts": {

        "structured_dev":
            str(
                OUT_JSONL
            ),

        "audit":
            str(
                OUT_AUDIT
            ),

        "positive_hint_audit":
            str(
                OUT_HINT_AUDIT
            ),
    },

    "next_stage":
        (
            "Stage 8 - item-specific "
            "legal retrieval / judgment packet"
        ),
}


with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 20. FINAL VALIDATION
# ============================================================

with open(
    OUT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    saved_manifest = (
        json.load(
            f
        )
    )


assert (
    saved_manifest[
        "status"
    ]
    ==
    "PASS"
)


assert (
    saved_manifest[
        "records"
    ]
    ==
    200
)


assert (
    saved_manifest[
        "important_policy"
    ][
        "regex_is_not_gate"
    ]
    is True
)


assert (
    total_exact_blocks
    ==
    int(
        audit_df[
            "exact_blocks"
        ].sum()
    )
)


# ============================================================
# 21. DISPLAY
# ============================================================

print()
print("=== Structured Extraction Summary ===")

display(
    audit_df[
        [
            "docs",
            "fully_observed",
            "dropped_docs",
            "product_codes",
            "qualification",
            "performance",
            "region",
            "product_model",
            "direct_production",
            "enterprise",
            "supply_commitment",
            "software",
            "joint_contract",
            "briefing",
            "amount",
            "date",
            "exact_blocks",
        ]
    ]
    .describe(
        include="all"
    )
)


print()
print("=== Positive Candidate Hint Audit ===")

display(
    hint_df
)


print()
print("=== Competitive Product Lookup ===")

print(
    product_status_counts
)


# ============================================================
# 22. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 7 FINAL RESULT")
print("=" * 100)


print(
    "RECORDS                 =",
    len(
        structured_dev
    )
)


print(
    "IDS UNIQUE              =",
    len(
        set(
            x["id"]
            for x
            in structured_dev
        )
    )
)


print(
    "EXACT BLOCKS CHECKED    =",
    total_exact_blocks
)


print(
    "FULLY OBSERVED          =",
    fully_observed_count
)


print(
    "HAS DROPPED DOCS        =",
    dropped_count
)


print(
    "SME CSV ROWS            =",
    len(
        sme_df
    )
)


print(
    "SME UNIQUE CODES        =",
    len(
        PRODUCT_CODE_INDEX
    )
)


print(
    "PRODUCT LOOKUP STATUS   =",
    product_status_counts
)


print(
    "REGEX USED AS HARD GATE =",
    False
)


print(
    "OUTPUT JSONL            =",
    OUT_JSONL
)


print(
    "AUDIT CSV               =",
    OUT_AUDIT
)


print(
    "HINT AUDIT              =",
    OUT_HINT_AUDIT
)


print(
    "STAGE7 MANIFEST         =",
    OUT_MANIFEST
)


print()
print(
    "[PASS] Stage 7 structured information extraction 완료"
)

print(
    "NEXT = Stage 8 item-specific legal retrieval / judgment packet"
)

[1] PATH PASS
[2] DEV / LABEL / SPEC LOAD PASS
[3] META SCHEMA PASS
    required meta = ['개찰예정일자', '계약방법', '공고게시일자', '공동도급구성방식', '낙찰방법', '면허업종제한목록', '배정예산금액', '세부품명번호목록', '소관구분', '업무구분', '업종제한여부', '입찰추정가격', '적용계약법', '정보화사업여부', '제한지역코드목록', '조항호내용', '지역제한여부']
[4] SME CSV PASS
    rows        = 616
    code columns= ['제품명번호', '세부품명번호']
    unique codes= 615
[extract] 25/200
[extract] 50/200
[extract] 75/200
[extract] 100/200
[extract] 125/200
[extract] 150/200
[extract] 175/200
[extract] 200/200
[5] DEV 200 STRUCTURED EXTRACTION PASS
[6] STRUCTURAL VALIDATION PASS
[7] JSONL SAVE / RELOAD PASS
[8] AUDIT TABLE 생성 PASS
[9] POSITIVE HINT AUDIT 생성 PASS

=== Structured Extraction Summary ===


,docs,fully_observed,dropped_docs,product_codes,qualification,performance,region,product_model,direct_production,enterprise,supply_commitment,software,joint_contract,briefing,amount,date,exact_blocks
count,200.000000,200,200,200.000000,200.00000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.00000,200.000000,200.000000,200.00000,200.00000
unique,NaN,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,190,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,1.870000,NaN,NaN,0.480000,11.80000,5.280000,1.550000,0.860000,0.825000,4.825000,1.640000,0.525000,5.01500,0.820000,1.745000,0.34500,66.99500
std,0.725071,NaN,NaN,0.729562,7.26802,9.120169,1.943292,3.458352,1.491812,4.189527,2.833823,1.403289,8.64456,1.482765,2.136069,0.82424,36.44787
min,1.000000,NaN,NaN,0.000000,2.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,16.00000
25%,1.000000,NaN,NaN,0.000000,6.00000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,1.00000,0.000000,0.000000,0.00000,39.00000
50%,2.000000,NaN,NaN,0.000000,10.00000,1.000000,1.000000,0.000000,0.000000,4.000000,1.000000,0.000000,2.00000,0.000000,1.000000,0.00000,55.50000
75%,2.000000,NaN,NaN,1.000000,16.00000,5.250000,2.000000,1.000000,1.000000,7.000000,2.000000,0.000000,4.25000,1.000000,2.000000,0.00000,91.25000



=== Positive Candidate Hint Audit ===


,item,positive_count,candidate_audit_applicable,candidate_hit,candidate_hit_rate,hard_gate
0,v1,7,7.0,7.0,1.000000,False
1,v2,7,7.0,7.0,1.000000,False
2,v3,8,8.0,8.0,1.000000,False
3,v4,6,6.0,6.0,1.000000,False
4,v5,7,7.0,6.0,0.857143,False
5,v6,6,6.0,5.0,0.833333,False
6,v7,7,7.0,7.0,1.000000,False
7,v8,6,6.0,5.0,0.833333,False
8,v9,6,6.0,3.0,0.500000,False
9,v10,7,NaN,NaN,NaN,False



=== Competitive Product Lookup ===
{'UNKNOWN_NO_CODE': 122, 'NO_MATCH': 61, 'MATCH': 17}

Stage 7 FINAL RESULT
RECORDS                 = 200
IDS UNIQUE              = 200
EXACT BLOCKS CHECKED    = 13399
FULLY OBSERVED          = 190
HAS DROPPED DOCS        = 10
SME CSV ROWS            = 616
SME UNIQUE CODES        = 615
PRODUCT LOOKUP STATUS   = {'UNKNOWN_NO_CODE': 122, 'NO_MATCH': 61, 'MATCH': 17}
REGEX USED AS HARD GATE = False
OUTPUT JSONL            = /home/alparka/nara-ai/artifacts/structured_facts_dev.jsonl
AUDIT CSV               = /home/alparka/nara-ai/artifacts/structured_extraction_audit.csv
HINT AUDIT              = /home/alparka/nara-ai/artifacts/structured_extraction_positive_hint_audit.csv
STAGE7 MANIFEST         = /home/alparka/nara-ai/artifacts/stage7_manifest.json

[PASS] Stage 7 structured information extraction 완료
NEXT = Stage 8 item-specific legal retrieval / judgment packet


In [4]:
# ============================================================
# Stage 8 FINAL
# Item-specific Retrieval + Judgment Packet Builder
#
# INPUT
#   artifacts/structured_facts_dev.jsonl
#   artifacts/judgment_specs_v1_v24.json
#   open/dev.jsonl.gz
#   open/dev_labels.csv        <- 검증용으로만 사용
#
# OUTPUT
#   artifacts/judgment_item_catalog.json
#   artifacts/judgment_packets_dev.jsonl
#   artifacts/judgment_packet_audit.csv
#   artifacts/stage8_manifest.json
#
# 원칙
# 1. 모든 공고에서 v1~v24 전부 유지
# 2. regex/candidate miss로 item 제거 금지
# 3. 원문 snippet은 exact substring 유지
# 4. 법령은 Stage 6에서 검증한 exact legal blocks만 사용
# 5. labels는 packet에 절대 넣지 않음
# 6. Stage 9에서 notice당 1회 Gemma 호출을 목표
# ============================================================

from pathlib import Path
import gzip
import json
import re
import math
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

DEV_PATH = (
    ROOT
    / "open"
    / "dev.jsonl.gz"
)

LABEL_PATH = (
    ROOT
    / "open"
    / "dev_labels.csv"
)

FACTS_PATH = (
    ROOT
    / "artifacts"
    / "structured_facts_dev.jsonl"
)

SPEC_PATH = (
    ROOT
    / "artifacts"
    / "judgment_specs_v1_v24.json"
)

OUT_CATALOG = (
    ROOT
    / "artifacts"
    / "judgment_item_catalog.json"
)

OUT_PACKETS = (
    ROOT
    / "artifacts"
    / "judgment_packets_dev.jsonl"
)

OUT_AUDIT = (
    ROOT
    / "artifacts"
    / "judgment_packet_audit.csv"
)

OUT_MANIFEST = (
    ROOT
    / "artifacts"
    / "stage8_manifest.json"
)


for p in [
    DEV_PATH,
    LABEL_PATH,
    FACTS_PATH,
    SPEC_PATH,
]:
    assert p.exists(), p


EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(1, 25)
]

ABSENCE_ITEMS = {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}


# ============================================================
# 1. LOAD
# ============================================================

def load_jsonl(path):

    rows = []

    opener = (
        gzip.open
        if str(path).endswith(".gz")
        else open
    )

    with opener(
        path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            rows.append(
                json.loads(line)
            )

    return rows


dev = load_jsonl(
    DEV_PATH
)

facts = load_jsonl(
    FACTS_PATH
)

labels = pd.read_csv(
    LABEL_PATH
)

with open(
    SPEC_PATH,
    "r",
    encoding="utf-8"
) as f:
    specs = json.load(f)


assert len(dev) == 200
assert len(facts) == 200
assert len(labels) == 200
assert list(specs.keys()) == EXPECTED_ITEMS


dev_ids = [
    x["id"]
    for x in dev
]

fact_ids = [
    x["id"]
    for x in facts
]

label_ids = (
    labels["id"]
    .astype(str)
    .tolist()
)


assert dev_ids == fact_ids
assert dev_ids == label_ids


DEV_BY_ID = {
    x["id"]: x
    for x in dev
}

FACTS_BY_ID = {
    x["id"]: x
    for x in facts
}


print("[1] INPUT LOAD PASS")


# ============================================================
# 2. ITEM → CANDIDATE CATEGORY MAP
#
# candidate는 retrieval priority용.
# 절대 hard gate 아님.
# ============================================================

ITEM_CATEGORIES = {

    "v1": [
        "qualification",
    ],

    "v2": [
        "performance",
        "qualification",
        "amount",
    ],

    "v3": [
        "performance",
        "amount",
    ],

    "v4": [
        "performance",
        "qualification",
        "amount",
    ],

    "v5": [
        "region",
        "qualification",
        "amount",
    ],

    "v6": [
        "region",
        "qualification",
        "amount",
    ],

    "v7": [
        "region",
        "qualification",
        "amount",
    ],

    "v8": [
        "performance",
        "region",
        "qualification",
    ],

    "v9": [
        "product_model",
    ],

    "v10": [
        "direct_production",
        "enterprise",
    ],

    "v11": [
        "enterprise",
    ],

    "v12": [
        "direct_production",
    ],

    "v13": [
        "enterprise",
    ],

    "v14": [
        "enterprise",
        "amount",
    ],

    "v15": [
        "enterprise",
        "amount",
    ],

    "v16": [
        "enterprise",
        "amount",
    ],

    "v17": [
        "enterprise",
        "amount",
    ],

    "v18": [
        "enterprise",
        "amount",
    ],

    "v19": [
        "supply_commitment",
        "qualification",
    ],

    "v20": [
        "software",
        "qualification",
        "amount",
    ],

    "v21": [
        "joint_contract",
    ],

    "v22": [
        "briefing",
        "qualification",
    ],

    "v23": [
        "briefing",
        "deadline",
    ],

    "v24": [
        "amount",
        "contract_method",
        "region",
        "industry",
    ],
}


assert set(
    ITEM_CATEGORIES
) == set(
    EXPECTED_ITEMS
)


# ============================================================
# 3. FAMILY MAP
# ============================================================

ITEM_FAMILY = {}

for i in range(1, 9):
    ITEM_FAMILY[f"v{i}"] = (
        "qualification_restriction"
    )

for i in range(9, 20):
    ITEM_FAMILY[f"v{i}"] = (
        "product_sme"
    )

for i in range(20, 24):
    ITEM_FAMILY[f"v{i}"] = (
        "special_conditions"
    )

ITEM_FAMILY["v24"] = (
    "consistency"
)


FAMILY_CATEGORIES = {

    "qualification_restriction": [
        "qualification",
        "performance",
        "region",
        "amount",
    ],

    "product_sme": [
        "product_model",
        "direct_production",
        "enterprise",
        "supply_commitment",
        "amount",
    ],

    "special_conditions": [
        "software",
        "joint_contract",
        "briefing",
        "deadline",
        "qualification",
        "amount",
    ],

    "consistency": [
        "amount",
        "contract_method",
        "region",
        "industry",
    ],
}


# ============================================================
# 4. TOKENIZER FOR LEGAL RETRIEVAL
# ============================================================

STOPWORDS = {

    "경우",
    "해당",
    "한다",
    "하는",
    "하여",
    "있는",
    "없는",
    "대한",
    "따른",
    "따라",
    "또는",
    "그리고",
    "입찰",
    "계약",
    "위반",
    "판단",
    "경우에",
}


def tokenize(text):

    tokens = re.findall(
        r"[가-힣A-Za-z0-9]{2,}",
        str(text)
    )

    return {
        t
        for t in tokens
        if t not in STOPWORDS
    }


def item_query_text(spec):

    d = spec[
        "decision_spec"
    ]

    pieces = [
        spec.get(
            "name",
            ""
        ),
        d.get(
            "violation_condition",
            ""
        ),
        d.get(
            "nonviolation_condition",
            ""
        ),
        d.get(
            "amount_condition",
            ""
        ),
    ]


    pieces.extend(
        d.get(
            "facts_to_extract",
            []
        )
    )

    pieces.extend(
        d.get(
            "exceptions",
            []
        )
    )


    official = spec.get(
        "official_basis",
        {}
    )

    if isinstance(
        official,
        dict
    ):
        pieces.extend(
            str(x)
            for x in official.values()
        )


    return "\n".join(
        str(x)
        for x in pieces
        if x
    )


# ============================================================
# 5. LEGAL TEXT RECURSIVE EXTRACTION
# ============================================================

def collect_text_nodes(
    obj,
    inherited_source="",
    inherited_locator=""
):

    result = []


    if isinstance(
        obj,
        dict
    ):

        source = obj.get(
            "source",
            inherited_source
        )

        locator = obj.get(
            "locator",
            obj.get(
                "official_locator",
                inherited_locator
            )
        )


        text = obj.get(
            "text"
        )

        if (
            isinstance(
                text,
                str
            )
            and
            text.strip()
        ):

            result.append({

                "source":
                    source,

                "locator":
                    locator,

                "text":
                    text.strip(),
            })


        for key, value in obj.items():

            if key == "text":
                continue

            result.extend(
                collect_text_nodes(
                    value,
                    source,
                    locator
                )
            )


    elif isinstance(
        obj,
        list
    ):

        for x in obj:

            result.extend(
                collect_text_nodes(
                    x,
                    inherited_source,
                    inherited_locator
                )
            )


    return result


# ============================================================
# 6. LEGAL CHUNKING
# ============================================================

def chunk_text(
    text,
    max_chars=1000
):

    lines = (
        str(text)
        .splitlines()
    )

    chunks = []

    current = []


    for line in lines:

        line = line.strip()

        if not line:
            continue


        proposed = (
            "\n".join(
                current + [line]
            )
        )


        if (
            current
            and
            len(proposed)
            > max_chars
        ):

            chunks.append(
                "\n".join(
                    current
                )
            )

            current = [
                line
            ]

        else:

            current.append(
                line
            )


    if current:

        chunks.append(
            "\n".join(
                current
            )
        )


    # 아주 긴 한 줄 방어
    final = []


    for chunk in chunks:

        if len(chunk) <= max_chars:

            final.append(
                chunk
            )

        else:

            for i in range(
                0,
                len(chunk),
                max_chars
            ):

                final.append(
                    chunk[
                        i:
                        i + max_chars
                    ]
                )


    return final


# ============================================================
# 7. COMPACT LEGAL RETRIEVAL
#
# Stage 6에서 이미 item별 법령 source는 확정됨.
# 여기서는 그 안에서 prompt에 넣을 부분을 선택.
# ============================================================

MAX_LEGAL_CHARS_PER_ITEM = 5000


def retrieve_legal_context(
    spec
):

    if not spec[
        "exact_legal_keys"
    ]:

        return []


    query_tokens = tokenize(
        item_query_text(
            spec
        )
    )


    candidates = []


    for legal_key in spec[
        "exact_legal_keys"
    ]:

        block = spec[
            "exact_legal_text"
        ][
            legal_key
        ]


        nodes = collect_text_nodes(
            block
        )


        for node in nodes:

            chunks = chunk_text(
                node[
                    "text"
                ]
            )


            for order, chunk in enumerate(
                chunks
            ):

                chunk_tokens = tokenize(
                    chunk
                )

                overlap = (
                    query_tokens
                    &
                    chunk_tokens
                )


                score = (
                    len(
                        overlap
                    )
                )


                # 조문 시작/금액/핵심 단어 약간 가산
                if re.search(
                    r"제\s*\d+\s*조",
                    chunk
                ):
                    score += 1


                candidates.append({

                    "legal_key":
                        legal_key,

                    "source":
                        node[
                            "source"
                        ],

                    "locator":
                        node[
                            "locator"
                        ],

                    "score":
                        score,

                    "order":
                        order,

                    "text":
                        chunk,
                })


    # 높은 점수 우선.
    # 동점이면 원문 앞쪽.
    candidates.sort(
        key=lambda x: (
            -x["score"],
            x["legal_key"],
            x["order"],
        )
    )


    selected = []

    total_chars = 0

    seen = set()


    for x in candidates:

        signature = re.sub(
            r"\s+",
            "",
            x["text"]
        )


        if signature in seen:
            continue


        additional = len(
            x["text"]
        )


        if (
            selected
            and
            total_chars + additional
            >
            MAX_LEGAL_CHARS_PER_ITEM
        ):
            continue


        seen.add(
            signature
        )

        selected.append(
            x
        )

        total_chars += (
            additional
        )


    # score가 모두 낮아도 법령을 아예 비우지 않는다
    if (
        not selected
        and
        candidates
    ):

        selected = [
            candidates[0]
        ]


    return selected


# ============================================================
# 8. BUILD STATIC ITEM CATALOG
#
# 24개 공통 정보.
# notice마다 중복 저장할 필요 없음.
# ============================================================

ITEM_CATALOG = {}


for v in EXPECTED_ITEMS:

    spec = specs[v]

    d = spec[
        "decision_spec"
    ]


    ITEM_CATALOG[v] = {

        "item":
            v,

        "name":
            spec[
                "name"
            ],

        "family":
            ITEM_FAMILY[v],

        "absence_detection":
            bool(
                spec[
                    "absence_detection"
                ]
            ),

        "applicability":
            d.get(
                "applicability",
                ""
            ),

        "required_meta":
            d.get(
                "required_meta",
                []
            ),

        "facts_to_extract":
            d.get(
                "facts_to_extract",
                []
            ),

        "amount_condition":
            d.get(
                "amount_condition",
                ""
            ),

        "violation_condition":
            d.get(
                "violation_condition",
                ""
            ),

        "nonviolation_condition":
            d.get(
                "nonviolation_condition",
                ""
            ),

        "exceptions":
            d.get(
                "exceptions",
                []
            ),

        "unresolved":
            d.get(
                "unresolved",
                []
            ),

        "retrieval_categories":
            ITEM_CATEGORIES[v],

        "legal_context":
            retrieve_legal_context(
                spec
            ),
    }


assert len(
    ITEM_CATALOG
) == 24


for v in EXPECTED_ITEMS:

    if v == "v24":

        assert (
            ITEM_CATALOG[v][
                "legal_context"
            ]
            ==
            []
        )

    else:

        assert (
            len(
                ITEM_CATALOG[v][
                    "legal_context"
                ]
            )
            > 0
        ), (
            f"{v}: legal retrieval empty"
        )


print("[2] ITEM CATALOG PASS")


# ============================================================
# 9. EXACT BLOCK DEDUPE
# ============================================================

def dedupe_blocks(
    blocks
):

    seen = set()

    out = []


    for b in blocks:

        key = (
            b[
                "doc_id"
            ],
            int(
                b[
                    "start"
                ]
            ),
            int(
                b[
                    "end"
                ]
            ),
        )


        if key in seen:
            continue


        seen.add(
            key
        )

        out.append(
            b
        )


    out.sort(
        key=lambda x: (
            str(
                x[
                    "doc_id"
                ]
            ),
            int(
                x[
                    "start"
                ]
            ),
        )
    )


    return out


# ============================================================
# 10. FAMILY CONTEXT
#
# 동일 snippet이 여러 item에서 반복되지 않도록
# family 단위로 원문을 모음.
# ============================================================

FAMILY_CHAR_BUDGET = {

    "qualification_restriction":
        9000,

    "product_sme":
        10000,

    "special_conditions":
        8000,

    "consistency":
        7000,
}


def build_family_context(
    facts
):

    categories = facts[
        "candidates"
    ]

    result = {}


    for family, family_categories in (
        FAMILY_CATEGORIES.items()
    ):

        blocks = []


        for category in family_categories:

            blocks.extend(
                categories.get(
                    category,
                    []
                )
            )


        blocks = dedupe_blocks(
            blocks
        )


        selected = []

        total_chars = 0


        for b in blocks:

            size = len(
                b[
                    "text"
                ]
            )


            if (
                selected
                and
                total_chars + size
                >
                FAMILY_CHAR_BUDGET[
                    family
                ]
            ):

                continue


            selected.append(
                b
            )

            total_chars += size


        result[
            family
        ] = {

            "categories":
                family_categories,

            "candidate_blocks":
                selected,

            "candidate_count":
                len(
                    blocks
                ),

            "selected_count":
                len(
                    selected
                ),

            "selected_chars":
                total_chars,
        }


    return result


# ============================================================
# 11. DOCUMENT FALLBACK CONTEXT
#
# regex miss를 보완하기 위해
# 모든 doc에서 최소한 head/tail을 보존.
#
# candidate blocks + head/tail
# → Stage 9에서 notice content를 충분히 전달.
# ============================================================

MAX_DOCUMENT_CONTEXT_CHARS = 30000


def build_document_segments(
    row,
    facts
):

    docs = row[
        "docs"
    ]


    # 전체 category block
    all_blocks = []


    for blocks in facts[
        "candidates"
    ].values():

        all_blocks.extend(
            blocks
        )


    all_blocks = dedupe_blocks(
        all_blocks
    )


    blocks_by_doc = {}


    for b in all_blocks:

        blocks_by_doc.setdefault(
            b[
                "doc_id"
            ],
            []
        ).append(
            b
        )


    segments = []


    doc_count = max(
        1,
        len(
            docs
        )
    )


    per_doc_budget = min(
        8000,
        max(
            2500,
            MAX_DOCUMENT_CONTEXT_CHARS
            //
            doc_count
        )
    )


    for doc in docs:

        text = (
            doc.get(
                "text"
            )
            or
            ""
        )

        doc_id = doc[
            "doc_id"
        ]


        candidates = []


        # --------------------------------------------
        # HEAD
        # --------------------------------------------

        head_end = min(
            len(text),
            1600
        )


        if head_end > 0:

            candidates.append({

                "kind":
                    "HEAD",

                "doc_id":
                    doc_id,

                "doc_type":
                    doc.get(
                        "type"
                    ),

                "start":
                    0,

                "end":
                    head_end,

                "text":
                    text[
                        0:
                        head_end
                    ],
            })


        # --------------------------------------------
        # CANDIDATES
        # --------------------------------------------

        for b in blocks_by_doc.get(
            doc_id,
            []
        ):

            candidates.append({

                "kind":
                    "CANDIDATE",

                "doc_id":
                    doc_id,

                "doc_type":
                    doc.get(
                        "type"
                    ),

                "start":
                    int(
                        b[
                            "start"
                        ]
                    ),

                "end":
                    int(
                        b[
                            "end"
                        ]
                    ),

                "text":
                    b[
                        "text"
                    ],
            })


        # --------------------------------------------
        # TAIL
        # --------------------------------------------

        if len(text) > 1600:

            tail_start = max(
                0,
                len(text) - 800
            )

            candidates.append({

                "kind":
                    "TAIL",

                "doc_id":
                    doc_id,

                "doc_type":
                    doc.get(
                        "type"
                    ),

                "start":
                    tail_start,

                "end":
                    len(text),

                "text":
                    text[
                        tail_start:
                    ],
            })


        # --------------------------------------------
        # exact validation + dedupe
        # --------------------------------------------

        deduped = []

        seen = set()


        for seg in candidates:

            key = (
                seg[
                    "start"
                ],
                seg[
                    "end"
                ],
            )


            if key in seen:
                continue


            seen.add(
                key
            )


            assert (
                text[
                    seg[
                        "start"
                    ]:
                    seg[
                        "end"
                    ]
                ]
                ==
                seg[
                    "text"
                ]
            )


            deduped.append(
                seg
            )


        # HEAD → CANDIDATE → TAIL 순서지만
        # budget 안에서 candidate를 최대한 보존
        #
        # head는 반드시 포함
        # candidate 우선
        # tail 마지막
        head_segments = [
            x
            for x in deduped
            if x["kind"] == "HEAD"
        ]

        candidate_segments = [
            x
            for x in deduped
            if x["kind"] == "CANDIDATE"
        ]

        tail_segments = [
            x
            for x in deduped
            if x["kind"] == "TAIL"
        ]


        ordered = (
            head_segments
            +
            candidate_segments
            +
            tail_segments
        )


        selected = []

        used = 0


        for seg in ordered:

            size = len(
                seg[
                    "text"
                ]
            )


            if (
                selected
                and
                used + size
                >
                per_doc_budget
            ):

                continue


            selected.append(
                seg
            )

            used += size


        # 각 doc 최소 하나
        assert selected


        segments.extend(
            selected
        )


    # 전체 budget 안전장치
    #
    # 처음부터 per-doc 배분했기 때문에
    # 크게 넘지 않아야 함.
    total = sum(
        len(
            x[
                "text"
            ]
        )
        for x in segments
    )


    if total > (
        MAX_DOCUMENT_CONTEXT_CHARS
        + 5000
    ):

        # 너무 큰 경우에도 doc 자체를 제거하지 않고
        # 각 doc HEAD만이라도 남긴다.
        compacted = []

        seen_docs = set()


        # 모든 doc HEAD 우선
        for seg in segments:

            if (
                seg[
                    "kind"
                ]
                ==
                "HEAD"
                and
                seg[
                    "doc_id"
                ]
                not in seen_docs
            ):

                compacted.append(
                    seg
                )

                seen_docs.add(
                    seg[
                        "doc_id"
                    ]
                )


        used = sum(
            len(
                x[
                    "text"
                ]
            )
            for x in compacted
        )


        for seg in segments:

            if seg in compacted:
                continue


            size = len(
                seg[
                    "text"
                ]
            )


            if (
                used + size
                >
                MAX_DOCUMENT_CONTEXT_CHARS
            ):
                continue


            compacted.append(
                seg
            )

            used += size


        segments = compacted


    return segments


# ============================================================
# 12. NON-HARD ITEM PRIORITY
#
# HIGH / MEDIUM만 사용.
# LOW / DROP 없음.
# ============================================================

def meaningful_meta(
    value
):

    if value is None:
        return False

    s = str(
        value
    ).strip()

    return (
        s
        not in {
            "",
            "미입력",
            "None",
            "null",
        }
    )


def item_priority(
    v,
    facts
):

    hints = facts[
        "candidate_hints"
    ]

    meta = facts[
        "meta"
    ][
        "core"
    ]

    product = facts[
        "product"
    ][
        "competitive_lookup"
    ]


    # --------------------------------------------
    # candidate hit
    # --------------------------------------------

    if any(
        hints.get(
            category,
            False
        )
        for category
        in ITEM_CATEGORIES[v]
    ):

        return "HIGH"


    # --------------------------------------------
    # product rules
    # --------------------------------------------

    if v in {
        "v10",
        "v11",
        "v13",
    }:

        if (
            product[
                "status"
            ]
            ==
            "MATCH"
        ):
            return "HIGH"


    if v in {
        "v12",
        "v14",
        "v15",
        "v16",
        "v17",
        "v18",
    }:

        if (
            product[
                "status"
            ]
            in {
                "NO_MATCH",
                "UNKNOWN_NO_CODE",
            }
        ):
            return "HIGH"


    # --------------------------------------------
    # SW
    # --------------------------------------------

    if v == "v20":

        if meaningful_meta(
            meta.get(
                "is_information_project"
            )
        ):
            return "HIGH"


    # --------------------------------------------
    # 공동계약
    # --------------------------------------------

    if v == "v21":

        if meaningful_meta(
            meta.get(
                "joint_contract_method"
            )
        ):
            return "HIGH"


    # --------------------------------------------
    # v24는 항상 적용
    # --------------------------------------------

    if v == "v24":
        return "HIGH"


    # 절대 제거하지 않음
    return "MEDIUM"


# ============================================================
# 13. BUILD PACKETS
#
# labels 절대 사용하지 않음.
# ============================================================

PACKETS = []


for idx, row in enumerate(
    dev,
    start=1
):

    row_id = row[
        "id"
    ]

    f = FACTS_BY_ID[
        row_id
    ]


    priorities = {

        v:
            item_priority(
                v,
                f
            )

        for v
        in EXPECTED_ITEMS
    }


    assert (
        set(
            priorities.keys()
        )
        ==
        set(
            EXPECTED_ITEMS
        )
    )


    assert all(
        x in {
            "HIGH",
            "MEDIUM",
        }
        for x
        in priorities.values()
    )


    family_context = (
        build_family_context(
            f
        )
    )


    document_segments = (
        build_document_segments(
            row,
            f
        )
    )


    # --------------------------------------------------------
    # compact structured facts
    # --------------------------------------------------------

    packet = {

        "id":
            row_id,


        "strategy": {

            "all_24_items_required":
                True,

            "item_filtering":
                False,

            "candidate_hard_gate":
                False,

            "target_llm_calls_per_notice":
                1,
        },


        "meta":
            f[
                "meta"
            ][
                "core"
            ],


        "completeness":
            f[
                "completeness"
            ],


        "product":
            {

                "codes":
                    f[
                        "product"
                    ][
                        "codes"
                    ],

                "competitive_lookup":
                    f[
                        "product"
                    ][
                        "competitive_lookup"
                    ],
            },


        "numeric_candidates":
            f[
                "numeric_candidates"
            ],


        "date_candidates":
            f[
                "date_candidates"
            ],


        "comparison_candidates":
            f[
                "comparison_candidates"
            ],


        "item_priority":
            priorities,


        "family_context":
            family_context,


        "document_segments":
            document_segments,


        # 원문 전체는 packet에 중복 저장하지 않음.
        # Stage 9에서는 dev/test input에서 id로 다시 가져옴.
        "document_manifest":
            {

                "doc_count":
                    len(
                        row[
                            "docs"
                        ]
                    ),

                "docs": [

                    {
                        "doc_id":
                            d[
                                "doc_id"
                            ],

                        "type":
                            d[
                                "type"
                            ],

                        "chars":
                            len(
                                d.get(
                                    "text",
                                    ""
                                )
                            ),
                    }

                    for d
                    in row[
                        "docs"
                    ]
                ],
            },
    }


    PACKETS.append(
        packet
    )


    if idx % 25 == 0:

        print(
            f"[packet] "
            f"{idx}/200"
        )


assert len(PACKETS) == 200


print("[3] PACKET BUILD PASS")


# ============================================================
# 14. EXACT SUBSTRING REVALIDATION
# ============================================================

total_packet_segments = 0


for packet in PACKETS:

    row = DEV_BY_ID[
        packet[
            "id"
        ]
    ]


    doc_map = {

        d[
            "doc_id"
        ]:
            d.get(
                "text",
                ""
            )

        for d
        in row[
            "docs"
        ]
    }


    # family blocks
    for family_obj in (
        packet[
            "family_context"
        ].values()
    ):

        for block in family_obj[
            "candidate_blocks"
        ]:

            original = doc_map[
                block[
                    "doc_id"
                ]
            ]


            assert (
                original[
                    int(
                        block[
                            "start"
                        ]
                    ):
                    int(
                        block[
                            "end"
                        ]
                    )
                ]
                ==
                block[
                    "text"
                ]
            )


    # document segments
    for seg in packet[
        "document_segments"
    ]:

        original = doc_map[
            seg[
                "doc_id"
            ]
        ]


        assert (
            original[
                int(
                    seg[
                        "start"
                    ]
                ):
                int(
                    seg[
                        "end"
                    ]
                )
            ]
            ==
            seg[
                "text"
            ]
        )


        total_packet_segments += 1


assert total_packet_segments > 0


print("[4] EXACT SUBSTRING REVALIDATION PASS")


# ============================================================
# 15. LABEL LEAKAGE GUARD
#
# packet 안에 v1=정답 등의 정보가 들어가면 안 됨.
# labels는 audit 계산에만 사용.
# ============================================================

FORBIDDEN_PACKET_KEYS = {
    "label",
    "labels",
    "ground_truth",
    "answer",
    "target",
}


def recursive_key_check(
    obj
):

    if isinstance(
        obj,
        dict
    ):

        for key, value in obj.items():

            assert (
                str(key)
                not in
                FORBIDDEN_PACKET_KEYS
            ), (
                f"label leakage key: {key}"
            )

            recursive_key_check(
                value
            )


    elif isinstance(
        obj,
        list
    ):

        for x in obj:

            recursive_key_check(
                x
            )


for packet in PACKETS:

    recursive_key_check(
        packet
    )


print("[5] LABEL LEAKAGE CHECK PASS")


# ============================================================
# 16. SAVE STATIC ITEM CATALOG
# ============================================================

with open(
    OUT_CATALOG,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ITEM_CATALOG,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 17. SAVE PACKETS
# ============================================================

with open(
    OUT_PACKETS,
    "w",
    encoding="utf-8"
) as f:

    for packet in PACKETS:

        f.write(
            json.dumps(
                packet,
                ensure_ascii=False
            )
            +
            "\n"
        )


# ============================================================
# 18. RELOAD VALIDATION
# ============================================================

with open(
    OUT_CATALOG,
    "r",
    encoding="utf-8"
) as f:

    saved_catalog = (
        json.load(
            f
        )
    )


saved_packets = []


with open(
    OUT_PACKETS,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():

            saved_packets.append(
                json.loads(
                    line
                )
            )


assert list(
    saved_catalog.keys()
) == EXPECTED_ITEMS

assert len(
    saved_packets
) == 200

assert [
    x["id"]
    for x in saved_packets
] == dev_ids


print("[6] SAVE / RELOAD PASS")


# ============================================================
# 19. AUDIT
# ============================================================

audit_rows = []


for packet in PACKETS:

    high = [
        v
        for v, p
        in packet[
            "item_priority"
        ].items()
        if p == "HIGH"
    ]

    medium = [
        v
        for v, p
        in packet[
            "item_priority"
        ].items()
        if p == "MEDIUM"
    ]


    family_chars = {

        family:
            obj[
                "selected_chars"
            ]

        for family, obj
        in packet[
            "family_context"
        ].items()
    }


    document_chars = sum(

        len(
            x[
                "text"
            ]
        )

        for x
        in packet[
            "document_segments"
        ]
    )


    audit_rows.append({

        "id":
            packet[
                "id"
            ],

        "high_items":
            len(
                high
            ),

        "medium_items":
            len(
                medium
            ),

        "all_items_present":
            (
                len(
                    high
                )
                +
                len(
                    medium
                )
                ==
                24
            ),

        "qualification_chars":
            family_chars[
                "qualification_restriction"
            ],

        "product_chars":
            family_chars[
                "product_sme"
            ],

        "special_chars":
            family_chars[
                "special_conditions"
            ],

        "consistency_chars":
            family_chars[
                "consistency"
            ],

        "document_context_chars":
            document_chars,

        "document_segments":
            len(
                packet[
                    "document_segments"
                ]
            ),

        "fully_observed":
            packet[
                "completeness"
            ][
                "fully_observed"
            ],

        "product_status":
            packet[
                "product"
            ][
                "competitive_lookup"
            ][
                "status"
            ],
    })


audit_df = pd.DataFrame(
    audit_rows
)


assert (
    audit_df[
        "all_items_present"
    ]
    ==
    True
).all()


audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


print("[7] AUDIT PASS")


# ============================================================
# 20. DEV POSITIVE PRIORITY AUDIT
#
# 정확도 확인용.
#
# HIGH가 아니더라도 item은 제거되지 않기 때문에
# MEDIUM positive가 존재해도 false negative로 직결되지 않음.
# ============================================================

priority_rows = []


packet_by_id = {
    x["id"]: x
    for x in PACKETS
}


for v in EXPECTED_ITEMS:

    pos_ids = (
        labels.loc[
            labels[v]
            .astype(int)
            == 1,
            "id"
        ]
        .astype(str)
        .tolist()
    )


    high_count = sum(

        packet_by_id[
            row_id
        ][
            "item_priority"
        ][
            v
        ]
        ==
        "HIGH"

        for row_id
        in pos_ids
    )


    priority_rows.append({

        "item":
            v,

        "positive":
            len(
                pos_ids
            ),

        "high_priority":
            high_count,

        "medium_priority":
            (
                len(
                    pos_ids
                )
                -
                high_count
            ),

        "items_dropped":
            0,
    })


priority_df = pd.DataFrame(
    priority_rows
)


# ============================================================
# 21. CATALOG SIZE AUDIT
# ============================================================

catalog_rows = []


for v in EXPECTED_ITEMS:

    legal_context = (
        ITEM_CATALOG[v][
            "legal_context"
        ]
    )


    legal_chars = sum(
        len(
            x[
                "text"
            ]
        )
        for x
        in legal_context
    )


    catalog_rows.append({

        "item":
            v,

        "name":
            ITEM_CATALOG[v][
                "name"
            ],

        "absence":
            ITEM_CATALOG[v][
                "absence_detection"
            ],

        "legal_chunks":
            len(
                legal_context
            ),

        "legal_chars":
            legal_chars,
    })


catalog_df = pd.DataFrame(
    catalog_rows
)


assert (
    catalog_df.loc[
        catalog_df[
            "item"
        ]
        !=
        "v24",
        "legal_chunks"
    ]
    > 0
).all()


# ============================================================
# 22. STAGE 8 MANIFEST
# ============================================================

manifest = {

    "stage":
        8,

    "status":
        "PASS",

    "records":
        200,

    "items":
        24,

    "strategy": {

        "target_llm_calls_per_notice":
            1,

        "all_items_always_judged":
            True,

        "candidate_hard_gate":
            False,

        "priority_levels": [
            "HIGH",
            "MEDIUM",
        ],

        "labels_in_packets":
            False,
    },

    "retrieval": {

        "legal_source":
            "Stage 6 exact legal grounding",

        "document_source":
            "Stage 7 exact candidate blocks + per-document fallback",

        "max_legal_chars_per_item":
            MAX_LEGAL_CHARS_PER_ITEM,

        "max_document_context_chars":
            MAX_DOCUMENT_CONTEXT_CHARS,
    },

    "exact_packet_segments_checked":
        total_packet_segments,

    "artifacts": {

        "item_catalog":
            str(
                OUT_CATALOG
            ),

        "packets_dev":
            str(
                OUT_PACKETS
            ),

        "audit":
            str(
                OUT_AUDIT
            ),
    },

    "next_stage":
        (
            "Stage 9 - fixed Gemma "
            "single-call judgment engine"
        ),
}


with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 23. FINAL VALIDATION
# ============================================================

with open(
    OUT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    saved_manifest = (
        json.load(
            f
        )
    )


assert (
    saved_manifest[
        "status"
    ]
    ==
    "PASS"
)

assert (
    saved_manifest[
        "records"
    ]
    ==
    200
)

assert (
    saved_manifest[
        "items"
    ]
    ==
    24
)

assert (
    saved_manifest[
        "strategy"
    ][
        "all_items_always_judged"
    ]
    is True
)

assert (
    saved_manifest[
        "strategy"
    ][
        "candidate_hard_gate"
    ]
    is False
)


# ============================================================
# 24. DISPLAY
# ============================================================

print()
print("=== Packet Size Audit ===")

display(
    audit_df[
        [
            "high_items",
            "medium_items",
            "qualification_chars",
            "product_chars",
            "special_chars",
            "consistency_chars",
            "document_context_chars",
            "document_segments",
        ]
    ]
    .describe()
)


print()
print("=== DEV Positive Priority Audit ===")

display(
    priority_df
)


print()
print("=== Legal Catalog Audit ===")

display(
    catalog_df
)


# ============================================================
# 25. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 8 FINAL RESULT")
print("=" * 100)


print(
    "RECORDS                    =",
    len(
        PACKETS
    )
)


print(
    "ITEM CATALOG               =",
    len(
        ITEM_CATALOG
    )
)


print(
    "ALL ITEMS / NOTICE         =",
    24
)


print(
    "ITEMS DROPPED              =",
    0
)


print(
    "CANDIDATE HARD GATE        =",
    False
)


print(
    "TARGET LLM CALLS / NOTICE  =",
    1
)


print(
    "EXACT SEGMENTS CHECKED     =",
    total_packet_segments
)


print(
    "AVG HIGH ITEMS             =",
    round(
        audit_df[
            "high_items"
        ].mean(),
        2
    )
)


print(
    "AVG DOCUMENT CHARS         =",
    round(
        audit_df[
            "document_context_chars"
        ].mean(),
        2
    )
)


print(
    "MAX DOCUMENT CHARS         =",
    int(
        audit_df[
            "document_context_chars"
        ].max()
    )
)


print(
    "CATALOG                    =",
    OUT_CATALOG
)


print(
    "PACKETS                    =",
    OUT_PACKETS
)


print(
    "AUDIT                      =",
    OUT_AUDIT
)


print(
    "MANIFEST                   =",
    OUT_MANIFEST
)


print()

print(
    "[PASS] Stage 8 item-specific retrieval / "
    "judgment packet 완료"
)

print(
    "NEXT = Stage 9 fixed Gemma single-call judgment engine"
)

[1] INPUT LOAD PASS
[2] ITEM CATALOG PASS
[packet] 25/200
[packet] 50/200
[packet] 75/200
[packet] 100/200
[packet] 125/200
[packet] 150/200
[packet] 175/200
[packet] 200/200
[3] PACKET BUILD PASS
[4] EXACT SUBSTRING REVALIDATION PASS


AssertionError: label leakage key: labels

In [5]:
# ============================================================
# Stage 8 RESUME
# Fix label-leakage false positive and finish Stage 8
#
# 현재 notebook memory에 이미 존재:
#   PACKETS
#   ITEM_CATALOG
#   dev_ids
#   labels
#   EXPECTED_ITEMS
#   OUT_CATALOG
#   OUT_PACKETS
#   OUT_AUDIT
#   OUT_MANIFEST
#   total_packet_segments
#   MAX_LEGAL_CHARS_PER_ITEM
#   MAX_DOCUMENT_CONTEXT_CHARS
#
# 문제 원인:
# numeric_candidates 안의 "labels"는
# ground-truth label이 아니라 금액 의미 태그임.
# ============================================================

import json
import pandas as pd


# ============================================================
# 15. SAFE LABEL LEAKAGE GUARD
#
# 모든 nested "labels"를 금지하지 않는다.
#
# 실제로 금지할 것은:
# - packet top-level의 label / labels
# - ground_truth
# - dev_labels
# - true_label
# - gold_label
# - answer_key
#
# amount 후보 내부의
# "labels": ["추정가격"]
# 같은 semantic annotation은 정상.
# ============================================================

STRICT_FORBIDDEN_KEYS = {
    "ground_truth",
    "ground_truths",
    "dev_labels",
    "true_label",
    "true_labels",
    "gold_label",
    "gold_labels",
    "answer_key",
    "correct_answer",
}


def recursive_leakage_check(
    obj,
    path=()
):

    if isinstance(obj, dict):

        for key, value in obj.items():

            key_str = str(key)

            current_path = (
                *path,
                key_str
            )

            # ------------------------------------------------
            # 어디에 있든 절대 들어가면 안 되는 key
            # ------------------------------------------------

            assert (
                key_str
                not in
                STRICT_FORBIDDEN_KEYS
            ), (
                "label leakage key: "
                +
                ".".join(
                    current_path
                )
            )


            # ------------------------------------------------
            # packet 최상위에 label / labels가 있으면 실패
            #
            # nested numeric_candidates.*.labels 는 허용
            # ------------------------------------------------

            if len(path) == 0:

                assert (
                    key_str
                    not in {
                        "label",
                        "labels",
                        "answer",
                        "target",
                    }
                ), (
                    "top-level label leakage key: "
                    f"{key_str}"
                )


            recursive_leakage_check(
                value,
                current_path
            )


    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            recursive_leakage_check(
                value,
                (
                    *path,
                    f"[{i}]"
                )
            )


for packet in PACKETS:

    recursive_leakage_check(
        packet
    )


print(
    "[5] LABEL LEAKAGE CHECK PASS"
)


# ============================================================
# 15-B. 추가 검증:
# PACKET 내부에 실제 v1~v24 정답 배열 같은 구조가
# 들어갔는지 확인
# ============================================================

def suspicious_prediction_dict(obj):

    """
    v1~v24가 한 dict에 모두 존재하면서
    값이 0/1이면 정답/prediction 누출 가능성이 있으므로 탐지.
    """

    if not isinstance(
        obj,
        dict
    ):
        return False

    item_keys = [
        f"v{i}"
        for i in range(1, 25)
    ]

    if not all(
        key in obj
        for key in item_keys
    ):
        return False

    values = [
        obj[key]
        for key in item_keys
    ]

    return all(
        value in {
            0,
            1,
            True,
            False,
        }
        for value in values
    )


def recursive_prediction_leak_check(
    obj,
    path=()
):

    if isinstance(
        obj,
        dict
    ):

        assert (
            not suspicious_prediction_dict(
                obj
            )
        ), (
            "possible v1~v24 label/prediction leakage at: "
            +
            ".".join(
                path
            )
        )

        for key, value in obj.items():

            recursive_prediction_leak_check(
                value,
                (
                    *path,
                    str(key)
                )
            )

    elif isinstance(
        obj,
        list
    ):

        for i, value in enumerate(
            obj
        ):

            recursive_prediction_leak_check(
                value,
                (
                    *path,
                    f"[{i}]"
                )
            )


for packet in PACKETS:

    recursive_prediction_leak_check(
        packet
    )


print(
    "[5-B] v1~v24 prediction leakage CHECK PASS"
)


# ============================================================
# 16. SAVE STATIC ITEM CATALOG
# ============================================================

with open(
    OUT_CATALOG,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ITEM_CATALOG,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 17. SAVE PACKETS
# ============================================================

with open(
    OUT_PACKETS,
    "w",
    encoding="utf-8"
) as f:

    for packet in PACKETS:

        f.write(
            json.dumps(
                packet,
                ensure_ascii=False
            )
            +
            "\n"
        )


# ============================================================
# 18. RELOAD VALIDATION
# ============================================================

with open(
    OUT_CATALOG,
    "r",
    encoding="utf-8"
) as f:

    saved_catalog = json.load(
        f
    )


saved_packets = []


with open(
    OUT_PACKETS,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():

            saved_packets.append(
                json.loads(
                    line
                )
            )


assert (
    list(
        saved_catalog.keys()
    )
    ==
    EXPECTED_ITEMS
)


assert (
    len(
        saved_packets
    )
    ==
    200
)


assert (
    [
        x["id"]
        for x
        in saved_packets
    ]
    ==
    dev_ids
)


# 저장된 packet에도 leakage 재검사
for packet in saved_packets:

    recursive_leakage_check(
        packet
    )

    recursive_prediction_leak_check(
        packet
    )


print(
    "[6] SAVE / RELOAD PASS"
)


# ============================================================
# 19. PACKET AUDIT
# ============================================================

audit_rows = []


for packet in PACKETS:

    high = [
        v
        for v, priority
        in packet[
            "item_priority"
        ].items()
        if priority == "HIGH"
    ]


    medium = [
        v
        for v, priority
        in packet[
            "item_priority"
        ].items()
        if priority == "MEDIUM"
    ]


    assert (
        len(high)
        +
        len(medium)
        ==
        24
    )


    assert (
        set(high)
        |
        set(medium)
        ==
        set(
            EXPECTED_ITEMS
        )
    )


    family_chars = {

        family:
            obj[
                "selected_chars"
            ]

        for family, obj
        in packet[
            "family_context"
        ].items()
    }


    document_chars = sum(

        len(
            segment[
                "text"
            ]
        )

        for segment
        in packet[
            "document_segments"
        ]
    )


    audit_rows.append({

        "id":
            packet[
                "id"
            ],

        "high_items":
            len(
                high
            ),

        "medium_items":
            len(
                medium
            ),

        "all_items_present":
            True,

        "qualification_chars":
            family_chars[
                "qualification_restriction"
            ],

        "product_chars":
            family_chars[
                "product_sme"
            ],

        "special_chars":
            family_chars[
                "special_conditions"
            ],

        "consistency_chars":
            family_chars[
                "consistency"
            ],

        "document_context_chars":
            document_chars,

        "document_segments":
            len(
                packet[
                    "document_segments"
                ]
            ),

        "fully_observed":
            packet[
                "completeness"
            ][
                "fully_observed"
            ],

        "product_status":
            packet[
                "product"
            ][
                "competitive_lookup"
            ][
                "status"
            ],
    })


audit_df = pd.DataFrame(
    audit_rows
)


assert (
    audit_df[
        "all_items_present"
    ]
    ==
    True
).all()


audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


print(
    "[7] AUDIT PASS"
)


# ============================================================
# 20. DEV POSITIVE PRIORITY AUDIT
#
# labels는 여기서만 사용.
# packet에는 들어가지 않음.
# ============================================================

packet_by_id = {

    packet[
        "id"
    ]:
        packet

    for packet
    in PACKETS
}


priority_rows = []


for v in EXPECTED_ITEMS:

    positive_ids = (

        labels.loc[
            labels[v]
            .astype(int)
            ==
            1,
            "id"
        ]
        .astype(str)
        .tolist()
    )


    high_count = sum(

        packet_by_id[
            row_id
        ][
            "item_priority"
        ][
            v
        ]
        ==
        "HIGH"

        for row_id
        in positive_ids
    )


    priority_rows.append({

        "item":
            v,

        "positive":
            len(
                positive_ids
            ),

        "high_priority":
            high_count,

        "medium_priority":
            (
                len(
                    positive_ids
                )
                -
                high_count
            ),

        # 중요:
        # 어떤 item도 제거하지 않는다.
        "items_dropped":
            0,
    })


priority_df = pd.DataFrame(
    priority_rows
)


assert (
    priority_df[
        "items_dropped"
    ]
    ==
    0
).all()


# ============================================================
# 21. LEGAL CATALOG AUDIT
# ============================================================

catalog_rows = []


for v in EXPECTED_ITEMS:

    legal_context = (
        ITEM_CATALOG[v][
            "legal_context"
        ]
    )


    legal_chars = sum(

        len(
            chunk[
                "text"
            ]
        )

        for chunk
        in legal_context
    )


    catalog_rows.append({

        "item":
            v,

        "name":
            ITEM_CATALOG[v][
                "name"
            ],

        "absence":
            ITEM_CATALOG[v][
                "absence_detection"
            ],

        "legal_chunks":
            len(
                legal_context
            ),

        "legal_chars":
            legal_chars,
    })


catalog_df = pd.DataFrame(
    catalog_rows
)


# v24만 legal RAG 없음
for v in EXPECTED_ITEMS:

    count = int(
        catalog_df.loc[
            catalog_df[
                "item"
            ]
            ==
            v,
            "legal_chunks"
        ].iloc[0]
    )

    if v == "v24":

        assert count == 0

    else:

        assert count > 0, (
            f"{v}: legal chunks empty"
        )


print(
    "[8] LEGAL CATALOG AUDIT PASS"
)


# ============================================================
# 22. PACKET SIZE SAFETY
# ============================================================

AVG_DOC_CHARS = float(
    audit_df[
        "document_context_chars"
    ].mean()
)

MAX_DOC_CHARS = int(
    audit_df[
        "document_context_chars"
    ].max()
)


# Stage 8의 hard design limit보다 비정상적으로 커지지 않았는지
assert (
    MAX_DOC_CHARS
    <=
    MAX_DOCUMENT_CONTEXT_CHARS
    +
    5000
), (
    f"document packet too large: "
    f"{MAX_DOC_CHARS}"
)


print(
    "[9] PACKET SIZE SAFETY PASS"
)


# ============================================================
# 23. STAGE 8 MANIFEST
# ============================================================

manifest = {

    "stage":
        8,

    "status":
        "PASS",

    "records":
        200,

    "items":
        24,

    "strategy": {

        "target_llm_calls_per_notice":
            1,

        "all_items_always_judged":
            True,

        "candidate_hard_gate":
            False,

        "priority_levels": [
            "HIGH",
            "MEDIUM",
        ],

        "labels_in_packets":
            False,

        "ground_truth_in_packets":
            False,
    },

    "retrieval": {

        "legal_source":
            (
                "Stage 6 exact legal grounding"
            ),

        "document_source":
            (
                "Stage 7 exact candidate blocks "
                "+ per-document fallback"
            ),

        "max_legal_chars_per_item":
            MAX_LEGAL_CHARS_PER_ITEM,

        "max_document_context_chars":
            MAX_DOCUMENT_CONTEXT_CHARS,
    },

    "exact_packet_segments_checked":
        total_packet_segments,

    "artifacts": {

        "item_catalog":
            str(
                OUT_CATALOG
            ),

        "packets_dev":
            str(
                OUT_PACKETS
            ),

        "audit":
            str(
                OUT_AUDIT
            ),
    },

    "next_stage":
        (
            "Stage 9 - fixed Gemma "
            "single-call judgment engine"
        ),
}


with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 24. FINAL MANIFEST VALIDATION
# ============================================================

with open(
    OUT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    saved_manifest = (
        json.load(
            f
        )
    )


assert (
    saved_manifest[
        "status"
    ]
    ==
    "PASS"
)

assert (
    saved_manifest[
        "records"
    ]
    ==
    200
)

assert (
    saved_manifest[
        "items"
    ]
    ==
    24
)

assert (
    saved_manifest[
        "strategy"
    ][
        "all_items_always_judged"
    ]
    is True
)

assert (
    saved_manifest[
        "strategy"
    ][
        "candidate_hard_gate"
    ]
    is False
)

assert (
    saved_manifest[
        "strategy"
    ][
        "labels_in_packets"
    ]
    is False
)


print(
    "[10] MANIFEST VALIDATION PASS"
)


# ============================================================
# 25. DISPLAY
# ============================================================

print()
print(
    "=== Packet Size Audit ==="
)

display(

    audit_df[
        [
            "high_items",
            "medium_items",
            "qualification_chars",
            "product_chars",
            "special_chars",
            "consistency_chars",
            "document_context_chars",
            "document_segments",
        ]
    ]
    .describe()
)


print()
print(
    "=== DEV Positive Priority Audit ==="
)

display(
    priority_df
)


print()
print(
    "=== Legal Catalog Audit ==="
)

display(
    catalog_df
)


# ============================================================
# 26. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 8 FINAL RESULT")
print("=" * 100)


print(
    "RECORDS                    =",
    len(
        PACKETS
    )
)


print(
    "ITEM CATALOG               =",
    len(
        ITEM_CATALOG
    )
)


print(
    "ALL ITEMS / NOTICE         =",
    24
)


print(
    "ITEMS DROPPED              =",
    0
)


print(
    "LABEL LEAKAGE              =",
    False
)


print(
    "CANDIDATE HARD GATE        =",
    False
)


print(
    "TARGET LLM CALLS / NOTICE  =",
    1
)


print(
    "EXACT SEGMENTS CHECKED     =",
    total_packet_segments
)


print(
    "AVG HIGH ITEMS             =",
    round(
        audit_df[
            "high_items"
        ].mean(),
        2
    )
)


print(
    "AVG DOCUMENT CHARS         =",
    round(
        AVG_DOC_CHARS,
        2
    )
)


print(
    "MAX DOCUMENT CHARS         =",
    MAX_DOC_CHARS
)


print(
    "CATALOG                    =",
    OUT_CATALOG
)


print(
    "PACKETS                    =",
    OUT_PACKETS
)


print(
    "AUDIT                      =",
    OUT_AUDIT
)


print(
    "MANIFEST                   =",
    OUT_MANIFEST
)


print()

print(
    "[PASS] Stage 8 item-specific retrieval / "
    "judgment packet 완료"
)

print(
    "NEXT = Stage 9 fixed Gemma single-call judgment engine"
)

[5] LABEL LEAKAGE CHECK PASS
[5-B] v1~v24 prediction leakage CHECK PASS
[6] SAVE / RELOAD PASS
[7] AUDIT PASS
[8] LEGAL CATALOG AUDIT PASS
[9] PACKET SIZE SAFETY PASS
[10] MANIFEST VALIDATION PASS

=== Packet Size Audit ===


,high_items,medium_items,qualification_chars,product_chars,special_chars,consistency_chars,document_context_chars,document_segments
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,22.770000,1.230000,3155.940000,1815.600000,4089.940000,2377.025000,9710.785000,55.345000
std,1.101438,1.101438,2309.767063,1419.239233,2216.198635,1663.808329,3699.153676,29.737139
min,19.000000,0.000000,183.000000,45.000000,417.000000,51.000000,3035.000000,16.000000
25%,22.750000,1.000000,1540.250000,874.000000,2352.750000,1148.500000,6974.000000,32.750000
50%,23.000000,1.000000,2397.500000,1393.000000,3669.500000,1833.000000,9230.000000,47.000000
75%,23.000000,1.250000,3997.250000,2443.500000,5548.000000,3362.000000,12441.750000,72.500000
max,24.000000,5.000000,8999.000000,7826.000000,8000.000000,6979.000000,19242.000000,178.000000



=== DEV Positive Priority Audit ===


,item,positive,high_priority,medium_priority,items_dropped
0,v1,7,7,0,0
1,v2,7,7,0,0
2,v3,8,8,0,0
3,v4,6,6,0,0
4,v5,7,7,0,0
5,v6,6,6,0,0
6,v7,7,7,0,0
7,v8,6,6,0,0
8,v9,6,3,3,0
9,v10,7,5,2,0



=== Legal Catalog Audit ===


,item,name,absence,legal_chunks,legal_chars
0,v1,참가자격 특정기관 제한,False,6,4846
1,v2,고시금액 미만 실적제한,False,6,4810
2,v3,실적제한 1배수 이상,False,6,4736
3,v4,"고시금액 이상 특정기관, 특정실적",False,5,4756
4,v5,고시금액 이상 지역제한,False,5,4461
5,v6,"고시금액 미만 지역제한 시,군,구",False,6,4787
6,v7,고시금액 미만 지역제한 인접 확대,False,6,4652
7,v8,중복제한 (실적+지역),False,6,4959
8,v9,과업지시서 특정 모델명 명시,False,5,4756
9,v10,중기간 경쟁제품 입찰 직생 없음,True,2,1390



Stage 8 FINAL RESULT
RECORDS                    = 200
ITEM CATALOG               = 24
ALL ITEMS / NOTICE         = 24
ITEMS DROPPED              = 0
LABEL LEAKAGE              = False
CANDIDATE HARD GATE        = False
TARGET LLM CALLS / NOTICE  = 1
EXACT SEGMENTS CHECKED     = 11069
AVG HIGH ITEMS             = 22.77
AVG DOCUMENT CHARS         = 9710.78
MAX DOCUMENT CHARS         = 19242
CATALOG                    = /home/alparka/nara-ai/artifacts/judgment_item_catalog.json
PACKETS                    = /home/alparka/nara-ai/artifacts/judgment_packets_dev.jsonl
AUDIT                      = /home/alparka/nara-ai/artifacts/judgment_packet_audit.csv
MANIFEST                   = /home/alparka/nara-ai/artifacts/stage8_manifest.json

[PASS] Stage 8 item-specific retrieval / judgment packet 완료
NEXT = Stage 9 fixed Gemma single-call judgment engine


In [1]:
# ============================================================
# Stage 9 FINAL - REBUILT
# Fixed Gemma Single-call Judgment Engine
#
# 목표
# ------------------------------------------------------------
# 1. 공고 1건당 Gemma 기본 1회 호출
# 2. v1~v24 전부 판단
# 3. 공식 정답스키마_디코딩.json 그대로 사용
# 4. prompt 중복 제거
# 5. 로컬: char budget 검증
# 6. 평가 서버: 실제 tokenizer token budget 재검증
# 7. schema가 flat/nested 어느 구조여도 output normalize
#
# 중요
# ------------------------------------------------------------
# - dev_labels 사용 안 함
# - candidate miss는 hard gate 아님
# - evidence는 Stage 11에서 최종 exact substring 강제
# - 실제 평가 서버에서는 tokenizer 기준으로 prompt를 자동 축소
# ============================================================

from pathlib import Path
import os
import json
import gzip
import re
import inspect
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

DEV_PATH = (
    ROOT
    / "open"
    / "dev.jsonl.gz"
)

CATALOG_PATH = (
    ROOT
    / "artifacts"
    / "judgment_item_catalog.json"
)

PACKETS_PATH = (
    ROOT
    / "artifacts"
    / "judgment_packets_dev.jsonl"
)

SCHEMA_PATH = (
    ROOT
    / "open"
    / "data"
    / "정답스키마_디코딩.json"
)

BASELINE_SCRIPT_PATH = (
    ROOT
    / "open"
    / "baseline"
    / "script.py"
)

OUT_PROMPTS = (
    ROOT
    / "artifacts"
    / "stage9_prompts_dev.jsonl"
)

OUT_AUDIT = (
    ROOT
    / "artifacts"
    / "stage9_prompt_audit.csv"
)

OUT_CONFIG = (
    ROOT
    / "artifacts"
    / "stage9_engine_config.json"
)

OUT_MANIFEST = (
    ROOT
    / "artifacts"
    / "stage9_manifest.json"
)


for p in [
    DEV_PATH,
    CATALOG_PATH,
    PACKETS_PATH,
    SCHEMA_PATH,
    BASELINE_SCRIPT_PATH,
]:
    assert p.exists(), p


EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(1, 25)
]

EXPECTED_EVIDENCE = [
    f"e{i}"
    for i in range(1, 25)
]

ABSENCE_ITEMS = {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}


# ------------------------------------------------------------
# prompt/runtime budgets
#
# max_model_len = 16384
#
# 실제 서버:
# input <= 13,500 tokens
# output <= 2,200 tokens
#
# 합계 15,700 이하로 두어 여유 확보
# ------------------------------------------------------------

PROMPT_CHAR_TARGET = 24_000

MAX_DOC_PROMPT_CHARS = 9_000

MIN_DOC_PROMPT_CHARS = 2_000

MAX_INPUT_TOKENS = 13_500

MAX_OUTPUT_TOKENS = 2_200


# ============================================================
# 1. LOAD
# ============================================================

def load_jsonl(path):

    rows = []

    opener = (
        gzip.open
        if str(path).endswith(".gz")
        else open
    )

    with opener(
        path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


dev = load_jsonl(
    DEV_PATH
)

packets = load_jsonl(
    PACKETS_PATH
)


with open(
    CATALOG_PATH,
    "r",
    encoding="utf-8"
) as f:

    catalog = json.load(f)


with open(
    SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:

    official_schema = json.load(f)


baseline_text = (
    BASELINE_SCRIPT_PATH
    .read_text(
        encoding="utf-8",
        errors="replace"
    )
)


assert len(dev) == 200
assert len(packets) == 200

assert list(
    catalog.keys()
) == EXPECTED_ITEMS


dev_ids = [
    x["id"]
    for x in dev
]

packet_ids = [
    x["id"]
    for x in packets
]


assert dev_ids == packet_ids


DEV_BY_ID = {
    x["id"]: x
    for x in dev
}

PACKET_BY_ID = {
    x["id"]: x
    for x in packets
}


print("[1] INPUT LOAD PASS")


# ============================================================
# 2. OFFICIAL JSON SCHEMA
#
# flat schema라고 가정하지 않는다.
# 실제 JSON schema 객체를 찾아 그대로 사용.
# ============================================================

def looks_like_json_schema(obj):

    if not isinstance(
        obj,
        dict
    ):
        return False

    return any(
        key in obj
        for key in [
            "type",
            "properties",
            "$schema",
            "$defs",
            "definitions",
            "oneOf",
            "anyOf",
            "allOf",
        ]
    )


def resolve_json_schema(obj):

    # root 자체가 schema
    if looks_like_json_schema(
        obj
    ):
        return obj, "root"


    # {"schema": {...}}
    child = obj.get(
        "schema"
    )

    if looks_like_json_schema(
        child
    ):
        return child, "schema"


    # {"json_schema": {...}}
    child = obj.get(
        "json_schema"
    )

    if isinstance(
        child,
        dict
    ):

        if looks_like_json_schema(
            child
        ):

            return (
                child,
                "json_schema"
            )


        nested = child.get(
            "schema"
        )

        if looks_like_json_schema(
            nested
        ):

            return (
                nested,
                "json_schema.schema"
            )


    raise AssertionError(
        "\n공식 JSON schema를 식별하지 못했습니다.\n"
        f"top-level keys="
        f"{list(obj.keys())}"
    )


STRUCTURED_SCHEMA, SCHEMA_LOCATION = (
    resolve_json_schema(
        official_schema
    )
)


assert isinstance(
    STRUCTURED_SCHEMA,
    dict
)

assert len(
    STRUCTURED_SCHEMA
) > 0


def collect_property_paths(
    obj,
    path=()
):

    result = []


    if isinstance(
        obj,
        dict
    ):

        props = obj.get(
            "properties"
        )


        if isinstance(
            props,
            dict
        ):

            for key, value in (
                props.items()
            ):

                current = (
                    *path,
                    str(key)
                )

                result.append(
                    ".".join(
                        current
                    )
                )

                result.extend(
                    collect_property_paths(
                        value,
                        current
                    )
                )


        for key in [
            "items",
            "oneOf",
            "anyOf",
            "allOf",
            "$defs",
            "definitions",
        ]:

            if key in obj:

                result.extend(
                    collect_property_paths(
                        obj[key],
                        path
                    )
                )


    elif isinstance(
        obj,
        list
    ):

        for x in obj:

            result.extend(
                collect_property_paths(
                    x,
                    path
                )
            )


    return result


SCHEMA_PROPERTY_PATHS = (
    collect_property_paths(
        STRUCTURED_SCHEMA
    )
)


print("[2] OFFICIAL JSON SCHEMA PASS")

print(
    "    location =",
    SCHEMA_LOCATION
)

print(
    "    root keys =",
    list(
        STRUCTURED_SCHEMA.keys()
    )
)

print(
    "    property paths =",
    len(
        SCHEMA_PROPERTY_PATHS
    )
)


# ============================================================
# 3. BASELINE CONTRACT AUDIT
# ============================================================

BASELINE_API = {

    "uses_vllm":
        (
            "from vllm import"
            in baseline_text
            or
            "import vllm"
            in baseline_text
        ),

    "uses_pps_model_dir":
        (
            "PPS_MODEL_DIR"
            in baseline_text
        ),

    "uses_structured_output":
        any(
            key in baseline_text
            for key in [
                "structured_outputs",
                "StructuredOutputsParams",
                "guided_decoding",
                "GuidedDecodingParams",
            ]
        ),

    "uses_int8_weight_only":
        (
            "int8_per_channel_weight_only"
            in baseline_text
        ),

    "uses_max_model_len":
        (
            "max_model_len"
            in baseline_text
        ),
}


assert BASELINE_API[
    "uses_vllm"
]

assert BASELINE_API[
    "uses_pps_model_dir"
]


print("[3] BASELINE API AUDIT PASS")

print(
    BASELINE_API
)


# ============================================================
# 4. TEXT HELPERS
# ============================================================

def compact_ws(text):

    return re.sub(
        r"\s+",
        " ",
        str(
            text
            or
            ""
        )
    ).strip()


def clip(
    text,
    n
):

    text = compact_ws(
        text
    )

    if len(text) <= n:
        return text

    return text[:n]


# ============================================================
# 5. LEGAL SOURCE NAME SUMMARY
#
# 법령 원문 전체를 24번 prompt에 반복하지 않는다.
#
# Stage 6에서 이미 exact legal grounding →
# decision spec을 확정했기 때문에
# 여기서는 source/locator만 짧게 보존.
# ============================================================

def legal_source_summary(
    item,
    max_sources=2
):

    contexts = (
        catalog[
            item
        ].get(
            "legal_context",
            []
        )
    )


    parts = []

    seen = set()


    for x in contexts:

        source = clip(
            x.get(
                "source",
                ""
            ),
            70
        )

        locator = clip(
            x.get(
                "locator",
                ""
            ),
            45
        )


        key = (
            source,
            locator
        )


        if key in seen:
            continue


        seen.add(
            key
        )


        if source and locator:

            parts.append(
                f"{source} {locator}"
            )

        elif source:

            parts.append(
                source
            )

        elif locator:

            parts.append(
                locator
            )


        if len(parts) >= max_sources:
            break


    return "; ".join(
        parts
    )


# ============================================================
# 6. COMPACT RULE SHEETS
#
# FULL:
#   기본 로컬/서버 prompt
#
# MIN:
#   실제 tokenizer에서 input token이 너무 클 때 fallback
# ============================================================

def make_rule_line(
    item,
    minimal=False
):

    x = catalog[
        item
    ]


    absence = (
        "ABS"
        if x[
            "absence_detection"
        ]
        else "POS"
    )


    if minimal:

        return (
            f"{item} {x['name']} "
            f"[{absence}] | "
            f"위반:{clip(x.get('violation_condition',''), 150)} | "
            f"비위반:{clip(x.get('nonviolation_condition',''), 80)}"
        )


    legal = (
        legal_source_summary(
            item
        )
    )


    parts = [

        (
            f"{item} "
            f"{x['name']} "
            f"[{absence}]"
        ),

        (
            "적용:"
            +
            clip(
                x.get(
                    "applicability",
                    ""
                ),
                85
            )
        ),

        (
            "위반:"
            +
            clip(
                x.get(
                    "violation_condition",
                    ""
                ),
                190
            )
        ),

        (
            "비위반:"
            +
            clip(
                x.get(
                    "nonviolation_condition",
                    ""
                ),
                105
            )
        ),
    ]


    if legal:

        parts.append(
            "근거:"
            +
            clip(
                legal,
                135
            )
        )


    return " | ".join(
        parts
    )


RULE_SHEET_FULL = "\n".join(

    make_rule_line(
        v,
        minimal=False
    )

    for v
    in EXPECTED_ITEMS
)


RULE_SHEET_MIN = "\n".join(

    make_rule_line(
        v,
        minimal=True
    )

    for v
    in EXPECTED_ITEMS
)


# 아주 긴 법령 원문이 다시 끼어드는 것 방지
assert len(
    RULE_SHEET_FULL
) < 12_000, (
    f"RULE_SHEET_FULL 과대: "
    f"{len(RULE_SHEET_FULL)}"
)


assert len(
    RULE_SHEET_MIN
) < len(
    RULE_SHEET_FULL
)


print(
    "[4] RULE SHEET FULL chars =",
    len(
        RULE_SHEET_FULL
    )
)

print(
    "[4] RULE SHEET MIN  chars =",
    len(
        RULE_SHEET_MIN
    )
)


# ============================================================
# 7. COMPACT STRUCTURED FACTS
#
# Stage 7 numeric/date 후보의 긴 "context"를
# 다시 prompt에 넣지 않는다.
#
# 원문은 아래 document_segments에서 별도로 들어감.
# ============================================================

def unique_take(
    rows,
    fields,
    limit
):

    result = []

    seen = set()


    for row in rows:

        compact_row = {

            field:
                row.get(
                    field
                )

            for field
            in fields
        }


        signature = json.dumps(
            compact_row,
            ensure_ascii=False,
            sort_keys=True,
            default=str
        )


        if signature in seen:
            continue


        seen.add(
            signature
        )

        result.append(
            compact_row
        )


        if len(result) >= limit:
            break


    return result


def make_compact_facts(
    packet
):

    product_lookup = (
        packet[
            "product"
        ][
            "competitive_lookup"
        ]
    )


    money = (
        packet[
            "numeric_candidates"
        ].get(
            "document_money",
            []
        )
    )


    percentages = (
        packet[
            "numeric_candidates"
        ].get(
            "joint_percentages",
            []
        )
    )


    dates = packet.get(
        "date_candidates",
        []
    )


    high_items = [

        v
        for v, priority
        in packet[
            "item_priority"
        ].items()

        if priority == "HIGH"
    ]


    result = {

        "meta":
            packet[
                "meta"
            ],

        "completeness": {

            "fully_observed":
                packet[
                    "completeness"
                ].get(
                    "fully_observed"
                ),

            "has_dropped_docs":
                packet[
                    "completeness"
                ].get(
                    "has_dropped_docs"
                ),

            "dropped_doc_counts":
                packet[
                    "completeness"
                ].get(
                    "dropped_doc_counts"
                ),
        },

        "product": {

            "codes":
                packet[
                    "product"
                ].get(
                    "codes",
                    []
                ),

            "status":
                product_lookup.get(
                    "status"
                ),

            "is_competitive_product":
                product_lookup.get(
                    "is_competitive_product"
                ),

            "matched_codes":
                product_lookup.get(
                    "matched_codes",
                    []
                ),

            "unmatched_codes":
                product_lookup.get(
                    "unmatched_codes",
                    []
                ),
        },

        "money_candidates":
            unique_take(

                money,

                [
                    "doc_id",
                    "raw",
                    "normalized_won",
                    "labels",
                ],

                limit=12
            ),

        "joint_percentages":
            unique_take(

                percentages,

                [
                    "doc_id",
                    "raw",
                    "percent",
                ],

                limit=10
            ),

        "date_candidates":
            unique_take(

                dates,

                [
                    "doc_id",
                    "raw",
                    "iso_date",
                ],

                limit=12
            ),

        # priority는 판정 결과가 아니라 retrieval hint
        "high_priority_items":
            high_items,
    }


    return result


def compact_json(
    obj
):

    return json.dumps(
        obj,
        ensure_ascii=False,
        separators=(
            ",",
            ":"
        ),
        default=str
    )


# ============================================================
# 8. DOCUMENT SEGMENT VALIDATION
# ============================================================

def validate_segment(
    seg,
    doc_text
):

    start = int(
        seg[
            "start"
        ]
    )

    end = int(
        seg[
            "end"
        ]
    )


    assert (
        0
        <=
        start
        <=
        end
        <=
        len(
            doc_text
        )
    )


    assert (
        doc_text[
            start:end
        ]
        ==
        seg[
            "text"
        ]
    )


# ============================================================
# 9. DOCUMENT CONTEXT SELECTION
#
# 정확도 전략:
#
# 1) 모든 문서 HEAD 최소 포함
# 2) CANDIDATE 우선 추가
# 3) 여유가 있으면 TAIL
#
# HEAD도 길이를 줄여 budget 확보.
# ============================================================

def truncate_exact_segment(
    seg,
    max_chars,
    original
):

    start = int(
        seg[
            "start"
        ]
    )

    end = int(
        seg[
            "end"
        ]
    )


    if (
        end - start
        <= max_chars
    ):

        new_seg = dict(
            seg
        )

        validate_segment(
            new_seg,
            original
        )

        return new_seg


    new_end = (
        start
        +
        max_chars
    )


    new_seg = dict(
        seg
    )

    new_seg[
        "end"
    ] = new_end

    new_seg[
        "text"
    ] = original[
        start:new_end
    ]


    validate_segment(
        new_seg,
        original
    )


    return new_seg


def select_document_context(
    packet,
    row,
    max_chars
):

    doc_map = {

        doc[
            "doc_id"
        ]:
            doc.get(
                "text",
                ""
            )

        for doc
        in row[
            "docs"
        ]
    }


    segments = (
        packet[
            "document_segments"
        ]
    )


    # --------------------------------------------------------
    # exact validation
    # --------------------------------------------------------

    for seg in segments:

        doc_id = seg[
            "doc_id"
        ]

        assert doc_id in doc_map


        validate_segment(
            seg,
            doc_map[
                doc_id
            ]
        )


    # --------------------------------------------------------
    # 모든 doc에 최소 HEAD 확보
    # --------------------------------------------------------

    heads_by_doc = {}


    for seg in segments:

        if seg[
            "kind"
        ] != "HEAD":
            continue


        doc_id = seg[
            "doc_id"
        ]


        if doc_id in heads_by_doc:
            continue


        heads_by_doc[
            doc_id
        ] = (
            truncate_exact_segment(

                seg,

                max_chars=550,

                original=doc_map[
                    doc_id
                ]
            )
        )


    # Stage 8에서 모든 docs HEAD를 만든 상태여야 함
    expected_docs = {
        doc[
            "doc_id"
        ]
        for doc
        in row[
            "docs"
        ]
    }


    assert (
        set(
            heads_by_doc.keys()
        )
        ==
        expected_docs
    ), (
        f"{packet['id']}: "
        "모든 document HEAD가 packet에 없음"
    )


    candidates = [

        seg
        for seg in segments

        if seg[
            "kind"
        ] == "CANDIDATE"
    ]


    tails = [

        seg
        for seg in segments

        if seg[
            "kind"
        ] == "TAIL"
    ]


    # --------------------------------------------------------
    # duplicate interval 방지
    # --------------------------------------------------------

    selected = []

    seen = set()

    used = 0


    def try_add(
        seg,
        allow_trim=False,
        trim_chars=None
    ):

        nonlocal used


        doc_id = seg[
            "doc_id"
        ]

        current = seg


        if (
            allow_trim
            and
            trim_chars is not None
        ):

            current = (
                truncate_exact_segment(

                    seg,

                    trim_chars,

                    doc_map[
                        doc_id
                    ]
                )
            )


        key = (
            current[
                "doc_id"
            ],
            int(
                current[
                    "start"
                ]
            ),
            int(
                current[
                    "end"
                ]
            ),
        )


        if key in seen:
            return


        size = len(
            current[
                "text"
            ]
        )


        if (
            used + size
            >
            max_chars
        ):

            return


        # 이미 완전히 포함된 interval이면 생략
        for old in selected:

            if (
                old[
                    "doc_id"
                ]
                ==
                current[
                    "doc_id"
                ]
                and
                int(
                    old[
                        "start"
                    ]
                )
                <=
                int(
                    current[
                        "start"
                    ]
                )
                and
                int(
                    old[
                        "end"
                    ]
                )
                >=
                int(
                    current[
                        "end"
                    ]
                )
            ):

                return


        seen.add(
            key
        )

        selected.append(
            current
        )

        used += size


    # --------------------------------------------------------
    # 1. 모든 docs HEAD
    # --------------------------------------------------------

    for doc in row[
        "docs"
    ]:

        try_add(
            heads_by_doc[
                doc[
                    "doc_id"
                ]
            ]
        )


    # --------------------------------------------------------
    # 2. candidate
    # --------------------------------------------------------

    for seg in candidates:

        try_add(
            seg
        )


    # --------------------------------------------------------
    # 3. tail은 짧게
    # --------------------------------------------------------

    for seg in tails:

        try_add(
            seg,
            allow_trim=True,
            trim_chars=300
        )


    selected.sort(
        key=lambda x: (
            str(
                x[
                    "doc_id"
                ]
            ),
            int(
                x[
                    "start"
                ]
            ),
        )
    )


    assert selected


    return selected


# ============================================================
# 10. DOCUMENT FORMAT
# ============================================================

def format_document_segments(
    segments
):

    parts = []


    for idx, seg in enumerate(
        segments,
        start=1
    ):

        parts.append(

            f"[D{idx}] "
            f"id={seg['doc_id']} "
            f"type={seg['doc_type']} "
            f"kind={seg['kind']} "
            f"offset={seg['start']}:{seg['end']}\n"
            f"{seg['text']}"
        )


    return "\n\n".join(
        parts
    )


# ============================================================
# 11. SYSTEM PROMPT
#
# 이전보다 짧게.
# ============================================================

SYSTEM_PROMPT = """
너는 나라장터 자체입찰 공고의 24개 검토항목 판정 엔진이다.
제공된 공고·첨부·meta·판정규칙만 사용하고 외부 지식을 추가하지 마라.

규칙:
- v1~v24를 모두 판단한다.
- 위반은 1, 비위반은 0이다.
- HIGH priority는 검색 힌트일 뿐 정답이 아니다.
- 검색 후보가 없다는 이유만으로 0으로 판단하지 않는다.
- 모든 제공 문서의 내용을 함께 고려한다.
- null과 "미입력"을 임의로 같은 의미로 처리하지 않는다.
- v10,v11,v16,v18,v20은 부재탐지형이다.
- 부재탐지에서 완전관측이 아니거나 첨부가 탈락했으면 단순한 문구 부재만으로 위반을 확정하지 않는다.
- v24는 공고/첨부와 나라장터 meta의 실질적 불일치 여부다.
- evidence는 문서 원문에서 그대로 복사한 연속 부분문자열만 사용한다.
- 위반이 아니면 evidence는 빈 문자열이다.
- v10,v11,v16,v18,v20 evidence는 항상 빈 문자열이다.
- 설명문은 출력하지 말고 structured JSON만 출력한다.
""".strip()


# ============================================================
# 12. PROMPT BUILDER
#
# fixed text 크기를 먼저 계산한 뒤
# 남는 budget만 docs에 할당.
# ============================================================

def build_prompt(
    packet,
    row,
    *,
    rule_sheet=None,
    doc_char_budget=None,
    target_chars=PROMPT_CHAR_TARGET
):

    if rule_sheet is None:

        rule_sheet = (
            RULE_SHEET_FULL
        )


    structured_text = (
        compact_json(
            make_compact_facts(
                packet
            )
        )
    )


    prefix = f"""
[공고 ID]
{packet['id']}

[구조화 사실]
{structured_text}

[판정 규칙]
{rule_sheet}

[공고 원문]
""".strip()


    suffix = """
[출력]
공식 JSON schema에 맞춰 24개 항목을 모두 판정하라.
각 위반 여부는 0 또는 1이다.
근거는 제공된 공고 원문의 exact substring만 사용한다.
부재탐지 v10,v11,v16,v18,v20의 근거는 빈 문자열이다.
""".strip()


    fixed_chars = (

        len(
            SYSTEM_PROMPT
        )
        +
        len(
            prefix
        )
        +
        len(
            suffix
        )
        +
        200
    )


    assert (
        fixed_chars
        <
        target_chars
    ), (
        "\n문서 원문을 넣기 전부터 prompt가 너무 큼\n"
        f"fixed_chars={fixed_chars}\n"
        f"target={target_chars}"
    )


    if doc_char_budget is None:

        doc_char_budget = min(

            MAX_DOC_PROMPT_CHARS,

            max(
                MIN_DOC_PROMPT_CHARS,
                target_chars
                -
                fixed_chars
                -
                500
            )
        )


    segments = (
        select_document_context(
            packet,
            row,
            max_chars=doc_char_budget
        )
    )


    document_text = (
        format_document_segments(
            segments
        )
    )


    user_prompt = (
        prefix
        +
        "\n"
        +
        document_text
        +
        "\n\n"
        +
        suffix
    )


    total_chars = (

        len(
            SYSTEM_PROMPT
        )
        +
        len(
            user_prompt
        )
    )


    # --------------------------------------------------------
    # format metadata 때문에 target 초과 시
    # docs budget을 조금씩 줄여 재작성
    # --------------------------------------------------------

    retry_budget = (
        doc_char_budget
    )


    while (
        total_chars
        >
        target_chars
        and
        retry_budget
        >
        MIN_DOC_PROMPT_CHARS
    ):

        retry_budget = max(
            MIN_DOC_PROMPT_CHARS,
            retry_budget - 750
        )


        segments = (
            select_document_context(
                packet,
                row,
                max_chars=retry_budget
            )
        )


        document_text = (
            format_document_segments(
                segments
            )
        )


        user_prompt = (
            prefix
            +
            "\n"
            +
            document_text
            +
            "\n\n"
            +
            suffix
        )


        total_chars = (

            len(
                SYSTEM_PROMPT
            )
            +
            len(
                user_prompt
            )
        )


        if (
            retry_budget
            ==
            MIN_DOC_PROMPT_CHARS
        ):
            break


    assert (
        total_chars
        <=
        target_chars
    ), (
        f"{packet['id']}: "
        f"prompt char budget 초과 "
        f"{total_chars} > {target_chars}"
    )


    return {

        "id":
            packet[
                "id"
            ],

        "system":
            SYSTEM_PROMPT,

        "user":
            user_prompt,

        "segments":
            segments,

        "rule_mode":
            (
                "MIN"
                if rule_sheet
                ==
                RULE_SHEET_MIN
                else
                "FULL"
            ),

        "document_char_budget":
            retry_budget,

        "total_chars":
            total_chars,
    }


# ============================================================
# 13. BUILD LOCAL DEV PROMPTS
#
# local에서는 FULL rule sheet로 24k char 안에
# 들어오는지 먼저 검증.
# ============================================================

PROMPTS = []


for idx, packet in enumerate(
    packets,
    start=1
):

    row = DEV_BY_ID[
        packet[
            "id"
        ]
    ]


    prompt = build_prompt(
        packet,
        row,
        rule_sheet=RULE_SHEET_FULL
    )


    PROMPTS.append(
        prompt
    )


    if idx % 50 == 0:

        print(
            f"[prompt] "
            f"{idx}/200"
        )


assert len(
    PROMPTS
) == 200


print("[5] PROMPT BUILD PASS")


# ============================================================
# 14. LOCAL PROMPT AUDIT
# ============================================================

prompt_rows = []


for x in PROMPTS:

    doc_chars = sum(

        len(
            seg[
                "text"
            ]
        )

        for seg
        in x[
            "segments"
        ]
    )


    prompt_rows.append({

        "id":
            x[
                "id"
            ],

        "system_chars":
            len(
                x[
                    "system"
                ]
            ),

        "user_chars":
            len(
                x[
                    "user"
                ]
            ),

        "total_chars":
            x[
                "total_chars"
            ],

        "document_chars":
            doc_chars,

        "document_segments":
            len(
                x[
                    "segments"
                ]
            ),

        "document_budget":
            x[
                "document_char_budget"
            ],

        "rule_mode":
            x[
                "rule_mode"
            ],
    })


prompt_audit = (
    pd.DataFrame(
        prompt_rows
    )
)


AVG_PROMPT_CHARS = float(
    prompt_audit[
        "total_chars"
    ].mean()
)

MAX_PROMPT_CHARS = int(
    prompt_audit[
        "total_chars"
    ].max()
)


assert (
    MAX_PROMPT_CHARS
    <=
    PROMPT_CHAR_TARGET
)


print("[6] LOCAL PROMPT SIZE PASS")

print(
    "    avg =",
    round(
        AVG_PROMPT_CHARS,
        2
    )
)

print(
    "    max =",
    MAX_PROMPT_CHARS
)


# ============================================================
# 15. OUTPUT PARSER
#
# official schema의 내부 구조를 flat이라고 가정하지 않음.
# ============================================================

ITEM_ID_KEYS = {
    "item",
    "item_id",
    "violation_id",
    "key",
    "항목",
    "항목번호",
}

VALUE_KEYS = {
    "violation",
    "violated",
    "value",
    "result",
    "label",
    "is_violation",
    "위반",
    "위반여부",
}

EVIDENCE_KEYS = {
    "evidence",
    "evidence_text",
    "quote",
    "ground",
    "근거",
    "근거문구",
}


def default_output():

    result = {}

    for i in range(
        1,
        25
    ):

        result[
            f"v{i}"
        ] = 0

        result[
            f"e{i}"
        ] = ""

    return result


def parse_json_output(
    raw
):

    if isinstance(
        raw,
        (
            dict,
            list,
        )
    ):

        return raw


    text = str(
        raw
    ).strip()


    try:

        return json.loads(
            text
        )

    except Exception:
        pass


    left = text.find(
        "{"
    )

    right = text.rfind(
        "}"
    )


    if (
        left >= 0
        and
        right > left
    ):

        return json.loads(
            text[
                left:
                right + 1
            ]
        )


    left = text.find(
        "["
    )

    right = text.rfind(
        "]"
    )


    if (
        left >= 0
        and
        right > left
    ):

        return json.loads(
            text[
                left:
                right + 1
            ]
        )


    raise ValueError(
        "JSON output parsing 실패"
    )


def normalize_binary(
    value
):

    if value is True:
        return 1

    if value is False:
        return 0


    if isinstance(
        value,
        str
    ):

        x = (
            value
            .strip()
            .lower()
        )


        if x in {
            "1",
            "1.0",
            "true",
            "yes",
            "y",
        }:

            return 1


        if x in {
            "0",
            "0.0",
            "false",
            "no",
            "n",
            "",
        }:

            return 0


    try:

        value = int(
            value
        )

    except Exception:

        return None


    if value in {
        0,
        1
    }:

        return value


    return None


def normalize_item_id(
    value
):

    value = str(
        value
    ).strip()


    if re.fullmatch(
        r"v(?:[1-9]|1\d|2[0-4])",
        value
    ):

        return value


    if re.fullmatch(
        r"(?:[1-9]|1\d|2[0-4])",
        value
    ):

        return (
            "v"
            +
            value
        )


    return None


def flatten_model_object(
    obj
):

    found_v = {}

    found_e = {}


    def put_v(
        item,
        value
    ):

        item = (
            normalize_item_id(
                item
            )
        )


        if item is None:
            return


        value = (
            normalize_binary(
                value
            )
        )


        if value is None:
            return


        found_v[
            item
        ] = value


    def put_e(
        item,
        value
    ):

        item = (
            normalize_item_id(
                item
            )
        )


        if item is None:
            return


        idx = int(
            item[
                1:
            ]
        )


        found_e[
            f"e{idx}"
        ] = (
            ""
            if value is None
            else str(
                value
            )
        )


    def walk(
        current,
        path=()
    ):

        # ----------------------------------------------------
        # dict
        # ----------------------------------------------------

        if isinstance(
            current,
            dict
        ):

            item_id = None


            for key in ITEM_ID_KEYS:

                if key in current:

                    candidate = (
                        normalize_item_id(
                            current[
                                key
                            ]
                        )
                    )


                    if candidate:

                        item_id = candidate

                        break


            # record 구조
            if item_id:

                for key in VALUE_KEYS:

                    if key in current:

                        put_v(
                            item_id,
                            current[
                                key
                            ]
                        )

                        break


                for key in EVIDENCE_KEYS:

                    if key in current:

                        put_e(
                            item_id,
                            current[
                                key
                            ]
                        )

                        break


            for key, value in (
                current.items()
            ):

                key_str = str(
                    key
                )


                # --------------------------------------------
                # e1 ~ e24
                # --------------------------------------------

                if re.fullmatch(
                    r"e(?:[1-9]|1\d|2[0-4])",
                    key_str
                ):

                    if not isinstance(
                        value,
                        (
                            dict,
                            list,
                        )
                    ):

                        found_e[
                            key_str
                        ] = (
                            ""
                            if value is None
                            else str(
                                value
                            )
                        )


                # --------------------------------------------
                # v1 ~ v24
                # --------------------------------------------

                elif re.fullmatch(
                    r"v(?:[1-9]|1\d|2[0-4])",
                    key_str
                ):

                    if isinstance(
                        value,
                        dict
                    ):

                        for value_key in (
                            VALUE_KEYS
                        ):

                            if (
                                value_key
                                in value
                            ):

                                put_v(
                                    key_str,
                                    value[
                                        value_key
                                    ]
                                )

                                break


                        for evidence_key in (
                            EVIDENCE_KEYS
                        ):

                            if (
                                evidence_key
                                in value
                            ):

                                put_e(
                                    key_str,
                                    value[
                                        evidence_key
                                    ]
                                )

                                break


                    elif not isinstance(
                        value,
                        list
                    ):

                        # evidence.v1 = "..." 구조 구분
                        lower_path = [

                            str(x)
                            .lower()

                            for x
                            in path
                        ]


                        evidence_container = any(

                            (
                                "evidence"
                                in p
                                or
                                "근거"
                                in p
                            )

                            for p
                            in lower_path
                        )


                        if evidence_container:

                            put_e(
                                key_str,
                                value
                            )

                        else:

                            put_v(
                                key_str,
                                value
                            )


                walk(
                    value,
                    (
                        *path,
                        key_str
                    )
                )


        # ----------------------------------------------------
        # list
        # ----------------------------------------------------

        elif isinstance(
            current,
            list
        ):

            for idx, value in enumerate(
                current
            ):

                walk(
                    value,
                    (
                        *path,
                        str(idx)
                    )
                )


    walk(
        obj
    )


    # --------------------------------------------------------
    # 24-element scalar arrays도 지원
    # --------------------------------------------------------

    if isinstance(
        obj,
        dict
    ):

        for key in [
            "violations",
            "labels",
            "results",
        ]:

            values = obj.get(
                key
            )


            if (
                isinstance(
                    values,
                    list
                )
                and
                len(
                    values
                )
                ==
                24
                and
                all(
                    not isinstance(
                        x,
                        (
                            dict,
                            list,
                        )
                    )
                    for x in values
                )
            ):

                for i, value in enumerate(
                    values,
                    start=1
                ):

                    put_v(
                        f"v{i}",
                        value
                    )


        for key in [
            "evidence",
            "evidences",
            "grounds",
        ]:

            values = obj.get(
                key
            )


            if (
                isinstance(
                    values,
                    list
                )
                and
                len(
                    values
                )
                ==
                24
                and
                all(
                    not isinstance(
                        x,
                        (
                            dict,
                            list,
                        )
                    )
                    for x in values
                )
            ):

                for i, value in enumerate(
                    values,
                    start=1
                ):

                    put_e(
                        f"v{i}",
                        value
                    )


    missing_v = [

        f"v{i}"

        for i in range(
            1,
            25
        )

        if (
            f"v{i}"
            not in found_v
        )
    ]


    # 여기서는 silent zero-fill 금지
    if missing_v:

        raise ValueError(
            "\n모델 출력에서 v1~v24를 모두 복원하지 못함\n"
            f"missing={missing_v}\n"
            f"type={type(obj).__name__}"
        )


    result = (
        default_output()
    )


    for i in range(
        1,
        25
    ):

        vkey = (
            f"v{i}"
        )

        ekey = (
            f"e{i}"
        )


        result[
            vkey
        ] = int(
            found_v[
                vkey
            ]
        )


        evidence = (
            found_e.get(
                ekey,
                ""
            )
        )


        if (
            result[
                vkey
            ]
            ==
            0
        ):

            evidence = ""


        if (
            vkey
            in ABSENCE_ITEMS
        ):

            evidence = ""


        result[
            ekey
        ] = str(
            evidence
        )[:500]


    return result


def normalize_model_output(
    raw
):

    return flatten_model_object(
        parse_json_output(
            raw
        )
    )


# ============================================================
# 16. PARSER TEST
# ============================================================

# ------------------------------------------------------------
# flat
# ------------------------------------------------------------

flat = {}


for i in range(
    1,
    25
):

    flat[
        f"v{i}"
    ] = (
        1
        if i == 2
        else 0
    )

    flat[
        f"e{i}"
    ] = (
        "실적제한 원문"
        if i == 2
        else ""
    )


parsed = (
    normalize_model_output(
        flat
    )
)


assert parsed[
    "v2"
] == 1

assert parsed[
    "e2"
] == "실적제한 원문"


# ------------------------------------------------------------
# nested item records
# ------------------------------------------------------------

nested = {

    "items": [

        {
            "item":
                f"v{i}",

            "violation":
                (
                    1
                    if i == 19
                    else 0
                ),

            "evidence":
                (
                    "물품공급 확약서"
                    if i == 19
                    else ""
                ),
        }

        for i in range(
            1,
            25
        )
    ]
}


parsed = (
    normalize_model_output(
        nested
    )
)


assert parsed[
    "v19"
] == 1

assert (
    parsed[
        "e19"
    ]
    ==
    "물품공급 확약서"
)


# ------------------------------------------------------------
# absence evidence 강제 제거
# ------------------------------------------------------------

absence_test = {

    "items": [

        {
            "item":
                f"v{i}",

            "violation":
                (
                    1
                    if i == 10
                    else 0
                ),

            "evidence":
                (
                    "없어져야 함"
                    if i == 10
                    else ""
                ),
        }

        for i in range(
            1,
            25
        )
    ]
}


parsed = (
    normalize_model_output(
        absence_test
    )
)


assert parsed[
    "v10"
] == 1

assert parsed[
    "e10"
] == ""


print("[7] OUTPUT PARSER PASS")


# ============================================================
# 17. LOCAL MODEL RUNTIME CHECK
# ============================================================

MODEL_DIR = (
    os.environ.get(
        "PPS_MODEL_DIR"
    )
)


try:

    import vllm  # noqa

    VLLM_AVAILABLE = True

except Exception:

    VLLM_AVAILABLE = False


MODEL_DIR_AVAILABLE = bool(

    MODEL_DIR
    and
    Path(
        MODEL_DIR
    ).exists()
)


MODEL_RUNTIME_AVAILABLE = (

    VLLM_AVAILABLE
    and
    MODEL_DIR_AVAILABLE
)


print()
print(
    "VLLM AVAILABLE      =",
    VLLM_AVAILABLE
)

print(
    "PPS_MODEL_DIR       =",
    MODEL_DIR
)

print(
    "MODEL DIR AVAILABLE =",
    MODEL_DIR_AVAILABLE
)

print(
    "REAL MODEL RUNTIME  =",
    MODEL_RUNTIME_AVAILABLE
)


# ============================================================
# 18. VLLM ENGINE
# ============================================================

def create_vllm_engine():

    model_dir = (
        os.environ.get(
            "PPS_MODEL_DIR"
        )
    )


    if not model_dir:

        raise RuntimeError(
            "PPS_MODEL_DIR가 설정되지 않았습니다."
        )


    from vllm import LLM


    return LLM(

        model=model_dir,

        quantization=(
            "int8_per_channel_weight_only"
        ),

        max_model_len=16384,

        trust_remote_code=True,
    )


# ============================================================
# 19. STRUCTURED SAMPLING PARAMS
#
# vLLM 버전에 따라 API를 자동 선택.
# ============================================================

def build_sampling_params(
    schema
):

    from vllm import SamplingParams


    signature = (
        inspect.signature(
            SamplingParams
        )
    )


    supported = set(
        signature.parameters.keys()
    )


    kwargs = {

        "temperature":
            0.0,

        "top_p":
            1.0,

        "max_tokens":
            MAX_OUTPUT_TOKENS,
    }


    if "seed" in supported:

        kwargs[
            "seed"
        ] = 42


    # --------------------------------------------------------
    # newer vLLM structured_outputs
    # --------------------------------------------------------

    if (
        "structured_outputs"
        in supported
    ):

        try:

            from vllm.sampling_params import (
                StructuredOutputsParams
            )


            kwargs[
                "structured_outputs"
            ] = (
                StructuredOutputsParams(
                    json=schema
                )
            )


        except ImportError:

            kwargs[
                "structured_outputs"
            ] = {
                "json":
                    schema
            }


        return SamplingParams(
            **kwargs
        )


    # --------------------------------------------------------
    # older guided_decoding API
    # --------------------------------------------------------

    if (
        "guided_decoding"
        in supported
    ):

        GuidedDecodingParams = None


        try:

            from vllm.sampling_params import (
                GuidedDecodingParams
            )

        except ImportError:

            try:

                from vllm import (
                    GuidedDecodingParams
                )

            except ImportError:

                pass


        if (
            GuidedDecodingParams
            is None
        ):

            raise RuntimeError(
                "guided_decoding API는 존재하지만 "
                "GuidedDecodingParams를 import할 수 없습니다."
            )


        kwargs[
            "guided_decoding"
        ] = (
            GuidedDecodingParams(
                json=schema
            )
        )


        return SamplingParams(
            **kwargs
        )


    raise RuntimeError(
        "\n현재 vLLM SamplingParams에서 "
        "structured_outputs/guided_decoding을 찾지 못했습니다.\n"
        f"supported={sorted(supported)}"
    )


# ============================================================
# 20. CHAT TEMPLATE
# ============================================================

def render_chat_prompt(
    tokenizer,
    prompt_obj
):

    messages = [

        {
            "role":
                "system",

            "content":
                prompt_obj[
                    "system"
                ],
        },

        {
            "role":
                "user",

            "content":
                prompt_obj[
                    "user"
                ],
        },
    ]


    return (
        tokenizer.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True,
        )
    )


# ============================================================
# 21. SERVER-SIDE TOKEN FIT
#
# 로컬 char budget만 믿지 않는다.
#
# 실제 평가 서버 tokenizer로 token 수를 계산하고
# 초과하면 docs → compact rules 순으로 줄임.
# ============================================================

def count_tokens(
    tokenizer,
    rendered
):

    token_ids = tokenizer.encode(

        rendered,

        add_special_tokens=False
    )


    return len(
        token_ids
    )


def fit_prompt_to_token_budget(
    tokenizer,
    packet,
    row
):

    attempts = [

        # full rules
        (
            RULE_SHEET_FULL,
            9000
        ),

        (
            RULE_SHEET_FULL,
            7000
        ),

        (
            RULE_SHEET_FULL,
            5500
        ),

        (
            RULE_SHEET_FULL,
            4000
        ),

        (
            RULE_SHEET_FULL,
            2800
        ),

        # minimal rules fallback
        (
            RULE_SHEET_MIN,
            6000
        ),

        (
            RULE_SHEET_MIN,
            4500
        ),

        (
            RULE_SHEET_MIN,
            3200
        ),

        (
            RULE_SHEET_MIN,
            2200
        ),
    ]


    audit = []


    for (
        rule_sheet,
        doc_budget
    ) in attempts:

        # char target은 충분히 높이고
        # 실제 token 기준으로 최종 판단
        prompt_obj = build_prompt(

            packet,

            row,

            rule_sheet=rule_sheet,

            doc_char_budget=doc_budget,

            target_chars=30_000
        )


        rendered = (
            render_chat_prompt(
                tokenizer,
                prompt_obj
            )
        )


        token_count = (
            count_tokens(
                tokenizer,
                rendered
            )
        )


        audit.append({

            "rule_mode":
                (
                    "MIN"
                    if rule_sheet
                    ==
                    RULE_SHEET_MIN
                    else
                    "FULL"
                ),

            "doc_budget":
                doc_budget,

            "input_tokens":
                token_count,
        })


        if (
            token_count
            <=
            MAX_INPUT_TOKENS
        ):

            return {

                "prompt_obj":
                    prompt_obj,

                "rendered":
                    rendered,

                "input_tokens":
                    token_count,

                "fit_audit":
                    audit,
            }


    raise RuntimeError(
        "\n실제 tokenizer 기준 prompt가 "
        "13,500 tokens 이하로 줄어들지 않았습니다.\n"
        f"id={packet['id']}\n"
        f"audit={audit}"
    )


# ============================================================
# 22. BATCH INFERENCE
# ============================================================

def run_vllm_batch(
    llm,
    packet_list,
    row_by_id,
    schema,
    batch_size=16
):

    sampling_params = (
        build_sampling_params(
            schema
        )
    )


    tokenizer = (
        llm.get_tokenizer()
    )


    results = []


    for start in range(
        0,
        len(
            packet_list
        ),
        batch_size
    ):

        batch = packet_list[
            start:
            start + batch_size
        ]


        fitted = []


        for packet in batch:

            row = row_by_id[
                packet[
                    "id"
                ]
            ]


            fitted.append(

                fit_prompt_to_token_budget(
                    tokenizer,
                    packet,
                    row
                )
            )


        rendered = [

            x[
                "rendered"
            ]

            for x
            in fitted
        ]


        outputs = llm.generate(

            rendered,

            sampling_params,
        )


        assert (
            len(
                outputs
            )
            ==
            len(
                batch
            )
        )


        for (
            packet,
            fit_obj,
            output
        ) in zip(
            batch,
            fitted,
            outputs
        ):

            assert output.outputs, (
                f"{packet['id']}: "
                "LLM output 없음"
            )


            raw_text = (
                output.outputs[
                    0
                ].text
            )


            parsed = (
                normalize_model_output(
                    raw_text
                )
            )


            assert set(
                parsed.keys()
            ) == set(
                EXPECTED_ITEMS
                +
                EXPECTED_EVIDENCE
            )


            results.append({

                "id":
                    packet[
                        "id"
                    ],

                "input_tokens":
                    fit_obj[
                        "input_tokens"
                    ],

                "rule_mode":
                    fit_obj[
                        "prompt_obj"
                    ][
                        "rule_mode"
                    ],

                "raw":
                    raw_text,

                "prediction":
                    parsed,
            })


    return results


# ============================================================
# 23. MODEL SMOKE
#
# 로컬에 모델이 없으면 SKIP 정상.
# ============================================================

MODEL_SMOKE_RESULT = (
    "SKIPPED_LOCAL_NO_MODEL"
)


if MODEL_RUNTIME_AVAILABLE:

    print()
    print(
        "[MODEL] Gemma 1건 smoke test"
    )


    llm = (
        create_vllm_engine()
    )


    smoke = (
        run_vllm_batch(

            llm,

            packets[:1],

            DEV_BY_ID,

            STRUCTURED_SCHEMA,

            batch_size=1
        )
    )


    assert len(
        smoke
    ) == 1


    MODEL_SMOKE_RESULT = (
        "PASS"
    )


    print(
        "[MODEL] smoke input tokens =",
        smoke[
            0
        ][
            "input_tokens"
        ]
    )


else:

    print()

    print(
        "[MODEL] 로컬에 실제 고정 모델 없음"
    )

    print(
        "[MODEL] smoke test 정상 SKIP"
    )


# ============================================================
# 24. SAVE LOCAL PROMPTS
# ============================================================

with open(
    OUT_PROMPTS,
    "w",
    encoding="utf-8"
) as f:

    for x in PROMPTS:

        record = {

            "id":
                x[
                    "id"
                ],

            "system":
                x[
                    "system"
                ],

            "user":
                x[
                    "user"
                ],

            "rule_mode":
                x[
                    "rule_mode"
                ],

            "total_chars":
                x[
                    "total_chars"
                ],
        }


        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            +
            "\n"
        )


prompt_audit.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 25. ENGINE CONFIG
# ============================================================

ENGINE_CONFIG = {

    "backend":
        "vllm offline",

    "model":
        "PPS_MODEL_DIR",

    "quantization":
        "int8_per_channel_weight_only",

    "max_model_len":
        16384,

    "max_input_tokens":
        MAX_INPUT_TOKENS,

    "max_output_tokens":
        MAX_OUTPUT_TOKENS,

    "temperature":
        0.0,

    "top_p":
        1.0,

    "calls_per_notice":
        1,

    "structured_output":
        True,

    "schema_location":
        SCHEMA_LOCATION,

    "prompt_char_target_local":
        PROMPT_CHAR_TARGET,

    "absence_items":
        sorted(
            ABSENCE_ITEMS,
            key=lambda x:
                int(
                    x[
                        1:
                    ]
                )
        ),

    "fallback_strategy": [
        "FULL rules + docs",
        "reduce docs",
        "MIN rules + docs",
        "tokenizer hard check",
    ],

    "local_prompt_stats": {

        "avg_chars":
            AVG_PROMPT_CHARS,

        "max_chars":
            MAX_PROMPT_CHARS,
    },
}


with open(
    OUT_CONFIG,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ENGINE_CONFIG,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 26. MANIFEST
# ============================================================

manifest = {

    "stage":
        9,

    "status":
        "PASS_ENGINE_READY",

    "records":
        200,

    "items_per_notice":
        24,

    "target_llm_calls_per_notice":
        1,

    "official_schema_validated":
        True,

    "schema_flat_assumption":
        False,

    "output_parser_validated":
        True,

    "local_prompt_char_budget_pass":
        True,

    "server_token_budget_guard":
        True,

    "max_input_tokens":
        MAX_INPUT_TOKENS,

    "max_output_tokens":
        MAX_OUTPUT_TOKENS,

    "model_runtime_available_local":
        MODEL_RUNTIME_AVAILABLE,

    "model_smoke_result":
        MODEL_SMOKE_RESULT,

    "artifacts": {

        "prompts":
            str(
                OUT_PROMPTS
            ),

        "audit":
            str(
                OUT_AUDIT
            ),

        "engine_config":
            str(
                OUT_CONFIG
            ),
    },

    "next_stage":
        (
            "Stage 10 deterministic "
            "postprocessing"
        ),
}


with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 27. RELOAD VALIDATION
# ============================================================

with open(
    OUT_CONFIG,
    "r",
    encoding="utf-8"
) as f:

    saved_config = (
        json.load(
            f
        )
    )


with open(
    OUT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    saved_manifest = (
        json.load(
            f
        )
    )


assert (
    saved_config[
        "max_model_len"
    ]
    ==
    16384
)


assert (
    saved_config[
        "max_input_tokens"
    ]
    ==
    MAX_INPUT_TOKENS
)


assert (
    saved_manifest[
        "status"
    ]
    ==
    "PASS_ENGINE_READY"
)


assert (
    saved_manifest[
        "schema_flat_assumption"
    ]
    is False
)


assert (
    saved_manifest[
        "server_token_budget_guard"
    ]
    is True
)


print("[8] SAVE / RELOAD PASS")


# ============================================================
# 28. DISPLAY
# ============================================================

print()
print(
    "=== Prompt Audit ==="
)


display(

    prompt_audit[
        [
            "system_chars",
            "user_chars",
            "total_chars",
            "document_chars",
            "document_segments",
            "document_budget",
        ]
    ].describe()
)


# ============================================================
# 29. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 9 FINAL RESULT")
print("=" * 100)


print(
    "PROMPTS                    =",
    len(
        PROMPTS
    )
)


print(
    "ITEMS / NOTICE             =",
    24
)


print(
    "TARGET LLM CALLS / NOTICE  =",
    1
)


print(
    "RULE FULL CHARS            =",
    len(
        RULE_SHEET_FULL
    )
)


print(
    "RULE MIN CHARS             =",
    len(
        RULE_SHEET_MIN
    )
)


print(
    "AVG PROMPT CHARS           =",
    round(
        AVG_PROMPT_CHARS,
        2
    )
)


print(
    "MAX PROMPT CHARS           =",
    MAX_PROMPT_CHARS
)


print(
    "LOCAL CHAR LIMIT           =",
    PROMPT_CHAR_TARGET
)


print(
    "SERVER MAX INPUT TOKENS    =",
    MAX_INPUT_TOKENS
)


print(
    "MAX OUTPUT TOKENS          =",
    MAX_OUTPUT_TOKENS
)


print(
    "STRUCTURED OUTPUT          =",
    True
)


print(
    "SCHEMA LOCATION            =",
    SCHEMA_LOCATION
)


print(
    "LOCAL MODEL AVAILABLE      =",
    MODEL_RUNTIME_AVAILABLE
)


print(
    "MODEL SMOKE                =",
    MODEL_SMOKE_RESULT
)


print(
    "PROMPTS                    =",
    OUT_PROMPTS
)


print(
    "ENGINE CONFIG              =",
    OUT_CONFIG
)


print(
    "MANIFEST                   =",
    OUT_MANIFEST
)


print()

print(
    "[PASS] Stage 9 fixed Gemma "
    "single-call judgment engine 준비 완료"
)

print(
    "NEXT = Stage 10 deterministic postprocessing"
)

[1] INPUT LOAD PASS
[2] OFFICIAL JSON SCHEMA PASS
    location = root
    root keys = ['$schema', '$id', 'title', 'description', 'type', 'additionalProperties', 'required', 'properties']
    property paths = 74
[3] BASELINE API AUDIT PASS
{'uses_vllm': True, 'uses_pps_model_dir': True, 'uses_structured_output': True, 'uses_int8_weight_only': True, 'uses_max_model_len': True}
[4] RULE SHEET FULL chars = 6991
[4] RULE SHEET MIN  chars = 3964
[prompt] 50/200
[prompt] 100/200
[prompt] 150/200
[prompt] 200/200
[5] PROMPT BUILD PASS
[6] LOCAL PROMPT SIZE PASS
    avg = 17797.43
    max = 23943
[7] OUTPUT PARSER PASS

VLLM AVAILABLE      = False
PPS_MODEL_DIR       = None
MODEL DIR AVAILABLE = False
REAL MODEL RUNTIME  = False

[MODEL] 로컬에 실제 고정 모델 없음
[MODEL] smoke test 정상 SKIP
[8] SAVE / RELOAD PASS

=== Prompt Audit ===


,system_chars,user_chars,total_chars,document_chars,document_segments,document_budget
count,200.0,200.000000,200.000000,200.000000,200.000000,200.000000
mean,530.0,17267.430000,17797.430000,6324.690000,48.205000,8988.750000
std,0.0,3148.077189,3148.077189,2131.914652,24.932128,91.393112
min,530.0,10564.000000,11094.000000,1467.000000,14.000000,8250.000000
25%,530.0,14951.250000,15481.250000,4639.750000,29.000000,9000.000000
50%,530.0,17483.000000,18013.000000,6645.000000,41.000000,9000.000000
75%,530.0,19530.250000,20060.250000,8233.500000,61.250000,9000.000000
max,530.0,23413.000000,23943.000000,9000.000000,122.000000,9000.000000



Stage 9 FINAL RESULT
PROMPTS                    = 200
ITEMS / NOTICE             = 24
TARGET LLM CALLS / NOTICE  = 1
RULE FULL CHARS            = 6991
RULE MIN CHARS             = 3964
AVG PROMPT CHARS           = 17797.43
MAX PROMPT CHARS           = 23943
LOCAL CHAR LIMIT           = 24000
SERVER MAX INPUT TOKENS    = 13500
MAX OUTPUT TOKENS          = 2200
STRUCTURED OUTPUT          = True
SCHEMA LOCATION            = root
LOCAL MODEL AVAILABLE      = False
MODEL SMOKE                = SKIPPED_LOCAL_NO_MODEL
PROMPTS                    = /home/alparka/nara-ai/artifacts/stage9_prompts_dev.jsonl
ENGINE CONFIG              = /home/alparka/nara-ai/artifacts/stage9_engine_config.json
MANIFEST                   = /home/alparka/nara-ai/artifacts/stage9_manifest.json

[PASS] Stage 9 fixed Gemma single-call judgment engine 준비 완료
NEXT = Stage 10 deterministic postprocessing


In [2]:
# ============================================================
# Stage 10 FINAL
# Safe Deterministic Postprocessing
#
# 목적
# ------------------------------------------------------------
# Gemma raw prediction 이후 적용할 안전한 deterministic layer.
#
# 현재 단계에서는 실제 Gemma prediction이 없으므로:
# - 실제 label을 사용한 튜닝 금지
# - 0 -> 1 positive override 금지
# - 논리적으로 명확한 applicability negative gate만 사용
#
# 핵심 safe gate
# ------------------------------------------------------------
# 경쟁제품 CSV + meta 세부품명번호로:
#
# ALL_GENERAL
#   -> v10, v11, v13 적용 불가 -> 0
#
# ALL_COMPETITIVE
#   -> v12, v14~v18 적용 불가 -> 0
#
# MIXED / UNKNOWN
#   -> 수정하지 않음
#
# 그리고:
# - v=0 evidence=""
# - absence item evidence=""
# - incomplete absence는 warning만 남김
# - v24는 절대 수정하지 않음
# ============================================================

from pathlib import Path
from copy import deepcopy
import json
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

FACTS_PATH = (
    ROOT
    / "artifacts"
    / "structured_facts_dev.jsonl"
)

SPEC_PATH = (
    ROOT
    / "artifacts"
    / "judgment_specs_v1_v24.json"
)

OUT_RULES = (
    ROOT
    / "artifacts"
    / "stage10_postprocess_rules.json"
)

OUT_AUDIT = (
    ROOT
    / "artifacts"
    / "stage10_postprocess_audit.csv"
)

OUT_MANIFEST = (
    ROOT
    / "artifacts"
    / "stage10_manifest.json"
)


for path in [
    FACTS_PATH,
    SPEC_PATH,
]:
    assert path.exists(), path


EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(1, 25)
]

EXPECTED_EVIDENCE = [
    f"e{i}"
    for i in range(1, 25)
]

ABSENCE_ITEMS = {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}


# ============================================================
# 1. LOAD
# ============================================================

def load_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


facts = load_jsonl(
    FACTS_PATH
)


with open(
    SPEC_PATH,
    "r",
    encoding="utf-8"
) as f:

    specs = json.load(f)


assert len(facts) == 200

assert list(
    specs.keys()
) == EXPECTED_ITEMS


assert {
    v
    for v in EXPECTED_ITEMS
    if specs[v][
        "absence_detection"
    ]
} == ABSENCE_ITEMS


print("[1] INPUT LOAD PASS")


# ============================================================
# 2. PRODUCT SCOPE
#
# 세부품명번호가 없으면 UNKNOWN.
#
# 모든 제공 코드가 경쟁제품:
#     ALL_COMPETITIVE
#
# 모든 제공 코드가 비경쟁제품:
#     ALL_GENERAL
#
# 둘 다 포함:
#     MIXED
#
# 절대:
# UNKNOWN -> GENERAL 로 바꾸지 않는다.
# ============================================================

def product_scope(
    structured_facts
):

    product = (
        structured_facts[
            "product"
        ]
    )


    codes = (
        product.get(
            "codes"
        )
        or
        []
    )


    lookup = (
        product.get(
            "competitive_lookup"
        )
        or
        {}
    )


    matched = (
        lookup.get(
            "matched_codes"
        )
        or
        []
    )


    unmatched = (
        lookup.get(
            "unmatched_codes"
        )
        or
        []
    )


    # --------------------------------------------
    # 코드 자체가 없음
    # --------------------------------------------

    if not codes:

        return "UNKNOWN"


    # --------------------------------------------
    # 경쟁 + 일반 혼합
    # --------------------------------------------

    if (
        matched
        and
        unmatched
    ):

        return "MIXED"


    # --------------------------------------------
    # 모든 코드가 경쟁제품
    # --------------------------------------------

    if (
        matched
        and
        not unmatched
    ):

        return "ALL_COMPETITIVE"


    # --------------------------------------------
    # 모든 코드가 일반제품
    # --------------------------------------------

    if (
        unmatched
        and
        not matched
    ):

        return "ALL_GENERAL"


    # 이상 상태는 UNKNOWN으로 보존
    return "UNKNOWN"


# ============================================================
# 3. PREDICTION VALIDATION
# ============================================================

def validate_prediction(
    prediction
):

    assert isinstance(
        prediction,
        dict
    )


    for v in EXPECTED_ITEMS:

        assert v in prediction, (
            f"prediction에 {v} 없음"
        )


        value = prediction[
            v
        ]


        # bool 방지
        if value is True:
            value = 1

        if value is False:
            value = 0


        assert value in {
            0,
            1
        }, (
            f"{v}: invalid value "
            f"{prediction[v]}"
        )


    # evidence는 없는 key를 허용하되
    # postprocess에서 빈칸 생성
    return True


# ============================================================
# 4. EVIDENCE POLICY
# ============================================================

def enforce_evidence_policy(
    prediction
):

    result = deepcopy(
        prediction
    )


    for i in range(
        1,
        25
    ):

        v = f"v{i}"
        e = f"e{i}"


        if e not in result:

            result[e] = ""


        if result[e] is None:

            result[e] = ""


        result[e] = str(
            result[e]
        )


        # ----------------------------------------
        # 비위반이면 evidence 없음
        # ----------------------------------------

        if result[v] == 0:

            result[e] = ""


        # ----------------------------------------
        # 부재탐지는 evidence 항상 없음
        # ----------------------------------------

        if v in ABSENCE_ITEMS:

            result[e] = ""


        # ----------------------------------------
        # 제출 규격 안전장치
        #
        # 앞부분을 자르면 여전히 원문 substring일 수 있지만,
        # Stage 11에서 exact validation을 다시 수행.
        # ----------------------------------------

        if len(
            result[e]
        ) > 500:

            result[e] = (
                result[e][
                    :500
                ]
            )


    return result


# ============================================================
# 5. POSTPROCESS ACTION
# ============================================================

def set_zero(
    result,
    item,
    reason,
    actions
):

    assert item in EXPECTED_ITEMS


    old = int(
        result[
            item
        ]
    )


    # 이미 0이면 action 필요 없음
    if old == 0:
        return


    result[
        item
    ] = 0


    idx = int(
        item[
            1:
        ]
    )


    result[
        f"e{idx}"
    ] = ""


    actions.append({

        "item":
            item,

        "from":
            old,

        "to":
            0,

        "reason":
            reason,
    })


# ============================================================
# 6. SAFE POSTPROCESS
# ============================================================

def postprocess_prediction(
    prediction,
    structured_facts
):

    validate_prediction(
        prediction
    )


    result = enforce_evidence_policy(
        prediction
    )


    before = deepcopy(
        result
    )


    actions = []

    warnings = []


    scope = product_scope(
        structured_facts
    )


    # ========================================================
    # A. ALL GENERAL PRODUCT
    #
    # 경쟁제품 전용 항목은 적용될 수 없음
    # ========================================================

    if scope == "ALL_GENERAL":

        for item in [
            "v10",
            "v11",
            "v13",
        ]:

            set_zero(

                result,

                item,

                (
                    "all provided detail-product codes "
                    "are NOT in the official competitive-product CSV"
                ),

                actions
            )


    # ========================================================
    # B. ALL COMPETITIVE PRODUCT
    #
    # 일반제품 전용 항목은 적용될 수 없음
    # ========================================================

    elif scope == "ALL_COMPETITIVE":

        for item in [
            "v12",
            "v14",
            "v15",
            "v16",
            "v17",
            "v18",
        ]:

            set_zero(

                result,

                item,

                (
                    "all provided detail-product codes "
                    "are in the official competitive-product CSV"
                ),

                actions
            )


    # ========================================================
    # C. MIXED / UNKNOWN
    #
    # 절대 hard correction 하지 않음
    # ========================================================

    elif scope in {
        "MIXED",
        "UNKNOWN",
    }:

        pass


    else:

        raise AssertionError(
            f"unexpected product scope: "
            f"{scope}"
        )


    # ========================================================
    # D. INCOMPLETE ABSENCE WARNING
    #
    # 중요:
    # 강제로 0 만들지 않는다.
    #
    # DEV에서 incomplete인데 실제 positive인
    # absence 사례가 이미 존재했기 때문.
    # ========================================================

    completeness = (
        structured_facts.get(
            "completeness",
            {}
        )
    )


    fully_observed = bool(
        completeness.get(
            "fully_observed",
            False
        )
    )


    has_dropped_docs = bool(
        completeness.get(
            "has_dropped_docs",
            False
        )
    )


    if (
        not fully_observed
        or
        has_dropped_docs
    ):

        for item in sorted(
            ABSENCE_ITEMS,
            key=lambda x:
                int(
                    x[
                        1:
                    ]
                )
        ):

            if result[
                item
            ] == 1:

                warnings.append({

                    "item":
                        item,

                    "type":
                        "INCOMPLETE_ABSENCE_INPUT",

                    "message":
                        (
                            "absence violation=1 but "
                            "input is not fully observed; "
                            "prediction preserved, not overridden"
                        ),
                })


    # ========================================================
    # E. v24 보호
    # ========================================================

    assert (
        result[
            "v24"
        ]
        ==
        before[
            "v24"
        ]
    ), (
        "Stage 10은 v24를 수정하면 안 됨"
    )


    # ========================================================
    # F. evidence policy 다시 적용
    # ========================================================

    result = enforce_evidence_policy(
        result
    )


    # ========================================================
    # G. 0 -> 1 변화 금지
    # ========================================================

    positive_overrides = []


    for item in EXPECTED_ITEMS:

        if (
            before[
                item
            ]
            ==
            0
            and
            result[
                item
            ]
            ==
            1
        ):

            positive_overrides.append(
                item
            )


    assert not positive_overrides, (
        "Stage 10에서 금지된 "
        f"0->1 override 발생: "
        f"{positive_overrides}"
    )


    validate_prediction(
        result
    )


    return {

        "prediction":
            result,

        "actions":
            actions,

        "warnings":
            warnings,

        "product_scope":
            scope,
    }


# ============================================================
# 7. SYNTHETIC TEST PREDICTION
#
# 실제 Gemma가 로컬에서 없으므로,
# 모두 1인 synthetic output으로
# postprocessor가 정확히 무엇을 지우는지 검사.
#
# 정답 label을 사용하지 않는다.
# ============================================================

def synthetic_all_positive():

    pred = {}


    for i in range(
        1,
        25
    ):

        v = f"v{i}"
        e = f"e{i}"


        pred[
            v
        ] = 1


        if v in ABSENCE_ITEMS:

            pred[
                e
            ] = ""

        else:

            pred[
                e
            ] = (
                "SYNTHETIC_EVIDENCE"
            )


    return pred


SYNTHETIC_PRED = (
    synthetic_all_positive()
)


# ============================================================
# 8. RUN STRUCTURAL TEST ON DEV 200
# ============================================================

audit_rows = []

all_actions = []


for f in facts:

    before = deepcopy(
        SYNTHETIC_PRED
    )


    out = (
        postprocess_prediction(
            before,
            f
        )
    )


    after = out[
        "prediction"
    ]


    # --------------------------------------------------------
    # invariants
    # --------------------------------------------------------

    for item in EXPECTED_ITEMS:

        assert after[
            item
        ] in {
            0,
            1
        }


        # 절대 0 -> 1 없음
        assert not (
            before[
                item
            ]
            ==
            0
            and
            after[
                item
            ]
            ==
            1
        )


    # absence evidence
    for item in ABSENCE_ITEMS:

        idx = int(
            item[
                1:
            ]
        )

        assert (
            after[
                f"e{idx}"
            ]
            ==
            ""
        )


    # v24 unchanged
    assert (
        after[
            "v24"
        ]
        ==
        before[
            "v24"
        ]
    )


    action_items = [

        x[
            "item"
        ]

        for x
        in out[
            "actions"
        ]
    ]


    for action in out[
        "actions"
    ]:

        all_actions.append({

            "id":
                f[
                    "id"
                ],

            **action,
        })


    audit_rows.append({

        "id":
            f[
                "id"
            ],

        "product_scope":
            out[
                "product_scope"
            ],

        "fully_observed":
            f[
                "completeness"
            ].get(
                "fully_observed"
            ),

        "has_dropped_docs":
            f[
                "completeness"
            ].get(
                "has_dropped_docs"
            ),

        "actions":
            len(
                out[
                    "actions"
            ]
        ),

        "action_items":
            ",".join(
                action_items
            ),

        "warnings":
            len(
                out[
                    "warnings"
            ]
        ),
    })


audit_df = pd.DataFrame(
    audit_rows
)


actions_df = pd.DataFrame(
    all_actions
)


assert len(
    audit_df
) == 200


print("[2] POSTPROCESS STRUCTURAL TEST PASS")


# ============================================================
# 9. PRODUCT-SCOPE VALIDATION
# ============================================================

scope_counts = (
    audit_df[
        "product_scope"
    ]
    .value_counts()
    .to_dict()
)


assert (
    sum(
        scope_counts.values()
    )
    ==
    200
)


assert set(
    scope_counts
) <= {
    "ALL_COMPETITIVE",
    "ALL_GENERAL",
    "MIXED",
    "UNKNOWN",
}


print(
    "[3] PRODUCT SCOPE PASS"
)

print(
    "    ",
    scope_counts
)


# ============================================================
# 10. ALLOWED ACTION VALIDATION
# ============================================================

ALLOWED_ZERO_OVERRIDE_ITEMS = {

    # competitive-only
    "v10",
    "v11",
    "v13",

    # general-product-only
    "v12",
    "v14",
    "v15",
    "v16",
    "v17",
    "v18",
}


if not actions_df.empty:

    assert set(
        actions_df[
            "item"
        ].unique()
    ) <= (
        ALLOWED_ZERO_OVERRIDE_ITEMS
    )


    assert (
        actions_df[
            "to"
        ]
        ==
        0
    ).all()


    assert (
        actions_df[
            "from"
        ]
        ==
        1
    ).all()


print("[4] SAFE ACTION WHITELIST PASS")


# ============================================================
# 11. EXPLICIT SCOPE TESTS
# ============================================================

def fake_facts(
    codes,
    matched,
    unmatched
):

    return {

        "product": {

            "codes":
                codes,

            "competitive_lookup": {

                "matched_codes":
                    matched,

                "unmatched_codes":
                    unmatched,
            },
        },

        "completeness": {

            "fully_observed":
                True,

            "has_dropped_docs":
                False,
        },
    }


# ------------------------------------------------------------
# all general
# ------------------------------------------------------------

test_general = (
    postprocess_prediction(

        SYNTHETIC_PRED,

        fake_facts(
            ["1111111111"],
            [],
            ["1111111111"]
        )
    )
)


assert (
    test_general[
        "product_scope"
    ]
    ==
    "ALL_GENERAL"
)


for v in [
    "v10",
    "v11",
    "v13",
]:

    assert (
        test_general[
            "prediction"
        ][
            v
        ]
        ==
        0
    )


# 일반제품 항목은 건드리면 안 됨
assert (
    test_general[
        "prediction"
    ][
        "v12"
    ]
    ==
    1
)


# ------------------------------------------------------------
# all competitive
# ------------------------------------------------------------

test_comp = (
    postprocess_prediction(

        SYNTHETIC_PRED,

        fake_facts(
            ["2222222222"],
            ["2222222222"],
            []
        )
    )
)


assert (
    test_comp[
        "product_scope"
    ]
    ==
    "ALL_COMPETITIVE"
)


for v in [
    "v12",
    "v14",
    "v15",
    "v16",
    "v17",
    "v18",
]:

    assert (
        test_comp[
            "prediction"
        ][
            v
        ]
        ==
        0
    )


# 경쟁제품 항목은 건드리면 안 됨
assert (
    test_comp[
        "prediction"
    ][
        "v10"
    ]
    ==
    1
)


# ------------------------------------------------------------
# mixed
# ------------------------------------------------------------

test_mixed = (
    postprocess_prediction(

        SYNTHETIC_PRED,

        fake_facts(
            [
                "1111111111",
                "2222222222",
            ],
            [
                "1111111111"
            ],
            [
                "2222222222"
            ]
        )
    )
)


assert (
    test_mixed[
        "product_scope"
    ]
    ==
    "MIXED"
)


assert (
    test_mixed[
        "actions"
    ]
    ==
    []
)


# ------------------------------------------------------------
# unknown
# ------------------------------------------------------------

test_unknown = (
    postprocess_prediction(

        SYNTHETIC_PRED,

        fake_facts(
            [],
            [],
            []
        )
    )
)


assert (
    test_unknown[
        "product_scope"
    ]
    ==
    "UNKNOWN"
)


assert (
    test_unknown[
        "actions"
    ]
    ==
    []
)


print("[5] EXPLICIT UNIT TESTS PASS")


# ============================================================
# 12. INCOMPLETE ABSENCE TEST
#
# incomplete라고 prediction을 0으로 바꾸면 안 됨.
# ============================================================

incomplete_facts = {

    "product": {

        "codes":
            [],

        "competitive_lookup": {

            "matched_codes":
                [],

            "unmatched_codes":
                [],
        },
    },

    "completeness": {

        "fully_observed":
            False,

        "has_dropped_docs":
            True,
    },
}


incomplete_test = (
    postprocess_prediction(

        SYNTHETIC_PRED,

        incomplete_facts
    )
)


for v in ABSENCE_ITEMS:

    # prediction 보존
    assert (
        incomplete_test[
            "prediction"
        ][
            v
        ]
        ==
        1
    )


    # evidence만 빈칸
    idx = int(
        v[
            1:
        ]
    )

    assert (
        incomplete_test[
            "prediction"
        ][
            f"e{idx}"
        ]
        ==
        ""
    )


assert (
    len(
        incomplete_test[
            "warnings"
        ]
    )
    ==
    len(
        ABSENCE_ITEMS
    )
)


print("[6] INCOMPLETE ABSENCE POLICY PASS")


# ============================================================
# 13. RULE MANIFEST
# ============================================================

POSTPROCESS_RULES = {

    "version":
        "stage10-v1",

    "philosophy":
        (
            "safe negative corrections only; "
            "no deterministic positive overrides before "
            "actual Gemma DEV inference"
        ),

    "positive_override_allowed":
        False,

    "v24_modified":
        False,

    "absence_items":
        sorted(
            ABSENCE_ITEMS,
            key=lambda x:
                int(
                    x[
                        1:
                    ]
                )
        ),

    "product_scope": {

        "ALL_GENERAL": {

            "definition":
                (
                    "all provided detail-product codes "
                    "are absent from the official "
                    "competitive-product CSV"
                ),

            "force_zero": [
                "v10",
                "v11",
                "v13",
            ],
        },

        "ALL_COMPETITIVE": {

            "definition":
                (
                    "all provided detail-product codes "
                    "match the official competitive-product CSV"
                ),

            "force_zero": [
                "v12",
                "v14",
                "v15",
                "v16",
                "v17",
                "v18",
            ],
        },

        "MIXED": {

            "force_zero":
                [],
        },

        "UNKNOWN": {

            "force_zero":
                [],
        },
    },

    "evidence_policy": {

        "v0":
            "empty",

        "absence_items":
            "empty",

        "max_chars":
            500,

        "exact_substring_validation":
            "Stage 11",
    },

    "incomplete_absence_policy": {

        "force_zero":
            False,

        "force_one":
            False,

        "action":
            "warning only",
    },

    "deferred_until_dev_model_predictions": [

        "amount-threshold deterministic corrections",

        "v22 negotiated-contract hard gate",

        "v23 local+negotiated hard gate",

        "SW applicability hard gate",

        "v24 deterministic mismatch override",

        "0->1 positive correction rules",
    ],
}


with open(
    OUT_RULES,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        POSTPROCESS_RULES,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 14. SAVE AUDIT
# ============================================================

audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 15. MANIFEST
# ============================================================

manifest = {

    "stage":
        10,

    "status":
        "PASS_POSTPROCESS_READY",

    "records_tested":
        200,

    "actual_model_predictions_used":
        False,

    "dev_labels_used":
        False,

    "safe_negative_overrides_only":
        True,

    "positive_overrides":
        False,

    "v24_modified":
        False,

    "product_scope_counts":
        scope_counts,

    "synthetic_override_actions":
        int(
            len(
                actions_df
            )
        ),

    "artifacts": {

        "rules":
            str(
                OUT_RULES
            ),

        "audit":
            str(
                OUT_AUDIT
            ),
    },

    "next_stage":
        (
            "Stage 11 exact-substring "
            "evidence validator"
        ),
}


with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 16. RELOAD VALIDATION
# ============================================================

with open(
    OUT_RULES,
    "r",
    encoding="utf-8"
) as f:

    saved_rules = json.load(f)


with open(
    OUT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    saved_manifest = json.load(f)


assert (
    saved_rules[
        "positive_override_allowed"
    ]
    is False
)


assert (
    saved_rules[
        "v24_modified"
    ]
    is False
)


assert (
    saved_manifest[
        "status"
    ]
    ==
    "PASS_POSTPROCESS_READY"
)


assert (
    saved_manifest[
        "dev_labels_used"
    ]
    is False
)


print("[7] SAVE / RELOAD PASS")


# ============================================================
# 17. DISPLAY
# ============================================================

print()
print(
    "=== Product Scope ==="
)

display(

    audit_df[
        "product_scope"
    ]
    .value_counts()
    .rename_axis(
        "product_scope"
    )
    .reset_index(
        name="count"
    )
)


print()
print(
    "=== Synthetic Safe Override Counts ==="
)


if actions_df.empty:

    display(
        pd.DataFrame(
            columns=[
                "item",
                "count",
            ]
        )
    )

else:

    action_summary = (

        actions_df[
            "item"
        ]
        .value_counts()
        .rename_axis(
            "item"
        )
        .reset_index(
            name="count"
        )
        .sort_values(
            "item",
            key=lambda s:
                s.str[
                    1:
                ].astype(
                    int
                )
        )
    )


    display(
        action_summary
    )


# ============================================================
# 18. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 10 FINAL RESULT")
print("=" * 100)


print(
    "RECORDS TESTED             =",
    len(
        facts
    )
)


print(
    "PRODUCT SCOPES             =",
    scope_counts
)


print(
    "SAFE ZERO-OVERRIDE ITEMS   =",
    sorted(
        ALLOWED_ZERO_OVERRIDE_ITEMS,
        key=lambda x:
            int(
                x[
                    1:
                ]
            )
    )
)


print(
    "POSITIVE OVERRIDES         =",
    False
)


print(
    "UNKNOWN HARD-GATED         =",
    False
)


print(
    "MIXED HARD-GATED           =",
    False
)


print(
    "INCOMPLETE ABSENCE ZEROED  =",
    False
)


print(
    "V24 MODIFIED               =",
    False
)


print(
    "DEV LABELS USED            =",
    False
)


print(
    "RULES                      =",
    OUT_RULES
)


print(
    "AUDIT                      =",
    OUT_AUDIT
)


print(
    "MANIFEST                   =",
    OUT_MANIFEST
)


print()

print(
    "[PASS] Stage 10 safe deterministic "
    "postprocessing 준비 완료"
)

print(
    "NEXT = Stage 11 exact-substring evidence validator"
)

[1] INPUT LOAD PASS
[2] POSTPROCESS STRUCTURAL TEST PASS
[3] PRODUCT SCOPE PASS
     {'UNKNOWN': 122, 'ALL_GENERAL': 61, 'ALL_COMPETITIVE': 16, 'MIXED': 1}
[4] SAFE ACTION WHITELIST PASS
[5] EXPLICIT UNIT TESTS PASS
[6] INCOMPLETE ABSENCE POLICY PASS
[7] SAVE / RELOAD PASS

=== Product Scope ===


,product_scope,count
0,UNKNOWN,122
1,ALL_GENERAL,61
2,ALL_COMPETITIVE,16
3,MIXED,1



=== Synthetic Safe Override Counts ===


,item,count
0,v10,61
1,v11,61
3,v12,16
2,v13,61
4,v14,16
5,v15,16
6,v16,16
7,v17,16
8,v18,16



Stage 10 FINAL RESULT
RECORDS TESTED             = 200
PRODUCT SCOPES             = {'UNKNOWN': 122, 'ALL_GENERAL': 61, 'ALL_COMPETITIVE': 16, 'MIXED': 1}
SAFE ZERO-OVERRIDE ITEMS   = ['v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18']
POSITIVE OVERRIDES         = False
UNKNOWN HARD-GATED         = False
MIXED HARD-GATED           = False
INCOMPLETE ABSENCE ZEROED  = False
V24 MODIFIED               = False
DEV LABELS USED            = False
RULES                      = /home/alparka/nara-ai/artifacts/stage10_postprocess_rules.json
AUDIT                      = /home/alparka/nara-ai/artifacts/stage10_postprocess_audit.csv
MANIFEST                   = /home/alparka/nara-ai/artifacts/stage10_manifest.json

[PASS] Stage 10 safe deterministic postprocessing 준비 완료
NEXT = Stage 11 exact-substring evidence validator


In [3]:
# ============================================================
# Stage 11 FINAL
# Exact-Substring Evidence Validator
#
# 목적
# ------------------------------------------------------------
# 최종 e1~e24가 반드시 제공 docs의 실제 부분문자열이 되도록 보장.
#
# 정책
# ------------------------------------------------------------
# 1. v=0 -> e=""
# 2. absence(v10,v11,v16,v18,v20) -> e=""
# 3. 모델 evidence exact -> 그대로 유지
# 4. whitespace 차이 -> 원문 exact substring으로 복원
# 5. invalid/missing evidence -> Stage 7 exact candidate에서 복구 시도
# 6. 복구 실패 -> e=""
# 7. evidence 실패 때문에 v를 1->0으로 바꾸지 않음
#
# dev_labels는 validator 품질 AUDIT에만 사용.
# 실제 inference rule에는 사용하지 않음.
#
# OUTPUT
# ------------------------------------------------------------
# artifacts/stage11_evidence_rules.json
# artifacts/stage11_evidence_audit.csv
# artifacts/stage11_manifest.json
# ============================================================

from pathlib import Path
from copy import deepcopy
import gzip
import json
import re
import pandas as pd


# ============================================================
# 0. PATH
# ============================================================

ROOT = Path.cwd()

DEV_PATH = (
    ROOT
    / "open"
    / "dev.jsonl.gz"
)

LABEL_PATH = (
    ROOT
    / "open"
    / "dev_labels.csv"
)

FACTS_PATH = (
    ROOT
    / "artifacts"
    / "structured_facts_dev.jsonl"
)

OUT_RULES = (
    ROOT
    / "artifacts"
    / "stage11_evidence_rules.json"
)

OUT_AUDIT = (
    ROOT
    / "artifacts"
    / "stage11_evidence_audit.csv"
)

OUT_MANIFEST = (
    ROOT
    / "artifacts"
    / "stage11_manifest.json"
)


for p in [
    DEV_PATH,
    LABEL_PATH,
    FACTS_PATH,
]:
    assert p.exists(), p


EXPECTED_ITEMS = [
    f"v{i}"
    for i in range(1, 25)
]

ABSENCE_ITEMS = {
    "v10",
    "v11",
    "v16",
    "v18",
    "v20",
}

MAX_EVIDENCE_CHARS = 500


# ============================================================
# 1. LOAD
# ============================================================

def load_jsonl(path):

    rows = []

    opener = (
        gzip.open
        if str(path).endswith(".gz")
        else open
    )

    with opener(
        path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


dev = load_jsonl(
    DEV_PATH
)

facts = load_jsonl(
    FACTS_PATH
)

labels = pd.read_csv(
    LABEL_PATH
)


assert len(dev) == 200
assert len(facts) == 200
assert len(labels) == 200


dev_ids = [
    x["id"]
    for x in dev
]

fact_ids = [
    x["id"]
    for x in facts
]

label_ids = (
    labels["id"]
    .astype(str)
    .tolist()
)


assert dev_ids == fact_ids
assert dev_ids == label_ids


DEV_BY_ID = {
    x["id"]: x
    for x in dev
}

FACTS_BY_ID = {
    x["id"]: x
    for x in facts
}


print("[1] INPUT LOAD PASS")


# ============================================================
# 2. ITEM → CANDIDATE CATEGORIES
#
# Stage 7 / Stage 8와 동일한 의미.
# evidence recovery용이며 판정 gate가 아님.
# ============================================================

ITEM_CATEGORIES = {

    "v1": [
        "qualification",
    ],

    "v2": [
        "performance",
        "qualification",
    ],

    "v3": [
        "performance",
        "amount",
    ],

    "v4": [
        "performance",
        "qualification",
    ],

    "v5": [
        "region",
        "qualification",
    ],

    "v6": [
        "region",
        "qualification",
    ],

    "v7": [
        "region",
        "qualification",
    ],

    "v8": [
        "performance",
        "region",
        "qualification",
    ],

    "v9": [
        "product_model",
    ],

    "v10": [],

    "v11": [],

    "v12": [
        "direct_production",
    ],

    "v13": [
        "enterprise",
    ],

    "v14": [
        "enterprise",
        "amount",
    ],

    "v15": [
        "enterprise",
        "amount",
    ],

    "v16": [],

    "v17": [
        "enterprise",
        "amount",
    ],

    "v18": [],

    "v19": [
        "supply_commitment",
        "qualification",
    ],

    "v20": [],

    "v21": [
        "joint_contract",
    ],

    "v22": [
        "briefing",
        "qualification",
    ],

    "v23": [
        "briefing",
        "deadline",
    ],

    "v24": [
        "amount",
        "contract_method",
        "region",
        "industry",
    ],
}


assert set(
    ITEM_CATEGORIES
) == set(
    EXPECTED_ITEMS
)


# ============================================================
# 3. ITEM-SPECIFIC EVIDENCE PATTERNS
#
# 위반 판정 규칙이 아님.
# 이미 v=1로 판단된 항목에서
# "가장 관련 있는 원문"을 찾기 위한 retrieval pattern.
# ============================================================

RAW_PATTERNS = {

    "v1": [
        r"기관",
        r"발주",
        r"공공기관",
        r"국가기관",
        r"지방자치단체",
        r"참가\s*자격",
    ],

    "v2": [
        r"실적",
        r"이행\s*실적",
        r"납품\s*실적",
        r"수행\s*실적",
    ],

    "v3": [
        r"실적",
        r"이상",
        r"배정\s*예산",
        r"예산",
    ],

    "v4": [
        r"실적",
        r"기관",
        r"발주",
        r"특정",
    ],

    "v5": [
        r"지역\s*제한",
        r"소재지",
        r"본점",
        r"주된\s*영업소",
    ],

    "v6": [
        r"소재지",
        r"본점",
        r"주된\s*영업소",
        r"시",
        r"군",
        r"구",
    ],

    "v7": [
        r"인접",
        r"소재지",
        r"본점",
        r"지역",
    ],

    "v8": [
        r"실적",
        r"지역",
        r"소재지",
        r"본점",
    ],

    "v9": [
        r"모델\s*명",
        r"모델명",
        r"모델\s*번호",
        r"제품\s*번호",
        r"품번",
        r"제조\s*사",
        r"브랜드",
        r"상표",
    ],

    "v12": [
        r"직접\s*생산",
        r"직접생산",
        r"직접\s*생산\s*확인",
        r"직접생산확인",
    ],

    "v13": [
        r"소기업",
        r"소상공인",
        r"중소기업",
    ],

    "v14": [
        r"중소기업",
        r"중소\s*기업",
        r"추정\s*가격",
        r"사업\s*금액",
    ],

    "v15": [
        r"소기업",
        r"소상공인",
        r"중소기업",
    ],

    "v17": [
        r"중소기업",
        r"중소\s*기업",
        r"소기업",
        r"소상공인",
    ],

    "v19": [
        r"물품\s*공급",
        r"기술\s*지원",
        r"확약서",
        r"협약서",
        r"공급\s*확인서",
    ],

    "v21": [
        r"공동\s*수급",
        r"공동\s*도급",
        r"출자\s*비율",
        r"분담\s*비율",
        r"지분율",
        r"\b5\s*%",
        r"\b10\s*%",
        r"100\s*분의\s*5",
        r"100\s*분의\s*10",
    ],

    "v22": [
        r"현장\s*설명",
        r"설명회",
        r"참석",
        r"불참",
        r"입찰\s*참가",
    ],

    "v23": [
        r"현장\s*설명",
        r"설명회",
        r"공고\s*기간",
        r"제안서\s*제출",
        r"마감",
        r"\d+\s*일",
    ],

    "v24": [
        r"추정\s*가격",
        r"사업\s*예산",
        r"계약\s*방법",
        r"지역\s*제한",
        r"소재지",
        r"업종",
        r"면허",
    ],
}


EVIDENCE_PATTERNS = {

    v: [
        re.compile(
            pattern,
            re.I
        )
        for pattern
        in RAW_PATTERNS.get(
            v,
            []
        )
    ]

    for v
    in EXPECTED_ITEMS
}


# ============================================================
# 4. DOC HELPERS
# ============================================================

def make_doc_map(
    row
):

    return {

        doc[
            "doc_id"
        ]:
            doc.get(
                "text",
                ""
            )

        for doc
        in row[
            "docs"
        ]
    }


def find_exact_evidence(
    evidence,
    row
):

    if evidence is None:
        return None


    evidence = str(
        evidence
    )


    if evidence == "":
        return None


    for doc in row[
        "docs"
    ]:

        text = doc.get(
            "text",
            ""
        )


        pos = text.find(
            evidence
        )


        if pos >= 0:

            return {

                "doc_id":
                    doc[
                        "doc_id"
                    ],

                "doc_type":
                    doc[
                        "type"
                    ],

                "start":
                    pos,

                "end":
                    pos
                    +
                    len(
                        evidence
                    ),

                "text":
                    evidence,
            }


    return None


# ============================================================
# 5. EVIDENCE CLEAN CANDIDATES
#
# 모델이:
#   "원문"
#   '원문'
#   ```원문```
#
# 처럼 wrapper를 붙인 경우 복원.
# ============================================================

def evidence_variants(
    evidence
):

    if evidence is None:
        return []


    raw = str(
        evidence
    )


    variants = []


    def add(x):

        if x is None:
            return

        x = str(
            x
        )

        if (
            x
            and
            x not in variants
        ):

            variants.append(
                x
            )


    add(
        raw
    )


    stripped = raw.strip()

    add(
        stripped
    )


    # markdown code wrapper
    if (
        stripped.startswith(
            "```"
        )
        and
        stripped.endswith(
            "```"
        )
    ):

        inner = (
            stripped[
                3:
                -3
            ]
            .strip()
        )

        add(
            inner
        )


    # paired quote wrapper만 제거
    wrappers = [
        ('"', '"'),
        ("'", "'"),
        ("“", "”"),
        ("‘", "’"),
        ("「", "」"),
        ("『", "』"),
    ]


    for left, right in wrappers:

        if (
            len(
                stripped
            )
            >= 2
            and
            stripped.startswith(
                left
            )
            and
            stripped.endswith(
                right
            )
        ):

            add(
                stripped[
                    len(
                        left
                    ):
                    len(
                        stripped
                    )
                    -
                    len(
                        right
                    )
                ]
            )


    return variants


# ============================================================
# 6. WHITESPACE-NORMALIZED EXACT RECOVERY
#
# 모델이:
#   "입찰 참가 자격"
#
# 원문:
#   "입찰참가자격"
#
# 처럼 공백만 다르게 출력한 경우,
# 원문 offset을 추적해 실제 substring으로 복원.
# ============================================================

def compact_nonspace_with_map(
    text
):

    chars = []

    positions = []


    for idx, ch in enumerate(
        text
    ):

        if ch.isspace():
            continue


        chars.append(
            ch
        )

        positions.append(
            idx
        )


    return (
        "".join(
            chars
        ),
        positions,
    )


def recover_whitespace_normalized(
    evidence,
    row
):

    variants = (
        evidence_variants(
            evidence
        )
    )


    for variant in variants:

        compact_ev = "".join(
            ch
            for ch in variant
            if not ch.isspace()
        )


        # 너무 짧으면 오탐 위험
        if len(
            compact_ev
        ) < 6:

            continue


        for doc in row[
            "docs"
        ]:

            text = (
                doc.get(
                    "text",
                    ""
                )
            )


            compact_doc, pos_map = (
                compact_nonspace_with_map(
                    text
                )
            )


            loc = (
                compact_doc.find(
                    compact_ev
                )
            )


            if loc < 0:
                continue


            compact_end = (
                loc
                +
                len(
                    compact_ev
                )
                -
                1
            )


            if (
                compact_end
                >=
                len(
                    pos_map
                )
            ):
                continue


            original_start = (
                pos_map[
                    loc
                ]
            )


            original_end = (
                pos_map[
                    compact_end
                ]
                +
                1
            )


            recovered = (
                text[
                    original_start:
                    original_end
                ]
            )


            # 최종 exact 확인
            assert (
                recovered
                in text
            )


            return {

                "doc_id":
                    doc[
                        "doc_id"
                    ],

                "doc_type":
                    doc[
                        "type"
                    ],

                "start":
                    original_start,

                "end":
                    original_end,

                "text":
                    recovered,
            }


    return None


# ============================================================
# 7. STAGE 7 EXACT CANDIDATES
# ============================================================

def candidate_blocks_for_item(
    facts_row,
    item
):

    result = []

    seen = set()


    for category in (
        ITEM_CATEGORIES[
            item
        ]
    ):

        blocks = (
            facts_row[
                "candidates"
            ].get(
                category,
                []
            )
        )


        for block in blocks:

            key = (
                block[
                    "doc_id"
                ],
                int(
                    block[
                        "start"
                ]),
                int(
                    block[
                        "end"
                    ]),
            )


            if key in seen:
                continue


            seen.add(
                key
            )

            result.append(
                block
            )


    return result


# ============================================================
# 8. BLOCK SCORE
# ============================================================

def block_score(
    block,
    item
):

    text = block[
        "text"
    ]


    patterns = (
        EVIDENCE_PATTERNS[
            item
        ]
    )


    score = 0


    for pattern in patterns:

        if pattern.search(
            text
        ):

            score += 3


    # Stage 7 matched terms도 작은 가산점
    score += min(
        len(
            block.get(
                "matched_terms",
                []
            )
        ),
        4
    )


    # 너무 긴 block보다 짧고 직접적인 block 선호
    if len(
        text
    ) <= 500:

        score += 2


    # 특수항목 보강
    if item == "v8":

        has_perf = bool(
            re.search(
                r"실적",
                text
            )
        )

        has_region = bool(
            re.search(
                r"지역|소재지|본점|영업소",
                text
            )
        )

        if (
            has_perf
            and
            has_region
        ):

            score += 8


    if item == "v4":

        has_perf = bool(
            re.search(
                r"실적",
                text
            )
        )

        has_inst = bool(
            re.search(
                r"기관|발주",
                text
            )
        )

        if (
            has_perf
            and
            has_inst
        ):

            score += 8


    if item == "v22":

        has_briefing = bool(
            re.search(
                r"설명",
                text
            )
        )

        has_attendance = bool(
            re.search(
                r"참석|불참|참가",
                text
            )
        )

        if (
            has_briefing
            and
            has_attendance
        ):

            score += 8


    if item == "v23":

        has_briefing = bool(
            re.search(
                r"설명",
                text
            )
        )

        has_period = bool(
            re.search(
                r"공고|기간|마감|제출|\d+\s*일",
                text
            )
        )

        if (
            has_briefing
            and
            has_period
        ):

            score += 8


    return score


# ============================================================
# 9. EXACT SPAN FROM BLOCK
#
# block 자체는 이미 exact substring.
#
# 그 안에서 관련 line을 골라 더 짧은
# exact substring evidence로 사용.
# ============================================================

def best_exact_span_from_block(
    block,
    item
):

    text = block[
        "text"
    ]


    patterns = (
        EVIDENCE_PATTERNS[
            item
        ]
    )


    if not patterns:
        return None


    # --------------------------------------------
    # line별 점수
    # --------------------------------------------

    lines = text.splitlines()


    candidates = []


    offset = 0


    for line in lines:

        # splitlines()가 newline을 제거했으므로
        # block 내부 위치를 find로 안전하게 계산
        pos = text.find(
            line,
            offset
        )


        if pos < 0:

            pos = offset


        offset = (
            pos
            +
            len(
                line
            )
        )


        stripped = (
            line.strip()
        )


        if not stripped:

            continue


        score = 0

        first_match = None


        for pattern in patterns:

            m = pattern.search(
                stripped
            )


            if m:

                score += 3


                if first_match is None:

                    first_match = m


        if score > 0:

            candidates.append({

                "text":
                    stripped,

                "score":
                    score,

                "first_match":
                    (
                        first_match.start()
                        if first_match
                        else 0
                    ),
            })


    # --------------------------------------------
    # line candidate 존재
    # --------------------------------------------

    if candidates:

        candidates.sort(
            key=lambda x: (
                -x[
                    "score"
                ],
                len(
                    x[
                        "text"
                    ]
                ),
            )
        )


        chosen = (
            candidates[
                0
            ][
                "text"
            ]
        )


        # line이 500자 이하면 그대로
        if len(
            chosen
        ) <= MAX_EVIDENCE_CHARS:

            return chosen


        # 너무 길면 첫 matching 주변 exact window
        first_match = None


        for pattern in patterns:

            m = pattern.search(
                chosen
            )

            if m:

                first_match = m

                break


        if first_match:

            center = (
                first_match.start()
            )


            start = max(
                0,
                center - 150
            )

            end = min(
                len(
                    chosen
                ),
                start
                +
                MAX_EVIDENCE_CHARS
            )


            return chosen[
                start:end
            ]


    # --------------------------------------------
    # block 전체에서 pattern은 있는데
    # line scoring이 실패한 경우
    # --------------------------------------------

    for pattern in patterns:

        m = pattern.search(
            text
        )


        if not m:
            continue


        center = (
            m.start()
        )


        start = max(
            0,
            center - 150
        )


        end = min(
            len(
                text
            ),
            start
            +
            MAX_EVIDENCE_CHARS
        )


        candidate = (
            text[
                start:end
            ]
            .strip()
        )


        if candidate:

            return candidate


    return None


# ============================================================
# 10. RECOVER FROM STAGE 7 CANDIDATES
# ============================================================

def recover_from_candidates(
    item,
    facts_row,
    raw_row
):

    blocks = (
        candidate_blocks_for_item(
            facts_row,
            item
        )
    )


    if not blocks:

        return None


    scored = [

        (
            block_score(
                block,
                item
            ),
            block,
        )

        for block
        in blocks
    ]


    scored.sort(
        key=lambda x: (
            -x[0],
            len(
                x[1][
                    "text"
                ]
            ),
        )
    )


    for score, block in scored:

        # item-specific signal이 전혀 없으면
        # 임의 근거를 만들지 않음
        if score <= 0:

            continue


        candidate = (
            best_exact_span_from_block(
                block,
                item
            )
        )


        if not candidate:

            continue


        exact = (
            find_exact_evidence(
                candidate,
                raw_row
            )
        )


        if exact:

            return exact


    return None


# ============================================================
# 11. SINGLE EVIDENCE VALIDATOR
# ============================================================

def validate_single_evidence(
    item,
    violation,
    evidence,
    raw_row,
    facts_row,
):

    assert item in EXPECTED_ITEMS

    violation = int(
        violation
    )


    assert violation in {
        0,
        1
    }


    # --------------------------------------------------------
    # v=0
    # --------------------------------------------------------

    if violation == 0:

        return {

            "evidence":
                "",

            "status":
                "EMPTY_NONVIOLATION",

            "source":
                None,
        }


    # --------------------------------------------------------
    # absence item
    # --------------------------------------------------------

    if item in ABSENCE_ITEMS:

        return {

            "evidence":
                "",

            "status":
                "EMPTY_ABSENCE",

            "source":
                None,
        }


    # --------------------------------------------------------
    # 1) exact model evidence
    # --------------------------------------------------------

    for variant in (
        evidence_variants(
            evidence
        )
    ):

        exact = (
            find_exact_evidence(
                variant,
                raw_row
            )
        )


        if exact:

            final_text = exact[
                "text"
            ]


            if len(
                final_text
            ) > MAX_EVIDENCE_CHARS:

                final_text = (
                    final_text[
                        :MAX_EVIDENCE_CHARS
                    ]
                )


                exact2 = (
                    find_exact_evidence(
                        final_text,
                        raw_row
                    )
                )


                assert exact2


            return {

                "evidence":
                    final_text,

                "status":
                    "MODEL_EXACT",

                "source":
                    exact,
            }


    # --------------------------------------------------------
    # 2) whitespace-normalized recovery
    # --------------------------------------------------------

    recovered = (
        recover_whitespace_normalized(
            evidence,
            raw_row
        )
    )


    if recovered:

        final_text = (
            recovered[
                "text"
            ]
        )


        if len(
            final_text
        ) > MAX_EVIDENCE_CHARS:

            final_text = (
                final_text[
                    :MAX_EVIDENCE_CHARS
                ]
            )


        exact = (
            find_exact_evidence(
                final_text,
                raw_row
            )
        )


        assert exact


        return {

            "evidence":
                final_text,

            "status":
                "MODEL_WHITESPACE_RECOVERED",

            "source":
                exact,
        }


    # --------------------------------------------------------
    # 3) candidate fallback
    # --------------------------------------------------------

    recovered = (
        recover_from_candidates(
            item,
            facts_row,
            raw_row
        )
    )


    if recovered:

        final_text = (
            recovered[
                "text"
            ]
        )


        assert len(
            final_text
        ) <= MAX_EVIDENCE_CHARS


        exact = (
            find_exact_evidence(
                final_text,
                raw_row
            )
        )


        assert exact


        return {

            "evidence":
                final_text,

            "status":
                "CANDIDATE_RECOVERED",

            "source":
                exact,
        }


    # --------------------------------------------------------
    # 4) 실패
    #
    # v는 절대 수정하지 않는다.
    # --------------------------------------------------------

    return {

        "evidence":
            "",

        "status":
            "MISSING_UNRECOVERED",

        "source":
            None,
    }


# ============================================================
# 12. FULL PREDICTION VALIDATOR
# ============================================================

def validate_prediction_evidence(
    prediction,
    raw_row,
    facts_row
):

    result = deepcopy(
        prediction
    )


    audit = []


    for i in range(
        1,
        25
    ):

        v = f"v{i}"
        e = f"e{i}"


        assert v in result


        violation = int(
            result[
                v
            ]
        )


        assert violation in {
            0,
            1
        }


        evidence = (
            result.get(
                e,
                ""
            )
        )


        checked = (
            validate_single_evidence(

                item=v,

                violation=violation,

                evidence=evidence,

                raw_row=raw_row,

                facts_row=facts_row,
            )
        )


        # 중요:
        # violation 값은 변경하지 않음
        assert (
            result[
                v
            ]
            ==
            violation
        )


        result[
            e
        ] = (
            checked[
                "evidence"
            ]
        )


        audit.append({

            "item":
                v,

            "violation":
                violation,

            "status":
                checked[
                    "status"
                ],

            "evidence_chars":
                len(
                    checked[
                        "evidence"
                    ]
                ),

            "doc_id":
                (
                    checked[
                        "source"
                    ].get(
                        "doc_id"
                    )
                    if checked[
                        "source"
                    ]
                    else None
                ),
        })


    return {

        "prediction":
            result,

        "audit":
            audit,
    }


# ============================================================
# 13. GLOBAL EXACTNESS ASSERT
# ============================================================

def assert_final_evidence_exact(
    prediction,
    raw_row
):

    for i in range(
        1,
        25
    ):

        v = f"v{i}"
        e = f"e{i}"


        evidence = str(
            prediction.get(
                e,
                ""
            )
            or
            ""
        )


        # v=0
        if int(
            prediction[
                v
            ]
        ) == 0:

            assert evidence == ""

            continue


        # absence
        if v in ABSENCE_ITEMS:

            assert evidence == ""

            continue


        # empty는 허용
        if evidence == "":

            continue


        assert (
            find_exact_evidence(
                evidence,
                raw_row
            )
            is not None
        ), (
            f"{raw_row['id']} "
            f"{v}: final evidence not exact"
        )


        assert (
            len(
                evidence
            )
            <=
            MAX_EVIDENCE_CHARS
        )


    return True


# ============================================================
# 14. DEV OFFICIAL LABEL AUDIT
#
# labels는 validator를 평가하는 용도일 뿐.
# inference rule 생성에는 사용하지 않는다.
# ============================================================

audit_records = []


for _, label_row in (
    labels.iterrows()
):

    row_id = str(
        label_row[
            "id"
        ]
    )


    raw_row = (
        DEV_BY_ID[
            row_id
        ]
    )


    facts_row = (
        FACTS_BY_ID[
            row_id
        ]
    )


    prediction = {}


    for i in range(
        1,
        25
    ):

        v = f"v{i}"
        e = f"e{i}"


        prediction[
            v
        ] = int(
            label_row[
                v
            ]
        )


        raw_e = (
            label_row[
                e
            ]
        )


        if pd.isna(
            raw_e
        ):

            raw_e = ""


        prediction[
            e
        ] = str(
            raw_e
        )


    checked = (
        validate_prediction_evidence(

            prediction,

            raw_row,

            facts_row
        )
    )


    final_prediction = (
        checked[
            "prediction"
        ]
    )


    assert_final_evidence_exact(
        final_prediction,
        raw_row
    )


    for item_audit in (
        checked[
            "audit"
        ]
    ):

        v = item_audit[
            "item"
        ]

        i = int(
            v[
                1:
            ]
        )

        original_e = (
            prediction[
                f"e{i}"
            ]
        )


        audit_records.append({

            "id":
                row_id,

            "item":
                v,

            "violation":
                prediction[
                    v
                ],

            "original_evidence_present":
                bool(
                    original_e
                ),

            "final_evidence_present":
                bool(
                    final_prediction[
                        f"e{i}"
                    ]
                ),

            "status":
                item_audit[
                    "status"
                ],

            "final_chars":
                item_audit[
                    "evidence_chars"
                ],
        })


print(
    "[2] DEV OFFICIAL EVIDENCE VALIDATION PASS"
)


# ============================================================
# 15. AUDIT SUMMARY PER ITEM
# ============================================================

audit_raw_df = pd.DataFrame(
    audit_records
)


summary_rows = []


for v in EXPECTED_ITEMS:

    sub = audit_raw_df[
        audit_raw_df[
            "item"
        ]
        ==
        v
    ]


    positive = sub[
        sub[
            "violation"
        ]
        ==
        1
    ]


    status_counts = (
        positive[
            "status"
        ]
        .value_counts()
        .to_dict()
    )


    summary_rows.append({

        "item":
            v,

        "positive":
            len(
                positive
            ),

        "official_evidence_present":
            int(
                positive[
                    "original_evidence_present"
                ].sum()
            ),

        "final_evidence_present":
            int(
                positive[
                    "final_evidence_present"
                ].sum()
            ),

        "model_exact":
            int(
                status_counts.get(
                    "MODEL_EXACT",
                    0
                )
            ),

        "whitespace_recovered":
            int(
                status_counts.get(
                    "MODEL_WHITESPACE_RECOVERED",
                    0
                )
            ),

        "candidate_recovered":
            int(
                status_counts.get(
                    "CANDIDATE_RECOVERED",
                    0
                )
            ),

        "missing_unrecovered":
            int(
                status_counts.get(
                    "MISSING_UNRECOVERED",
                    0
                )
            ),

        "absence":
            (
                v
                in ABSENCE_ITEMS
            ),
    })


audit_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# 16. STRONG INVARIANTS
# ============================================================

# ------------------------------------------------------------
# absence positive는 final evidence가 0개여야 함
# ------------------------------------------------------------

for v in ABSENCE_ITEMS:

    row = audit_df[
        audit_df[
            "item"
        ]
        ==
        v
    ].iloc[0]


    assert (
        int(
            row[
                "final_evidence_present"
            ]
        )
        ==
        0
    )


# ------------------------------------------------------------
# official non-empty evidence는 최종적으로 잃어버리면 안 됨
# ------------------------------------------------------------

for _, row in (
    audit_df.iterrows()
):

    if row[
        "absence"
    ]:

        continue


    assert (
        int(
            row[
                "final_evidence_present"
            ]
        )
        >=
        int(
            row[
                "official_evidence_present"
            ]
        )
    ), (
        f"{row['item']}: "
        "기존 공식 evidence 손실"
    )


# ------------------------------------------------------------
# validator는 violation을 변경하지 않음
# ------------------------------------------------------------

assert (
    audit_raw_df[
        "violation"
    ]
    .isin(
        [
            0,
            1,
        ]
    )
    .all()
)


print("[3] EVIDENCE INVARIANTS PASS")


# ============================================================
# 17. UNIT TEST: INVALID MODEL EVIDENCE
# ============================================================

# 실제 positive 중 evidence가 있는 첫 사례를 하나 골라
# 공백을 망가뜨려도 원문으로 복구되는지 확인.

whitespace_test_pass = False


for _, label_row in (
    labels.iterrows()
):

    row_id = str(
        label_row[
            "id"
        ]
    )


    raw_row = DEV_BY_ID[
        row_id
    ]


    facts_row = FACTS_BY_ID[
        row_id
    ]


    for i in range(
        1,
        25
    ):

        v = f"v{i}"
        e = f"e{i}"


        if v in ABSENCE_ITEMS:
            continue


        if int(
            label_row[
                v
            ]
        ) != 1:
            continue


        original = (
            label_row[
                e
            ]
        )


        if pd.isna(
            original
        ):

            continue


        original = str(
            original
        )


        if len(
            original
        ) < 8:

            continue


        # 원문 내부 whitespace를 제거한 형태
        broken = re.sub(
            r"\s+",
            "",
            original
        )


        result = (
            validate_single_evidence(

                item=v,

                violation=1,

                evidence=broken,

                raw_row=raw_row,

                facts_row=facts_row,
            )
        )


        if result[
            "evidence"
        ]:

            assert (
                find_exact_evidence(
                    result[
                        "evidence"
                    ],
                    raw_row
                )
                is not None
            )


            whitespace_test_pass = True

            break


    if whitespace_test_pass:
        break


assert whitespace_test_pass


print(
    "[4] WHITESPACE RECOVERY UNIT TEST PASS"
)


# ============================================================
# 18. UNIT TEST: COMPLETELY INVALID EVIDENCE
# ============================================================

test_row = dev[
    0
]

test_facts = facts[
    0
]


# item은 non-absence 하나 사용
invalid_result = (
    validate_single_evidence(

        item="v2",

        violation=1,

        evidence=(
            "THIS STRING DEFINITELY "
            "DOES NOT EXIST IN SOURCE"
        ),

        raw_row=test_row,

        facts_row=test_facts,
    )
)


# candidate가 있으면 candidate recovery,
# 없으면 empty.
# 어떤 경우든 final evidence는 exact여야 함.
if invalid_result[
    "evidence"
]:

    assert (
        find_exact_evidence(

            invalid_result[
                "evidence"
            ],

            test_row
        )
        is not None
    )


print(
    "[5] INVALID EVIDENCE SAFETY PASS"
)


# ============================================================
# 19. SAVE RULES
# ============================================================

RULES = {

    "version":
        "stage11-v1",

    "max_evidence_chars":
        MAX_EVIDENCE_CHARS,

    "absence_items":
        sorted(
            ABSENCE_ITEMS,
            key=lambda x:
                int(
                    x[
                        1:
                    ]
                )
        ),

    "order": [

        "v=0 -> empty",

        "absence item -> empty",

        "exact model evidence",

        "wrapper-stripped exact evidence",

        "whitespace-normalized recovery",

        "Stage 7 exact candidate recovery",

        "unrecoverable -> empty",
    ],

    "prediction_change":
        False,

    "important_policy": {

        "evidence_failure_does_not_flip_v":
            True,

        "generated_or_summarized_evidence":
            False,

        "all_nonempty_final_evidence_exact":
            True,
    },
}


with open(
    OUT_RULES,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        RULES,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 20. SAVE AUDIT
# ============================================================

audit_df.to_csv(
    OUT_AUDIT,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 21. MANIFEST
# ============================================================

total_positive = int(
    audit_df[
        "positive"
    ].sum()
)


total_official_e = int(
    audit_df[
        "official_evidence_present"
    ].sum()
)


total_final_e = int(
    audit_df[
        "final_evidence_present"
    ].sum()
)


total_candidate_recovered = int(
    audit_df[
        "candidate_recovered"
    ].sum()
)


total_missing = int(
    audit_df[
        "missing_unrecovered"
    ].sum()
)


manifest = {

    "stage":
        11,

    "status":
        "PASS_EVIDENCE_VALIDATOR_READY",

    "records_tested":
        200,

    "positive_labels_tested":
        total_positive,

    "official_evidence_present":
        total_official_e,

    "final_evidence_present":
        total_final_e,

    "candidate_recovered":
        total_candidate_recovered,

    "missing_unrecovered":
        total_missing,

    "prediction_values_changed":
        False,

    "exact_substring_required":
        True,

    "dev_labels_use":
        (
            "AUDIT_ONLY; never used by runtime "
            "evidence recovery rules"
        ),

    "artifacts": {

        "rules":
            str(
                OUT_RULES
            ),

        "audit":
            str(
                OUT_AUDIT
            ),
    },

    "next_stage":
        "Stage 12 first submission-ready script.py",
}


with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 22. RELOAD
# ============================================================

with open(
    OUT_RULES,
    "r",
    encoding="utf-8"
) as f:

    saved_rules = json.load(f)


with open(
    OUT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    saved_manifest = json.load(f)


assert (
    saved_rules[
        "prediction_change"
    ]
    is False
)


assert (
    saved_rules[
        "important_policy"
    ][
        "all_nonempty_final_evidence_exact"
    ]
    is True
)


assert (
    saved_manifest[
        "status"
    ]
    ==
    "PASS_EVIDENCE_VALIDATOR_READY"
)


print("[6] SAVE / RELOAD PASS")


# ============================================================
# 23. DISPLAY
# ============================================================

display(
    audit_df
)


# ============================================================
# 24. FINAL RESULT
# ============================================================

print()
print("=" * 100)
print("Stage 11 FINAL RESULT")
print("=" * 100)


print(
    "RECORDS TESTED              =",
    200
)


print(
    "POSITIVE LABELS             =",
    total_positive
)


print(
    "OFFICIAL EVIDENCE PRESENT   =",
    total_official_e
)


print(
    "FINAL EVIDENCE PRESENT      =",
    total_final_e
)


print(
    "CANDIDATE RECOVERED         =",
    total_candidate_recovered
)


print(
    "MISSING UNRECOVERED         =",
    total_missing
)


print(
    "ABSENCE EVIDENCE            =",
    "ALWAYS EMPTY"
)


print(
    "PREDICTION VALUES CHANGED   =",
    False
)


print(
    "FINAL NONEMPTY EVIDENCE     =",
    "EXACT SUBSTRING ONLY"
)


print(
    "RULES                       =",
    OUT_RULES
)


print(
    "AUDIT                       =",
    OUT_AUDIT
)


print(
    "MANIFEST                    =",
    OUT_MANIFEST
)


print()
print(
    "[PASS] Stage 11 exact-substring "
    "evidence validator 준비 완료"
)

print(
    "NEXT = Stage 12 first submission-ready script.py"
)

[1] INPUT LOAD PASS
[2] DEV OFFICIAL EVIDENCE VALIDATION PASS
[3] EVIDENCE INVARIANTS PASS
[4] WHITESPACE RECOVERY UNIT TEST PASS
[5] INVALID EVIDENCE SAFETY PASS
[6] SAVE / RELOAD PASS


,item,positive,official_evidence_present,final_evidence_present,model_exact,whitespace_recovered,candidate_recovered,missing_unrecovered,absence
0,v1,7,1,7,1,0,6,0,False
1,v2,7,7,7,7,0,0,0,False
2,v3,8,8,8,8,0,0,0,False
3,v4,6,5,6,5,0,1,0,False
4,v5,7,1,6,1,0,5,1,False
5,v6,6,1,6,1,0,5,0,False
6,v7,7,7,7,7,0,0,0,False
7,v8,6,2,6,2,0,4,0,False
8,v9,6,3,4,3,0,1,2,False
9,v10,7,0,0,0,0,0,0,True



Stage 11 FINAL RESULT
RECORDS TESTED              = 200
POSITIVE LABELS             = 153
OFFICIAL EVIDENCE PRESENT   = 54
FINAL EVIDENCE PRESENT      = 119
CANDIDATE RECOVERED         = 65
MISSING UNRECOVERED         = 3
ABSENCE EVIDENCE            = ALWAYS EMPTY
PREDICTION VALUES CHANGED   = False
FINAL NONEMPTY EVIDENCE     = EXACT SUBSTRING ONLY
RULES                       = /home/alparka/nara-ai/artifacts/stage11_evidence_rules.json
AUDIT                       = /home/alparka/nara-ai/artifacts/stage11_evidence_audit.csv
MANIFEST                    = /home/alparka/nara-ai/artifacts/stage11_manifest.json

[PASS] Stage 11 exact-substring evidence validator 준비 완료
NEXT = Stage 12 first submission-ready script.py
